# T3: a slip-detection dataset, driven by the real OpenVLA-OFT policy (Openvla38_T3)

Derived from `Openvla37_T2.ipynb` (same dual-GelSight robot setup, same pinned D3 checkpoint). T2 built a dataset of clean, fully-held successes; T3 builds a dataset for training a **tactile slip detector**, so its whole collection philosophy is different -- see the "Plan B" design discussed with the user for the full reasoning. Per explicit request, **OpenVLA-OFT (the same pinned D3 checkpoint) still drives every rollout** -- nothing here is scripted -- because slip dynamics under the real policy's own (imperfect, replanning) behavior are what the eventual deployment will actually look like. Only the physics/controller environment underneath the policy is randomized.

**What changed in Cell P, on top of Openvla37_T2:**

- `CONTROL_RATE_SCALE_FACTOR = 1.0` (10 Hz, matching D3's original collection rate) instead of 0.5 (5 Hz), per request, to check whether it reduces motion lag. It also happens to reintroduce some of the real fast-motion slip dynamics 5 Hz was originally chosen to suppress.
- **The keep criterion is inverted from T2's.** T2 kept a demo only if the task succeeded AND both sensors showed a strong, sustained grip -- selecting *against* slip. T3 keeps any demo with a real grasp attempt (meaningful contact on at least one sensor), success or not: `DATASET_KEEP_ONLY_QUALIFYING`, `DATASET_TARGET_DEMOS = 200`, `DATASET_MIN_CONTACT_FRAMES = 10`, `DATASET_MIN_PEAK_TAXELS = 200`.
- **New per-frame ground truth, saved at the same 30 Hz as the tactile RGB (perfectly aligned by construction)**, addressing the 5-field vector from the design discussion:
  1. **Slippage (binary)** -- object pose relative to the hand, rolling `SLIP_WINDOW_S`-window displacement/rotation vs `SLIP_TRANSLATION_THRESHOLD_MM`/`SLIP_ROTATION_THRESHOLD_DEG`, only evaluated in `SLIP_HOLDING_PHASES`.
  2. **Grasp pose** -- captured once, at the instant the controller's own grip-plateau lock first engages (a read-only observation of existing controller state, not a new algorithm).
  3. **In-hand object pose** -- the continuous version of (1), saved every frame.
  4. **Contact force (normal, per finger)** -- read directly from MuJoCo's own contact solver (`mj_contactForce`), not a fabricated "virtual loadcell".
  5. **Contact state (binary)** -- `active_taxel_count > 0`, already existed, now also saved under this name.
  All in `<demo>/slip_ground_truth.npz` (global fields) + two new arrays in each sensor's own `fots_quantitative_data.npz` (`normal_force_n`, `contact_state`).
- **Randomization**, so the same OpenVLA policy produces a real spread of grip outcomes instead of T2's near-uniform clean successes: live (no scene rebuild) per-rollout randomization of gel/cube torsional friction and cube mass, a randomized grip-lock margin, and an independent external perturbation force on the cube during carry (known onset time, doesn't touch the policy at all).
- **Disk usage trimmed ~14x.** `SAVE_RAW_DEPTH_STACK = False` drops T2's dominant cost (measured: ~94% of a demo, ~150 MB/sensor, the full uncompressed per-frame depth image) since it isn't needed for slip labeling. Measured: **~22 MB/demo** here vs. T2's ~318 MB.
- **Cell D1** now also plots, per selected demo: slip binary + the underlying continuous translation/rotation deltas vs. threshold, per-finger normal force, and per-finger contact state, all phase-shaded -- not just the still-frame/video summary T2 had.

**Slip threshold, recalibrated from the smoke test.** The first pass of the smoke test (at the originally-requested 1.5 mm/1.5 deg) showed that even frames where force and contact state both say "rock-solid hold" have a measured baseline translation delta of ~3.5 mm (median) up to ~10.4 mm (99th percentile) in this compliant-gel simulation -- the 1.5 mm threshold was below the simulator's own ambient noise floor, so `slip_binary` was "1" for 75-96% of holding frames in every demo, successes included. `SLIP_TRANSLATION_THRESHOLD_MM`/`SLIP_ROTATION_THRESHOLD_DEG` are now **12 mm / 10 deg** (just above the measured 99th-percentile noise floor). Re-running the smoke test at this threshold shows a clean, informative signal: all 7 successful demos now show 3-9% of holding frames flagged as slip, all 3 failed demos show 43-56% -- and one failed demo's slip window now tightly and correctly isolates an actual, unambiguous drop event (~300-440 mm displacement, ~175 deg rotation spike, force and both fingers' contact dropping to zero and staying there). The raw continuous fields (`translation_delta_mm`, `rotation_delta_deg`) are saved regardless of the threshold, so it can be revisited later without re-collecting anything.

**Dataset quality filter, added after the smoke test.** `DATASET_KEEP_ONLY_QUALIFYING`'s bar was raised from "10 contact frames" (~0.3 s) to **`DATASET_MIN_CONTACT_SECONDS = 2.0`** (60 frames at 30 Hz) on at least one sensor: a demo where the gripper never achieved a real, sustained touch (missed the cube, or only brushed it) has no useful tactile content for a slip detector either way, so it is discarded rather than saved. Applied per side, not both -- a strongly asymmetric grip (one finger solidly on, one barely touching) is itself a real slip signature, not something to filter out.

---

## Inherited from Openvla37_T2

Derived from `Openvla36_T1.ipynb` (same dual-GelSight robot setup, same pinned D3 checkpoint, no occluder plant, baseline/in-distribution cube starts). This notebook's purpose is different: instead of evaluating success rate, Cell P now **collects a dataset** -- it runs closed-loop OpenVLA-OFT rollouts until at least 200 of them succeed, keeping only the successful, dual-sensor-verified ones (with every file Cell P already saves: world/wrist/grasp videos, per-query third-person images, both fingers' tactile RGB/marker-flow/quantitative npz, the full `closed_loop_log.json`), and deleting the rest.

**What changed compared with Openvla36_T1's Cell P**

- `N_ROLLOUTS = 400` -- a ceiling on attempts, not a target.
- `EXPERIMENT_DATASET = "T2"` (was `"T1_duo"`) -- its own dataset folder, saved to `PROJECT_DIR/T2/baseline/`.
- New knobs: `DATASET_KEEP_ONLY_SUCCESSFUL = True`, `DATASET_TARGET_SUCCESSFUL_DEMOS = 200`. Right after each attempt, Cell P decides whether it qualifies as a dataset demo -- **both** the policy actually placing the cube at the far side **and** both GelSight sensors registering a real, sustained grip (reusing Cell T3's own audit thresholds: `DATASET_MIN_SUSTAINED_CONTACT_FRAMES = 30`, `DATASET_MIN_PEAK_TAXELS = 1000`) -- so a rollout that reached the target region without a clean two-finger grasp is excluded even though the placement check alone would call it a success.
  - Qualifying attempts are renamed from `rollout_NNN` (attempt order) to `demo_NNN` (dense dataset order, `0..199+`).
  - Non-qualifying attempts are deleted immediately (`shutil.rmtree`), so disk usage never exceeds one extra attempt's worth of files above the running dataset size, and the loop stops as soon as `DATASET_TARGET_SUCCESSFUL_DEMOS` is reached (well before 400, if the ~70% success rate observed on `T1_duo/baseline` holds).
  - Re-running the cell **resumes**: it detects existing `demo_*` folders, continues numbering after them, and merges its own attempt log into the existing `results.csv` instead of overwriting it.
- `DISPLAY_ROLLOUT_VIDEOS = DISPLAY_GELSIGHT_VIDEOS = False` -- inline-rendering hundreds of video widgets isn't useful at this scale; the saved files (and `results.csv`) are the actual output.
- Cell P2/P4's rollout auto-discovery now also recognizes `demo_*` folders (was `rollout_*`-only), so they still work unmodified when pointed at a T2 demo.

Everything else -- the scene, both GelSight sensors, the D3 checkpoint pinning, the controller -- is unchanged from Openvla36_T1. The two-sensor sanity test at the end (Cells T1-T3) is unchanged and still applies to this notebook's own hardware setup.

---

## Inherited from OpenVLA35_T1 / Openvla36_T1

Derived from `OpenVLA35_T1.ipynb`. The robot setup used by the closed-loop T1 tests is replaced by the two-sensor setup of `TactileModel05_Duo.ipynb`, and the notebook now ends with a test of both sensors.

**What changed compared with OpenVLA35_T1**

- **Robot setup (Cell P).** Both Panda fingers carry a GelSight Mini assembly (gel pad, housing, FOTS virtual tactile camera) instead of only the right one. Two FOTS sensors record every rollout into `rollout_NNN/right/` and `rollout_NNN/left/`. The cube, contact materials, controller and policy inputs are unchanged. Results go to the new folder `T1_duo/`, so the single-sensor rollouts in `T1/` are never overwritten. Setting `RUN_CLOSED_LOOP_ROLLOUTS = False` in Cell P builds and self-tests the scene and both sensors without loading OpenVLA.
- **Checkpoint (Cell M, re-verified in Cell P).** The checkpoint is pinned to the D3 OpenVLA-OFT checkpoint (`openvla_oft_d3_runs/...--50000_chkpt`, dataset `panda_pickplace_d3`) instead of "the newest folder by modification time". The newer D4 checkpoints are listed but never used, and the run stops if the resolved checkpoint is not the D3 one.
- **Cell P4.** One judgment video per sensor (right and left).
- **Final section: two-sensor test.** Cell T1 records both sensors through a scripted grasp, lift, carry, place and release; Cell T2 judges the output of both sensors (PASS/FAIL table, figures, side-by-side video); Cell T3 audits the tactile data saved by the policy rollouts.

**Running the sensor test on its own:** run Cells 2, A, B, C, F, M and P0, then Cell P with `RUN_CLOSED_LOOP_ROLLOUTS = False`, then Cells T1 and T2.

---

## OpenVLA-OFT D3: joint proprioception and continuous action chunks

This notebook preserves the existing Panda simulation and collection cells, then fine-tunes the official OpenVLA-OFT implementation on **D3**:

- 700 nominal + 100 boundary demonstrations; all recovery demonstrations are excluded.
- Input: fixed third-person image, language instruction, and `[q1, q2, q3, q4, q5, q6, q7, gripper_open_fraction]`.
- Output: an 8-step chunk of continuous 7D Cartesian actions.
- Objective: continuous L1 regression with parallel decoding, LoRA rank 32, and train-only q01/q99 normalization.

For OFT fine-tuning, start at the server section after Cell E. Use `TRAINING_MODE = "smoke"` first, then change it to `"full"`.

In [ ]:
# Cell 1: Laptop-safe dependency, driver, and thermal preflight.
# Run this before every fresh VS Code/Jupyter kernel.
import importlib.metadata as importlib_metadata
import importlib.util
import json
import os
import platform
import re
import shutil
import subprocess
import sys
from pathlib import Path

# GitHub sync: save locally, then rerun Cell 1 to publish the latest source snapshot.
GITHUB_WEB_URL = "https://github.com/Fariborz-Eshraghi/force-openvla"
GITHUB_HTTPS_REMOTE = f"{GITHUB_WEB_URL}.git"
GITHUB_SSH_REMOTE = "git@github-force-openvla:Fariborz-Eshraghi/force-openvla.git"
GITHUB_SSH_KEY = Path.home() / ".ssh" / "force_openvla_deploy"
GITHUB_REMOTE = GITHUB_SSH_REMOTE if GITHUB_SSH_KEY.is_file() else GITHUB_HTTPS_REMOTE
GITHUB_BRANCH = "main"
GITHUB_USER = "Fariborz-Eshraghi"
GITHUB_CHECKOUT = Path.home() / ".cache" / "force-openvla-github"
GITHUB_SOURCE_DIR = Path.cwd().resolve()
GITHUB_NOTEBOOK = Path(os.environ.get("OPENVLA_NOTEBOOK_PATH", globals().get("__vsc_ipynb_file__", GITHUB_SOURCE_DIR / "OpenVLA26_OFT.ipynb"))).expanduser().resolve()
GITHUB_ROOT_FILES = ("README.md", "assets/force-openvla-architecture.png")
GITHUB_EXTRA_FILES = (
    "prepare_panda_d3.py",
    "openvla_oft_d3_setup.py",
    "openvla_oft_d3_verify.py",
    "openvla_oft_heldout_visual_eval.py",
    "rlds_dataset_builder/panda_pickplace_d3/__init__.py",
    "rlds_dataset_builder/panda_pickplace_d3/panda_pickplace_d3_dataset_builder.py",
)


def _optional_secret(name):
    value = os.environ.get(name, "")
    if value:
        return value
    try:
        from google.colab import userdata
        return userdata.get(name) or ""
    except Exception:
        return ""


def _git(*args, cwd=GITHUB_CHECKOUT, check=True):
    env = os.environ.copy()
    env.update({"GIT_TERMINAL_PROMPT": "0", "GCM_INTERACTIVE": "never"})
    return subprocess.run(["git", *map(str, args)], cwd=cwd, env=env, text=True, check=check)


def _write_source_only_notebook(destination_dir):
    notebook = None
    try:
        from google.colab import _message
        reply = _message.blocking_request("get_ipynb", timeout_sec=30)
        notebook = reply.get("ipynb") if isinstance(reply, dict) else None
    except Exception:
        pass
    if not isinstance(notebook, dict):
        if not GITHUB_NOTEBOOK.is_file():
            raise FileNotFoundError(f"Save the notebook first or set OPENVLA_NOTEBOOK_PATH: {GITHUB_NOTEBOOK}")
        notebook = json.loads(GITHUB_NOTEBOOK.read_text(encoding="utf-8"))
    raw_name = notebook.get("metadata", {}).get("colab", {}).get("name", GITHUB_NOTEBOOK.name)
    notebook_name = Path(raw_name).name
    if not notebook_name.endswith(".ipynb"):
        notebook_name += ".ipynb"
    for cell in notebook.get("cells", []):
        if cell.get("cell_type") == "code":
            cell["outputs"] = []
            cell["execution_count"] = None
        cell.get("metadata", {}).pop("execution", None)
    destination = destination_dir / notebook_name
    destination.write_text(json.dumps(notebook, ensure_ascii=False, indent=1) + "\n", encoding="utf-8")
    return destination


def sync_to_github():
    use_ssh = GITHUB_SSH_KEY.is_file()
    token = "" if use_ssh else _optional_secret("GITHUB_TOKEN")
    if not use_ssh and not token:
        print("GitHub sync skipped: local SSH key not found; add GITHUB_TOKEN to Colab Secrets and rerun Cell 1.")
        return
    print("GitHub authentication:", "permanent SSH deploy key" if use_ssh else "Colab token")
    GITHUB_CHECKOUT.parent.mkdir(parents=True, exist_ok=True)
    if not (GITHUB_CHECKOUT / ".git").is_dir():
        if GITHUB_CHECKOUT.exists():
            raise RuntimeError(f"GitHub checkout path exists but is not a Git repository: {GITHUB_CHECKOUT}")
        _git("clone", "--branch", GITHUB_BRANCH, "--single-branch", GITHUB_REMOTE, GITHUB_CHECKOUT, cwd=GITHUB_CHECKOUT.parent)
    _git("remote", "set-url", "origin", GITHUB_REMOTE)
    _git("config", "user.name", GITHUB_USER)
    _git("config", "user.email", f"{GITHUB_USER}@users.noreply.github.com")
    if token:
        _git("config", "credential.helper", "cache --timeout=21600")
        subprocess.run(
            ["git", "credential", "approve"],
            cwd=GITHUB_CHECKOUT,
            input=f"protocol=https\nhost=github.com\nusername={GITHUB_USER}\npassword={token}\n\n",
            text=True,
            check=True,
        )
        token = None
    _git("pull", "--rebase", "origin", GITHUB_BRANCH)
    simulation_dir = GITHUB_CHECKOUT / "Simulation"
    simulation_dir.mkdir(parents=True, exist_ok=True)
    staged = [_write_source_only_notebook(simulation_dir).relative_to(GITHUB_CHECKOUT)]
    for relative_name in GITHUB_ROOT_FILES:
        source = GITHUB_SOURCE_DIR / relative_name
        if source.is_file():
            destination = GITHUB_CHECKOUT / relative_name
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, destination)
            staged.append(destination.relative_to(GITHUB_CHECKOUT))
    for relative_name in GITHUB_EXTRA_FILES:
        source = GITHUB_SOURCE_DIR / relative_name
        if source.is_file():
            destination = simulation_dir / relative_name
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, destination)
            staged.append(destination.relative_to(GITHUB_CHECKOUT))
    _git("add", "--", *staged)
    diff = _git("diff", "--cached", "--quiet", check=False)
    if diff.returncode == 1:
        _git("commit", "-m", f"Sync {staged[0].name}, README, and simulation helpers")
    elif diff.returncode != 0:
        raise RuntimeError("Git could not inspect the staged changes.")
    _git("push", "origin", GITHUB_BRANCH)
    print(f"GitHub sync complete: {GITHUB_WEB_URL}/tree/{GITHUB_BRANCH}/Simulation")


sync_to_github()

SAFE_LAPTOP_MODE = True
INSTALL_LIGHTWEIGHT_SIM_PACKAGES = True
INSTALL_HEAVY_VLA_STACK = False  # Flip to True only after updating the NVIDIA driver and closing other apps.

# Keep the i7-7700HQ and cooling system out of the danger zone.
os.environ.setdefault("OMP_NUM_THREADS", "8")
os.environ.setdefault("MKL_NUM_THREADS", "8")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "8")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "8")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:64")

LOCAL_HF_CACHE_DIR = Path("/home/fariborz/projects/force-vla-colab/.hf_cache")
LOCAL_HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(LOCAL_HF_CACHE_DIR)
os.environ["HF_HUB_CACHE"] = str(LOCAL_HF_CACHE_DIR / "hub")
os.environ["TRANSFORMERS_CACHE"] = str(LOCAL_HF_CACHE_DIR / "transformers")
for cache_subdir in [os.environ["HF_HUB_CACHE"], os.environ["TRANSFORMERS_CACHE"]]:
    Path(cache_subdir).mkdir(parents=True, exist_ok=True)

print("Python:", sys.executable)
print("Python version:", sys.version.split()[0])
print("Platform:", platform.platform())
print("CPU threads limited to:", os.environ.get("OMP_NUM_THREADS"))
print("HF cache:", LOCAL_HF_CACHE_DIR)
try:
    cache_drive = shutil.disk_usage(LOCAL_HF_CACHE_DIR.anchor)
    print("HF cache drive free GiB:", round(cache_drive.free / 1024**3, 2))
except Exception as exc:
    print("HF cache drive space check unavailable:", exc)

try:
    import psutil
    print("System RAM GiB:", round(psutil.virtual_memory().total / 1024**3, 2))
except Exception as exc:
    print("psutil RAM check unavailable:", exc)


def module_status(module_name, package_name=None):
    package_name = package_name or module_name
    installed = importlib.util.find_spec(module_name) is not None
    version = None
    if installed:
        try:
            version = importlib_metadata.version(package_name)
        except Exception:
            version = "installed"
    return installed, version


def pip_install(args):
    cmd = [sys.executable, "-m", "pip", "install", "--no-cache-dir", *args]
    print("Running:", " ".join(cmd))
    subprocess.check_call(cmd)


LIGHTWEIGHT_PACKAGES = {
    "mujoco": "mujoco",
    "mediapy": "mediapy",
    "imageio_ffmpeg": "imageio-ffmpeg",
    "PIL": "pillow",
    "ipywidgets": "ipywidgets",
    "psutil": "psutil",
}

missing_light = []
print("\nLightweight simulation/display packages")
for module_name, pip_name in LIGHTWEIGHT_PACKAGES.items():
    installed, version = module_status(module_name, pip_name)
    print(f"  {module_name:16s}: {version if installed else 'missing'}")
    if not installed:
        missing_light.append(pip_name)

if missing_light and INSTALL_LIGHTWEIGHT_SIM_PACKAGES:
    pip_install(missing_light)
    print("Lightweight packages installed. If imports still fail, restart the kernel and rerun Cell 1.")
elif missing_light:
    print("Missing lightweight packages:", missing_light)

# OpenVLA's official minimal requirements pin transformers/tokenizers/timm; PyTorch is installed from the PyTorch CUDA 11.8 wheel index.
TORCH_INSTALL_COMMAND = [
    sys.executable, "-m", "pip", "install", "--no-cache-dir",
    "torch==2.5.1", "torchvision==0.20.1", "torchaudio==2.5.1",
    "--index-url", "https://download.pytorch.org/whl/cu118",
]
VLA_PACKAGE_COMMAND = [
    sys.executable, "-m", "pip", "install", "--no-cache-dir",
    "transformers==4.40.1",
    "tokenizers==0.19.1",
    "timm==0.9.10",
    "accelerate>=0.30.1",
    "bitsandbytes>=0.46.0",
    "safetensors",
    "huggingface_hub",
    "einops",
    "sentencepiece",
]

HEAVY_MODULES = {
    "torch": "torch",
    "torchvision": "torchvision",
    "transformers": "transformers",
    "tokenizers": "tokenizers",
    "accelerate": "accelerate",
    "bitsandbytes": "bitsandbytes",
    "timm": "timm",
    "huggingface_hub": "huggingface_hub",
    "einops": "einops",
}

print("\nHeavy VLA packages")
missing_heavy = []
for module_name, package_name in HEAVY_MODULES.items():
    installed, version = module_status(module_name, package_name)
    print(f"  {module_name:16s}: {version if installed else 'missing'}")
    if not installed:
        missing_heavy.append(module_name)


def public_nvidia_driver_version(wmi_driver_version):
    # WMI often reports NVIDIA 451.67 as 27.21.14.5167, 536.40 as 31.0.15.3640, etc.
    parts = str(wmi_driver_version).split(".")
    if len(parts) >= 4 and parts[2].isdigit() and parts[3].isdigit():
        branch = (int(parts[2]) - 10) * 100
        return branch + int(parts[3]) / 100.0
    return None


def windows_gpu_report():
    if platform.system() != "Windows":
        return []
    powershell_exe = (
        shutil.which("powershell.exe")
        or shutil.which("powershell")
        or shutil.which("pwsh")
        or r"C:\Windows\System32\WindowsPowerShell\v1.0\powershell.exe"
    )
    if not Path(powershell_exe).exists() and shutil.which(powershell_exe) is None:
        print("PowerShell was not found, so Windows GPU details could not be queried.")
        return []
    ps = "Get-CimInstance Win32_VideoController | Select-Object Name,AdapterRAM,DriverVersion | ConvertTo-Json"
    result = subprocess.run(
        [powershell_exe, "-NoProfile", "-ExecutionPolicy", "Bypass", "-Command", ps],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
        timeout=10,
    )
    if result.returncode != 0 or not result.stdout.strip():
        print("Windows GPU report unavailable:", result.stderr.strip())
        return []
    data = json.loads(result.stdout)
    if isinstance(data, dict):
        data = [data]
    return data

print("\nWindows GPU / driver report")
NVIDIA_DRIVER_PUBLIC_VERSION = None
for gpu in windows_gpu_report():
    name = gpu.get("Name", "unknown")
    ram_gib = (int(gpu.get("AdapterRAM") or 0) / 1024**3) if gpu.get("AdapterRAM") else None
    driver = gpu.get("DriverVersion", "unknown")
    public_driver = public_nvidia_driver_version(driver) if "NVIDIA" in name.upper() else None
    if "NVIDIA" in name.upper():
        NVIDIA_DRIVER_PUBLIC_VERSION = public_driver
    ram_text = f", VRAM~{ram_gib:.2f} GiB" if ram_gib else ""
    public_text = f", public driver~{public_driver:.2f}" if public_driver else ""
    print(f"  {name}{ram_text}, WMI driver={driver}{public_text}")

if NVIDIA_DRIVER_PUBLIC_VERSION is not None and NVIDIA_DRIVER_PUBLIC_VERSION < 452.39:
    print("\nWARNING: Your NVIDIA driver appears older than the CUDA 11.x Windows compatibility floor (452.39).")
    print("Update the NVIDIA driver before installing/running CUDA PyTorch. The current driver may make torch.cuda unavailable.")

print("\nPyTorch CUDA check")
torch_cuda_ok = False
if importlib.util.find_spec("torch") is None:
    print("  torch is not installed in this kernel.")
else:
    try:
        import torch
        print("  torch:", torch.__version__)
        print("  torch CUDA runtime:", torch.version.cuda)
        print("  cuda available:", torch.cuda.is_available())
        if torch.cuda.is_available():
            props = torch.cuda.get_device_properties(0)
            print("  device:", torch.cuda.get_device_name(0))
            print("  VRAM GiB:", round(props.total_memory / 1024**3, 2))
            print("  compute capability:", f"{props.major}.{props.minor}")
            x = torch.randn(256, 256, device="cuda", dtype=torch.float16)
            y = x @ x.T
            torch.cuda.synchronize()
            del x, y
            torch.cuda.empty_cache()
            torch_cuda_ok = True
            print("  tiny CUDA tensor test: ok")
    except Exception as exc:
        print("  torch import/CUDA check failed:", repr(exc))

print("\nBitsAndBytes 4-bit CUDA check")
if importlib.util.find_spec("bitsandbytes") is None:
    print("  bitsandbytes is not installed in this kernel.")
elif not torch_cuda_ok:
    print("  skipped because torch CUDA is not ready.")
else:
    try:
        import bitsandbytes as bnb
        layer = bnb.nn.Linear4bit(16, 8, bias=False, quant_type="nf4", compute_dtype=torch.float16).cuda()
        inp = torch.randn(2, 16, device="cuda", dtype=torch.float16)
        with torch.no_grad():
            out = layer(inp)
        torch.cuda.synchronize()
        del layer, inp, out
        torch.cuda.empty_cache()
        print("  tiny NF4 Linear4bit test: ok")
    except Exception as exc:
        print("  bitsandbytes 4-bit CUDA check failed:", repr(exc))

if INSTALL_HEAVY_VLA_STACK:
    print("\nInstalling heavy PyTorch CUDA/OpenVLA packages. This can take a while and several GB of disk/network.")
    subprocess.check_call(TORCH_INSTALL_COMMAND)
    subprocess.check_call(VLA_PACKAGE_COMMAND)
    print("Heavy stack installed. Restart the kernel before loading OpenVLA.")
elif missing_heavy:
    print("\nHeavy VLA stack install is disabled for safety.")
    print("After updating the NVIDIA driver, you can either set INSTALL_HEAVY_VLA_STACK=True and rerun this cell, or run these commands in VS Code's terminal:")
    print(" ".join(TORCH_INSTALL_COMMAND))
    print(" ".join(VLA_PACKAGE_COMMAND))
else:
    print("\nHeavy VLA stack is installed. Keep INSTALL_HEAVY_VLA_STACK=False unless you intentionally want to reinstall it.")

print("\nHF token setup")
print("  Store HF_TOKEN in the environment or Colab Secrets if the model download asks for one.")
print("  Never paste access tokens into a notebook that may be published.")
if os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN"):
    print("  An environment token is also present and can be used as fallback.")


In [ ]:
# Cell 2: Imports and rendering setup for local Windows/VS Code or Linux.
import contextlib
import gc
import json
import os
import platform
import shutil
import subprocess
import time
import xml.etree.ElementTree as ET
from base64 import b64encode
from html import escape
from pathlib import Path

# Windows local notebooks should use GLFW. Linux/headless GPU workstations usually use EGL.
if "MUJOCO_GL" not in os.environ:
    os.environ["MUJOCO_GL"] = "glfw" if platform.system() == "Windows" else "egl"

import imageio_ffmpeg
import mediapy as media
import ipywidgets as widgets
import mujoco
import numpy as np
from IPython.display import HTML, clear_output, display
from PIL import Image

PROJECT_DIR = Path(os.environ.get("OPENVLA_PROJECT_DIR", Path.cwd())).expanduser().resolve()
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

ffmpeg_path = shutil.which("ffmpeg") or imageio_ffmpeg.get_ffmpeg_exe()
media.set_ffmpeg(ffmpeg_path)

print("Project:", PROJECT_DIR)
print("MuJoCo version:", mujoco.__version__)
print("Rendering backend:", os.environ.get("MUJOCO_GL"))
print("ffmpeg:", ffmpeg_path)


In [ ]:
# Cell 3: Build/load a slim Panda scene with only a wrist camera, table, and red block.
MENAGERIE_DIR = PROJECT_DIR / "mujoco_menagerie"
PANDA_DIR = MENAGERIE_DIR / "franka_emika_panda"
SOURCE_PANDA_XML = PANDA_DIR / "panda.xml"
PANDA_WITH_CAMERA_XML = PANDA_DIR / "panda_openvla04_wrist_camera.xml"
PANDA_XML = PANDA_DIR / "openvla04_pickplace_scene.xml"


def download_mujoco_menagerie_without_git():
    import urllib.request

    api_url = "https://api.github.com/repos/google-deepmind/mujoco_menagerie/git/trees/main?recursive=1"
    prefix = "franka_emika_panda/"
    print("git is not available; downloading only mujoco_menagerie/franka_emika_panda from GitHub.")
    MENAGERIE_DIR.mkdir(parents=True, exist_ok=True)

    with urllib.request.urlopen(api_url, timeout=120) as response:
        tree_payload = json.loads(response.read().decode("utf-8"))

    panda_files = [
        item["path"]
        for item in tree_payload.get("tree", [])
        if item.get("type") == "blob" and item.get("path", "").startswith(prefix)
    ]
    if not panda_files:
        raise RuntimeError("Could not find franka_emika_panda files in the GitHub tree response.")

    for rel_path in panda_files:
        dest = MENAGERIE_DIR / rel_path
        dest.parent.mkdir(parents=True, exist_ok=True)
        if dest.exists() and dest.stat().st_size > 0:
            continue
        raw_url = f"https://raw.githubusercontent.com/google-deepmind/mujoco_menagerie/main/{rel_path}"
        print("  downloading", rel_path)
        with urllib.request.urlopen(raw_url, timeout=120) as response:
            dest.write_bytes(response.read())


if not SOURCE_PANDA_XML.exists():
    git_exe = shutil.which("git")
    if MENAGERIE_DIR.exists() and (MENAGERIE_DIR / ".git").exists() and git_exe:
        subprocess.check_call([git_exe, "-C", str(MENAGERIE_DIR), "pull", "--ff-only"])
    elif MENAGERIE_DIR.exists() and PANDA_DIR.exists():
        pass
    elif MENAGERIE_DIR.exists():
        raise RuntimeError(
            f"{MENAGERIE_DIR} exists but does not look like a complete mujoco_menagerie install. "
            "Rename or remove that folder, then rerun this cell."
        )
    elif git_exe:
        subprocess.check_call([
            git_exe,
            "clone",
            "--depth",
            "1",
            "https://github.com/google-deepmind/mujoco_menagerie.git",
            str(MENAGERIE_DIR),
        ])
    else:
        download_mujoco_menagerie_without_git()

if not SOURCE_PANDA_XML.exists():
    raise FileNotFoundError(f"Could not find Panda model at {SOURCE_PANDA_XML}")


def find_body(root, body_name):
    for body in root.iter("body"):
        if body.get("name") == body_name:
            return body
    raise ValueError(f"Could not find body named {body_name!r}")


panda_tree = ET.parse(SOURCE_PANDA_XML)
panda_root = panda_tree.getroot()
hand_body = find_body(panda_root, "hand")

for child in list(hand_body):
    if child.tag == "body" and child.get("name") == "agent_camera_mount":
        hand_body.remove(child)

mount_body = ET.Element("body", name="agent_camera_mount", pos="0.08 0 -0.12")
ET.SubElement(
    mount_body,
    "geom",
    name="agent_camera_boom_geom",
    type="capsule",
    fromto="0 0 0 -0.08 0 0.12",
    size="0.008",
    rgba="0.02 0.02 0.02 1",
    contype="0",
    conaffinity="0",
    group="2",
)
ET.SubElement(
    mount_body,
    "geom",
    name="agent_camera_body_geom",
    type="box",
    size="0.020 0.016 0.012",
    rgba="0.02 0.02 0.02 1",
    contype="0",
    conaffinity="0",
    group="2",
)
ET.SubElement(
    mount_body,
    "camera",
    name="agent_wrist_camera",
    mode="fixed",
    fovy="82",
    xyaxes="-0.000007 -1.000000 0.000034 -0.976592 -0.000000 -0.215100",
)
hand_body.insert(0, mount_body)
ET.indent(panda_tree, space="  ")
panda_tree.write(PANDA_WITH_CAMERA_XML, encoding="unicode")

PANDA_XML.write_text(
    f"""<mujoco model="openvla04 slim panda pick place scene">
  <include file="{PANDA_WITH_CAMERA_XML.name}"/>

  <statistic center="0.50 0 0.35" extent="1.1"/>

  <visual>
    <headlight diffuse="0.6 0.6 0.6" ambient="0.3 0.3 0.3" specular="0 0 0"/>
    <rgba haze="0.15 0.25 0.35 1"/>
    <global azimuth="120" elevation="-20"/>
  </visual>

  <asset>
    <texture type="skybox" builtin="gradient" rgb1="0.3 0.5 0.7" rgb2="0 0 0" width="512" height="3072"/>
    <texture type="2d" name="groundplane" builtin="checker" mark="edge" rgb1="0.2 0.3 0.4" rgb2="0.1 0.2 0.3" markrgb="0.8 0.8 0.8" width="300" height="300"/>
    <material name="groundplane" texture="groundplane" texuniform="true" texrepeat="5 5" reflectance="0.2"/>
    <material name="table_mat" rgba="0.48 0.38 0.26 1"/>
    <material name="red_block_mat" rgba="1 0.03 0.02 1"/>
  </asset>

  <worldbody>
    <light pos="0 0 1.5" dir="0 0 -1" directional="true"/>
    <geom name="floor" size="0 0 0.05" type="plane" material="groundplane"/>
    <body name="table" pos="0.55 0 0.025">
      <geom name="table_top" type="box" size="0.35 0.45 0.025" material="table_mat" friction="1 0.005 0.0001"/>
    </body>
    <body name="red_block" pos="0.50 0 0.085">
      <freejoint name="red_block_freejoint"/>
      <geom name="red_block_geom" type="box" size="0.035 0.035 0.035" material="red_block_mat" mass="0.05" friction="1 0.005 0.0001"/>
    </body>
  </worldbody>
</mujoco>
""",
    encoding="utf-8",
)

model = mujoco.MjModel.from_xml_path(str(PANDA_XML))
data = mujoco.MjData(model)
mujoco.mj_forward(model, data)

print("Loaded:", PANDA_XML)
print("Wrist camera:", "agent_wrist_camera")
print("Task block:", "red_block")
print("Tactile/GelSight path: disabled in this slim VLA test notebook")
print("nq:", model.nq, "nv:", model.nv, "nu:", model.nu)
print("timestep:", model.opt.timestep)

In [ ]:
# Cell 4: Task instruction and important scene IDs.
TASK_INSTRUCTION = "pick up the red block from the table and place it on the far side of the table"
OPENVLA_PROMPT = f"In: What action should the robot take to {TASK_INSTRUCTION}?\nOut: "

AGENT_CAMERA_NAME = "agent_wrist_camera"
START_BLOCK_POS = np.array([0.50, 0.00, 0.085], dtype=np.float64)
PLACE_BLOCK_POS = np.array([0.78, 0.00, 0.085], dtype=np.float64)

body_ids = {
    "hand": mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, "hand"),
    "left_finger": mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, "left_finger"),
    "right_finger": mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, "right_finger"),
    "red_block": mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, "red_block"),
    "agent_camera_mount": mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, "agent_camera_mount"),
}
camera_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_CAMERA, AGENT_CAMERA_NAME)

print("Instruction:", TASK_INSTRUCTION)
print("OpenVLA prompt:", repr(OPENVLA_PROMPT))
print("Body IDs:", body_ids)
print("Camera ID:", camera_id)

print("\nActuators:")
for actuator_id in range(model.nu):
    name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_ACTUATOR, actuator_id)
    trntype = mujoco.mjtTrn(model.actuator_trntype[actuator_id])
    target_id = model.actuator_trnid[actuator_id, 0]
    if trntype == mujoco.mjtTrn.mjTRN_JOINT:
        target = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_JOINT, target_id)
    elif trntype == mujoco.mjtTrn.mjTRN_TENDON:
        target = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_TENDON, target_id)
    else:
        target = f"id={target_id}"
    print(f" - {actuator_id}: {name} -> {trntype.name}:{target}, ctrlrange={model.actuator_ctrlrange[actuator_id]}")

In [ ]:
# Cell 5: OpenVLA execution setup.
# Default local mode is chunked worker isolation: OpenVLA loads only inside an external child process.
# The worker answers a few policy queries, exits to release VRAM, then restarts if needed.
RUN_OPENVLA = True
HF_TOKEN_IN_NOTEBOOK = _optional_secret("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN", "")

OPENVLA_EXECUTION_MODE = "chunked_worker"  # Recommended for GTX 1050 Ti. "kernel_hybrid" is riskier.
OPENVLA_LOAD_MODE = "hybrid_4bit_vision_cpu"
RAISE_ON_OPENVLA_LOAD_ERROR = False

LOCAL_HF_CACHE_DIR = Path(r"D:\Origins\Polimi\Thesis\Codes\Cache")
LOCAL_HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(LOCAL_HF_CACHE_DIR)
os.environ["HF_HUB_CACHE"] = str(LOCAL_HF_CACHE_DIR / "hub")
os.environ["TRANSFORMERS_CACHE"] = str(LOCAL_HF_CACHE_DIR / "transformers")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:64")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
for cache_subdir in [os.environ["HF_HUB_CACHE"], os.environ["TRANSFORMERS_CACHE"]]:
    Path(cache_subdir).mkdir(parents=True, exist_ok=True)

processor = None
vla = None
COMPUTE_DTYPE = None
OPENVLA_INPUT_PLACEMENT = "chunked_worker"
OPENVLA_LOAD_STATUS = {"loaded": False, "reason": "chunked external worker configured"}
UNNORM_KEY = "bridge_orig"
MODEL_ID = "openvla/openvla-7b"
OFFLOAD_DIR = LOCAL_HF_CACHE_DIR / "openvla_offload"
OFFLOAD_DIR.mkdir(parents=True, exist_ok=True)

print("RUN_OPENVLA:", RUN_OPENVLA)
print("OpenVLA execution mode:", OPENVLA_EXECUTION_MODE)
print("HF/cache directory:", LOCAL_HF_CACHE_DIR)
print("Note: in chunked-worker mode, Cell 5 does not keep OpenVLA loaded in the kernel.")
print("The model loads inside an external worker for a few queries, then exits to release VRAM.")

In [ ]:
# Cell 6: Slim controller, observation, and OpenVLA subprocess helper functions.
HAND_BODY_ID = body_ids["hand"]
LEFT_FINGER_BODY_ID = body_ids["left_finger"]
RIGHT_FINGER_BODY_ID = body_ids["right_finger"]
BLOCK_BODY_ID = body_ids["red_block"]
BLOCK_JOINT_ID = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, "red_block_freejoint")
BLOCK_QPOS_ADR = model.jnt_qposadr[BLOCK_JOINT_ID]

ARM_JOINT_NAMES = [f"joint{i}" for i in range(1, 8)]
ARM_JOINT_IDS = [mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, name) for name in ARM_JOINT_NAMES]
ARM_QPOS_ADR = [model.jnt_qposadr[joint_id] for joint_id in ARM_JOINT_IDS]
ARM_DOF_ADR = [model.jnt_dofadr[joint_id] for joint_id in ARM_JOINT_IDS]

FINGER_JOINT_NAMES = ["finger_joint1", "finger_joint2"]
FINGER_JOINT_IDS = [mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, name) for name in FINGER_JOINT_NAMES]
FINGER_QPOS_ADR = [model.jnt_qposadr[joint_id] for joint_id in FINGER_JOINT_IDS]

HOME_QPOS = {
    "joint1": 0.0,
    "joint2": -0.785,
    "joint3": 0.0,
    "joint4": -2.356,
    "joint5": 0.0,
    "joint6": 1.571,
    "joint7": 0.785,
    "finger_joint1": 0.04,
    "finger_joint2": 0.04,
}


def reset_task_state():
    mujoco.mj_resetData(model, data)
    for joint_name, value in HOME_QPOS.items():
        joint_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, joint_name)
        data.qpos[model.jnt_qposadr[joint_id]] = value
    data.qpos[BLOCK_QPOS_ADR:BLOCK_QPOS_ADR + 7] = np.array([*START_BLOCK_POS, 1.0, 0.0, 0.0, 0.0])
    mujoco.mj_forward(model, data)


def current_arm_qpos_dict():
    return {
        joint_name: float(data.qpos[model.jnt_qposadr[joint_id]])
        for joint_name, joint_id in zip(ARM_JOINT_NAMES, ARM_JOINT_IDS)
    }


def set_arm_position_targets(joint_targets):
    for actuator_id in range(model.nu):
        if model.actuator_trntype[actuator_id] != mujoco.mjtTrn.mjTRN_JOINT:
            continue
        joint_id = model.actuator_trnid[actuator_id, 0]
        joint_name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_JOINT, joint_id)
        if joint_name not in joint_targets:
            continue
        value = joint_targets[joint_name]
        if model.actuator_ctrllimited[actuator_id]:
            low, high = model.actuator_ctrlrange[actuator_id]
            value = np.clip(value, low, high)
        data.ctrl[actuator_id] = value


def set_gripper_opening(opening_m):
    ctrl = float(np.clip(opening_m / 0.04 * 255.0, 0.0, 255.0))
    for actuator_id in range(model.nu):
        if model.actuator_trntype[actuator_id] == mujoco.mjtTrn.mjTRN_TENDON:
            data.ctrl[actuator_id] = ctrl


def solve_hand_position_ik(target_pos, seed_qpos=None, max_iter=250, tolerance=0.002):
    if seed_qpos is not None:
        for joint_name, value in seed_qpos.items():
            joint_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, joint_name)
            data.qpos[model.jnt_qposadr[joint_id]] = value
    mujoco.mj_forward(model, data)

    jacp = np.zeros((3, model.nv), dtype=np.float64)
    jacr = np.zeros((3, model.nv), dtype=np.float64)

    for _ in range(max_iter):
        error = target_pos - data.xpos[HAND_BODY_ID]
        if np.linalg.norm(error) < tolerance:
            break

        mujoco.mj_jacBody(model, data, jacp, jacr, HAND_BODY_ID)
        jacobian = jacp[:, ARM_DOF_ADR]
        damping = 0.08
        delta_q = jacobian.T @ np.linalg.solve(
            jacobian @ jacobian.T + damping * damping * np.eye(3),
            error,
        )
        delta_q = np.clip(delta_q, -0.035, 0.035)

        for idx, qpos_adr in enumerate(ARM_QPOS_ADR):
            joint_id = ARM_JOINT_IDS[idx]
            low, high = model.jnt_range[joint_id]
            data.qpos[qpos_adr] = np.clip(data.qpos[qpos_adr] + delta_q[idx], low, high)

        mujoco.mj_forward(model, data)

    return current_arm_qpos_dict()


def capture_agent_observation(renderer):
    renderer.update_scene(data, camera=AGENT_CAMERA_NAME)
    rgb = renderer.render()
    return rgb, Image.fromarray(rgb).convert("RGB")


def _openvla_subprocess_script():
    return r"""
import gc, json, os, sys, time, traceback
from pathlib import Path
import numpy as np
import torch
from PIL import Image
from transformers import AutoModelForVision2Seq, AutoProcessor, BitsAndBytesConfig

image_path = Path(sys.argv[1])
prompt = sys.argv[2]
unnorm_key = sys.argv[3]
model_id = sys.argv[4]
cache_dir = Path(sys.argv[5])
offload_dir = Path(sys.argv[6])

os.environ["HF_HOME"] = str(cache_dir)
os.environ["HF_HUB_CACHE"] = str(cache_dir / "hub")
os.environ["TRANSFORMERS_CACHE"] = str(cache_dir / "transformers")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:64")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

result = {"ok": False}
try:
    if not torch.cuda.is_available():
        raise RuntimeError("torch.cuda is not available in OpenVLA subprocess")
    dtype = torch.float16
    gc.collect()
    torch.cuda.empty_cache()
    load_start = time.perf_counter()
    processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=dtype,
        bnb_4bit_use_double_quant=True,
        llm_int8_enable_fp32_cpu_offload=True,
    )
    vla = AutoModelForVision2Seq.from_pretrained(
        model_id,
        attn_implementation="eager",
        torch_dtype=dtype,
        quantization_config=quantization_config,
        device_map={"vision_backbone": "cpu", "projector": 0, "language_model": 0},
        offload_folder=str(offload_dir),
        offload_state_dict=True,
        low_cpu_mem_usage=True,
        trust_remote_code=True,
    )
    vla.eval()
    load_s = time.perf_counter() - load_start

    image = Image.open(image_path).convert("RGB")
    preprocess_start = time.perf_counter()
    processed = processor(prompt, image)
    inputs = {}
    for key, value in processed.items():
        if not torch.is_tensor(value):
            inputs[key] = value
        elif key == "pixel_values":
            inputs[key] = value.to("cpu", dtype=dtype)
        elif torch.is_floating_point(value):
            inputs[key] = value.to("cuda:0", dtype=dtype)
        else:
            inputs[key] = value.to("cuda:0")
    torch.cuda.synchronize()
    preprocess_s = time.perf_counter() - preprocess_start

    model_start = time.perf_counter()
    with torch.inference_mode():
        action = vla.predict_action(**inputs, unnorm_key=unnorm_key, do_sample=False, use_cache=False)
    torch.cuda.synchronize()
    model_s = time.perf_counter() - model_start
    result = {
        "ok": True,
        "action": np.asarray(action, dtype=np.float32).reshape(-1).tolist(),
        "timings": {
            "vla_load_s": float(load_s),
            "vla_preprocess_s": float(preprocess_s),
            "vla_model_inference_s": float(model_s),
            "openvla_total_s": float(preprocess_s + model_s),
            "subprocess_total_s": float(load_s + preprocess_s + model_s),
        },
        "device_map": {"vision_backbone": "cpu", "projector": 0, "language_model": 0},
    }
except Exception as exc:
    result = {"ok": False, "error": repr(exc), "traceback": traceback.format_exc()[-4000:]}
print("OPENVLA_RESULT_JSON=" + json.dumps(result), flush=True)
sys.exit(0 if result.get("ok") else 2)
"""


def openvla_predict_from_image(image):
    if not globals().get("RUN_OPENVLA", False):
        return None
    import tempfile
    with tempfile.TemporaryDirectory(prefix="openvla04_query_") as tmp:
        image_path = Path(tmp) / "query.png"
        image.convert("RGB").save(image_path)
        env = os.environ.copy()
        env["HF_HOME"] = str(LOCAL_HF_CACHE_DIR)
        env["HF_HUB_CACHE"] = str(LOCAL_HF_CACHE_DIR / "hub")
        env["TRANSFORMERS_CACHE"] = str(LOCAL_HF_CACHE_DIR / "transformers")
        token = globals().get("HF_TOKEN_IN_NOTEBOOK", "").strip()
        if token:
            env["HF_TOKEN"] = token
            env["HUGGINGFACE_HUB_TOKEN"] = token
        result = subprocess.run(
            [
                sys.executable,
                "-c",
                _openvla_subprocess_script(),
                str(image_path),
                OPENVLA_PROMPT,
                UNNORM_KEY,
                MODEL_ID,
                str(LOCAL_HF_CACHE_DIR),
                str(OFFLOAD_DIR),
            ],
            env=env,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            timeout=900,
            check=False,
        )
        marker = "OPENVLA_RESULT_JSON="
        payload = None
        for line in result.stdout.splitlines():
            if line.startswith(marker):
                payload = json.loads(line[len(marker):])
        if payload is None:
            raise RuntimeError("OpenVLA subprocess produced no result. stderr tail:\n" + result.stderr[-1600:])
        if not payload.get("ok"):
            raise RuntimeError("OpenVLA subprocess failed: " + payload.get("error", "unknown") + "\n" + payload.get("traceback", ""))
        timings = payload["timings"]
        timings["pipeline_total_s"] = timings.get("subprocess_total_s", timings.get("openvla_total_s", 0.0))
        return np.asarray(payload["action"], dtype=np.float32).reshape(-1), timings


def _openvla_chunked_worker_script():
    return r"""
import gc, json, os, sys, time, traceback
from pathlib import Path
import numpy as np
import torch
from PIL import Image
from transformers import AutoModelForVision2Seq, AutoProcessor, BitsAndBytesConfig

prompt_default = sys.argv[1]
unnorm_key_default = sys.argv[2]
model_id = sys.argv[3]
cache_dir = Path(sys.argv[4])
offload_dir = Path(sys.argv[5])

os.environ["HF_HOME"] = str(cache_dir)
os.environ["HF_HUB_CACHE"] = str(cache_dir / "hub")
os.environ["TRANSFORMERS_CACHE"] = str(cache_dir / "transformers")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:64")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

ready_marker = "OPENVLA_WORKER_READY_JSON="
result_marker = "OPENVLA_WORKER_RESULT_JSON="

try:
    if not torch.cuda.is_available():
        raise RuntimeError("torch.cuda is not available in OpenVLA worker")

    dtype = torch.float16
    gc.collect()
    torch.cuda.empty_cache()

    load_start = time.perf_counter()
    processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=dtype,
        bnb_4bit_use_double_quant=True,
        llm_int8_enable_fp32_cpu_offload=True,
    )
    vla = AutoModelForVision2Seq.from_pretrained(
        model_id,
        attn_implementation="eager",
        torch_dtype=dtype,
        quantization_config=quantization_config,
        device_map={"vision_backbone": "cpu", "projector": 0, "language_model": 0},
        offload_folder=str(offload_dir),
        offload_state_dict=True,
        low_cpu_mem_usage=True,
        trust_remote_code=True,
    )
    vla.eval()
    torch.cuda.synchronize()
    load_s = time.perf_counter() - load_start

    print(ready_marker + json.dumps({
        "ok": True,
        "vla_load_s": float(load_s),
        "device_map": {"vision_backbone": "cpu", "projector": 0, "language_model": 0},
    }), flush=True)

    query_count = 0
    for raw_line in sys.stdin:
        raw_line = raw_line.strip()
        if not raw_line:
            continue
        request = json.loads(raw_line)
        if request.get("cmd") == "quit":
            break

        request_id = request.get("request_id")
        response = {"ok": False, "request_id": request_id}
        try:
            image_path = Path(request["image_path"])
            prompt = request.get("prompt") or prompt_default
            unnorm_key = request.get("unnorm_key") or unnorm_key_default

            image = Image.open(image_path).convert("RGB")
            preprocess_start = time.perf_counter()
            processed = processor(prompt, image)
            inputs = {}
            for key, value in processed.items():
                if not torch.is_tensor(value):
                    inputs[key] = value
                elif key == "pixel_values":
                    inputs[key] = value.to("cpu", dtype=dtype)
                elif torch.is_floating_point(value):
                    inputs[key] = value.to("cuda:0", dtype=dtype)
                else:
                    inputs[key] = value.to("cuda:0")
            torch.cuda.synchronize()
            preprocess_s = time.perf_counter() - preprocess_start

            model_start = time.perf_counter()
            with torch.inference_mode():
                action = vla.predict_action(**inputs, unnorm_key=unnorm_key, do_sample=False, use_cache=False)
            torch.cuda.synchronize()
            model_s = time.perf_counter() - model_start

            response = {
                "ok": True,
                "request_id": request_id,
                "action": np.asarray(action, dtype=np.float32).reshape(-1).tolist(),
                "timings": {
                    "vla_load_s": float(load_s if query_count == 0 else 0.0),
                    "worker_load_s": float(load_s),
                    "vla_preprocess_s": float(preprocess_s),
                    "vla_model_inference_s": float(model_s),
                    "openvla_total_s": float(preprocess_s + model_s),
                    "worker_query_index": int(query_count),
                },
                "device_map": {"vision_backbone": "cpu", "projector": 0, "language_model": 0},
            }
            query_count += 1
            del image, processed, inputs
            gc.collect()
        except Exception as exc:
            response = {
                "ok": False,
                "request_id": request_id,
                "error": repr(exc),
                "traceback": traceback.format_exc()[-4000:],
            }
        print(result_marker + json.dumps(response), flush=True)

except Exception as exc:
    print(ready_marker + json.dumps({
        "ok": False,
        "error": repr(exc),
        "traceback": traceback.format_exc()[-4000:],
    }), flush=True)
    sys.exit(2)

try:
    del vla, processor
except Exception:
    pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
sys.exit(0)
"""


class OpenVLAChunkedWorker:
    READY_MARKER = "OPENVLA_WORKER_READY_JSON="
    RESULT_MARKER = "OPENVLA_WORKER_RESULT_JSON="

    def __init__(self, chunk_index, start_timeout_s=420, query_timeout_s=420):
        import queue
        import threading

        self.chunk_index = int(chunk_index)
        self.query_count = 0
        self.start_timeout_s = float(start_timeout_s)
        self.query_timeout_s = float(query_timeout_s)
        self.stdout_queue = queue.Queue()
        self.stdout_tail = []
        self.stderr_path = PROJECT_DIR / f"openvla05_worker_chunk_{self.chunk_index:03d}.stderr.log"
        self.stderr_file = self.stderr_path.open("w", encoding="utf-8")

        env = os.environ.copy()
        env["HF_HOME"] = str(LOCAL_HF_CACHE_DIR)
        env["HF_HUB_CACHE"] = str(LOCAL_HF_CACHE_DIR / "hub")
        env["TRANSFORMERS_CACHE"] = str(LOCAL_HF_CACHE_DIR / "transformers")
        token = globals().get("HF_TOKEN_IN_NOTEBOOK", "").strip()
        if token:
            env["HF_TOKEN"] = token
            env["HUGGINGFACE_HUB_TOKEN"] = token

        self.process = subprocess.Popen(
            [
                sys.executable,
                "-u",
                "-c",
                _openvla_chunked_worker_script(),
                OPENVLA_PROMPT,
                UNNORM_KEY,
                MODEL_ID,
                str(LOCAL_HF_CACHE_DIR),
                str(OFFLOAD_DIR),
            ],
            env=env,
            text=True,
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            stderr=self.stderr_file,
            bufsize=1,
        )

        def reader():
            try:
                for line in self.process.stdout:
                    self.stdout_queue.put(line)
            finally:
                self.stdout_queue.put(None)

        self.reader_thread = threading.Thread(target=reader, daemon=True)
        self.reader_thread.start()

        ready = self._read_marker(self.READY_MARKER, self.start_timeout_s)
        if not ready.get("ok"):
            self.close(kill=True)
            raise RuntimeError(
                "OpenVLA worker failed to load: "
                + ready.get("error", "unknown")
                + "\n"
                + ready.get("traceback", "")
                + "\nStderr tail:\n"
                + self.stderr_tail()
            )
        self.load_timing = ready
        print(
            f"[openvla_worker] chunk={self.chunk_index} ready "
            f"load={ready.get('vla_load_s', float('nan')):.1f}s stderr={self.stderr_path.name}"
        )

    def _read_marker(self, marker, timeout_s):
        import queue
        import time as _time

        deadline = _time.perf_counter() + float(timeout_s)
        while True:
            remaining = deadline - _time.perf_counter()
            if remaining <= 0:
                self.close(kill=True)
                raise TimeoutError(
                    f"Timed out waiting for OpenVLA worker marker {marker!r}. "
                    f"Recent stdout: {self.stdout_tail[-8:]}\nStderr tail:\n{self.stderr_tail()}"
                )
            try:
                line = self.stdout_queue.get(timeout=remaining)
            except queue.Empty:
                continue
            if line is None:
                raise RuntimeError(
                    "OpenVLA worker exited before returning a result. "
                    f"Recent stdout: {self.stdout_tail[-8:]}\nStderr tail:\n{self.stderr_tail()}"
                )
            line = line.rstrip("\r\n")
            if line:
                self.stdout_tail.append(line)
                self.stdout_tail = self.stdout_tail[-30:]
            if line.startswith(marker):
                return json.loads(line[len(marker):])

    def stderr_tail(self, chars=2400):
        try:
            self.stderr_file.flush()
            if self.stderr_path.exists():
                text = self.stderr_path.read_text(encoding="utf-8", errors="replace")
                return text[-chars:]
        except Exception:
            pass
        return ""

    def predict_from_image_path(self, image_path, request_id):
        if self.process.poll() is not None:
            raise RuntimeError("OpenVLA worker is not running before prediction request.")
        payload = {
            "cmd": "predict",
            "request_id": int(request_id),
            "image_path": str(image_path),
            "prompt": OPENVLA_PROMPT,
            "unnorm_key": UNNORM_KEY,
        }
        send_start = time.perf_counter()
        self.process.stdin.write(json.dumps(payload) + "\n")
        self.process.stdin.flush()
        response = self._read_marker(self.RESULT_MARKER, self.query_timeout_s)
        if response.get("request_id") != int(request_id):
            self.close(kill=True)
            raise RuntimeError(f"OpenVLA worker returned mismatched request id: {response}")
        if not response.get("ok"):
            raise RuntimeError(
                "OpenVLA worker prediction failed: "
                + response.get("error", "unknown")
                + "\n"
                + response.get("traceback", "")
                + "\nStderr tail:\n"
                + self.stderr_tail()
            )
        timings = response["timings"]
        timings["worker_chunk_index"] = float(self.chunk_index)
        timings["worker_query_index"] = float(self.query_count)
        timings["worker_roundtrip_s"] = float(time.perf_counter() - send_start)
        timings["pipeline_total_s"] = timings["worker_roundtrip_s"]
        self.query_count += 1
        return np.asarray(response["action"], dtype=np.float32).reshape(-1), timings

    def close(self, kill=False):
        proc = getattr(self, "process", None)
        try:
            if proc is not None and proc.poll() is None:
                if kill:
                    proc.kill()
                else:
                    try:
                        proc.stdin.write(json.dumps({"cmd": "quit"}) + "\n")
                        proc.stdin.flush()
                        proc.wait(timeout=30)
                    except Exception:
                        proc.kill()
        finally:
            try:
                if getattr(self, "stderr_file", None):
                    self.stderr_file.close()
            except Exception:
                pass


class OpenVLAChunkedWorkerManager:
    def __init__(self, queries_per_load=3, cooldown_s=20, start_timeout_s=420, query_timeout_s=420):
        self.queries_per_load = int(queries_per_load)
        self.cooldown_s = float(cooldown_s)
        self.start_timeout_s = float(start_timeout_s)
        self.query_timeout_s = float(query_timeout_s)
        self.chunk_index = 0
        self.worker = None

    def _ensure_worker(self):
        if self.worker is not None and self.worker.query_count < self.queries_per_load:
            return
        self.close_current()
        if self.chunk_index > 0 and self.cooldown_s > 0:
            print(f"[openvla_worker] cooling down {self.cooldown_s:.0f}s before next chunk")
            time.sleep(self.cooldown_s)
        self.worker = OpenVLAChunkedWorker(
            self.chunk_index,
            start_timeout_s=self.start_timeout_s,
            query_timeout_s=self.query_timeout_s,
        )
        self.chunk_index += 1

    def predict_from_image_path(self, image_path, request_id):
        self._ensure_worker()
        return self.worker.predict_from_image_path(image_path, request_id)

    def close_current(self):
        if self.worker is not None:
            self.worker.close()
            self.worker = None

    def close(self):
        self.close_current()



OPENVLA_ACTION_LABELS = ["delta_x", "delta_y", "delta_z", "delta_roll", "delta_pitch", "delta_yaw", "gripper"]


def safe_hz(seconds):
    seconds = float(seconds)
    return float(1.0 / seconds) if seconds > 0 else float("nan")


def summarize_samples(samples):
    samples = np.asarray(samples, dtype=np.float64)
    samples = samples[np.isfinite(samples)]
    if samples.size == 0:
        return {"count": 0, "mean_s": None, "median_s": None, "min_s": None, "max_s": None, "std_s": None, "mean_hz": None}
    mean_s = float(samples.mean())
    return {
        "count": int(samples.size),
        "mean_s": mean_s,
        "median_s": float(np.median(samples)),
        "min_s": float(samples.min()),
        "max_s": float(samples.max()),
        "std_s": float(samples.std()),
        "mean_hz": safe_hz(mean_s),
    }


def build_timing_summary(vla_log, low_level_step_times=None, low_level_render_times=None):
    low_level_step_times = low_level_step_times or []
    low_level_render_times = low_level_render_times or []
    return {
        "vla_model_inference": summarize_samples([entry.get("vla_model_inference_s", entry.get("inference_time_s")) for entry in vla_log]),
        "vla_load": summarize_samples([entry.get("vla_load_s") for entry in vla_log]),
        "vla_preprocess": summarize_samples([entry.get("vla_preprocess_s") for entry in vla_log]),
        "openvla_total": summarize_samples([entry.get("openvla_total_s") for entry in vla_log]),
        "whole_vla_pipeline_step": summarize_samples([entry.get("pipeline_total_s") for entry in vla_log]),
        "low_level_mujoco_step": summarize_samples(low_level_step_times),
        "video_frame_render_step": summarize_samples(low_level_render_times),
    }


def print_timing_summary(summary):
    print("\nTiming summary")
    labels = [
        ("VLA worker/chunk load", "vla_load"),
        ("VLA model inference only", "vla_model_inference"),
        ("VLA preprocessing", "vla_preprocess"),
        ("OpenVLA total", "openvla_total"),
        ("Whole VLA pipeline step", "whole_vla_pipeline_step"),
        ("Low-level MuJoCo step", "low_level_mujoco_step"),
        ("Video frame render step", "video_frame_render_step"),
    ]
    for label, key in labels:
        stats = summary.get(key, {})
        if not stats or stats.get("count", 0) == 0:
            print(f"  {label:28s}: no samples")
            continue
        print(
            f"  {label:28s}: mean={stats['mean_s']:.4f} s "
            f"({stats['mean_hz']:.2f} Hz), median={stats['median_s']:.4f} s, "
            f"min={stats['min_s']:.4f} s, max={stats['max_s']:.4f} s, n={stats['count']}"
        )


def format_action_lines(action):
    action = np.asarray(action, dtype=np.float32).reshape(-1)
    return [f"{label:>11s}: {value: .5f}" for label, value in zip(OPENVLA_ACTION_LABELS, action)]


def make_action_record(time_s, waypoint, action, timings):
    action = np.asarray(action, dtype=np.float32).reshape(-1)
    record = {
        "time_s": float(time_s),
        "waypoint": str(waypoint),
        "action": action.tolist(),
        "labels": OPENVLA_ACTION_LABELS[: len(action)],
    }
    record.update({key: float(value) for key, value in timings.items()})
    record["inference_time_s"] = record.get("vla_model_inference_s", record.get("openvla_total_s", 0.0))
    record["pipeline_hz"] = safe_hz(record.get("pipeline_total_s", 0.0))
    record["vla_model_hz"] = safe_hz(record.get("vla_model_inference_s", 0.0))
    return record


# === OpenVLA closed-loop controller adapter ===
OPENVLA_ROTATION_CONTROL_ENABLED = False
OPENVLA_ROTATION_NOTE = "rotation deltas are logged but ignored; current end-effector orientation is held implicitly by position-only IK"
OPENVLA_GRIPPER_THRESHOLD = 0.5  # OpenVLA/Bridge convention observed here: near 1=open, near 0=close.
OPENVLA_GRIPPER_OPENING_M = 0.04
OPENVLA_GRIPPER_CLOSED_M = 0.0
OPENVLA_TRANSLATION_SCALE = 1.0
OPENVLA_MAX_TRANSLATION_M = 0.02
OPENVLA_MAX_ROTATION_RAD = np.deg2rad(3.0)
OPENVLA_WORKSPACE_LOW = np.array([0.24, -0.25, 0.12], dtype=np.float64)
OPENVLA_WORKSPACE_HIGH = np.array([0.86, 0.25, 0.62], dtype=np.float64)


def get_ee_pose():
    pos = data.xpos[HAND_BODY_ID].copy()
    rotmat = data.xmat[HAND_BODY_ID].reshape(3, 3).copy()
    return {
        "position": pos,
        "rotation_matrix": rotmat,
        "position_list": pos.tolist(),
        "rotation_matrix_list": rotmat.tolist(),
    }


def clip_openvla_action(action):
    raw = np.asarray(action, dtype=np.float64).reshape(-1)
    if raw.size != 7:
        raise ValueError(f"Expected 7D OpenVLA action, got shape {raw.shape}: {raw}")
    clipped = raw.copy()
    clipped[:3] = np.clip(
        clipped[:3] * OPENVLA_TRANSLATION_SCALE,
        -OPENVLA_MAX_TRANSLATION_M,
        OPENVLA_MAX_TRANSLATION_M,
    )
    clipped[3:6] = np.clip(clipped[3:6], -OPENVLA_MAX_ROTATION_RAD, OPENVLA_MAX_ROTATION_RAD)
    clipped[6] = float(np.clip(clipped[6], 0.0, 1.0))
    return clipped.astype(np.float32)


def openvla_gripper_to_opening(gripper_value):
    gripper_value = float(np.clip(gripper_value, 0.0, 1.0))
    return OPENVLA_GRIPPER_OPENING_M if gripper_value >= OPENVLA_GRIPPER_THRESHOLD else OPENVLA_GRIPPER_CLOSED_M


def cartesian_delta_to_joint_target(delta_xyz, seed_qpos=None):
    ee_pose = get_ee_pose()
    unclipped_target = ee_pose["position"] + np.asarray(delta_xyz, dtype=np.float64)
    target_pos = np.clip(unclipped_target, OPENVLA_WORKSPACE_LOW, OPENVLA_WORKSPACE_HIGH)
    seed = current_arm_qpos_dict() if seed_qpos is None else seed_qpos

    # The existing IK helper uses the live MuJoCo data object internally.
    # Save and restore state so target planning does not secretly move the robot.
    saved_qpos = data.qpos.copy()
    saved_qvel = data.qvel.copy()
    saved_ctrl = data.ctrl.copy()
    try:
        qpos_target = solve_hand_position_ik(target_pos, seed_qpos=seed)
    finally:
        data.qpos[:] = saved_qpos
        data.qvel[:] = saved_qvel
        data.ctrl[:] = saved_ctrl
        mujoco.mj_forward(model, data)
    return qpos_target, target_pos, unclipped_target


def apply_openvla_action(action):
    ee_before = get_ee_pose()
    clipped = clip_openvla_action(action)
    qpos_target, target_pos, unclipped_target = cartesian_delta_to_joint_target(clipped[:3])
    gripper_opening = openvla_gripper_to_opening(clipped[6])
    return {
        "raw_action": np.asarray(action, dtype=np.float32).reshape(-1),
        "clipped_action": clipped,
        "ee_before": ee_before,
        "target_pos": target_pos,
        "unclipped_target_pos": np.asarray(unclipped_target, dtype=np.float64),
        "qpos_target": qpos_target,
        "gripper_opening": float(gripper_opening),
        "rotation_ignored": not OPENVLA_ROTATION_CONTROL_ENABLED,
        "rotation_note": OPENVLA_ROTATION_NOTE,
    }


def step_low_level_controller(qpos_target, gripper_opening, horizon_s=0.6, frame_dt=0.1, qpos_video_frames=None, observation_log=None, waypoint="openvla_closed_loop"):
    control_dt = model.opt.timestep
    steps = max(1, int(horizon_s / control_dt))
    next_frame_time = 0.0
    local_time = 0.0
    low_level_step_times = []
    start_qpos = current_arm_qpos_dict()
    start_gripper = float(np.mean(data.qpos[FINGER_QPOS_ADR]))

    for step_idx in range(steps):
        step_start = time.perf_counter()
        alpha = (step_idx + 1) / steps
        alpha = alpha * alpha * (3.0 - 2.0 * alpha)
        qpos_cmd = {
            joint_name: (1.0 - alpha) * start_qpos[joint_name] + alpha * qpos_target[joint_name]
            for joint_name in ARM_JOINT_NAMES
        }
        gripper_cmd = (1.0 - alpha) * start_gripper + alpha * gripper_opening
        set_arm_position_targets(qpos_cmd)
        set_gripper_opening(gripper_cmd)
        mujoco.mj_step(model, data)
        low_level_step_times.append(time.perf_counter() - step_start)
        local_time += control_dt

        if qpos_video_frames is not None and local_time >= next_frame_time:
            qpos_video_frames.append(data.qpos.copy())
            next_frame_time += frame_dt
        if observation_log is not None and local_time >= next_frame_time - frame_dt:
            observation_log.append({
                "local_time_s": float(local_time),
                "waypoint": waypoint,
                "hand_pos": data.xpos[HAND_BODY_ID].copy().tolist(),
                "block_pos_eval_only": data.xpos[BLOCK_BODY_ID].copy().tolist(),
                "gripper_qpos": data.qpos[FINGER_QPOS_ADR].copy().tolist(),
            })

    mujoco.mj_forward(model, data)
    return {
        "low_level_step_times": low_level_step_times,
        "ee_after": get_ee_pose(),
        "block_pos_eval_only": data.xpos[BLOCK_BODY_ID].copy().tolist(),
        "gripper_qpos_after": data.qpos[FINGER_QPOS_ADR].copy().tolist(),
    }


def render_agent_image_once(width=224, height=224):
    renderer = mujoco.Renderer(model, height=height, width=width)
    try:
        renderer.update_scene(data, camera=AGENT_CAMERA_NAME)
        rgb = renderer.render()
        return rgb, Image.fromarray(rgb).convert("RGB")
    finally:
        renderer.close()


def render_videos_from_qpos_frames(qpos_frames, world_video_path, agent_video_path, fps=10, world_size=(320, 240), agent_size=(224, 224)):
    if not qpos_frames:
        raise RuntimeError("No qpos frames were recorded; cannot render videos.")
    world_w, world_h = world_size
    agent_w, agent_h = agent_size
    saved_qpos = data.qpos.copy()
    world_renderer = mujoco.Renderer(model, height=world_h, width=world_w)
    agent_renderer = mujoco.Renderer(model, height=agent_h, width=agent_w)
    world_camera = mujoco.MjvCamera()
    world_camera.distance = 2.2
    world_camera.azimuth = 135
    world_camera.elevation = -25
    world_camera.lookat[:] = np.array([0.50, 0.00, 0.35])
    world_frames = []
    agent_frames = []
    try:
        for qpos in qpos_frames:
            data.qpos[:] = qpos
            mujoco.mj_forward(model, data)
            world_renderer.update_scene(data, camera=world_camera)
            world_frames.append(world_renderer.render())
            agent_renderer.update_scene(data, camera=AGENT_CAMERA_NAME)
            agent_frames.append(agent_renderer.render())
    finally:
        world_renderer.close()
        agent_renderer.close()
        data.qpos[:] = saved_qpos
        mujoco.mj_forward(model, data)
    media.write_video(world_video_path, world_frames, fps=fps)
    media.write_video(agent_video_path, agent_frames, fps=fps)
    return len(world_frames)


reset_task_state()
print("Initial hand position:", data.xpos[HAND_BODY_ID])
print("Initial block position:", data.xpos[BLOCK_BODY_ID])
print("OpenVLA execution mode:", globals().get("OPENVLA_EXECUTION_MODE", "subprocess"))
print("Tactile path: disabled")

In [ ]:
# Cell 7: scripted_baseline waypoint plan only.
# This is retained for comparison. It is not used by the OpenVLA closed-loop experiment.
SCRIPTED_BASELINE_CONTROL_MODE = "scripted_baseline"
RUN_SCRIPTED_BASELINE = False

WAYPOINT_SPECS = [
    ("home", np.array([0.307, 0.00, 0.590]), 0.04, 0.4),
    ("pregrasp", np.array([0.50, 0.00, 0.280]), 0.04, 1.5),
    ("grasp", np.array([0.50, 0.00, 0.180]), 0.04, 1.0),
    ("close_gripper", np.array([0.50, 0.00, 0.180]), 0.00, 0.8),
    ("lift", np.array([0.50, 0.00, 0.360]), 0.00, 1.0),
    ("transfer_far_side", np.array([0.72, 0.00, 0.360]), 0.00, 1.5),
    ("place", np.array([0.72, 0.00, 0.180]), 0.00, 1.0),
    ("open_gripper", np.array([0.72, 0.00, 0.180]), 0.04, 0.7),
    ("retreat", np.array([0.72, 0.00, 0.320]), 0.04, 1.0),
]

reset_task_state()
seed = current_arm_qpos_dict()
scripted_baseline_waypoints = []

for name, target_pos, gripper_opening, duration_s in WAYPOINT_SPECS:
    qpos_target = solve_hand_position_ik(target_pos, seed_qpos=seed)
    position_error = float(np.linalg.norm(target_pos - data.xpos[HAND_BODY_ID]))
    scripted_baseline_waypoints.append({
        "name": name,
        "target_pos": target_pos,
        "qpos": qpos_target,
        "gripper_opening": gripper_opening,
        "duration_s": duration_s,
        "ik_error_m": position_error,
    })
    seed = qpos_target
    print(f"[scripted_baseline] {name:18s} target={target_pos} hand={data.xpos[HAND_BODY_ID]} ik_error={position_error:.4f} m")

print("\nScripted baseline waypoints planned:", len(scripted_baseline_waypoints))
print("RUN_SCRIPTED_BASELINE:", RUN_SCRIPTED_BASELINE)
print("The OpenVLA closed-loop experiment does not call scripted_baseline_waypoints.")

In [ ]:
# Cell 8: Optional scripted_baseline runner.
# Disabled by default. The main experiment is Cell 9: openvla_closed_loop.
if not RUN_SCRIPTED_BASELINE:
    print("scripted_baseline is disabled. Set RUN_SCRIPTED_BASELINE=True only for baseline comparison.")
else:
    video_fps = 10
    qpos_video_frames = []
    low_level_step_times = []
    reset_task_state()
    set_arm_position_targets(scripted_baseline_waypoints[0]["qpos"])
    set_gripper_opening(scripted_baseline_waypoints[0]["gripper_opening"])
    mujoco.mj_forward(model, data)
    qpos_video_frames.append(data.qpos.copy())
    previous_qpos = scripted_baseline_waypoints[0]["qpos"].copy()
    previous_gripper = scripted_baseline_waypoints[0]["gripper_opening"]

    for waypoint in scripted_baseline_waypoints[1:]:
        result = step_low_level_controller(
            waypoint["qpos"],
            waypoint["gripper_opening"],
            horizon_s=waypoint["duration_s"],
            frame_dt=1.0 / video_fps,
            qpos_video_frames=qpos_video_frames,
            observation_log=None,
            waypoint="scripted_baseline:" + waypoint["name"],
        )
        low_level_step_times.extend(result["low_level_step_times"])
        print(f"[scripted_baseline] after {waypoint['name']:18s} hand={data.xpos[HAND_BODY_ID]} block={data.xpos[BLOCK_BODY_ID]}")

    final_block_pos = data.xpos[BLOCK_BODY_ID].copy()
    success = bool(final_block_pos[0] > 0.70 and abs(final_block_pos[1]) < 0.08 and 0.07 < final_block_pos[2] < 0.12)
    world_video_path = PROJECT_DIR / "openvla05_scripted_baseline_world.mp4"
    agent_video_path = PROJECT_DIR / "openvla05_scripted_baseline_agent_view.mp4"
    log_path = PROJECT_DIR / "openvla05_scripted_baseline_log.json"
    render_videos_from_qpos_frames(qpos_video_frames, world_video_path, agent_video_path, fps=video_fps)
    log_payload = {
        "control_mode": "scripted_baseline",
        "success": success,
        "final_block_pos": final_block_pos.tolist(),
        "scripted_waypoint_commands_used": len(scripted_baseline_waypoints),
        "openvla_queries_used_for_control": 0,
    }
    log_path.write_text(json.dumps(log_payload, indent=2), encoding="utf-8")
    print("scripted_baseline complete")
    print("Success:", success)
    print("Saved:", world_video_path, agent_video_path, log_path)

In [ ]:
# Cell 9: openvla_closed_loop main experiment.
# No scripted pick/place waypoints, no privileged object coordinates for control, no phase labels.
CONTROL_MODE = "openvla_closed_loop"
RUN_OPENVLA_CLOSED_LOOP = True
USE_ZERO_ACTIONS = False
USE_RANDOM_ACTIONS = False

# Laptop-safe chunked-worker defaults.
# 8 queries is still a short task attempt, but it is enough to see whether motion trends toward the block.
OPENVLA_CLOSED_LOOP_MAX_QUERIES = 16
CONTROL_HORIZON_S = 0.6
VIDEO_FPS = 10
STOP_ON_WORKSPACE_VIOLATION = True

OPENVLA_WORKER_ENABLED = True
OPENVLA_WORKER_QUERIES_PER_LOAD = 8
OPENVLA_COOLDOWN_BETWEEN_WORKERS_S = 20
OPENVLA_WORKER_START_TIMEOUT_S = 600
OPENVLA_WORKER_QUERY_TIMEOUT_S = 600

if USE_ZERO_ACTIONS and USE_RANDOM_ACTIONS:
    raise ValueError("Choose at most one sanity ablation: USE_ZERO_ACTIONS or USE_RANDOM_ACTIONS.")
if not RUN_OPENVLA_CLOSED_LOOP:
    raise RuntimeError("RUN_OPENVLA_CLOSED_LOOP is False. Enable it to run the OpenVLA closed-loop experiment.")
if not RUN_OPENVLA and not (USE_ZERO_ACTIONS or USE_RANDOM_ACTIONS):
    raise RuntimeError("OpenVLA is unavailable and no ablation is enabled. This cell will not fall back to scripted control.")

reset_task_state()
qpos_video_frames = [data.qpos.copy()]
closed_loop_log = []
observation_log = []
all_low_level_step_times = []
scripted_waypoint_commands_used = 0
openvla_actions_applied_to_controller = False
episode_stop_reason = "max_policy_queries_reached"
openvla_worker_manager = None

print("Control mode:", CONTROL_MODE)
print("Policy queries planned:", OPENVLA_CLOSED_LOOP_MAX_QUERIES)
print("Control horizon per action:", CONTROL_HORIZON_S, "s")
print("Sanity ablations: USE_ZERO_ACTIONS=", USE_ZERO_ACTIONS, "USE_RANDOM_ACTIONS=", USE_RANDOM_ACTIONS)
print("OpenVLA worker enabled:", OPENVLA_WORKER_ENABLED)
print("Worker queries per load:", OPENVLA_WORKER_QUERIES_PER_LOAD)
print("Worker cooldown between chunks:", OPENVLA_COOLDOWN_BETWEEN_WORKERS_S, "s")
print("Rotation handling:", OPENVLA_ROTATION_NOTE)
print("Gripper convention: action[6] >= ", OPENVLA_GRIPPER_THRESHOLD, "=> open, otherwise close")

try:
    if RUN_OPENVLA and OPENVLA_WORKER_ENABLED and not (USE_ZERO_ACTIONS or USE_RANDOM_ACTIONS):
        openvla_worker_manager = OpenVLAChunkedWorkerManager(
            queries_per_load=OPENVLA_WORKER_QUERIES_PER_LOAD,
            cooldown_s=OPENVLA_COOLDOWN_BETWEEN_WORKERS_S,
            start_timeout_s=OPENVLA_WORKER_START_TIMEOUT_S,
            query_timeout_s=OPENVLA_WORKER_QUERY_TIMEOUT_S,
        )

    for query_idx in range(OPENVLA_CLOSED_LOOP_MAX_QUERIES):
        sim_time = float(query_idx * CONTROL_HORIZON_S)
        ee_before = get_ee_pose()
        camera_rgb, camera_image = render_agent_image_once(width=224, height=224)
        camera_image_path = PROJECT_DIR / f"openvla05_closed_loop_query_{query_idx:03d}.png"
        camera_image.save(camera_image_path)

        if USE_ZERO_ACTIONS:
            raw_action = np.zeros(7, dtype=np.float32)
            raw_action[6] = 1.0
            timings = {"vla_model_inference_s": 0.0, "openvla_total_s": 0.0, "pipeline_total_s": 0.0}
            action_source = "zero_ablation"
        elif USE_RANDOM_ACTIONS:
            raw_action = np.random.uniform(-1.0, 1.0, size=7).astype(np.float32)
            raw_action[6] = np.random.uniform(0.0, 1.0)
            timings = {"vla_model_inference_s": 0.0, "openvla_total_s": 0.0, "pipeline_total_s": 0.0}
            action_source = "random_ablation"
        else:
            pipeline_start = time.perf_counter()
            if OPENVLA_WORKER_ENABLED:
                prediction = openvla_worker_manager.predict_from_image_path(camera_image_path, request_id=query_idx)
                action_source = "openvla_chunked_worker"
            else:
                prediction = openvla_predict_from_image(camera_image)
                action_source = "openvla_one_shot_subprocess"
            if prediction is None:
                raise RuntimeError("OpenVLA prediction returned None. Refusing to fall back to scripted control.")
            raw_action, timings = prediction
            timings["pipeline_total_s"] = float(time.perf_counter() - pipeline_start)

        adapter = apply_openvla_action(raw_action)
        low_level = step_low_level_controller(
            adapter["qpos_target"],
            adapter["gripper_opening"],
            horizon_s=CONTROL_HORIZON_S,
            frame_dt=1.0 / VIDEO_FPS,
            qpos_video_frames=qpos_video_frames,
            observation_log=observation_log,
            waypoint=CONTROL_MODE,
        )
        all_low_level_step_times.extend(low_level["low_level_step_times"])
        openvla_actions_applied_to_controller = openvla_actions_applied_to_controller or action_source.startswith("openvla")

        record = {
            "query_index": int(query_idx),
            "sim_time_s": sim_time,
            "camera_image_source": str(camera_image_path),
            "action_source": action_source,
            "raw_openvla_action": np.asarray(raw_action, dtype=np.float32).reshape(-1).tolist(),
            "clipped_action": adapter["clipped_action"].tolist(),
            "ee_pose_before": adapter["ee_before"]["position_list"],
            "ee_target_after_action": adapter["target_pos"].tolist(),
            "ee_unclipped_target_after_action": adapter["unclipped_target_pos"].tolist(),
            "ee_pose_after_low_level": low_level["ee_after"]["position_list"],
            "rotation_ignored": adapter["rotation_ignored"],
            "rotation_note": adapter["rotation_note"],
            "gripper_command_opening_m": adapter["gripper_opening"],
            "gripper_qpos_after": low_level["gripper_qpos_after"],
            "block_pose_eval_only": low_level["block_pos_eval_only"],
            "timings": {key: float(value) for key, value in timings.items()},
            "effective_policy_hz_including_load": safe_hz(timings.get("pipeline_total_s", timings.get("openvla_total_s", 0.0))),
        }
        closed_loop_log.append(record)

        print(
            f"[{CONTROL_MODE}] query={query_idx} source={action_source} "
            f"raw={record['raw_openvla_action']} clipped={record['clipped_action']} "
            f"target={record['ee_target_after_action']} after={record['ee_pose_after_low_level']} "
            f"gripper={record['gripper_command_opening_m']:.3f}"
        )

        ee_after = np.asarray(low_level["ee_after"]["position"], dtype=np.float64)
        if STOP_ON_WORKSPACE_VIOLATION and (np.any(ee_after < OPENVLA_WORKSPACE_LOW - 0.03) or np.any(ee_after > OPENVLA_WORKSPACE_HIGH + 0.03)):
            episode_stop_reason = "workspace_violation"
            print("Stopping: end-effector left the safety workspace.")
            break
finally:
    if openvla_worker_manager is not None:
        openvla_worker_manager.close()

final_block_pos = data.xpos[BLOCK_BODY_ID].copy()
success = bool(final_block_pos[0] > 0.70 and abs(final_block_pos[1]) < 0.08 and 0.07 < final_block_pos[2] < 0.12)

world_video_path = PROJECT_DIR / "openvla05_closed_loop_world.mp4"
agent_video_path = PROJECT_DIR / "openvla05_closed_loop_agent_view.mp4"
log_path = PROJECT_DIR / "openvla05_closed_loop_log.json"
frames_rendered = render_videos_from_qpos_frames(qpos_video_frames, world_video_path, agent_video_path, fps=VIDEO_FPS)

timing_summary = build_timing_summary(
    [
        {
            **entry["timings"],
            "pipeline_total_s": entry["timings"].get("pipeline_total_s", 0.0),
        }
        for entry in closed_loop_log
    ],
    all_low_level_step_times,
    [],
)

control_provenance = {
    "control_mode": CONTROL_MODE,
    "number_of_openvla_queries_used_for_control": int(sum(1 for entry in closed_loop_log if entry["action_source"].startswith("openvla"))),
    "number_of_scripted_waypoint_commands_used": scripted_waypoint_commands_used,
    "openvla_actions_applied_to_controller": bool(openvla_actions_applied_to_controller),
    "zero_action_ablation": bool(USE_ZERO_ACTIONS),
    "random_action_ablation": bool(USE_RANDOM_ACTIONS),
    "openvla_worker_enabled": bool(OPENVLA_WORKER_ENABLED),
    "openvla_worker_queries_per_load": int(OPENVLA_WORKER_QUERIES_PER_LOAD),
    "openvla_worker_cooldown_s": float(OPENVLA_COOLDOWN_BETWEEN_WORKERS_S),
}

log_payload = {
    "instruction": TASK_INSTRUCTION,
    "control_provenance": control_provenance,
    "stop_reason": episode_stop_reason,
    "success_eval_only": success,
    "start_block_pos_eval_only": START_BLOCK_POS.tolist(),
    "place_block_pos_eval_only": PLACE_BLOCK_POS.tolist(),
    "final_block_pos_eval_only": final_block_pos.tolist(),
    "workspace_low": OPENVLA_WORKSPACE_LOW.tolist(),
    "workspace_high": OPENVLA_WORKSPACE_HIGH.tolist(),
    "max_translation_m": OPENVLA_MAX_TRANSLATION_M,
    "rotation_control_enabled": OPENVLA_ROTATION_CONTROL_ENABLED,
    "rotation_note": OPENVLA_ROTATION_NOTE,
    "gripper_threshold": OPENVLA_GRIPPER_THRESHOLD,
    "control_horizon_s": CONTROL_HORIZON_S,
    "policy_queries": closed_loop_log,
    "observations": observation_log,
    "timing_summary": timing_summary,
    "videos": {
        "world": str(world_video_path),
        "agent": str(agent_video_path),
        "frames_rendered": int(frames_rendered),
    },
}
log_path.write_text(json.dumps(log_payload, indent=2), encoding="utf-8")

print("\nControl mode: OpenVLA closed-loop")
print("Number of OpenVLA queries used for control:", control_provenance["number_of_openvla_queries_used_for_control"])
print("Number of scripted waypoint commands used:", control_provenance["number_of_scripted_waypoint_commands_used"])
print("OpenVLA actions applied to controller:", control_provenance["openvla_actions_applied_to_controller"])
print("Task success from OpenVLA-driven motion:", success)
print("Stop reason:", episode_stop_reason)
print("Saved world video:", world_video_path)
print("Saved agent video:", agent_video_path)
print("Saved JSON log:", log_path)


In [ ]:
# Cell 10: Display the OpenVLA closed-loop videos and action provenance.
videos = [
    ("OpenVLA closed-loop external view", PROJECT_DIR / "openvla05_closed_loop_world.mp4"),
    ("OpenVLA closed-loop wrist camera", PROJECT_DIR / "openvla05_closed_loop_agent_view.mp4"),
]
log_path = PROJECT_DIR / "openvla05_closed_loop_log.json"

missing = [str(path) for _, path in videos if not path.exists()]
if missing:
    raise FileNotFoundError("Run Cell 9 first. Missing: " + ", ".join(missing))
if not log_path.exists():
    raise FileNotFoundError("Run Cell 9 first. Missing: " + str(log_path))

saved_log = json.loads(log_path.read_text(encoding="utf-8"))
provenance = saved_log.get("control_provenance", {})
policy_queries = saved_log.get("policy_queries", [])

video_cards = []
for title, path in videos:
    encoded_video = b64encode(path.read_bytes()).decode("ascii")
    video_cards.append(
        f"""
        <div style="min-width: 280px; max-width: 560px; flex: 1;">
          <div style="font-weight: 600; margin: 0 0 6px 0;">{escape(title)}</div>
          <video controls loop muted playsinline
                 style="width: 100%; border: 1px solid #ccc; background: #111;"
                 src="data:video/mp4;base64,{encoded_video}"></video>
          <div style="font-family: monospace; font-size: 12px; color: #666; margin-top: 4px;">
            {escape(str(path))}
          </div>
        </div>
        """
    )


def query_table_html(entries):
    if not entries:
        return "<div style='font-family: monospace; color: #666;'>No policy queries logged.</div>"
    blocks = []
    for entry in entries:
        raw = entry.get("raw_openvla_action", [])
        clipped = entry.get("clipped_action", [])
        rows = "".join(
            f"<tr><td>{escape(label)}</td><td style='text-align:right'>{float(r): .5f}</td><td style='text-align:right'>{float(c): .5f}</td></tr>"
            for label, r, c in zip(OPENVLA_ACTION_LABELS, raw, clipped)
        )
        timings = entry.get("timings", {})
        blocks.append(
            f"""
            <details open style="margin-bottom: 10px; border-bottom: 1px solid #ddd; padding-bottom: 8px;">
              <summary style="cursor: pointer; font-weight: 600;">
                query={entry.get('query_index')} | source={escape(str(entry.get('action_source')))} | pipeline={float(timings.get('pipeline_total_s', 0.0)):.2f}s
              </summary>
              <div style="font-family: monospace; font-size: 12px; margin-top: 6px;">
                ee before: {escape(str(entry.get('ee_pose_before')))}<br/>
                ee target: {escape(str(entry.get('ee_target_after_action')))}<br/>
                ee after: {escape(str(entry.get('ee_pose_after_low_level')))}<br/>
                gripper opening m: {float(entry.get('gripper_command_opening_m', 0.0)):.3f}
              </div>
              <table style="width:100%; font-family: monospace; font-size: 12px; border-collapse: collapse; margin-top: 6px;">
                <tr><th style='text-align:left'>Action</th><th>Raw</th><th>Clipped</th></tr>
                {rows}
              </table>
            </details>
            """
        )
    return "\n".join(blocks)


panel_html = f"""
<div style="display: grid; grid-template-columns: minmax(320px, 2fr) minmax(320px, 1fr); gap: 18px; align-items: start;">
  <div style="display: flex; gap: 16px; flex-wrap: wrap;">
    {"".join(video_cards)}
  </div>
  <div style="border: 1px solid #ccc; padding: 10px; max-height: 760px; overflow: auto; background: #fafafa;">
    <div style="font-weight: 700; margin-bottom: 8px;">Control provenance</div>
    <div style="font-family: monospace; font-size: 12px; line-height: 1.45;">
      mode: {escape(str(provenance.get('control_mode')))}<br/>
      OpenVLA queries used: {escape(str(provenance.get('number_of_openvla_queries_used_for_control')))}<br/>
      scripted waypoint commands used: {escape(str(provenance.get('number_of_scripted_waypoint_commands_used')))}<br/>
      OpenVLA actions applied: {escape(str(provenance.get('openvla_actions_applied_to_controller')))}<br/>
      success eval only: {escape(str(saved_log.get('success_eval_only')))}<br/>
      stop reason: {escape(str(saved_log.get('stop_reason')))}
    </div>
    <div style="font-weight: 700; margin: 14px 0 8px 0;">Policy Queries</div>
    {query_table_html(policy_queries)}
  </div>
</div>
"""

display(HTML(panel_html))

In [ ]:
# Cell 11: Native MuJoCo GUI viewer is disabled in this slim laptop notebook.
OPEN_NATIVE_GUI = False
if OPEN_NATIVE_GUI:
    raise RuntimeError("The native GUI path is intentionally disabled for this laptop VLA test. Use the videos from Cell 8 instead.")
print("Native MuJoCo GUI is disabled. Use Cell 8 videos instead.")

In [ ]:
# Cell 12: Native MuJoCo GUI replay is disabled in this slim laptop notebook.
OPEN_NATIVE_GUI_REPLAY = False
if OPEN_NATIVE_GUI_REPLAY:
    raise RuntimeError("The native GUI replay path is intentionally disabled for this laptop VLA test. Use the videos from Cell 8 instead.")
print("Native MuJoCo GUI replay is disabled. Use Cell 8 videos instead.")

## LoRA data collection append cells

Run the original notebook through Cell 6 first. Then run these cells.

These cells do not query OpenVLA. They use the original scripted expert trajectory only to collect demonstrations for later LoRA fine-tuning, with 10 Hz image/action samples and explicit metadata.


In [ ]:
# Cell A: Rebuild the Panda scene with the working right-finger grasp contact geometry.
# This intentionally keeps tactile rendering/data disabled. The GelSight-shaped body is used only
# as the physical right-finger contact pad that made the original scripted grasp work reliably.
import importlib.util

if importlib.util.find_spec("imageio") is None:
    pip_install(["imageio"])
import imageio.v2 as imageio

MENAGERIE_DIR = PROJECT_DIR / "mujoco_menagerie"
PANDA_DIR = MENAGERIE_DIR / "franka_emika_panda"
SOURCE_PANDA_XML = PANDA_DIR / "panda.xml"
PANDA_WITH_DATA_GRASP_XML = PANDA_DIR / "panda_openvla07_lora_data_grasp.xml"
PANDA_XML = PANDA_DIR / "openvla07_lora_data_scene.xml"

GELSIGHT_BODY_NAME = "gelsight_mini_right"
GELSIGHT_SITE_NAME = "gelsight_mini_right_site"
GELSIGHT_MOUNT_BODY_NAME = "right_finger"
GELSIGHT_GEL_PAD_GEOM_NAME = "gelsight_mini_right_gel_pad_geom"
GELSIGHT_GEOM_NAMES = [
    "gelsight_mini_right_case_geom",
    "gelsight_mini_right_gel_pad_geom",
    "gelsight_mini_right_label_geom",
    "gelsight_mini_right_cable_geom",
    "gelsight_mini_right_mount_plate_geom",
    "gelsight_mini_right_standoff_top_geom",
    "gelsight_mini_right_standoff_bottom_geom",
]


def _find_body(root, body_name):
    for body in root.iter("body"):
        if body.get("name") == body_name:
            return body
    raise ValueError(f"Could not find body named {body_name!r}")


def _remove_named_child_body(parent, child_name):
    for child in list(parent):
        if child.tag == "body" and child.get("name") == child_name:
            parent.remove(child)


if not SOURCE_PANDA_XML.exists():
    raise FileNotFoundError(f"Could not find Panda model at {SOURCE_PANDA_XML}. Run original Cell 3 first.")

panda_tree = ET.parse(SOURCE_PANDA_XML)
panda_root = panda_tree.getroot()
hand_body = _find_body(panda_root, "hand")
right_finger_body = _find_body(panda_root, GELSIGHT_MOUNT_BODY_NAME)

_remove_named_child_body(hand_body, "agent_camera_mount")
_remove_named_child_body(right_finger_body, GELSIGHT_BODY_NAME)

# Match the successful OpenVLA05 setup: remove the stock right-finger collision pads and replace
# that contact surface with a small, soft, high-friction gel pad. Left finger remains stock.
removed_right_contact_geoms = []
for child in list(right_finger_body):
    if child.tag != "geom":
        continue
    geom_class = child.get("class", "")
    is_stock_contact = geom_class == "collision" or geom_class.startswith("fingertip_pad_collision")
    if is_stock_contact:
        removed_right_contact_geoms.append(child.get("name", geom_class))
        right_finger_body.remove(child)

mount_body = ET.Element("body", name="agent_camera_mount", pos="0.08 0 -0.12")
ET.SubElement(
    mount_body,
    "geom",
    name="agent_camera_boom_geom",
    type="capsule",
    fromto="0 0 0 -0.08 0 0.12",
    size="0.008",
    rgba="0.02 0.02 0.02 1",
    contype="0",
    conaffinity="0",
    group="2",
)
ET.SubElement(
    mount_body,
    "geom",
    name="agent_camera_body_geom",
    type="box",
    size="0.020 0.016 0.012",
    rgba="0.02 0.02 0.02 1",
    contype="0",
    conaffinity="0",
    group="2",
)
ET.SubElement(
    mount_body,
    "camera",
    name="agent_wrist_camera",
    mode="fixed",
    fovy="82",
    xyaxes="-0.000007 -1.000000 0.000034 -0.976592 -0.000000 -0.215100",
)

sensor_body = ET.Element("body", name=GELSIGHT_BODY_NAME, pos="0.0 0.01 0.045")
ET.SubElement(sensor_body, "inertial", mass="0.025", pos="0 0 0", diaginertia="3.6e-6 4.8e-6 1.8e-6")
ET.SubElement(
    sensor_body,
    "geom",
    name="gelsight_mini_right_mount_plate_geom",
    type="box",
    size="0.013 0.0012 0.019",
    pos="0 -0.011 0",
    rgba="0.08 0.08 0.08 1",
    contype="0",
    conaffinity="0",
    group="2",
)
ET.SubElement(
    sensor_body,
    "geom",
    name="gelsight_mini_right_standoff_top_geom",
    type="capsule",
    fromto="0.007 -0.010 0.012 0.007 -0.005 0.012",
    size="0.0016",
    rgba="0.02 0.02 0.025 1",
    contype="0",
    conaffinity="0",
    group="2",
)
ET.SubElement(
    sensor_body,
    "geom",
    name="gelsight_mini_right_standoff_bottom_geom",
    type="capsule",
    fromto="0.007 -0.010 -0.012 0.007 -0.005 -0.012",
    size="0.0016",
    rgba="0.02 0.02 0.025 1",
    contype="0",
    conaffinity="0",
    group="2",
)
ET.SubElement(
    sensor_body,
    "geom",
    name="gelsight_mini_right_case_geom",
    type="box",
    size="0.012 0.005 0.018",
    rgba="0.02 0.02 0.025 1",
    contype="0",
    conaffinity="0",
    group="2",
)
ET.SubElement(
    sensor_body,
    "geom",
    name=GELSIGHT_GEL_PAD_GEOM_NAME,
    type="box",
    size="0.009 0.0012 0.013",
    pos="0 -0.0072 0",
    rgba="0.0 0.75 0.95 0.70",
    contype="1",
    conaffinity="1",
    condim="4",
    friction="1.0 0.005 0.0001",
    solimp="0.90 0.95 0.001",
    solref="0.004 1",
    group="3",
)
ET.SubElement(
    sensor_body,
    "geom",
    name="gelsight_mini_right_label_geom",
    type="box",
    size="0.008 0.0010 0.0025",
    pos="0 -0.0074 0.015",
    rgba="0.92 0.92 0.88 1",
    contype="0",
    conaffinity="0",
    group="2",
)
ET.SubElement(
    sensor_body,
    "geom",
    name="gelsight_mini_right_cable_geom",
    type="capsule",
    fromto="0 0.006 -0.016 0 0.018 -0.038",
    size="0.0018",
    rgba="0.01 0.01 0.01 1",
    contype="0",
    conaffinity="0",
    group="2",
)
ET.SubElement(
    sensor_body,
    "site",
    name=GELSIGHT_SITE_NAME,
    type="box",
    size="0.009 0.0010 0.013",
    pos="0 -0.0086 0",
    rgba="0.0 0.9 1.0 0.35",
    group="4",
)
right_finger_body.append(sensor_body)
hand_body.insert(0, mount_body)

ET.indent(panda_tree, space="  ")
panda_tree.write(PANDA_WITH_DATA_GRASP_XML, encoding="unicode")

PANDA_XML.write_text(
    f"""<mujoco model="openvla07 lora data panda pick place scene">
  <include file="{PANDA_WITH_DATA_GRASP_XML.name}"/>

  <statistic center="0.50 0 0.35" extent="1.1"/>

  <visual>
    <headlight diffuse="0.6 0.6 0.6" ambient="0.3 0.3 0.3" specular="0 0 0"/>
    <rgba haze="0.15 0.25 0.35 1"/>
    <global azimuth="120" elevation="-20" offwidth="1920" offheight="1080"/>
  </visual>

  <asset>
    <texture type="skybox" builtin="gradient" rgb1="0.3 0.5 0.7" rgb2="0 0 0" width="512" height="3072"/>
    <texture type="2d" name="groundplane" builtin="checker" mark="edge" rgb1="0.2 0.3 0.4" rgb2="0.1 0.2 0.3" markrgb="0.8 0.8 0.8" width="300" height="300"/>
    <material name="groundplane" texture="groundplane" texuniform="true" texrepeat="5 5" reflectance="0.2"/>
    <material name="table_mat" rgba="0.48 0.38 0.26 1"/>
    <material name="red_block_mat" rgba="1 0.03 0.02 1"/>
  </asset>

  <worldbody>
    <light pos="0 0 1.5" dir="0 0 -1" directional="true"/>
    <geom name="floor" size="0 0 0.05" type="plane" material="groundplane"/>
    <body name="table" pos="0.55 0 0.025">
      <geom name="table_top" type="box" size="0.35 0.45 0.025" material="table_mat" friction="1 0.005 0.0001"/>
    </body>
    <body name="red_block" pos="0.50 0 0.085">
      <freejoint name="red_block_freejoint"/>
      <geom name="red_block_geom" type="box" size="0.035 0.035 0.035" material="red_block_mat" mass="0.05" friction="1 0.005 0.0001"/>
    </body>
  </worldbody>
</mujoco>
""",
    encoding="utf-8",
)

model = mujoco.MjModel.from_xml_path(str(PANDA_XML))
data = mujoco.MjData(model)
mujoco.mj_forward(model, data)

print("Loaded LoRA data scene:", PANDA_XML)
print("Right-finger stock contact geoms removed:", removed_right_contact_geoms)
print("Gel pad contact geom:", GELSIGHT_GEL_PAD_GEOM_NAME)
print("Tactile images/data: disabled; this is only grasp contact geometry.")
print("nq:", model.nq, "nv:", model.nv, "nu:", model.nu, "timestep:", model.opt.timestep)


In [ ]:
# Cell B: Refresh scene IDs and controller helpers after replacing model/data in Cell A.
AGENT_CAMERA_NAME = "agent_wrist_camera"
TASK_INSTRUCTION = "pick up the red block from the table and place it on the far side of the table"
OPENVLA_PROMPT = f"In: What action should the robot take to {TASK_INSTRUCTION}?\nOut: "

START_BLOCK_POS = np.array([0.50, 0.00, 0.085], dtype=np.float64)
PLACE_BLOCK_POS = np.array([0.78, 0.00, 0.085], dtype=np.float64)


def require_mj_id(obj_type, name):
    obj_id = mujoco.mj_name2id(model, obj_type, name)
    if obj_id < 0:
        raise RuntimeError(f"Missing MuJoCo object {name!r} of type {obj_type}")
    return obj_id


body_ids = {
    "hand": require_mj_id(mujoco.mjtObj.mjOBJ_BODY, "hand"),
    "left_finger": require_mj_id(mujoco.mjtObj.mjOBJ_BODY, "left_finger"),
    "right_finger": require_mj_id(mujoco.mjtObj.mjOBJ_BODY, "right_finger"),
    "red_block": require_mj_id(mujoco.mjtObj.mjOBJ_BODY, "red_block"),
    "agent_camera_mount": require_mj_id(mujoco.mjtObj.mjOBJ_BODY, "agent_camera_mount"),
    GELSIGHT_BODY_NAME: require_mj_id(mujoco.mjtObj.mjOBJ_BODY, GELSIGHT_BODY_NAME),
}
site_ids = {GELSIGHT_SITE_NAME: require_mj_id(mujoco.mjtObj.mjOBJ_SITE, GELSIGHT_SITE_NAME)}
camera_id = require_mj_id(mujoco.mjtObj.mjOBJ_CAMERA, AGENT_CAMERA_NAME)

HAND_BODY_ID = body_ids["hand"]
LEFT_FINGER_BODY_ID = body_ids["left_finger"]
RIGHT_FINGER_BODY_ID = body_ids["right_finger"]
GELSIGHT_BODY_ID = body_ids[GELSIGHT_BODY_NAME]
GELSIGHT_SITE_ID = site_ids[GELSIGHT_SITE_NAME]
BLOCK_BODY_ID = body_ids["red_block"]
BLOCK_JOINT_ID = require_mj_id(mujoco.mjtObj.mjOBJ_JOINT, "red_block_freejoint")
BLOCK_QPOS_ADR = model.jnt_qposadr[BLOCK_JOINT_ID]

ARM_JOINT_NAMES = [f"joint{i}" for i in range(1, 8)]
ARM_JOINT_IDS = [require_mj_id(mujoco.mjtObj.mjOBJ_JOINT, name) for name in ARM_JOINT_NAMES]
ARM_QPOS_ADR = [model.jnt_qposadr[joint_id] for joint_id in ARM_JOINT_IDS]
ARM_DOF_ADR = [model.jnt_dofadr[joint_id] for joint_id in ARM_JOINT_IDS]

FINGER_JOINT_NAMES = ["finger_joint1", "finger_joint2"]
FINGER_JOINT_IDS = [require_mj_id(mujoco.mjtObj.mjOBJ_JOINT, name) for name in FINGER_JOINT_NAMES]
FINGER_QPOS_ADR = [model.jnt_qposadr[joint_id] for joint_id in FINGER_JOINT_IDS]

HOME_QPOS = {
    "joint1": 0.0,
    "joint2": -0.785,
    "joint3": 0.0,
    "joint4": -2.356,
    "joint5": 0.0,
    "joint6": 1.571,
    "joint7": 0.785,
    "finger_joint1": 0.04,
    "finger_joint2": 0.04,
}


def current_arm_qpos_dict():
    return {
        joint_name: float(data.qpos[model.jnt_qposadr[joint_id]])
        for joint_name, joint_id in zip(ARM_JOINT_NAMES, ARM_JOINT_IDS)
    }


def current_finger_opening_m():
    return float(np.mean([data.qpos[qpos_adr] for qpos_adr in FINGER_QPOS_ADR]))


def set_arm_position_targets(joint_targets):
    for actuator_id in range(model.nu):
        if model.actuator_trntype[actuator_id] != mujoco.mjtTrn.mjTRN_JOINT:
            continue
        joint_id = model.actuator_trnid[actuator_id, 0]
        joint_name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_JOINT, joint_id)
        if joint_name not in joint_targets:
            continue
        value = float(joint_targets[joint_name])
        if model.actuator_ctrllimited[actuator_id]:
            low, high = model.actuator_ctrlrange[actuator_id]
            value = float(np.clip(value, low, high))
        data.ctrl[actuator_id] = value


def set_gripper_opening(opening_m):
    # Panda menagerie maps 0.04 m open to actuator ctrl=255 and 0 m closed to ctrl=0.
    ctrl = float(np.clip(opening_m / 0.04 * 255.0, 0.0, 255.0))
    for actuator_id in range(model.nu):
        if model.actuator_trntype[actuator_id] == mujoco.mjtTrn.mjTRN_TENDON:
            data.ctrl[actuator_id] = ctrl


def reset_task_state_randomized(cube_pos=None, robot_joint_noise=None, settle_steps=80):
    cube_pos = START_BLOCK_POS if cube_pos is None else np.asarray(cube_pos, dtype=np.float64)
    robot_joint_noise = {} if robot_joint_noise is None else robot_joint_noise
    mujoco.mj_resetData(model, data)
    data.qvel[:] = 0.0
    for joint_name, value in HOME_QPOS.items():
        joint_id = require_mj_id(mujoco.mjtObj.mjOBJ_JOINT, joint_name)
        qpos_adr = model.jnt_qposadr[joint_id]
        if joint_name in ARM_JOINT_NAMES:
            value = float(value) + float(robot_joint_noise.get(joint_name, 0.0))
            low, high = model.jnt_range[joint_id]
            value = float(np.clip(value, low, high))
        data.qpos[qpos_adr] = value
    data.qpos[BLOCK_QPOS_ADR:BLOCK_QPOS_ADR + 7] = np.array([*cube_pos, 1.0, 0.0, 0.0, 0.0])
    set_arm_position_targets(current_arm_qpos_dict())
    set_gripper_opening(0.04)
    mujoco.mj_forward(model, data)
    for _ in range(settle_steps):
        set_arm_position_targets(current_arm_qpos_dict())
        set_gripper_opening(0.04)
        mujoco.mj_step(model, data)
    return cube_pos.copy()


def reset_task_state():
    return reset_task_state_randomized()


def solve_hand_position_ik(target_pos, seed_qpos=None, max_iter=300, tolerance=0.0015):
    target_pos = np.asarray(target_pos, dtype=np.float64)
    if seed_qpos is not None:
        for joint_name, value in seed_qpos.items():
            joint_id = require_mj_id(mujoco.mjtObj.mjOBJ_JOINT, joint_name)
            data.qpos[model.jnt_qposadr[joint_id]] = float(value)
    mujoco.mj_forward(model, data)

    jacp = np.zeros((3, model.nv), dtype=np.float64)
    jacr = np.zeros((3, model.nv), dtype=np.float64)

    for _ in range(max_iter):
        error = target_pos - data.xpos[HAND_BODY_ID]
        if np.linalg.norm(error) < tolerance:
            break
        mujoco.mj_jacBody(model, data, jacp, jacr, HAND_BODY_ID)
        jacobian = jacp[:, ARM_DOF_ADR]
        damping = 0.08
        delta_q = jacobian.T @ np.linalg.solve(
            jacobian @ jacobian.T + damping * damping * np.eye(3),
            error,
        )
        delta_q = np.clip(delta_q, -0.035, 0.035)
        for idx, qpos_adr in enumerate(ARM_QPOS_ADR):
            joint_id = ARM_JOINT_IDS[idx]
            low, high = model.jnt_range[joint_id]
            data.qpos[qpos_adr] = np.clip(data.qpos[qpos_adr] + delta_q[idx], low, high)
        mujoco.mj_forward(model, data)
    return current_arm_qpos_dict()


def get_ee_pose():
    quat = np.empty(4, dtype=np.float64)
    mujoco.mju_mat2Quat(quat, data.xmat[HAND_BODY_ID].copy())
    return {
        "position": data.xpos[HAND_BODY_ID].copy(),
        "quat_wxyz": quat.copy(),
    }


def get_cube_pose():
    quat = np.empty(4, dtype=np.float64)
    mujoco.mju_mat2Quat(quat, data.xmat[BLOCK_BODY_ID].copy())
    return {
        "position": data.xpos[BLOCK_BODY_ID].copy(),
        "quat_wxyz": quat.copy(),
    }


def make_third_person_camera(lookat=None):
    cam = mujoco.MjvCamera()
    cam.distance = 2.2
    cam.azimuth = 135
    cam.elevation = -25
    cam.lookat[:] = np.array([0.55, 0.00, 0.35], dtype=np.float64) if lookat is None else lookat
    return cam


reset_task_state_randomized()
print("Refreshed IDs for LoRA data scene.")
print("Body IDs:", body_ids)
print("Camera ID:", camera_id)
print("Initial hand:", data.xpos[HAND_BODY_ID])
print("Initial cube:", data.xpos[BLOCK_BODY_ID])


In [ ]:
# Cell C: LoRA raw-data collection knobs.
# For smoke testing, keep this at 2 demos with videos ON. For the real 100-demo collection,
# set COLLECT_SMOKE_TEST=False and usually SAVE_EPISODE_VIDEOS=False to save disk.
from datetime import datetime

DATA_ROOT = Path(r"/home/fariborz/projects/force-vla-colab/loradata")
RUN_NAME = "panda_pickplace_commanded_gripper_contact"
COLLECT_SMOKE_TEST = False
SMOKE_TEST_DEMOS = 2
NUM_DEMONSTRATIONS = 300
TRAIN_FRACTION = 0.90

POLICY_HZ = 10.0
IMAGE_FORMAT = "jpg"
JPEG_QUALITY = 95

AGENTIC_IMAGE_SIZE = (256, 256)       # wrist/agentic image for OpenVLA-style observations
THIRD_PERSON_IMAGE_SIZE = (640, 480)  # useful for debugging and RLDS side observations

SAVE_EPISODE_VIDEOS = True            # requested ON for the first smoke tests
VIDEO_FPS = 10
AGENTIC_VIDEO_SIZE = (1080, 1080)
THIRD_PERSON_VIDEO_SIZE = (1920, 1080)

# Conservative randomization first. Widen only after smoke demos look physically correct.
CUBE_X_RANGE = (0.492, 0.508)
CUBE_Y_RANGE = (-0.010, 0.010)
TARGET_X_RANGE = (0.720, 0.750)
TARGET_Y_RANGE = (-0.012, 0.012)
ROBOT_JOINT_NOISE_RAD = 0.010

SUCCESS_X_MIN_MARGIN_M = -0.030
SUCCESS_Y_TOLERANCE_M = 0.090
SUCCESS_Z_RANGE_M = (0.070, 0.125)
REQUIRE_SUCCESSFUL_DEMOS = True
MAX_ATTEMPTS_MULTIPLIER = 4

GRIPPER_ACTION_CONVENTION = "commanded_open_fraction: 1.0=open, 0.0=closed; measured gripper_qpos is saved separately for debugging"
ROTATION_ACTION_CONVENTION = "droll/dpitch/dyaw are saved as 0.0 because this expert keeps wrist orientation implicit through IK seeds."

RUN_DIR = DATA_ROOT / f"{RUN_NAME}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
RUN_DIR.mkdir(parents=True, exist_ok=True)

print("Data run directory:", RUN_DIR)
print("Demos to collect now:", SMOKE_TEST_DEMOS if COLLECT_SMOKE_TEST else NUM_DEMONSTRATIONS)
print("Policy/data sample rate Hz:", POLICY_HZ)
print("Save full-HD videos:", SAVE_EPISODE_VIDEOS)
print("Still image sizes:", {"agentic": AGENTIC_IMAGE_SIZE, "third_person": THIRD_PERSON_IMAGE_SIZE})


In [ ]:
# Cell D: Contact-faithful scripted expert recorder.
# This is for collecting imitation/fine-tuning demonstrations only. OpenVLA is not used here.


def sample_episode_randomization(rng):
    cube_pos = np.array([
        rng.uniform(*CUBE_X_RANGE),
        rng.uniform(*CUBE_Y_RANGE),
        0.085,
    ], dtype=np.float64)
    target_pos = np.array([
        rng.uniform(*TARGET_X_RANGE),
        rng.uniform(*TARGET_Y_RANGE),
        0.085,
    ], dtype=np.float64)
    robot_noise = {
        name: rng.uniform(-ROBOT_JOINT_NOISE_RAD, ROBOT_JOINT_NOISE_RAD)
        for name in ARM_JOINT_NAMES
    }
    return cube_pos, target_pos, robot_noise


def build_pickplace_waypoint_specs(cube_pos, target_pos):
    # The high-level expert is intentionally the original pick-place path, parameterized only
    # by randomized cube and target positions for data collection.
    cube_xy = np.asarray(cube_pos[:2], dtype=np.float64)
    target_xy = np.asarray(target_pos[:2], dtype=np.float64)
    return [
        ("home", np.array([0.307, 0.00, 0.590], dtype=np.float64), 0.04, 0.4),
        ("pregrasp", np.array([cube_xy[0], cube_xy[1], 0.280], dtype=np.float64), 0.04, 1.5),
        ("grasp", np.array([cube_xy[0], cube_xy[1], 0.180], dtype=np.float64), 0.04, 1.0),
        ("close_gripper", np.array([cube_xy[0], cube_xy[1], 0.180], dtype=np.float64), 0.00, 0.8),
        ("lift", np.array([cube_xy[0], cube_xy[1], 0.360], dtype=np.float64), 0.00, 1.0),
        ("transfer_far_side", np.array([target_xy[0], target_xy[1], 0.360], dtype=np.float64), 0.00, 1.5),
        ("place", np.array([target_xy[0], target_xy[1], 0.180], dtype=np.float64), 0.00, 1.0),
        ("open_gripper", np.array([target_xy[0], target_xy[1], 0.180], dtype=np.float64), 0.04, 0.7),
        ("retreat", np.array([target_xy[0], target_xy[1], 0.320], dtype=np.float64), 0.04, 1.0),
    ]


def plan_expert_waypoints(cube_pos, target_pos, robot_noise=None):
    reset_task_state_randomized(cube_pos=cube_pos, robot_joint_noise=robot_noise)
    seed = current_arm_qpos_dict()
    waypoints = []
    for name, target_pos_m, gripper_opening, duration_s in build_pickplace_waypoint_specs(cube_pos, target_pos):
        qpos_target = solve_hand_position_ik(target_pos_m, seed_qpos=seed)
        ik_error_m = float(np.linalg.norm(target_pos_m - data.xpos[HAND_BODY_ID]))
        waypoints.append({
            "name": name,
            "target_pos": target_pos_m.copy(),
            "qpos": qpos_target.copy(),
            "gripper_opening": float(gripper_opening),
            "duration_s": float(duration_s),
            "ik_error_m": ik_error_m,
        })
        seed = qpos_target
    return waypoints


def episode_success(final_cube_pos, target_pos):
    final_cube_pos = np.asarray(final_cube_pos, dtype=np.float64)
    target_pos = np.asarray(target_pos, dtype=np.float64)
    far_enough = final_cube_pos[0] >= target_pos[0] + SUCCESS_X_MIN_MARGIN_M
    y_ok = abs(final_cube_pos[1] - target_pos[1]) <= SUCCESS_Y_TOLERANCE_M
    z_ok = SUCCESS_Z_RANGE_M[0] <= final_cube_pos[2] <= SUCCESS_Z_RANGE_M[1]
    return bool(far_enough and y_ok and z_ok)


def save_jpg(rgb, path, quality=JPEG_QUALITY):
    Image.fromarray(np.asarray(rgb, dtype=np.uint8)).save(path, quality=quality, optimize=True)


def relative_to_episode(path, episode_dir):
    return str(Path(path).resolve().relative_to(Path(episode_dir).resolve())).replace("\\", "/")


def open_video_writer(path, fps):
    path.parent.mkdir(parents=True, exist_ok=True)
    return imageio.get_writer(
        path,
        fps=fps,
        codec="libx264",
        quality=8,
        macro_block_size=1,
    )


def render_with_renderer(renderer, camera):
    renderer.update_scene(data, camera=camera)
    return renderer.render()


def collect_one_expert_episode(episode_id, split, seed, run_dir=RUN_DIR):
    rng = np.random.default_rng(seed)
    cube_pos, target_pos, robot_noise = sample_episode_randomization(rng)
    waypoints = plan_expert_waypoints(cube_pos, target_pos, robot_noise=robot_noise)

    episode_dir = run_dir / split / f"episode_{episode_id:06d}"
    agentic_dir = episode_dir / "images" / "agentic"
    third_dir = episode_dir / "images" / "third_person"
    agentic_dir.mkdir(parents=True, exist_ok=True)
    third_dir.mkdir(parents=True, exist_ok=True)

    agentic_renderer = mujoco.Renderer(model, height=AGENTIC_IMAGE_SIZE[1], width=AGENTIC_IMAGE_SIZE[0])
    third_renderer = mujoco.Renderer(model, height=THIRD_PERSON_IMAGE_SIZE[1], width=THIRD_PERSON_IMAGE_SIZE[0])
    third_camera = make_third_person_camera()

    agentic_video_renderer = None
    third_video_renderer = None
    agentic_video_writer = None
    third_video_writer = None
    video_paths = {}
    if SAVE_EPISODE_VIDEOS:
        video_dir = episode_dir / "videos"
        agentic_video_path = video_dir / "agentic_full_hd.mp4"
        third_video_path = video_dir / "third_person_full_hd.mp4"
        agentic_video_renderer = mujoco.Renderer(model, height=AGENTIC_VIDEO_SIZE[1], width=AGENTIC_VIDEO_SIZE[0])
        third_video_renderer = mujoco.Renderer(model, height=THIRD_PERSON_VIDEO_SIZE[1], width=THIRD_PERSON_VIDEO_SIZE[0])
        agentic_video_writer = open_video_writer(agentic_video_path, VIDEO_FPS)
        third_video_writer = open_video_writer(third_video_path, VIDEO_FPS)
        video_paths = {
            "agentic_full_hd": relative_to_episode(agentic_video_path, episode_dir),
            "third_person_full_hd": relative_to_episode(third_video_path, episode_dir),
        }

    sample_dt = 1.0 / float(POLICY_HZ)
    snapshots = []
    qpos_samples = []
    step_id = 0
    next_sample_time = 0.0
    elapsed_s = 0.0

    def capture_snapshot(phase_name, commanded_gripper_opening):
        nonlocal step_id
        commanded_gripper_open_fraction = float(np.clip(commanded_gripper_opening / 0.04, 0.0, 1.0))
        measured_gripper_open_fraction = float(np.clip(current_finger_opening_m() / 0.04, 0.0, 1.0))
        agentic_path = agentic_dir / f"step_{step_id:06d}.{IMAGE_FORMAT}"
        third_path = third_dir / f"step_{step_id:06d}.{IMAGE_FORMAT}"

        agentic_rgb = render_with_renderer(agentic_renderer, AGENT_CAMERA_NAME)
        third_rgb = render_with_renderer(third_renderer, third_camera)
        save_jpg(agentic_rgb, agentic_path)
        save_jpg(third_rgb, third_path)

        if SAVE_EPISODE_VIDEOS:
            agentic_video_writer.append_data(render_with_renderer(agentic_video_renderer, AGENT_CAMERA_NAME))
            third_video_writer.append_data(render_with_renderer(third_video_renderer, third_camera))

        ee_pose = get_ee_pose()
        cube_pose = get_cube_pose()
        snapshot = {
            "episode_id": int(episode_id),
            "step_id": int(step_id),
            "time_s": float(elapsed_s),
            "phase": phase_name,
            "image_before_action": {
                "agentic": relative_to_episode(agentic_path, episode_dir),
                "third_person": relative_to_episode(third_path, episode_dir),
            },
            "language_instruction": TASK_INSTRUCTION,
            "end_effector_pose": {
                "position": ee_pose["position"].astype(float).tolist(),
                "quat_wxyz": ee_pose["quat_wxyz"].astype(float).tolist(),
            },
            "joint_qpos": {name: float(data.qpos[qpos_adr]) for name, qpos_adr in zip(ARM_JOINT_NAMES, ARM_QPOS_ADR)},
            "gripper_qpos": {name: float(data.qpos[qpos_adr]) for name, qpos_adr in zip(FINGER_JOINT_NAMES, FINGER_QPOS_ADR)},
            "cube_pose": {
                "position": cube_pose["position"].astype(float).tolist(),
                "quat_wxyz": cube_pose["quat_wxyz"].astype(float).tolist(),
            },
            "target_corner_pose": {
                "position": target_pos.astype(float).tolist(),
                "quat_wxyz": [1.0, 0.0, 0.0, 0.0],
            },
            "camera_name": {
                "agentic": AGENT_CAMERA_NAME,
                "third_person": "free_camera_az135_el-25",
            },
            "camera_parameters": {
                "agentic": {
                    "width": AGENTIC_IMAGE_SIZE[0],
                    "height": AGENTIC_IMAGE_SIZE[1],
                    "fovy": float(model.cam_fovy[camera_id]),
                },
                "third_person": {
                    "width": THIRD_PERSON_IMAGE_SIZE[0],
                    "height": THIRD_PERSON_IMAGE_SIZE[1],
                    "distance": float(third_camera.distance),
                    "azimuth": float(third_camera.azimuth),
                    "elevation": float(third_camera.elevation),
                    "lookat": third_camera.lookat.astype(float).tolist(),
                },
            },
            "random_seed": int(seed),
            "randomization": {
                "cube_start_position": cube_pos.astype(float).tolist(),
                "target_corner_position": target_pos.astype(float).tolist(),
                "robot_joint_noise_rad": {k: float(v) for k, v in robot_noise.items()},
            },
            "commanded_gripper_open_fraction": commanded_gripper_open_fraction,
            "measured_gripper_open_fraction": measured_gripper_open_fraction,
            "qpos": data.qpos.copy(),
        }
        snapshots.append(snapshot)
        qpos_samples.append(data.qpos.copy())
        step_id += 1

    try:
        reset_task_state_randomized(cube_pos=cube_pos, robot_joint_noise=robot_noise)
        set_arm_position_targets(waypoints[0]["qpos"])
        set_gripper_opening(waypoints[0]["gripper_opening"])
        mujoco.mj_forward(model, data)
        capture_snapshot("start", waypoints[0]["gripper_opening"])
        next_sample_time = sample_dt

        previous_qpos = waypoints[0]["qpos"].copy()
        previous_gripper = waypoints[0]["gripper_opening"]
        for waypoint in waypoints[1:]:
            steps = max(1, int(round(waypoint["duration_s"] / model.opt.timestep)))
            for low_step in range(steps):
                alpha = (low_step + 1) / steps
                joint_targets = {
                    name: (1.0 - alpha) * previous_qpos[name] + alpha * waypoint["qpos"][name]
                    for name in ARM_JOINT_NAMES
                }
                gripper_opening = (1.0 - alpha) * previous_gripper + alpha * waypoint["gripper_opening"]
                set_arm_position_targets(joint_targets)
                set_gripper_opening(gripper_opening)
                mujoco.mj_step(model, data)
                elapsed_s += float(model.opt.timestep)
                if elapsed_s + 1e-9 >= next_sample_time:
                    capture_snapshot(waypoint["name"], gripper_opening)
                    next_sample_time += sample_dt
            previous_qpos = waypoint["qpos"].copy()
            previous_gripper = waypoint["gripper_opening"]
            print(
                f"[episode {episode_id:06d}] after {waypoint['name']:18s} "
                f"hand={data.xpos[HAND_BODY_ID]} cube={data.xpos[BLOCK_BODY_ID]}"
            )

        final_cube_pos = data.xpos[BLOCK_BODY_ID].copy()
        success = episode_success(final_cube_pos, target_pos)

        # Convert snapshots to DROID/RLDS-builder-friendly JSONL records.
        steps_jsonl = episode_dir / "steps.jsonl"
        with steps_jsonl.open("w", encoding="utf-8") as f:
            for idx, snapshot in enumerate(snapshots):
                if idx < len(snapshots) - 1:
                    next_snapshot = snapshots[idx + 1]
                    cur_pos = np.asarray(snapshot["end_effector_pose"]["position"], dtype=np.float64)
                    next_pos = np.asarray(next_snapshot["end_effector_pose"]["position"], dtype=np.float64)
                    dpos = next_pos - cur_pos
                    gripper_action = float(next_snapshot["commanded_gripper_open_fraction"])
                else:
                    dpos = np.zeros(3, dtype=np.float64)
                    gripper_action = float(snapshot["commanded_gripper_open_fraction"])
                action_7d = [
                    float(dpos[0]),
                    float(dpos[1]),
                    float(dpos[2]),
                    0.0,
                    0.0,
                    0.0,
                    gripper_action,
                ]
                row = {k: v for k, v in snapshot.items() if k != "qpos"}
                row["action"] = action_7d
                row["done"] = bool(idx == len(snapshots) - 1)
                row["success"] = bool(success) if row["done"] else None
                row["control_source"] = "scripted_expert_for_lora_data"
                f.write(json.dumps(row) + "\n")

        np.savez_compressed(
            episode_dir / "trajectory_arrays.npz",
            qpos=np.asarray(qpos_samples, dtype=np.float64),
            final_cube_pos=final_cube_pos.astype(np.float64),
            target_pos=target_pos.astype(np.float64),
            cube_start_pos=cube_pos.astype(np.float64),
        )

        metadata = {
            "episode_id": int(episode_id),
            "split": split,
            "seed": int(seed),
            "success": bool(success),
            "num_steps": int(len(snapshots)),
            "policy_hz": float(POLICY_HZ),
            "language_instruction": TASK_INSTRUCTION,
            "action_format": "[dx, dy, dz, droll, dpitch, dyaw, gripper]",
            "gripper_action_convention": GRIPPER_ACTION_CONVENTION,
            "gripper_action_source": "expert commanded gripper opening, not measured gripper qpos",
            "rotation_action_convention": ROTATION_ACTION_CONVENTION,
            "image_modalities": ["agentic", "third_person"],
            "video_paths": video_paths,
            "scene_xml": str(PANDA_XML),
            "contact_note": "Right finger uses OpenVLA05 GelSight-shaped contact pad; tactile rendering is not saved.",
            "cube_start_position": cube_pos.astype(float).tolist(),
            "target_corner_position": target_pos.astype(float).tolist(),
            "final_cube_position": final_cube_pos.astype(float).tolist(),
            "waypoints": [
                {
                    "name": w["name"],
                    "target_pos": w["target_pos"].astype(float).tolist(),
                    "gripper_opening": float(w["gripper_opening"]),
                    "duration_s": float(w["duration_s"]),
                    "ik_error_m": float(w["ik_error_m"]),
                }
                for w in waypoints
            ],
        }
        (episode_dir / "episode_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
        return metadata

    finally:
        for writer in [agentic_video_writer, third_video_writer]:
            if writer is not None:
                writer.close()
        for renderer in [agentic_renderer, third_renderer, agentic_video_renderer, third_video_renderer]:
            if renderer is not None:
                renderer.close()


def collect_lora_dataset():
    total = SMOKE_TEST_DEMOS if COLLECT_SMOKE_TEST else NUM_DEMONSTRATIONS
    train_count = int(round(total * TRAIN_FRACTION))
    if total >= 2:
        train_count = min(max(1, train_count), total - 1)
    max_attempts = max(total, total * MAX_ATTEMPTS_MULTIPLIER)
    successful_metadata = []
    manifest_rows = []
    base_seed = 7707
    attempt = 0
    while len(successful_metadata) < total and attempt < max_attempts:
        episode_id = len(successful_metadata)
        split = "train" if episode_id < train_count else "test"
        seed = base_seed + attempt
        print(f"\nCollecting attempt {attempt + 1}/{max_attempts}: episode={episode_id:06d}, split={split}, seed={seed}")
        try:
            metadata = collect_one_expert_episode(episode_id=episode_id, split=split, seed=seed)
        except Exception as exc:
            print("Episode failed with exception:", repr(exc))
            metadata = None
        attempt += 1
        if metadata is None:
            continue
        print("Episode success:", metadata["success"], "final cube:", metadata["final_cube_position"])
        if REQUIRE_SUCCESSFUL_DEMOS and not metadata["success"]:
            print("Discarding failed demo because REQUIRE_SUCCESSFUL_DEMOS=True")
            continue
        successful_metadata.append(metadata)
        manifest_rows.append({
            "episode_id": metadata["episode_id"],
            "split": metadata["split"],
            "episode_dir": f"{metadata['split']}/episode_{metadata['episode_id']:06d}",
            "success": metadata["success"],
            "num_steps": metadata["num_steps"],
            "seed": metadata["seed"],
        })

    if len(successful_metadata) < total:
        raise RuntimeError(f"Collected {len(successful_metadata)} demos, but requested {total}. Check grasp quality and randomization ranges.")

    with (RUN_DIR / "manifest.jsonl").open("w", encoding="utf-8") as f:
        for row in manifest_rows:
            f.write(json.dumps(row) + "\n")
    dataset_metadata = {
        "run_dir": str(RUN_DIR),
        "num_episodes": len(successful_metadata),
        "policy_hz": float(POLICY_HZ),
        "task_instruction": TASK_INSTRUCTION,
        "action_format": "[dx, dy, dz, droll, dpitch, dyaw, gripper]",
        "gripper_action_convention": GRIPPER_ACTION_CONVENTION,
        "gripper_action_source": "expert commanded gripper opening, not measured gripper qpos",
        "rotation_action_convention": ROTATION_ACTION_CONVENTION,
        "image_before_action_modalities": ["agentic", "third_person"],
        "saved_video": bool(SAVE_EPISODE_VIDEOS),
        "ready_for_rlds_builder": True,
        "note": "Raw JSONL/JPG structure. Convert to RLDS with a dataset builder before OpenVLA LoRA fine-tuning.",
    }
    (RUN_DIR / "dataset_metadata.json").write_text(json.dumps(dataset_metadata, indent=2), encoding="utf-8")
    print("\nCollection complete:", RUN_DIR)
    print("Episodes:", len(successful_metadata))
    return successful_metadata


In [ ]:
# Cell E: Run the smoke collection now.
# Expected for default settings: 2 successful demos, 10 Hz image/action samples, and full-HD videos.
collected_episode_metadata = collect_lora_dataset()

print("\nSmoke run saved at:", RUN_DIR)
for metadata in collected_episode_metadata:
    print(
        f"episode={metadata['episode_id']:06d} split={metadata['split']} "
        f"success={metadata['success']} steps={metadata['num_steps']} "
        f"final_cube={np.round(metadata['final_cube_position'], 4).tolist()} "
        f"target={np.round(metadata['target_corner_position'], 4).tolist()}"
    )

first_ep = collected_episode_metadata[0]
first_episode_dir = RUN_DIR / first_ep["split"] / f"episode_{first_ep['episode_id']:06d}"
print("\nFirst episode files:")
print("  steps:", first_episode_dir / "steps.jsonl")
print("  metadata:", first_episode_dir / "episode_metadata.json")
print("  agentic images:", first_episode_dir / "images" / "agentic")
print("  third-person images:", first_episode_dir / "images" / "third_person")
if SAVE_EPISODE_VIDEOS:
    print("  videos:", first_episode_dir / "videos")

sample_third = sorted((first_episode_dir / "images" / "third_person").glob("*.jpg"))
sample_agentic = sorted((first_episode_dir / "images" / "agentic").glob("*.jpg"))
if sample_third:
    print("Preview third-person:", sample_third[min(10, len(sample_third) - 1)])
if sample_agentic:
    print("Preview agentic:", sample_agentic[min(10, len(sample_agentic) - 1)])


## Server: D3 OpenVLA-OFT fine-tuning

Run Cells F through K in order. Cell K defaults to three optimizer steps and does not save a checkpoint. After that succeeds, set `TRAINING_MODE = "full"` in Cell K.

In [ ]:
# Cell F: Paths and subprocess helpers for the dedicated OpenVLA-OFT workflow.
from pathlib import Path
import importlib.metadata as importlib_metadata
import json
import os
import shlex
import shutil
import subprocess
import sys
import time

PROJECT_DIR = Path("/home/fariborz/projects/force-vla-colab")  # Adjust if this project moves.
OFT_REPO = Path("/home/fariborz/projects/openvla-oft")  # Official moojink/openvla-oft checkout.
OFT_PYTHON = Path("/home/fariborz/miniforge3/envs/openvla/bin/python")  # Python with RTX 5090 PyTorch.
if not OFT_PYTHON.exists():
    OFT_PYTHON = Path(sys.executable)
OFT_BIN_DIR = OFT_PYTHON.parent

D2_RAW_DIR = PROJECT_DIR / "loradata/panda_pickplace_rotation_recovery_v1_20260803_235336"
D3_RAW_DIR = PROJECT_DIR / "loradata/panda_pickplace_d3_nominal_boundary_v1"
D3_BUILDER_DIR = PROJECT_DIR / "rlds_dataset_builder/panda_pickplace_d3"
DATA_ROOT_DIR = PROJECT_DIR / "RLDS_TFDS"
DATASET_NAME = "panda_pickplace_d3"

PREPARE_D3_SCRIPT = PROJECT_DIR / "prepare_panda_d3.py"
OFT_SETUP_SCRIPT = PROJECT_DIR / "openvla_oft_d3_setup.py"
OFT_VERIFY_SCRIPT = PROJECT_DIR / "openvla_oft_d3_verify.py"
RUN_ROOT_DIR = PROJECT_DIR / "openvla_oft_d3_runs"
HF_CACHE_DIR = PROJECT_DIR / "hf_cache"

HF_TOKEN_IN_NOTEBOOK = ""  # Optional: use an environment variable instead when possible.
USE_WANDB_OFFLINE = False  # Set True to log locally without contacting wandb.ai.
RUN_OFT_TRAINING = True  # Set False to print/check configuration without launching training.

for path in (RUN_ROOT_DIR, HF_CACHE_DIR):
    path.mkdir(parents=True, exist_ok=True)

def oft_env(extra=None):
    env = os.environ.copy()
    env["HF_HOME"] = str(HF_CACHE_DIR)
    env["HF_HUB_CACHE"] = str(HF_CACHE_DIR / "hub")
    env.pop("TRANSFORMERS_CACHE", None)
    env["TOKENIZERS_PARALLELISM"] = "false"
    env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    env["TF_CPP_MIN_LOG_LEVEL"] = "3"
    env["WANDB_MODE"] = "offline" if USE_WANDB_OFFLINE else "online"
    if HF_TOKEN_IN_NOTEBOOK.strip():
        env["HF_TOKEN"] = HF_TOKEN_IN_NOTEBOOK.strip()
        env["HUGGINGFACE_HUB_TOKEN"] = HF_TOKEN_IN_NOTEBOOK.strip()
    repo_path = str(OFT_REPO)
    old_pythonpath = env.get("PYTHONPATH", "")
    env["PYTHONPATH"] = repo_path + (os.pathsep + old_pythonpath if old_pythonpath else "")
    cuda_libs = sorted((OFT_PYTHON.parent.parent / "lib").glob("python*/site-packages/nvidia/cu13/lib"))
    if cuda_libs:
        old_ld = env.get("LD_LIBRARY_PATH", "")
        env["LD_LIBRARY_PATH"] = str(cuda_libs[0]) + (os.pathsep + old_ld if old_ld else "")
    if extra:
        env.update({str(key): str(value) for key, value in extra.items()})
    return env

def run(cmd, cwd=None, check=True, env=None):
    pretty = " ".join(shlex.quote(str(part)) for part in cmd)
    print("$", pretty)
    result = subprocess.run(
        [str(part) for part in cmd],
        cwd=str(cwd) if cwd else None,
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(result.stdout)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed ({result.returncode}): {pretty}")
    return result

print("Notebook Python:", sys.executable)
print("OFT Python:", OFT_PYTHON)
print("OFT repository:", OFT_REPO)
print("D3 raw directory:", D3_RAW_DIR)
print("D3 RLDS directory:", DATA_ROOT_DIR / DATASET_NAME / "1.0.0")
print("Run directory:", RUN_ROOT_DIR)

In [ ]:
# Cell G: Create D3 as an exact filtered view of D2 and audit its mode counts.
# Adjustable only if the source/output dataset locations change.
for required in (PREPARE_D3_SCRIPT, D2_RAW_DIR / "manifest.jsonl"):
    assert required.exists(), f"Missing: {required}"

prepared = run(
    [
        OFT_PYTHON,
        PREPARE_D3_SCRIPT,
        "--source", D2_RAW_DIR,
        "--output", D3_RAW_DIR,
    ],
    env=oft_env({"CUDA_VISIBLE_DEVICES": ""}),
)
assert "PANDA_D3_RAW_READY" in prepared.stdout

rows = [json.loads(line) for line in (D3_RAW_DIR / "manifest.jsonl").read_text().splitlines()]
assert len(rows) == 800
assert sum(row["trajectory_mode"] == "nominal" for row in rows) == 700
assert sum(row["trajectory_mode"] == "boundary" for row in rows) == 100
assert not any("recovery" in row["trajectory_mode"] for row in rows)
D3_RAW_VERIFIED = True
print("D3 confirmed: 700 nominal + 100 boundary + 0 recovery episodes.")

In [ ]:
# Cell H: Build the independent D3 RLDS dataset, or reuse its verified build.
REBUILD_D3_RLDS = False  # Set True only after intentionally changing the D3 builder or source selection.
D3_VERSION_DIR = DATA_ROOT_DIR / DATASET_NAME / "1.0.0"
builder_file = D3_BUILDER_DIR / "panda_pickplace_d3_dataset_builder.py"
assert globals().get("D3_RAW_VERIFIED", False), "Run Cell G first."
assert builder_file.is_file(), f"Missing builder: {builder_file}"

if REBUILD_D3_RLDS or not (D3_VERSION_DIR / "dataset_info.json").is_file():
    build_cmd = [
        OFT_PYTHON,
        "-m", "tensorflow_datasets.scripts.cli.main",
        "build",
        "--data_dir", DATA_ROOT_DIR,
    ]
    if REBUILD_D3_RLDS:
        build_cmd.append("--overwrite")
    run(
        build_cmd,
        cwd=D3_BUILDER_DIR,
        env=oft_env({"PANDA_D3_RAW_DIR": D3_RAW_DIR, "CUDA_VISIBLE_DEVICES": ""}),
    )
else:
    print("Reusing:", D3_VERSION_DIR)

split_probe = run(
    [
        OFT_PYTHON,
        "-c",
        (
            "import tensorflow_datasets as tfds; "
            f"b=tfds.builder('{DATASET_NAME}', data_dir=r'{DATA_ROOT_DIR}'); "
            "print(b.info.splits['train'].num_examples, b.info.splits['test'].num_examples)"
        ),
    ],
    env=oft_env({"CUDA_VISIBLE_DEVICES": ""}),
)
assert split_probe.stdout.strip().endswith("720 80")
D3_RLDS_VERIFIED = True
print("D3 RLDS confirmed: train=720, test=80.")

In [ ]:
# Cell I: Clone and install the official OpenVLA-OFT code without replacing RTX 5090 PyTorch.
if not OFT_REPO.exists():
    run(["git", "clone", "https://github.com/moojink/openvla-oft.git", OFT_REPO])
else:
    print("Reusing existing OFT checkout; no reset/pull is performed:", OFT_REPO)

dependency_specs = [
    "accelerate>=0.25.0", "draccus==0.8.0", "einops", "huggingface_hub",
    "json-numpy", "jsonlines", "matplotlib", "peft==0.11.1", "protobuf<4",
    "rich", "sentencepiece==0.1.99", "timm==0.9.10", "tokenizers==0.19.1",
    "wandb", "tensorflow==2.15.0", "tensorflow_datasets==4.9.3",
    "tensorflow_graphics==2021.12.3", "diffusers==0.30.3", "imageio",
]
run([OFT_PYTHON, "-m", "pip", "install", "--upgrade-strategy", "only-if-needed", *dependency_specs])

# OFT requires this Transformers fork for bidirectional attention and parallel decoding.
transformers_probe = run(
    [
        OFT_PYTHON,
        "-c",
        (
            "from importlib.metadata import distribution; "
            "u=distribution('transformers').read_text('direct_url.json') or ''; "
            "print(u); raise SystemExit(0 if 'transformers-openvla-oft' in u else 2)"
        ),
    ],
    check=False,
    env=oft_env({"CUDA_VISIBLE_DEVICES": ""}),
)
if transformers_probe.returncode != 0:
    run([
        OFT_PYTHON, "-m", "pip", "install", "--force-reinstall", "--no-deps",
        "transformers @ git+https://github.com/moojink/transformers-openvla-oft.git",
    ])

run([OFT_PYTHON, "-m", "pip", "install", "--no-deps", "-e", OFT_REPO])
environment_probe = run(
    [
        OFT_PYTHON,
        "-c",
        (
            "import torch, transformers; "
            "from importlib.metadata import distribution; "
            "print('torch', torch.__version__, 'cuda', torch.cuda.is_available()); "
            "print('gpu', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'); "
            "print('transformers', distribution('transformers').read_text('direct_url.json'))"
        ),
    ],
    env=oft_env(),
)
assert "transformers-openvla-oft" in environment_probe.stdout
assert "cuda True" in environment_probe.stdout
OFT_ENV_VERIFIED = True

In [ ]:
# Cell J: Register D3, 8D joint proprioception, and Panda OFT constants.
assert globals().get("D3_RLDS_VERIFIED", False), "Run Cell H first."
assert globals().get("OFT_ENV_VERIFIED", False), "Run Cell I first."

registration = run(
    [OFT_PYTHON, OFT_SETUP_SCRIPT, "--repo", OFT_REPO],
    env=oft_env({"CUDA_VISIBLE_DEVICES": ""}),
)
assert "OPENVLA_OFT_D3_REGISTRATION_INSTALLED" in registration.stdout

constant_probe = run(
    [
        OFT_PYTHON,
        "-c",
        (
            "import sys; sys.argv.append('panda_d3'); "
            "from prismatic.vla.constants import ACTION_DIM, PROPRIO_DIM, NUM_ACTIONS_CHUNK; "
            "print(ACTION_DIM, PROPRIO_DIM, NUM_ACTIONS_CHUNK)"
        ),
    ],
    cwd=OFT_REPO,
    env=oft_env({"CUDA_VISIBLE_DEVICES": ""}),
)
assert constant_probe.stdout.strip().endswith("7 8 8")
OFT_D3_REGISTERED = True

In [ ]:
# Cell K: Exercise real D3 statistics, proprio normalization, and OFT chunking before model loading.
assert globals().get("OFT_D3_REGISTERED", False), "Run Cell J first."
verification = run(
    [
        OFT_PYTHON,
        OFT_VERIFY_SCRIPT,
        "--repo", OFT_REPO,
        "--data-root", DATA_ROOT_DIR,
        "--dataset-name", DATASET_NAME,
    ],
    env=oft_env({"CUDA_VISIBLE_DEVICES": ""}),
)
assert "OPENVLA_OFT_D3_VERIFIED" in verification.stdout
OFT_D3_VERIFIED = True
print("Cell K passed: nonzero 8D proprio -> one proprio token; action chunks are 8 x 7.")

## OFT training smoke test and full run

The default below runs three optimizer steps with no checkpoint save or merge. A full run uses the same command and changes only the adjustable training values.

In [ ]:
# Cell L: Launch OpenVLA-OFT with continuous L1 chunks and joint proprioception.
TRAINING_MODE = "full"  # Change to "full" only after the three-step smoke test passes.
SMOKE_MAX_STEPS = 3  # Adjustable smoke-test optimizer steps.
FULL_MAX_STEPS = 50_000  # Adjustable full-run optimizer steps; evaluate saved checkpoints.

BATCH_SIZE = 1  # Keep at 1 for a 32 GB RTX 5090; raise only after measuring VRAM.
GRAD_ACCUMULATION_STEPS = 8  # Adjustable effective batch multiplier.
LORA_RANK = 32  # Official OFT LoRA rank.
LEARNING_RATE = 5e-4  # Official OFT starting learning rate.
IMAGE_AUG = True  # Recommended; inference must use the matching center crop.
SHUFFLE_BUFFER_SIZE = 1000  # Raise if host RAM permits.
FULL_SAVE_FREQ = 10_000  # Adjustable full-run checkpoint interval.
FULL_VAL_FREQ = 10_000  # Adjustable held-out test-split evaluation interval.
MERGE_LORA_DURING_FULL_TRAINING = False # Requires roughly 30+ GB free disk for a merged 7B checkpoint.
MIN_FREE_DISK_GB_FOR_MERGE = 30  # Adjustable safety threshold.

assert globals().get("OFT_D3_VERIFIED", False), "Run Cells F-K before training."
assert TRAINING_MODE in {"smoke", "full"}

is_smoke = TRAINING_MODE == "smoke"
max_steps = SMOKE_MAX_STEPS if is_smoke else FULL_MAX_STEPS
save_freq = max_steps + 1000 if is_smoke else FULL_SAVE_FREQ
use_val_set = not is_smoke
merge_lora = False if is_smoke else MERGE_LORA_DURING_FULL_TRAINING
run_note = f"panda-d3-joint-proprio-oft-{'smoke' if is_smoke else 'full'}"

if merge_lora:
    free_gb = shutil.disk_usage(RUN_ROOT_DIR).free / 1024**3
    print("Free disk space:", round(free_gb, 1), "GB")
    assert free_gb >= MIN_FREE_DISK_GB_FOR_MERGE, (
        f"Free at least {MIN_FREE_DISK_GB_FOR_MERGE} GB before merged checkpoint saves; "
        "or set MERGE_LORA_DURING_FULL_TRAINING=False and merge later."
    )

torchrun = OFT_BIN_DIR / "torchrun"
torchrun = torchrun if torchrun.exists() else Path("torchrun")
cmd = [
    torchrun,
    "--standalone", "--nnodes", "1", "--nproc-per-node", "1",
    OFT_REPO / "vla-scripts/finetune.py",
    "--vla_path", "openvla/openvla-7b",
    "--data_root_dir", DATA_ROOT_DIR,
    "--dataset_name", DATASET_NAME,
    "--run_root_dir", RUN_ROOT_DIR,
    "--use_l1_regression", "True",
    "--use_diffusion", "False",
    "--use_film", "False",
    "--num_images_in_input", "1",
    "--use_proprio", "True",
    "--batch_size", str(BATCH_SIZE),
    "--grad_accumulation_steps", str(GRAD_ACCUMULATION_STEPS),
    "--lora_rank", str(LORA_RANK),
    "--learning_rate", str(LEARNING_RATE),
    "--image_aug", str(IMAGE_AUG),
    "--shuffle_buffer_size", str(SHUFFLE_BUFFER_SIZE),
    "--max_steps", str(max_steps),
    "--save_freq", str(save_freq),
    "--use_val_set", str(use_val_set),
    "--val_freq", str(FULL_VAL_FREQ),
    "--merge_lora_during_training", str(merge_lora),
    "--wandb_project", "openvla_oft_panda_d3",
    "--wandb_entity", "family-of-note-politecnico-di-milano",
    "--run_id_note", run_note,
]

print("Training mode:", TRAINING_MODE)
print("OFT input: third-person RGB + language + 8D joint/gripper proprio")
print("OFT output: 8 future actions x 7 continuous dimensions")
print("Effective batch size:", BATCH_SIZE * GRAD_ACCUMULATION_STEPS)
if is_smoke:
    print("Smoke mode: no checkpoint save and no LoRA merge.")

if not RUN_OFT_TRAINING:
    print("RUN_OFT_TRAINING=False; command is configured but was not launched.")
else:
    train_env = oft_env({"WANDB_MODE": "disabled" if is_smoke else os.environ.get("WANDB_MODE", "online")})
    start = time.time()
    process = subprocess.Popen(
        [str(part) for part in cmd],
        cwd=str(OFT_REPO),
        env=train_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=0,
    )

    while True:
        chunk = os.read(process.stdout.fileno(), 4096)
        if not chunk:
            break
        print(chunk.decode("utf-8", errors="replace"), end="", flush=True)

    return_code = process.wait()
    assert return_code == 0
    print("Elapsed minutes:", round((time.time() - start) / 60, 2))
    OFT_TRAINING_FINISHED = True

In [ ]:
# Cell L2: Three training curves for only the run that produced the 50,000-step checkpoint.
from pathlib import Path
import json

import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import pandas as pd
from wandb.proto import wandb_internal_pb2
from wandb.sdk.internal import datastore

L3_RUN_ID = "thfwkiyw"  # The single full run that ended at checkpoint step 50,000.
L3_FINAL_STEP = 50_000
L3_SMOOTH_ROWS = 101  # Logs are every 10 steps: trailing mean over about 1,000 steps.

assert "OFT_REPO" in globals(), "Run Cell F first."
l3_run_matches = list((Path(OFT_REPO) / "wandb").glob(f"run-*-{L3_RUN_ID}"))
assert len(l3_run_matches) == 1, f"Could not uniquely locate local W&B run {L3_RUN_ID}."
l3_run_dir = l3_run_matches[0]
l3_wandb_files = list(l3_run_dir.glob(f"run-{L3_RUN_ID}.wandb"))
assert len(l3_wandb_files) == 1, f"Missing local history in {l3_run_dir}."

l3_reader = datastore.DataStore()
l3_reader.open_for_scan(str(l3_wandb_files[0]))
l3_rows = []
while True:
    l3_record_bytes = l3_reader.scan_data()
    if l3_record_bytes is None:
        break
    l3_record = wandb_internal_pb2.Record()
    l3_record.ParseFromString(l3_record_bytes)
    if l3_record.WhichOneof("record_type") != "history":
        continue
    l3_row = {}
    for item in l3_record.history.item:
        key = item.key or "/".join(item.nested_key)
        if key:
            l3_row[key] = json.loads(item.value_json)
    l3_rows.append(l3_row)

l3_metric_panels = [
    ("train_loss", "VLA Train/Loss", "overall 8 × 7 continuous-action L1"),
    ("current_action_l1_loss", "VLA Train/Curr Action L1 Loss", "current action: horizon 0"),
    ("next_actions_l1_loss", "VLA Train/Next Actions L1 Loss", "next actions: horizons 1–7"),
]
l3_columns = ["_step", *[metric for _, metric, _ in l3_metric_panels]]
l3_history = pd.DataFrame(l3_rows)
l3_missing = [column for column in l3_columns if column not in l3_history]
assert not l3_missing, "Missing saved metrics: " + ", ".join(l3_missing)
for column in l3_columns:
    l3_history[column] = pd.to_numeric(l3_history[column], errors="coerce")
l3_history = (
    l3_history.dropna(subset=l3_columns)[l3_columns]
    .sort_values("_step")
    .drop_duplicates("_step", keep="last")
)
assert int(l3_history["_step"].min()) == 0
assert int(l3_history["_step"].max()) == L3_FINAL_STEP, (
    f"Run {L3_RUN_ID} ends at {int(l3_history['_step'].max()):,}, not {L3_FINAL_STEP:,}."
)

raw_color = "#F06AD7"
smooth_color = "#C026B8"
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), constrained_layout=True)
for ax, (title, metric, description) in zip(axes, l3_metric_panels):
    raw_values = l3_history[metric]
    smooth_values = raw_values.rolling(L3_SMOOTH_ROWS, min_periods=1).mean()
    ax.plot(
        l3_history["_step"], raw_values,
        color=raw_color, alpha=0.28, linewidth=0.7, label="raw logged value",
    )
    ax.plot(
        l3_history["_step"], smooth_values,
        color=smooth_color, linewidth=2.0, label="~1,000-step trailing mean",
    )
    final_mean = float(raw_values.tail(L3_SMOOTH_ROWS).mean())
    ax.scatter([L3_FINAL_STEP], [final_mean], color=smooth_color, s=30, zorder=3)
    ax.annotate(
        f"final ~1k mean: {final_mean:.4f}",
        (L3_FINAL_STEP, final_mean),
        xytext=(-8, 10), textcoords="offset points", ha="right", fontsize=9,
    )
    ax.set(
        title=f"{title}\n{description}",
        xlabel="Step",
        ylabel="L1 mean absolute error\n(normalized training coordinates)",
        xlim=(0, L3_FINAL_STEP),
        ylim=(0, None),
    )
    ax.set_xticks([0, 10_000, 20_000, 30_000, 40_000, 50_000])
    ax.xaxis.set_major_formatter(
        FuncFormatter(lambda value, _: "0" if value == 0 else f"{int(value / 1000)}k")
    )
    ax.grid(axis="y", alpha=0.25)
    ax.legend(loc="upper right", fontsize=8)

fig.suptitle(
    "Single continuous-OFT training run ending at checkpoint 50,000\n"
    f"W&B run {L3_RUN_ID}; no intermediate-checkpoint series",
    fontsize=15,
)
plt.show()

print("Plotted only run:", l3_run_dir.name)
print(f"History rows: {len(l3_history):,}; range: 0–{L3_FINAL_STEP:,} optimizer steps.")
print(
    "No action_accuracy curve exists for this run: it predicts continuous actions with L1 regression, "
    "whereas the older action_accuracy plot measured exact discrete action-token matches."
)

In [ ]:
# Cell M: Locate and validate the D3 OpenVLA-OFT checkpoint used by every test below.
#
# The checkpoint is PINNED to D3, not "whichever folder was modified most recently":
#   * D3 is the one-to-last trained generation. D4 (openvla_oft_d4_runs/, trained on
#     panda_pickplace_d4) is newer, and must never be picked up by this D3 notebook.
#   * A modification time is not a reliable "latest" signal anyway: every OFT model load
#     rewrites config.json inside the checkpoint folder (leaving config.json.back.*
#     copies), which bumps that folder's mtime.
# D3 was trained once (50,000 optimizer steps, checkpoint interval 10,000) and only the
# final step was kept, so `--50000_chkpt` is the only D3 checkpoint on disk. If earlier
# D3 steps are ever kept, set D3_CHECKPOINT_STEP to choose one of them.
# This cell reports no checkpoint after the default smoke run, which intentionally saves nothing.
import json
import re
from pathlib import Path

D3_CHECKPOINT_STEP = 50_000
D3_RUN_NAME_MARKERS = ("panda_pickplace_d3", "panda-d3-")  # both appear in every D3 run folder name
_CHECKPOINT_STEP_PATTERN = re.compile(r"--(\d+)_chkpt$")


def list_saved_checkpoints(run_root):
    """{training_step: folder} for every saved `--<step>_chkpt` folder directly under run_root."""
    found = {}
    if not Path(run_root).is_dir():
        return found
    for path in Path(run_root).iterdir():
        match = _CHECKPOINT_STEP_PATTERN.search(path.name)
        if path.is_dir() and match and (path / "dataset_statistics.json").is_file():
            found[int(match.group(1))] = path
    return dict(sorted(found.items()))


def verify_d3_checkpoint(checkpoint_dir, expected_step, require_merged=False):
    """Raise AssertionError unless `checkpoint_dir` is the D3 checkpoint at `expected_step`.
    Checks the folder location and name, the dataset key inside dataset_statistics.json, the
    step encoded in the action-head / proprio-projector file names, 8D proprio statistics
    and (optionally) that the merged model shards are present. Returns a small summary dict."""
    checkpoint_dir = Path(checkpoint_dir).resolve()
    assert checkpoint_dir.is_dir(), f"Checkpoint folder does not exist: {checkpoint_dir}"
    assert checkpoint_dir.parent == Path(RUN_ROOT_DIR).resolve(), (
        f"Checkpoint is not directly under the D3 run root {RUN_ROOT_DIR}: {checkpoint_dir}"
    )
    assert all(marker in checkpoint_dir.name for marker in D3_RUN_NAME_MARKERS), (
        f"Not a D3 run folder name (expected {D3_RUN_NAME_MARKERS}): {checkpoint_dir.name}"
    )
    assert "panda_pickplace_d4" not in checkpoint_dir.name and "panda-d4-" not in checkpoint_dir.name, (
        f"D4 checkpoint selected by mistake: {checkpoint_dir.name}"
    )
    step_match = _CHECKPOINT_STEP_PATTERN.search(checkpoint_dir.name)
    assert step_match and int(step_match.group(1)) == int(expected_step), (
        f"Folder step {step_match.group(1) if step_match else None} != pinned D3 step {expected_step}"
    )

    stats = json.loads((checkpoint_dir / "dataset_statistics.json").read_text())
    assert list(stats) == [DATASET_NAME], (
        f"dataset_statistics.json holds {list(stats)}, expected exactly [{DATASET_NAME!r}]"
    )
    proprio_stats = stats[DATASET_NAME]["proprio"]
    assert len(proprio_stats["q01"]) == 8 and len(proprio_stats["q99"]) == 8, "Expected 8D proprio statistics"

    for pattern in ("action_head--*checkpoint.pt", "proprio_projector--*checkpoint.pt"):
        matches = list(checkpoint_dir.glob(pattern))
        assert len(matches) == 1, (pattern, matches)
        assert f"--{int(expected_step)}_checkpoint.pt" in matches[0].name, (
            f"{matches[0].name} is not the step-{expected_step} weights file"
        )

    merged = (checkpoint_dir / "model.safetensors.index.json").is_file()
    if require_merged:
        assert merged, "Checkpoint is unmerged; run the LoRA merge cell first."
        shard_names = sorted(set(json.loads((checkpoint_dir / "model.safetensors.index.json").read_text())["weight_map"].values()))
        assert shard_names and all((checkpoint_dir / name).is_file() for name in shard_names), "Missing merged model shard(s)"
        for name in ("config.json", "preprocessor_config.json", "tokenizer_config.json"):
            assert (checkpoint_dir / name).is_file(), f"Incomplete merged checkpoint: missing {name}"
    return {
        "checkpoint_dir": str(checkpoint_dir),
        "dataset": DATASET_NAME,
        "step": int(expected_step),
        "merged": bool(merged),
        "num_trajectories": stats[DATASET_NAME].get("num_trajectories"),
    }


d3_checkpoints = list_saved_checkpoints(RUN_ROOT_DIR)
print("Saved D3 checkpoints (step -> folder):", {step: path.name for step, path in d3_checkpoints.items()} or "none")
d4_checkpoints = list_saved_checkpoints(PROJECT_DIR / "openvla_oft_d4_runs")
if d4_checkpoints:
    print(f"Newer D4 checkpoints exist (steps {sorted(d4_checkpoints)}); intentionally NOT used by this D3 notebook.")

if not d3_checkpoints:
    print("No saved checkpoint yet. This is expected after TRAINING_MODE='smoke'.")
    D3_OFT_CHECKPOINT = None
    LATEST_OFT_CHECKPOINT = None
else:
    assert D3_CHECKPOINT_STEP in d3_checkpoints, (
        f"Pinned D3 step {D3_CHECKPOINT_STEP} not found; saved D3 steps: {sorted(d3_checkpoints)}"
    )
    D3_OFT_CHECKPOINT = d3_checkpoints[D3_CHECKPOINT_STEP]
    D3_CHECKPOINT_SUMMARY = verify_d3_checkpoint(D3_OFT_CHECKPOINT, D3_CHECKPOINT_STEP)
    # Name kept for the LoRA-merge cell, Cell O and Cell P, which all read LATEST_OFT_CHECKPOINT.
    LATEST_OFT_CHECKPOINT = D3_OFT_CHECKPOINT
    print(f"Using D3 OFT checkpoint (step {D3_CHECKPOINT_STEP:,}):", D3_OFT_CHECKPOINT)
    print(
        f"  dataset={D3_CHECKPOINT_SUMMARY['dataset']}  trajectories={D3_CHECKPOINT_SUMMARY['num_trajectories']}  "
        "8D proprio statistics confirmed."
    )
    if not D3_CHECKPOINT_SUMMARY["merged"]:
        print("Checkpoint is unmerged; merge LoRA before using the standard OFT inference loader.")


In [ ]:
# Merge the saved LoRA adapter into the OpenVLA-7B base checkpoint.
import json
import shutil
import subprocess
from pathlib import Path

assert LATEST_OFT_CHECKPOINT is not None, "Run Cell M first."
LATEST_OFT_CHECKPOINT = Path(LATEST_OFT_CHECKPOINT)

merge_script = OFT_REPO / "vla-scripts/merge_lora_weights_and_save.py"
merged_index = LATEST_OFT_CHECKPOINT / "model.safetensors.index.json"

assert merge_script.is_file(), f"Missing merge script: {merge_script}"
assert (LATEST_OFT_CHECKPOINT / "lora_adapter/adapter_model.safetensors").is_file()

free_gib = shutil.disk_usage(LATEST_OFT_CHECKPOINT).free / (1024**3)
print(f"Available disk space: {free_gib:.1f} GiB")
assert free_gib >= 18, "Insufficient disk space for the merged OpenVLA-7B checkpoint."

if merged_index.is_file():
    print("Checkpoint is already merged:", LATEST_OFT_CHECKPOINT)
else:
    merge_command = [
        str(OFT_PYTHON),
        str(merge_script),
        "--base_checkpoint",
        "openvla/openvla-7b",
        "--lora_finetuned_checkpoint_dir",
        str(LATEST_OFT_CHECKPOINT),
    ]

    print("Starting LoRA merge. This may take several minutes...")
    merge_process = subprocess.Popen(
        merge_command,
        cwd=str(OFT_REPO),
        env=oft_env(),
    )
    merge_return_code = merge_process.wait()
    assert merge_return_code == 0, f"LoRA merge failed with exit code {merge_return_code}"

# Validate every merged model shard.
assert merged_index.is_file(), "Merge completed without creating the model index."
index_data = json.loads(merged_index.read_text())
model_shards = sorted(set(index_data["weight_map"].values()))

assert model_shards, "No model shards listed in the merged index."
for shard_name in model_shards:
    shard_path = LATEST_OFT_CHECKPOINT / shard_name
    assert shard_path.is_file() and shard_path.stat().st_size > 0, (
        f"Missing or empty merged shard: {shard_path}"
    )

assert (LATEST_OFT_CHECKPOINT / "config.json").is_file()
assert any(LATEST_OFT_CHECKPOINT.glob("action_head--*checkpoint.pt"))
assert any(LATEST_OFT_CHECKPOINT.glob("proprio_projector--*checkpoint.pt"))

print(f"LoRA merge verified: {len(model_shards)} model shard(s)")
print("Merged checkpoint ready for Cell N:", LATEST_OFT_CHECKPOINT)

# Testing the fine-tuned model

These tests use the merged 50,000-step OpenVLA-OFT checkpoint, its continuous action head, and its proprio projector.

## 1. In-distribution test

Teacher-forced evaluation on untouched D3 test demonstrations. Each image is paired with its measured `[q1..q7, gripper_open_fraction]`, and every predicted `8 x 7` chunk is compared with the corresponding eight future expert actions.

In [ ]:
# Cell O: Teacher-forced OFT evaluation on held-out D3 demonstrations.
from datetime import datetime
from pathlib import Path
import json
import subprocess

from IPython.display import Image as NotebookImage
from IPython.display import Video, display

# Evaluate one nominal and one boundary demonstration by default.
TEST_EPISODE_INDICES = [10]
MAX_STEPS_PER_EPISODE = None  # Set a small positive integer for a quick smoke test.
TEST_VIDEO_FPS = 10
DISPLAY_ID_VIDEO_INLINE = True

assert LATEST_OFT_CHECKPOINT is not None, "Run Cell M first."
O_CHECKPOINT_DIR = Path(LATEST_OFT_CHECKPOINT)
O_EVALUATOR = PROJECT_DIR / "openvla_oft_heldout_visual_eval.py"

required_globals = ["OFT_REPO", "DATA_ROOT_DIR", "DATASET_NAME", "OFT_PYTHON", "oft_env"]
missing_globals = [name for name in required_globals if name not in globals()]
assert not missing_globals, "Run Cell F first. Missing: " + ", ".join(missing_globals)

required_checkpoint_files = [
    "config.json",
    "dataset_statistics.json",
    "model.safetensors.index.json",
    "preprocessor_config.json",
    "tokenizer_config.json",
]
missing_files = [
    name for name in required_checkpoint_files if not (O_CHECKPOINT_DIR / name).is_file()
]
assert not missing_files, "The merged checkpoint is incomplete. Missing: " + ", ".join(missing_files)
assert any(O_CHECKPOINT_DIR.glob("action_head--*checkpoint.pt"))
assert any(O_CHECKPOINT_DIR.glob("proprio_projector--*checkpoint.pt"))
assert O_EVALUATOR.is_file(), f"Evaluator script is missing: {O_EVALUATOR}"
assert TEST_EPISODE_INDICES, "Choose at least one D3 test episode."

O_OUTPUT_DIR = (
    PROJECT_DIR
    / "test_eval_visuals"
    / "oft-d3-in-distribution"
    / datetime.now().strftime("%Y%m%d_%H%M%S")
)
O_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

command = [
    str(OFT_PYTHON),
    str(O_EVALUATOR),
    "--checkpoint", str(O_CHECKPOINT_DIR),
    "--data-root", str(DATA_ROOT_DIR),
    "--dataset-name", DATASET_NAME,
    "--openvla-repo", str(OFT_REPO),
    "--output-dir", str(O_OUTPUT_DIR),
    "--episode-indices", *[str(index) for index in TEST_EPISODE_INDICES],
    "--fps", str(TEST_VIDEO_FPS),
]
if MAX_STEPS_PER_EPISODE is not None:
    command.extend(["--max-steps", str(MAX_STEPS_PER_EPISODE)])

evaluation_env = oft_env({
    "TF_CPP_MIN_LOG_LEVEL": "3",
    "MPLBACKEND": "Agg",
    "MPLCONFIGDIR": "/tmp/openvla_oft_id_matplotlib",
    "WANDB_MODE": "disabled",
})

print("Held-out split: D3 test")
print("Input: third-person RGB + language + measured 8D joint/gripper proprio")
print("Output: 8 future actions x 7 continuous dimensions")
print("Checkpoint:", O_CHECKPOINT_DIR)
print("Output directory:", O_OUTPUT_DIR)
print("Launching OFT evaluator...\n")

process = subprocess.Popen(
    command,
    cwd=str(OFT_REPO),
    env=evaluation_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert process.stdout is not None
for line in process.stdout:
    print(line, end="")

return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f"Held-out OFT visual evaluation failed with exit code {return_code}.")

summary_path = O_OUTPUT_DIR / "summary.json"
summary = json.loads(summary_path.read_text(encoding="utf-8"))
for episode in summary["episodes"]:
    print("\n" + "=" * 78)
    print(
        f"Held-out episode {episode['episode_index']:03d} | "
        f"{episode['evaluated_queries']} valid 8-step chunks | "
        f"phase boundaries {episode['phase_boundaries']}"
    )
    print("Solid lines = test demonstration; dashed lines = OFT horizon-0 prediction.")
    display(NotebookImage(filename=episode["commands_plot"]))
    display(NotebookImage(filename=episode["chunk_plot"]))
    display(NotebookImage(filename=episode["proprio_plot"]))
    display(NotebookImage(filename=episode["contact_sheet"]))
    if DISPLAY_ID_VIDEO_INLINE:
        display(Video(episode["video"], embed=True, html_attributes="controls loop"))
    else:
        print("Video:", episode["video"])

print("\nSaved summary:", summary_path)
print("OPENVLA_OFT_IN_DISTRIBUTION_TEST_COMPLETE")

## 2. Out-of-distribution test (MuJoKo Enviornment)

Closed-loop MuJoCo evaluation with cube starts outside the D3 training envelope. The policy is queried only from live third-person RGB, language, and measured 8D proprio; the full eight-action chunk is executed before the next query.

In [ ]:
# Cell P0: Quaternion helpers and 6D pose IK for closed-loop execution.
import math
import numpy as np


def quat_normalize(quat):
    quat = np.asarray(quat, dtype=np.float64)
    norm = float(np.linalg.norm(quat))
    if norm < 1e-12:
        raise ValueError("Cannot normalize a zero quaternion")
    return quat / norm


def quat_conjugate(quat):
    w, x, y, z = quat_normalize(quat)
    return np.array([w, -x, -y, -z], dtype=np.float64)


def quat_multiply(left, right):
    w1, x1, y1, z1 = quat_normalize(left)
    w2, x2, y2, z2 = quat_normalize(right)
    return quat_normalize(np.array([
        w1 * w2 - x1 * x2 - y1 * y2 - z1 * z2,
        w1 * x2 + x1 * w2 + y1 * z2 - z1 * y2,
        w1 * y2 - x1 * z2 + y1 * w2 + z1 * x2,
        w1 * z2 + x1 * y2 - y1 * x2 + z1 * w2,
    ], dtype=np.float64))


def quat_from_rotvec(rotvec):
    rotvec = np.asarray(rotvec, dtype=np.float64)
    angle = float(np.linalg.norm(rotvec))
    if angle < 1e-12:
        return np.array([1.0, 0.0, 0.0, 0.0], dtype=np.float64)
    axis = rotvec / angle
    return np.array([
        math.cos(angle / 2.0),
        *(axis * math.sin(angle / 2.0)),
    ], dtype=np.float64)


def quat_apply_world_delta(quat, rotvec_world):
    return quat_multiply(quat_from_rotvec(rotvec_world), quat)


def quat_error_world(target_quat, current_quat):
    error_quat = quat_multiply(target_quat, quat_conjugate(current_quat))
    if error_quat[0] < 0.0:
        error_quat = -error_quat
    vector_norm = float(np.linalg.norm(error_quat[1:]))
    if vector_norm < 1e-12:
        return np.zeros(3, dtype=np.float64)
    angle = 2.0 * math.atan2(vector_norm, float(error_quat[0]))
    return error_quat[1:] / vector_norm * angle


def solve_hand_pose_ik(
    target_pos,
    target_quat,
    seed_qpos=None,
    max_iter=450,
    position_tolerance=0.00005,
    orientation_tolerance=np.deg2rad(0.01),
):
    """Solve one 6D hand-pose target with damped least squares."""
    target_pos = np.asarray(target_pos, dtype=np.float64)
    target_quat = quat_normalize(target_quat)
    if seed_qpos is not None:
        for name, value in seed_qpos.items():
            joint_id = require_mj_id(mujoco.mjtObj.mjOBJ_JOINT, name)
            data.qpos[model.jnt_qposadr[joint_id]] = float(value)
    mujoco.mj_forward(model, data)

    jacp = np.zeros((3, model.nv), dtype=np.float64)
    jacr = np.zeros((3, model.nv), dtype=np.float64)
    iterations = 0
    for iterations in range(1, max_iter + 1):
        current = get_ee_pose()
        position_error = target_pos - current["position"]
        rotation_error = quat_error_world(target_quat, current["quat_wxyz"])
        # Always perform at least one DLS update for a nonzero policy command.
        if (
            iterations > 1
            and np.linalg.norm(position_error) < position_tolerance
            and np.linalg.norm(rotation_error) < orientation_tolerance
        ):
            break

        mujoco.mj_jacBody(model, data, jacp, jacr, HAND_BODY_ID)
        jacobian = np.vstack([
            jacp[:, ARM_DOF_ADR],
            jacr[:, ARM_DOF_ADR],
        ])
        error = np.concatenate([position_error, rotation_error])
        damping = 0.055
        delta_q = jacobian.T @ np.linalg.solve(
            jacobian @ jacobian.T + damping * damping * np.eye(6),
            error,
        )
        delta_q = np.clip(delta_q, -0.035, 0.035)

        for index, address in enumerate(ARM_QPOS_ADR):
            low, high = model.jnt_range[ARM_JOINT_IDS[index]]
            data.qpos[address] = np.clip(
                data.qpos[address] + delta_q[index],
                low,
                high,
            )
        mujoco.mj_forward(model, data)

    final_pose = get_ee_pose()
    diagnostics = {
        "iterations": int(iterations),
        "position_error_m": float(
            np.linalg.norm(target_pos - final_pose["position"])
        ),
        "orientation_error_rad": float(np.linalg.norm(
            quat_error_world(target_quat, final_pose["quat_wxyz"])
        )),
    }
    diagnostics["converged"] = bool(
        diagnostics["position_error_m"] < position_tolerance
        and diagnostics["orientation_error_rad"] < orientation_tolerance
    )
    return current_arm_qpos_dict(), diagnostics


def plan_pose_qpos(target_pos, target_quat):
    """Plan in temporary MuJoCo state, then restore the live rollout state."""
    saved_qpos = data.qpos.copy()
    saved_qvel = data.qvel.copy()
    saved_ctrl = data.ctrl.copy()
    saved_time = float(data.time)
    try:
        return solve_hand_pose_ik(
            target_pos,
            target_quat,
            seed_qpos=current_arm_qpos_dict(),
        )
    finally:
        data.qpos[:] = saved_qpos
        data.qvel[:] = saved_qvel
        data.ctrl[:] = saved_ctrl
        data.time = saved_time
        mujoco.mj_forward(model, data)


In [ ]:
# Cell P: Out-of-distribution closed-loop MuJoCo rollout with OpenVLA-OFT.
# Every policy query uses live third-person RGB, language, and measured 8D proprio.
# Only horizon 0 is executed before replanning at 10 Hz; horizons 1-7 are logged
# for diagnostics and never substituted into the physical controller.
#
# Dual tactile sensors/gripper/cube: ported from TactileModel05_Duo.ipynb (this
# project's dual-GelSight FOTS notebook), replacing the earlier single-sensor
# (right finger only) build ported from TactileModel04.ipynb. Scoped to this
# cell only -- Cell A/B/C/D (LoRA data collection, used to build the dataset the
# loaded checkpoint was actually trained on) are left completely untouched, so
# this cell builds its OWN self-contained scene from panda.xml (same source Cell A
# uses) instead of reloading/extending Cell A's canonical model. What changed
# compared with OpenVLA35_T1's Cell P:
#   1. BOTH fingers now carry a GelSight Mini assembly (mount plate/standoffs/case/
#      gel pad/label/cable/site + a FOTS virtual tactile camera), built by one
#      shared helper called once per finger. Mirroring needs no sign flips:
#      panda.xml already gives right_finger an extra 180 degree about-Z quat
#      relative to left_finger, which turns identical local offsets into a
#      correctly mirrored, facing-each-other placement (an assertion below guards
#      it). The stock left-finger mesh/collision pads are replaced by the same
#      primitive carrier + gel pad the right finger already used.
#   2. Two FOTSTactileSensor objects (one per finger) share the cube's reference
#      site; every rollout records both through one DualFOTSRecorder into
#      rollout_NNN/right/ and rollout_NNN/left/ (tactile RGB + marker-flow video,
#      quantitative npz, validation json).
#   3. Nothing else about the rollout changes: the arm's 7 joints/actuators, the
#      wrist camera, the controller and the cube (size, bump lattice, gel/cube
#      torsional friction 0.1, soft-gel / rigid-cube contact materials) are
#      identical to OpenVLA35_T1 -- TactileModel05_Duo already carried those over.
#   4. The checkpoint is the D3 checkpoint pinned in Cell M, re-verified below.
#   5. RUN_CLOSED_LOOP_ROLLOUTS=False builds + self-tests the two sensors and stops
#      before OpenVLA loads, so the two-sensor test cells at the end of the notebook
#      can run on their own, without any policy rollout.
# The FOTS tactile pipeline itself (virtual camera behind the gel,
# depth-diff -> contact mask -> FOTS's pretrained MLP + shadow model ->
# tactile RGB -> FOTS's marker-motion model) is unchanged -- see
# external/fots/NOTICE.md for FOTS's own provenance (Zhao et al., RA-L 2024,
# MIT-licensed, vendored into external/fots/).

from pathlib import Path
from datetime import datetime
import csv
import hashlib
import io
import json
import os
import select
import shutil
import subprocess
import sys
import time
import xml.etree.ElementTree as ET

import cv2
import imageio.v2 as imageio
import numpy as np
import torch
from PIL import Image, ImageDraw
from scipy.ndimage import gaussian_filter, median_filter
from IPython.display import Video, display

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
TORCH_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ----- User-facing rollout knobs -----
TASK_PROMPT = "pick up the red block from the table and place it on the far side of the table"
OPENVLA_ROLLOUT_PROMPT = f"In: What action should the robot take to {TASK_PROMPT}?\nOut: "   # \n means start a new line.

# Ceiling on attempts, not a target: DATASET_TARGET_DEMOS below stops the loop as
# soon as enough successful demos are collected, so 400 is only spent in full if the success
# rate is much lower than the ~70% observed on T1_duo/baseline (28/40 accepted).
N_ROLLOUTS = 800
ACTIONS_PER_CHUNK = 1  # Re-observe and execute only the newly predicted horizon-0 action.

# D3 demonstrations were collected at 10 Hz (CONTROL_HORIZON_S = 0.1 s/action). Repeated OOD
# rollouts showed the cube slipping out of the grasp when the arm moves fast -- inertial force
# on the still-settling grasp exceeding what friction/normal force can hold. CONTROL_RATE_SCALE_FACTOR
# slows the robot down to fight that: it stretches how long the controller is given to reach each
# commanded target (see step_controller's `steps` below), which lowers both velocity and
# acceleration for the exact same commanded motion, and -- since ACTIONS_PER_CHUNK == 1 replans
# every control period -- proportionally lowers the OpenVLA query rate as a direct consequence,
# not a separately-tuned knob. 1.0 reproduces the original 10 Hz behavior; 0.5 halves the control
# rate to 5 Hz (each action now takes 0.2 s instead of 0.1 s, i.e. half the average velocity and
# a quarter of the peak acceleration for the same target displacement). Lower = gentler/slower.
CONTROL_RATE_SCALE_FACTOR = 1.0  # 10 Hz -- matches D3's original collection rate; testing whether
# this removes the slow-motion lag CONTROL_RATE_SCALE_FACTOR=0.5 introduced. Also relevant here for
# a second reason: 0.5 was specifically chosen to SUPPRESS the fast-motion slip that this notebook
# now wants some of -- 10 Hz likely reintroduces a bit of that real slip dynamic too.
CONTROL_HORIZON_S = 0.1 / CONTROL_RATE_SCALE_FACTOR

# Time-budget stopping condition (the other is cube_at_destination, below). Expressed directly in
# simulated seconds rather than a hardcoded action count so it stays correct at any
# CONTROL_RATE_SCALE_FACTOR -- MAX_EXECUTED_ACTIONS is just however many control periods fit in it.
MAX_ROLLOUT_SECONDS = 25.0
MAX_EXECUTED_ACTIONS = round(MAX_ROLLOUT_SECONDS / CONTROL_HORIZON_S)

# Deliberately NOT tied to CONTROL_HORIZON_S. OpenVLA only makes a new decision once per control
# period (5 Hz at scale 0.5), but the physics in between is still simulated at full resolution --
# step_controller() smoothly interpolates the arm toward each target over many physics substeps
# regardless of how long that period is. Fixing VIDEO_FPS at the original 10 means step_controller
# and settle_after_release (both below) capture more than one frame per control period whenever
# CONTROL_HORIZON_S > 0.1 s, so saved video stays as smooth as the original 10 Hz rollouts and
# still plays back in real time -- only the robot's own decision rate actually changes.
VIDEO_FPS = 10
ROLLOUT_SEED = 7
DISPLAY_ROLLOUT_VIDEOS = False  # 400 rollouts: saved files are the output, not inline widgets.

# True: run the closed-loop OpenVLA rollouts (the normal behaviour of this cell).
# False: only build the dual-GelSight scene and both FOTS sensors, run their self-test, and
# stop before the OpenVLA worker starts -- no model load on the GPU, and no run folder is
# created or cleaned. Use it to run the two-sensor test cells at the end of the notebook on
# their own.
RUN_CLOSED_LOOP_ROLLOUTS = True

print(
    f"Control rate: {1.0 / CONTROL_HORIZON_S:.2f} Hz "
    f"(CONTROL_HORIZON_S={CONTROL_HORIZON_S:.3f} s, scale factor={CONTROL_RATE_SCALE_FACTOR}); "
    f"time budget per rollout: {MAX_EXECUTED_ACTIONS * CONTROL_HORIZON_S:.1f} simulated seconds "
    f"over {MAX_EXECUTED_ACTIONS} actions; video: {VIDEO_FPS} fps"
)

# D3 trained on cube x=[0.47, 0.53], y=[-0.04, 0.04]. These starts are outside it.
# z is re-derived below from the new (2/3-scale) cube half-size, since the
# original 0.085 assumed the old 70 mm cube's half-size (table 0.05 + 0.035).
OOD_CUBE_STARTS_XY = np.array([
[0.455, -0.060, 0.085],
[0.545, 0.060, 0.085],
[0.455, 0.060, 0.085],
[0.545, -0.060, 0.085],
[0.465, -0.050, 0.085],
[0.535, 0.050, 0.085],
[0.445, 0.075, 0.085],
[0.555, -0.075, 0.085],
[0.440, -0.045, 0.085],
[0.560, 0.045, 0.085],

    
], dtype=np.float64)

# Which OOD_CUBE_STARTS corner(s) to actually run, in order. Leave as None to
# run every corner once, indices 0..N_ROLLOUTS-1 (the original behavior).
# Set to an explicit list of indices into OOD_CUBE_STARTS to run only (and
# exactly) those corners, e.g. [2] for just the third corner, or [0, 2, 0] to
# repeat corner 0 with corner 2 in between. Output folders are still named
# rollout_000, rollout_001, ... in the order run, regardless of which corner
# each one used; N_ROLLOUTS is ignored whenever this is set.
SELECTED_OOD_START_INDICES = None  # e.g. [2]
OOD_EE_JITTER_M = 0.0

# ----- T1 multi-condition dataset knobs -----
# Set EXPERIMENT_DATASET to save this run's rollouts under a persistent, named dataset
# folder (PROJECT_DIR/EXPERIMENT_DATASET/EXPERIMENT_CONDITION/rollout_NNN) instead of the
# scratch openvla_oft_ood_closed_loop_<timestamp> folder above, which is wiped on every run.
# Leave EXPERIMENT_DATASET as None to keep that original scratch OOD-corner behavior.
# "T3" -- a fresh dataset, separate from T2. Deliberately not reusing/growing T2: T2's own keep
# criterion (see below) specifically selected FOR clean, fully-held grasps, which is close to the
# opposite of what a slip-detection dataset needs.
EXPERIMENT_DATASET = "T3"
EXPERIMENT_CONDITION = "baseline"

# ----- T3 slip-detection dataset knobs -----
# T2 kept a demo only if BOTH the task succeeded AND both sensors showed a strong, sustained grip --
# selecting FOR clean holds and against everything a slip detector actually needs. T3's keep
# criterion is deliberately much looser: keep if a real grasp attempt happened at all (meaningful
# contact on at least one sensor at some point), regardless of whether the task ultimately
# succeeded -- a partial slip or an outright drop mid-carry is exactly the positive-class data this
# dataset is for, not something to discard. "Kept" therefore no longer means "succeeded"; both are
# tracked separately in attempt_summary/results.csv below.
DATASET_KEEP_ONLY_QUALIFYING = True
DATASET_TARGET_DEMOS = 900          # the loop stops once this many demos are kept (any outcome).
DATASET_MIN_CONTACT_SECONDS = 2.0   # a real, sustained grasp attempt happened -- brief/glancing touches
                                     # (the gripper missed the cube, or barely brushed it) are excluded:
                                     # they have no useful tactile content for a slip detector to learn
                                     # from either way, so keeping them was pure waste. Applied per side
                                     # (kept if EITHER sensor clears the bar -- a strongly asymmetric grip
                                     # is itself a real slip signature, not something to discard).
DATASET_MIN_PEAK_TAXELS = 200       # a light floor on peak contact area, independent of duration.

# ----- Slip ground truth (Plan B items 1-3, 5): computed every tactile frame (30 Hz), perfectly
# aligned with the tactile RGB/marker-flow videos by construction -- see InstrumentedDualFOTSRecorder
# below, which captures it in the same call that triggers each tactile sample, not on a separate
# timer. Saved to <demo>/slip_ground_truth.npz (global/per-episode: hand & cube pose, in-hand
# relative pose, slip binary) and two new arrays appended into each sensor's own
# fots_quantitative_data.npz (normal_force_n, contact_state -- item 4 and item 5).
# Calibrated from the 10-demo smoke test, not the originally-requested 1.5 mm/1.5 deg: even during
# frames where BOTH normal force and contact state say "solid, unbroken hold", the measured baseline
# translation delta in this compliant-gel simulation is ~3.5 mm (median) up to ~10.4 mm (99th
# percentile) -- i.e. 1.5 mm was below the simulator's own ambient noise floor and flagged 75-96% of
# genuinely solid holding frames as "slip" across every smoke-test demo. 12 mm/10 deg sits just above
# the measured 99th-percentile noise floor (0% false-positive rate on the same force-stable frames)
# while still well below the ~48 mm/22 deg spike measured at an actual release event in the same data.
SLIP_TRANSLATION_THRESHOLD_MM = 12.0  # object-vs-hand displacement over SLIP_WINDOW_S.
SLIP_ROTATION_THRESHOLD_DEG = 10.0    # object-vs-hand rotation over SLIP_WINDOW_S.
SLIP_WINDOW_S = 0.15                  # ~4-5 tactile frames at 30 Hz. A ROLLING window, not cumulative
                                       # drift since grasp -- so the label means "slipping right now",
                                       # not "ever moved since the grasp closed" (which would latch at
                                       # 1 forever after a single early settle).
SLIP_HOLDING_PHASES = frozenset({"closed_hold", "lift_bootstrap", "carrying"})  # slip is only
                                       # evaluated in phases where the object should be rigidly held.
SAVE_RAW_DEPTH_STACK = False          # T2's dominant disk cost by far (measured: ~94% of a demo,
                                       # ~150 MB per sensor) is the full per-frame raw depth image,
                                       # saved uncompressed. Not needed for slip labeling -- the
                                       # derived scalars (max_depth_diff_mm, active_taxel_count) and
                                       # the new relative-pose/force channels already capture what
                                       # matters -- so it is off here, which is what makes a dataset of
                                       # this size sustainable. Set True to re-run Cell P4's
                                       # depth_difference panel against a specific T3 demo.

# ----- Randomization, so the SAME OpenVLA policy produces a real spread of grip outcomes instead of
# the almost entirely clean-success dataset T2 collected. Nothing here touches what OpenVLA itself
# sees or decides -- only how the physics/controller responds underneath it, per your priority that
# this should look like the real OpenVLA deployment. Friction and mass are edited directly on the
# already-compiled `model` arrays (no scene rebuild -- MuJoCo allows live edits to physical
# parameters, just not structure), so these can vary every single rollout at negligible cost.
SLIP_RANDOMIZE_MATERIALS = True
SLIP_TORSIONAL_FRICTION_RANGE = (0.02, 0.10)      # tuned-safe (0.1) down to a documented-fragile value.
SLIP_CUBE_MASS_SCALE_RANGE = (1.0, 2.0)           # heavier cube, same grip force -> more likely to slip.
SLIP_RANDOMIZE_GRIP_MARGIN = True
SLIP_GRIP_LOCK_EXTRA_MARGIN_RANGE = (-0.01, 0.05)  # replaces the fixed GRIP_LOCK_EXTRA_MARGIN (0.03)
                                       # per rollout; negative values deliberately under-squeeze past
                                       # what the policy's own plateau already established.
SLIP_PERTURBATION_ENABLED = True      # a brief external force on the cube mid-carry -- independent of
                                       # OpenVLA entirely, so it gives a slip event with a KNOWN onset
                                       # time without touching the policy's own behavior at all.
SLIP_PERTURBATION_PROBABILITY = 0.35  # fraction of rollouts that get a perturbation at all.
SLIP_PERTURBATION_FORCE_RANGE_N = (3.0, 12.0)
SLIP_PERTURBATION_DURATION_S = 0.15
SLIP_PERTURBATION_PHASES = frozenset({"lift_bootstrap", "carrying"})  # only while it should be held.

# True: start the (still resized) cube INSIDE the D3 training envelope (CUBE_X_RANGE/
# CUBE_Y_RANGE from Cell C -- see IN_DISTRIBUTION_CUBE_STARTS_XY below), so the resized
# cube is the only out-of-distribution factor this condition tests -- it is not combined
# with an out-of-distribution start position too. False restores the original 10-corner
# OOD-start behavior (OOD_CUBE_STARTS_XY above).
USE_IN_DISTRIBUTION_STARTS = True
ID_START_SAMPLING_SEED = 4242
WRIST_IMAGE_SIZE = 256

DISPLAY_GELSIGHT_VIDEOS = False  # same reasoning as DISPLAY_ROLLOUT_VIDEOS above.

# The policy predicts D3 Cartesian deltas, a world-frame rotation vector, and a
# continuous gripper open fraction. These gains compensate the physical actuator
# response measured against exact D3 expert-action replay.
OPENVLA_MAX_TRANSLATION_M = 0.040
OPENVLA_MAX_ROTATION_RAD = np.deg2rad(20.0)
OPENVLA_GRIPPER_OPENING_M = 0.04
OPENVLA_GRIPPER_CLOSED_M = 0.00
WORKSPACE_LOW = np.array([0.24, -0.25, 0.180], dtype=np.float64)
WORKSPACE_HIGH = np.array([0.86, 0.25, 0.62], dtype=np.float64)
APPROACH_TRANSLATION_GAIN = np.array([0.9, 1.05, 1.0], dtype=np.float64)
CARRY_TRANSLATION_GAIN = np.array([2.5, 1.05, 1.0], dtype=np.float64)
CARRY_MAX_TRANSLATION_M = 0.030
ROTATION_GAIN = 0.4
ACTUATOR_LEAD = 2.0
FINE_APPROACH_START_Z_M = 0.0
FINE_APPROACH_MAX_XY_STEP_M = 0.0025
FINE_APPROACH_MAX_DESCENT_STEP_M = 0.005
CARRY_FILTER_ALPHA = 0.5
CARRY_ROTATION_FILTER_ALPHA = 0.35
CARRY_AXIS_MAX_STEP_M = np.array([0.02, 0.004, 0.008], dtype=np.float64)
PLACEMENT_AXIS_MAX_STEP_M = np.array([0.01, 0.003, 0.004], dtype=np.float64)
WRIST_VISUAL_SERVO_ENABLED = True
WRIST_VISUAL_SERVO_START_Z_M = 0.3
WRIST_VISUAL_SERVO_STOP_Z_M = 0.0
WRIST_VISUAL_SERVO_GAIN = 0.35
WRIST_VISUAL_SERVO_MAX_STEP_M = 0.004
WRIST_ALIGNMENT_PLANE_Z_M = 0.120
WRIST_ESTIMATE_TO_GRASP_OFFSET_M = np.array([-0.009, 0.001], dtype=np.float64)
WRIST_ALIGNMENT_TOLERANCE_M = 0.003
SAFE_CARRY_Y_BOUNDS_M = np.array([-0.08, 0.08], dtype=np.float64)
MAX_VALID_GRASP_CLOSE_Z_M = 0.205

# Track grasp/release phases for evaluation and physical gripper control while
# applying the policy's continuous gripper fraction directly, as in the D3
# demonstrations. NOTE: these phases no longer drive rollout termination --
# see PLACEMENT_X_MIN_M / cube_at_destination below for the only two ways a
# rollout ends.
GRIPPER_CLOSE_TRIGGER = 0.50
GRIPPER_OPEN_TRIGGER = 0.65
GRIPPER_CLOSE_PROFILE_STEPS = 8
GRIPPER_OPEN_PROFILE_STEPS = 7
LIFT_LOOKAHEAD_MIN_DZ_M = 0.005
LIFT_BOOTSTRAP_HEIGHT_M = 0.12
CARRY_Z_FLOOR_OFFSET_M = 0.12
CARRY_DESCENT_START_X_M = 0.65
RELEASE_CLEARANCE_M = 0.03
RELEASE_SETTLE_S = 0.5

# --- Grip-plateau gripper-command hold ---------------------------------------
# Diagnosed from real OOD rollouts (torsional-friction sweep against the 2/3
# -scale cube): once past the initial "closing" ramp, the raw policy gripper
# output frequently pins at (or very near) 0.000 -- commanding fully-closed --
# for many consecutive control steps in a row, even though the fingers
# physically stopped against the object within the first step or two of
# contact and never moved again. Because the gripper actuator is
# position-controlled, that sustained near-zero *target* against an
# already-blocked *actual* position keeps a large position error alive
# continuously; under our compliant contact model that creeps the gel's
# indentation deeper and deeper over real time, eventually destabilizing the
# grip. Confirmed empirically: raising torsional friction from 0.005 to 0.1
# only ever delayed this failure (never eliminated it), and 0.1 gave no
# further benefit over 0.05 -- the signature of a time/creep-driven failure,
# not a friction-insufficiency one.
#
# The fix is not "stop pushing the instant contact is felt" -- a gripper that
# stops exactly at an object's rigid width exerts zero normal force and
# therefore zero friction grip, no matter the friction coefficient; real
# grasping needs deliberate overtravel past that point to build genuine
# clamping force. An earlier version tried to construct that overtravel
# explicitly (a fixed margin past the measured plateau position) and cap the
# command there -- that abruptly released most of the *already-established*
# squeeze the instant it engaged, since the raw policy schedule had already
# built up a far larger implicit squeeze by then, and the cube dropped
# almost immediately. Fixed design: don't recompute a target from geometry
# at all. Wait for the measured finger opening to genuinely plateau (stop
# changing by more than GRIP_PLATEAU_TOLERANCE_M for
# GRIP_PLATEAU_STABLE_STEPS consecutive steps -- i.e. the initial settle,
# which already includes the policy's own natural overtravel, is over),
# then LOCK the tightest commanded fraction already seen during that
# stability window as a floor for the rest of the grip -- freezing exactly
# the squeeze level that was already naturally established, with no jump in
# either direction, rather than substituting an independently-derived one.
# The policy's own gripper output is still followed normally whenever it
# calls for *more* opening than the locked floor (release is never blocked).
GRIP_PLATEAU_STABLE_STEPS = 3
GRIP_PLATEAU_TOLERANCE_M = 0.0005

# Small extra tightening applied ONLY as a delta *from* the already-locked
# fraction (see capped_grip_fraction) -- never a freshly-recomputed target
# from geometry, which is exactly what caused the v1 abrupt-release bug.
# Diagnosed from a real 4-corner test of the plain lock (0 extra margin):
# it held perfectly for one corner (44+ steps through the whole carry) but
# was lost mid-lift for another despite a clean, unwavering lock at a
# similar value -- a real disturbance beating a grip that was simply a
# little light, not a creep failure. In fraction space, 0.03 corresponds to
# about 1.2 mm of additional commanded overtravel (0.03 * 0.04 m) -- small
# enough not to reintroduce the v1 problem, but enough to add real margin.
GRIP_LOCK_EXTRA_MARGIN = 0.03

#   ----------------- Internal rollout parameters -----------------
POLICY_CAMERA_VIEW = "third_person"

required_globals = [
    "model", "data", "mujoco", "AGENT_CAMERA_NAME", "START_BLOCK_POS", "PLACE_BLOCK_POS",
    "RUN_ROOT_DIR", "OFT_REPO", "OFT_PYTHON", "DATASET_NAME", "oft_env", "LATEST_OFT_CHECKPOINT",
    "reset_task_state_randomized", "current_arm_qpos_dict", "set_arm_position_targets",
    "set_gripper_opening", "current_finger_opening_m", "solve_hand_position_ik", "get_ee_pose", "get_cube_pose",
    "HAND_BODY_ID", "require_mj_id", "ARM_JOINT_NAMES", "ARM_JOINT_IDS",
    "ARM_QPOS_ADR", "ARM_DOF_ADR", "FINGER_QPOS_ADR", "make_third_person_camera",
    "THIRD_PERSON_IMAGE_SIZE", "quat_apply_world_delta", "quat_error_world",
    "quat_normalize", "solve_hand_pose_ik", "plan_pose_qpos",
    "PANDA_DIR", "PANDA_XML", "GELSIGHT_BODY_NAME", "GELSIGHT_SITE_NAME",
    "GELSIGHT_GEL_PAD_GEOM_NAME", "FINGER_JOINT_NAMES",
    "CUBE_X_RANGE", "CUBE_Y_RANGE",
    "verify_d3_checkpoint", "D3_CHECKPOINT_STEP",
]
missing = [name for name in required_globals if name not in globals()]
assert not missing, "Run Cells 2, 3, 4, A, B, C, F, M, and the LoRA merge cell first. Missing: " + ", ".join(missing)

# 40 (N_ROLLOUTS) in-distribution start positions -- sampled from the same CUBE_X_RANGE/
# CUBE_Y_RANGE the D3 demonstrations were collected from in Cell C -- used only when
# USE_IN_DISTRIBUTION_STARTS is True (see the T1 dataset knobs above). z is re-derived
# below from the new (2/3-scale) cube half-size, same as OOD_CUBE_STARTS_XY above.
_id_start_rng = np.random.default_rng(ID_START_SAMPLING_SEED)
IN_DISTRIBUTION_CUBE_STARTS_XY = np.stack(
    [
        _id_start_rng.uniform(*CUBE_X_RANGE, size=N_ROLLOUTS),
        _id_start_rng.uniform(*CUBE_Y_RANGE, size=N_ROLLOUTS),
    ],
    axis=1,
)

# ----- Rollout stopping conditions -----
# A rollout ends for exactly one of two reasons, checked every executed
# action: the cube is (more or less) at the destination, or
# MAX_EXECUTED_ACTIONS has been used up (MAX_EXECUTED_ACTIONS * CONTROL_HORIZON_S
# simulated seconds -- see the control-rate print above). There is
# deliberately no other early-exit path. An
# earlier version also broke out the instant the gripper's release profile
# finished (CONTROLLER_STATE["mode"] == "released_hold"), which fired
# whenever the policy's predicted gripper fraction crossed
# GRIPPER_OPEN_TRIGGER -- including while the arm was still approaching, well
# before the cube had ever been grasped. That early exit has been removed.
PLACEMENT_X_MIN_M = PLACE_BLOCK_POS[0] - 0.06
PLACEMENT_Y_TOLERANCE_M = 0.08
PLACEMENT_Z_RANGE_M = (0.06, 0.14)


def cube_at_destination(cube_pos):
    """Same tolerance box used both to stop a rollout early and to score it."""
    return bool(
        cube_pos[0] >= PLACEMENT_X_MIN_M
        and abs(cube_pos[1] - PLACE_BLOCK_POS[1]) <= PLACEMENT_Y_TOLERANCE_M
        and PLACEMENT_Z_RANGE_M[0] <= cube_pos[2] <= PLACEMENT_Z_RANGE_M[1]
    )


POLICY_IMAGE_WIDTH = THIRD_PERSON_IMAGE_SIZE[0]
POLICY_IMAGE_HEIGHT = THIRD_PERSON_IMAGE_SIZE[1]

# ----- D3 checkpoint -----------------------------------------------------------
# Pinned to the D3 checkpoint by Cell M (step D3_CHECKPOINT_STEP under openvla_oft_d3_runs/)
# and re-verified here, so neither a stale kernel nor a newer run folder (e.g. D4) can be picked
# up silently. The old "newest folder by modification time" helper is gone on purpose: a model
# load rewrites config.json inside the checkpoint folder, which bumps that folder's mtime.
assert LATEST_OFT_CHECKPOINT is not None, "Run Cell M first."
checkpoint_dir = Path(LATEST_OFT_CHECKPOINT)
checkpoint_info = verify_d3_checkpoint(checkpoint_dir, D3_CHECKPOINT_STEP, require_merged=True)
assert ACTIONS_PER_CHUNK == 1, "The validated controller replans after every horizon-0 action."

print("Using merged OpenVLA-OFT checkpoint (D3, pinned in Cell M):")
print(checkpoint_dir)
print(
    f"  dataset={checkpoint_info['dataset']}  step={checkpoint_info['step']:,}  "
    f"training trajectories={checkpoint_info['num_trajectories']}"
)
print("Policy input: third-person RGB + language + live 8D proprio")
print("Policy replanning: execute horizon 0, then acquire a new image and proprio state")
print("Action selection: only horizon 0 is controlled; horizons 1-7 are diagnostic")

#########################

rollout_root = Path(PROJECT_DIR if "PROJECT_DIR" in globals() else Path.cwd())
if not RUN_CLOSED_LOOP_ROLLOUTS:
    print("RUN_CLOSED_LOOP_ROLLOUTS=False: this cell creates or cleans no run folder.")
elif EXPERIMENT_DATASET:
    # Named dataset conditions (e.g. T1_duo/size) are a persistent, growing collection --
    # never auto-deleted like the scratch OOD folder below.
    run_dir = rollout_root / EXPERIMENT_DATASET / EXPERIMENT_CONDITION
    run_dir.mkdir(parents=True, exist_ok=True)
    print(f"Saving to persistent dataset folder (not auto-cleaned): {run_dir}")
    _existing_rollouts = sorted(run_dir.glob("rollout_*"))
    if _existing_rollouts:
        print(
            f"WARNING: {run_dir} already holds {len(_existing_rollouts)} rollout folder(s); rollouts with "
            "the same index are overwritten. Change EXPERIMENT_DATASET/EXPERIMENT_CONDITION to keep them."
        )
    _existing_demos = sorted(run_dir.glob("demo_*")) if DATASET_KEEP_ONLY_QUALIFYING else []
    _resume_dataset_kept_count = len(_existing_demos)
    if _existing_demos:
        print(
            f"Resuming an existing T2 dataset: {_resume_dataset_kept_count} demo_* folder(s) already saved "
            f"under {run_dir}; new successful demos continue at demo_{_resume_dataset_kept_count:03d}."
        )
else:
    # Delete every previous OOD rollout run before starting this one, so only the
    # most recent run's videos/logs/tactile data stay on disk (each run is several GB).
    for old_run_dir in rollout_root.glob("openvla_oft_ood_closed_loop_*"):
        if old_run_dir.is_dir():
            shutil.rmtree(old_run_dir)

    run_dir = rollout_root / f"openvla_oft_ood_closed_loop_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    run_dir.mkdir(parents=True, exist_ok=True)

# -----------------------------------------------------------------------------
# Self-contained scene: primitive-only GelSight Mini housings on BOTH fingers +
# the dots-only cube (ported from TactileModel05_Duo.ipynb's scene cell). Builds
# fresh from panda.xml -- the same source Cell A uses -- rather than reloading or
# mutating Cell A's canonical model, so Cell A/B/C/D (and the D3 dataset the
# loaded checkpoint was trained on) are completely unaffected by anything in
# this cell. Only the fingers' attached geometry and the cube differ; the arm's
# 7 joints/actuators and the wrist camera are identical to Cell A's build (same
# body/joint/camera names), so every controller helper from Cell B keeps working
# unchanged once `model`/`data` below are rebound to this new scene.
# -----------------------------------------------------------------------------

# Defined explicitly (not inherited from Cell 3/A's globals) so this cell's
# scene build has no implicit ordering dependency on exactly which earlier
# cell last set SOURCE_PANDA_XML -- both already point here regardless.
SOURCE_PANDA_XML = PANDA_DIR / "panda.xml"

# 1.0 = the primary/original size the D3 demonstrations and the fine-tuned checkpoint
# were collected on (TACTILEMODEL03_CUBE_HALF_SIZE_M below, 70 mm cube). 2/3 reproduces
# the resized-cube OOD condition (T1/size). This notebook runs the T1 baseline at the
# primary size, with the in-distribution start position above -- no out-of-distribution
# factor at all -- to measure the reference success rate other T1 conditions compare to.
CUBE_SIZE_SCALE = 1.0
TACTILEMODEL03_CUBE_HALF_SIZE_M = 0.035
CUBE_HALF_SIZE_M = TACTILEMODEL03_CUBE_HALF_SIZE_M * CUBE_SIZE_SCALE
TABLE_TOP_SURFACE_Z_M = 0.050

# Kept at 1.0 (original brightness) -- this is the baseline condition, no
# out-of-distribution factor under test. See T1/lighting for the dimmed-lighting
# condition and T1/occlusion (still on disk under T1_duo/occlusion) for the plant.
LIGHT_DIM_FACTOR = 1.0

# Baseline condition: the occluder plant is disabled below (OCCLUDER_PLANT_ENABLED =
# False). This class/comment block is kept so re-enabling it (e.g. to reproduce
# T1_duo/occlusion) is a one-line change; it is otherwise inert here. The plant, when
# enabled, is a static, non-colliding potted-plant prop placed in the
# scene (added to _scene_xml_text below) so OpenVLA's actual camera view -- and the
# saved videos -- show the cube partly blocked, not a post-hoc image filter.
# contype="0" conaffinity="0" on every plant geom makes it a pure visual prop: MuJoCo
# still renders it in every camera, but it can never be touched or collided with, so
# it cannot affect the robot's motion at any point in the task, by construction --
# not just by careful placement.
#
# Shape and placement picked interactively via Cell P-preview: ten broad-leaf plants
# (small pot + a fan of flat leaf ellipsoids, far more pixel-efficient per unit size
# than an earlier round-bush version that looked like a hedge) were rendered beside
# the cube; "v6_fuller" (11 leaves, tilt=22 deg) was picked, then a second preview pass
# scaled that exact shape up and moved it into the table corner nearest the camera
# (opposite the robot's own base) instead of beside the cube -- it reads large purely
# from being much closer to the camera than the cube is, so it can sit far from the
# cube (safe for grasping) while still occluding a real chunk of it. "c1_gentle" was
# the final pick: segmentation-rendered at ~34% cube occlusion, 0.00% of the
# arm/gripper's own pixels lost, and 46cm from the cube center at the in-distribution
# start box's center (and everywhere else in the box, since the box barely moves the
# cube). Dimensions below are the exact v6 leaf/pot shape scaled by
# OCCLUDER_PLANT_SCALE, matching Cell P-preview's build_plant_xml precisely.
OCCLUDER_PLANT_ENABLED = False
OCCLUDER_PLANT_SCALE = 2.10
OCCLUDER_PLANT_POS_M = np.array([0.80, -0.35])  # pot (x, y); table corner nearest the camera
OCCLUDER_BASE_POT_RADIUS_M = 0.045
OCCLUDER_BASE_POT_HEIGHT_M = 0.060
OCCLUDER_BASE_LEAF_LENGTH_M = 0.110
OCCLUDER_BASE_LEAF_WIDTH_M = 0.048
OCCLUDER_BASE_LEAF_THICKNESS_M = 0.007
OCCLUDER_LEAF_COUNT = 11
OCCLUDER_LEAF_TILT_DEG = 22
OCCLUDER_LEAF_SEED = 6
OCCLUDER_LEAF_BIAS_SPREAD_DEG = 160

# Torsional friction coefficient (meters -- MuJoCo's torsional friction is a
# length scale: max resistive torque = this * normal force) shared by the
# gel pad and the cube/bump lattice. Diagnosed from a real OOD rollout: the
# cube held a firm, stable grip (23.3mm/finger, unchanged) for ~30 control
# steps while being lifted and carried, then visibly *rotated* out of the
# gripper over 2-3 steps once the policy's commanded closing target reached
# exactly 0.000 (fully closed) and kept the actuator's squeeze force
# growing with nothing left to compress against. The escape was a twist,
# not a straight pull-out, so torsional friction (previously 0.005, the
# same value sliding friction=1.0/rolling=0.0001 originally shipped with)
# is the material parameter most directly relevant -- being tuned upward
# here, one empirical step at a time, re-testing the real rollout after
# each change. A physically-motivated ceiling: for a Hertzian-like contact,
# torsional friction torque scales roughly as (contact patch radius) x
# (sliding friction) -- the gel pad's own half-dimensions are 9x13mm, so a
# value corresponding to the *entire* pad face being in full contact
# (radius ~13mm, i.e. ~0.013) is already a generous upper bound for what a
# real compliant gel could produce; going further stops being a "material"
# explanation and starts being a brute-force weld.
GEL_CUBE_TORSIONAL_FRICTION = 0.1

# Local-only cube pose used purely to give the built XML a valid, non
# -degenerate starting geometry -- randomize_start() below always teleports
# the cube to one of OOD_CUBE_STARTS before any rollout, so this value is
# never actually used as a task-relevant position. NOT the same as the
# notebook-wide START_BLOCK_POS global (still 0.085 m, the old 70 mm cube's
# height, required by other cells / the D3 training data's semantics).
_CELL_P_CUBE_BUILD_POS = np.array(
    [0.50, 0.00, TABLE_TOP_SURFACE_Z_M + CUBE_HALF_SIZE_M], dtype=np.float64
)

PANDA_WITH_GRASP_XML = PANDA_DIR / "openvla36_cellp_dual_fots_grasp.xml"
PRETEXTURE_SCENE_XML = PANDA_DIR / "openvla36_cellp_dual_fots_pretexture_scene.xml"
PANDA_XML = PANDA_DIR / "openvla36_cellp_dual_fots_scene.xml"

# Both fingers carry a GelSight Mini assembly (ported from TactileModel05_Duo.ipynb). Cell A only
# defines the right-finger names (the unsuffixed GELSIGHT_*_NAME globals); this cell defines
# both sides explicitly and re-points the unsuffixed names at the RIGHT sensor -- the same string
# values Cell A sets -- so Cell A/B and the required_globals check above keep working unchanged.
GELSIGHT_MOUNT_BODY_NAME_RIGHT = "right_finger"
GELSIGHT_BODY_NAME_RIGHT = "gelsight_mini_right"
GELSIGHT_SITE_NAME_RIGHT = "gelsight_mini_right_site"
GELSIGHT_GEL_PAD_GEOM_NAME_RIGHT = "gelsight_mini_right_gel_pad_geom"
GELSIGHT_TACTILE_CAMERA_NAME_RIGHT = "gelsight_mini_right_tactile_camera"

GELSIGHT_MOUNT_BODY_NAME_LEFT = "left_finger"
GELSIGHT_BODY_NAME_LEFT = "gelsight_mini_left"
GELSIGHT_SITE_NAME_LEFT = "gelsight_mini_left_site"
GELSIGHT_GEL_PAD_GEOM_NAME_LEFT = "gelsight_mini_left_gel_pad_geom"
GELSIGHT_TACTILE_CAMERA_NAME_LEFT = "gelsight_mini_left_tactile_camera"

GELSIGHT_MOUNT_BODY_NAME = GELSIGHT_MOUNT_BODY_NAME_RIGHT
GELSIGHT_BODY_NAME = GELSIGHT_BODY_NAME_RIGHT
GELSIGHT_SITE_NAME = GELSIGHT_SITE_NAME_RIGHT
GELSIGHT_GEL_PAD_GEOM_NAME = GELSIGHT_GEL_PAD_GEOM_NAME_RIGHT
GELSIGHT_TACTILE_CAMERA_NAME = GELSIGHT_TACTILE_CAMERA_NAME_RIGHT

# --- FOTS virtual tactile cameras (new in this cell; Cell A's native-sensor
# pipeline never needed one) -------------------------------------------------
# A camera mounted *behind* each gel pad (same body, opposite side from the
# object), looking through it toward the contact region -- the same concept
# FOTS-mujoco's `digit:camera` uses (see external/fots/NOTICE.md, and
# TactileModel04/05's scene cell for the full derivation of the numbers
# below, including why the standoff was tuned against this scene's
# near-clipping plane). Both sensors' shell/gel/site geoms share the same three
# groups: each tactile camera's ~50 degree frustum only ever frames its own gel
# pad, never the other finger, so there is no reason to tell them apart camera-side.
SENSOR_SHELL_GEOM_GROUP = 2  # existing group used by mount/standoffs/case/label/cable
GEL_PAD_GEOM_GROUP = 3       # existing group used by the gel pad geom itself
GELSIGHT_SITE_GROUP = 4      # existing group used by the gel pad's visualization site
FOTS_TACTILE_CAM_LOCAL_Y_M = 0.022
FOTS_TACTILE_CAM_FOVY_DEG = 50.0

# Reference/object site used only to compute the object's pose relative to
# the sensor for FOTS's marker-flow model (mirrors FOTS-mujoco's
# "digit:site" / "object1:site" pair).
OBJECT_REF_SITE_NAME = "red_block_center_site"

# Fine, physical verification texture for the cube -- "dots" only (the other
# patterns TactileModel04 tested were only for comparing tactile signal
# across shapes, not needed for this OOD policy test). Grid SPACING scales
# down with the cube so the lattice still fits the smaller face; bump RADIUS
# stays at its original absolute size -- TactileModel04 found that scaling
# radius down too made contact detection unreliable (see that notebook's
# cell 2 for the full explanation).
CUBE_TEXTURE_GRID_SIZE = 7
CUBE_TEXTURE_GRID_HALF_SPAN_M = 0.027 * CUBE_SIZE_SCALE
CUBE_TEXTURE_BUMP_RADIUS_M = 0.00150
CUBE_TEXTURE_GEOM_PREFIX = "red_block_tactile_bump_"
_CUBE_TEXTURE_FACE_SPECS = (
    ("px", 0, +1, 1, 2),
    ("nx", 0, -1, 1, 2),
    ("py", 1, +1, 0, 2),
    ("ny", 1, -1, 0, 2),
    ("pz", 2, +1, 0, 1),
    ("nz", 2, -1, 0, 1),
)


def _find_body(root, body_name):
    for body in root.iter("body"):
        if body.get("name") == body_name:
            return body
    raise ValueError(f"Could not find body named {body_name!r}")


def _remove_named_child_body(parent, child_name):
    for child in list(parent):
        if child.tag == "body" and child.get("name") == child_name:
            parent.remove(child)


def _strip_stock_finger_geoms(finger_body):
    """Remove the stock collision mesh/fingertip-pad boxes AND the stock visual meshes
    (finger_0 off-white body, finger_1 black rubber-pad tip) from `finger_body`. With a
    GelSight sensor's own primitives added in their place, those stock meshes would
    render/collide at roughly the same location and either visually poke through the
    sensor or double up its contact surface, instead of the sensor being the sole outer
    contact layer. Shared by both fingers so left and right end up with identical stripped
    geometry -- from here on neither finger has any mesh geom left, only MuJoCo primitives."""
    removed_contact, removed_visual = [], []
    for child in list(finger_body):
        if child.tag != "geom":
            continue
        geom_class = child.get("class", "")
        is_stock_contact = geom_class == "collision" or geom_class.startswith("fingertip_pad_collision")
        is_stock_visual = geom_class == "visual"
        if is_stock_contact:
            removed_contact.append(child.get("name", geom_class))
            finger_body.remove(child)
        elif is_stock_visual:
            removed_visual.append(child.get("mesh", geom_class))
            finger_body.remove(child)
    return removed_contact, removed_visual


def _add_finger_carrier_geom(finger_body, side):
    """Primitive-only stand-in for the removed stock finger body: a plain box connecting the
    hand's finger mount to where the GelSight sensor starts (sensor_body mounts at local
    z=0.045 below), dark plastic-like color to match the rest of the (unmodified) gripper and
    stay visually distinct from the lighter sensor. Identical size/pos on both fingers: the
    right_finger body's own 180 degree about-Z quat (see the header) is what mirrors it
    correctly, not a sign flip on the numbers themselves. z spans finger-local [0, 0.026], up
    to just below where the sensor case starts; y is matched to the case's own depth (finger
    -local [0.005, 0.015]) -- NOT centered on the finger's own y=0 origin, which would extend
    the carrier past the case *and* the gel pad toward the object."""
    ET.SubElement(
        finger_body,
        "geom",
        name=f"{side}_finger_carrier_geom",
        type="box",
        size="0.010 0.005 0.013",
        pos="0 0.010 0.013",
        rgba="0.12 0.12 0.13 1",
        contype="0",
        conaffinity="0",
        group="2",
    )


# Sensor shell colors: light gray/silver, deliberately far lighter than the
# gripper's own dark plastic, so the GelSight housings read as visually
# distinct from the gripper at a glance.
SENSOR_SHELL_RGBA = "0.82 0.83 0.87 1"
SENSOR_STANDOFF_RGBA = "0.72 0.73 0.78 1"


def _add_gelsight_sensor(finger_body, side):
    """Build one GelSight Mini sensor assembly -- mount plate, standoffs, case, gel pad (the
    actual contact geom), label, cable, a visualization site, and a FOTS virtual tactile
    camera -- entirely from MuJoCo primitives (boxes/capsules), and append it to `finger_body`.

    Called once for each of "right"/"left". Every local pos/size number below is IDENTICAL for
    both calls -- the mirroring that makes the two assemblies face each other comes entirely
    from right_finger's extra 180 degree about-Z quat in panda.xml relative to left_finger.
    """
    body_name = f"gelsight_mini_{side}"
    gel_pad_geom_name = f"gelsight_mini_{side}_gel_pad_geom"
    site_name = f"gelsight_mini_{side}_site"
    camera_name = f"gelsight_mini_{side}_tactile_camera"

    sensor_body = ET.Element("body", name=body_name, pos="0.0 0.01 0.045")
    ET.SubElement(sensor_body, "inertial", mass="0.025", pos="0 0 0", diaginertia="3.6e-6 4.8e-6 1.8e-6")
    # Positioned so its front face sits *behind* the gel pad's own front face, framing the gel
    # like a bezel instead of occluding it -- the gel is the true outermost surface.
    ET.SubElement(
        sensor_body,
        "geom",
        name=f"gelsight_mini_{side}_mount_plate_geom",
        type="box",
        size="0.013 0.0012 0.019",
        pos="0 -0.006 0",
        rgba=SENSOR_SHELL_RGBA,
        contype="0",
        conaffinity="0",
        group="2",
    )
    ET.SubElement(
        sensor_body,
        "geom",
        name=f"gelsight_mini_{side}_standoff_top_geom",
        type="capsule",
        fromto="0.007 -0.006 0.012 0.007 -0.005 0.012",
        size="0.0016",
        rgba=SENSOR_STANDOFF_RGBA,
        contype="0",
        conaffinity="0",
        group="2",
    )
    ET.SubElement(
        sensor_body,
        "geom",
        name=f"gelsight_mini_{side}_standoff_bottom_geom",
        type="capsule",
        fromto="0.007 -0.006 -0.012 0.007 -0.005 -0.012",
        size="0.0016",
        rgba=SENSOR_STANDOFF_RGBA,
        contype="0",
        conaffinity="0",
        group="2",
    )
    ET.SubElement(
        sensor_body,
        "geom",
        name=f"gelsight_mini_{side}_case_geom",
        type="box",
        size="0.012 0.005 0.018",
        rgba=SENSOR_SHELL_RGBA,
        contype="0",
        conaffinity="0",
        group="2",
    )
    ET.SubElement(
        sensor_body,
        "geom",
        name=gel_pad_geom_name,
        type="box",
        size="0.009 0.0012 0.013",
        pos="0 -0.0072 0",
        rgba="0.0 0.75 0.95 0.70",
        contype="1",
        conaffinity="1",
        condim="4",
        friction=f"1.0 {GEL_CUBE_TORSIONAL_FRICTION:.17g} 0.0001",
        # Deliberately SOFT contact material (paired with the rigid cube material below): a real
        # GelSight gel is a compliant silicone layer that conforms around whatever presses into
        # it, while the probed object is effectively rigid by comparison. Empirically tuned in
        # TactileModel04.ipynb to fix a single-point-contact bug where the gel pad's own solver
        # params were, if anything, stiffer than the cube's -- a rigid-feeling flat pad meeting a
        # field of coplanar bumps pivoted on whichever bump touched first instead of conforming,
        # so contact stopped building after only one bump.
        solimp="0.50 0.85 0.0035",
        solref="0.045 1",
        group="3",
    )
    ET.SubElement(
        sensor_body,
        "geom",
        name=f"gelsight_mini_{side}_label_geom",
        type="box",
        size="0.008 0.0010 0.0025",
        pos="0 -0.0074 0.015",
        rgba="0.92 0.92 0.88 1",
        contype="0",
        conaffinity="0",
        group="2",
    )
    ET.SubElement(
        sensor_body,
        "geom",
        name=f"gelsight_mini_{side}_cable_geom",
        type="capsule",
        fromto="0 0.006 -0.016 0 0.018 -0.038",
        size="0.0018",
        rgba="0.55 0.56 0.60 1",
        contype="0",
        conaffinity="0",
        group="2",
    )
    ET.SubElement(
        sensor_body,
        "site",
        name=site_name,
        type="box",
        size="0.009 0.0010 0.013",
        pos="0 -0.0086 0",
        rgba="0.0 0.9 1.0 0.35",
        group="4",
    )
    # FOTS virtual tactile camera: behind the gel pad, looking toward -y (through the gel, toward
    # the object). xyaxes chosen so image "up" = sensor body local +z and the view direction
    # (-Z_camera) = local -y.
    ET.SubElement(
        sensor_body,
        "camera",
        name=camera_name,
        mode="fixed",
        pos=f"0 {FOTS_TACTILE_CAM_LOCAL_Y_M:.17g} 0",
        xyaxes="-1 0 0 0 0 1",
        fovy=f"{FOTS_TACTILE_CAM_FOVY_DEG:.17g}",
    )
    finger_body.append(sensor_body)
    return {
        "body_name": body_name,
        "gel_pad_geom_name": gel_pad_geom_name,
        "site_name": site_name,
        "camera_name": camera_name,
    }


panda_tree = ET.parse(SOURCE_PANDA_XML)
panda_root = panda_tree.getroot()
hand_body = _find_body(panda_root, "hand")
right_finger_body = _find_body(panda_root, GELSIGHT_MOUNT_BODY_NAME_RIGHT)
left_finger_body = _find_body(panda_root, GELSIGHT_MOUNT_BODY_NAME_LEFT)

_remove_named_child_body(hand_body, "agent_camera_mount")
_remove_named_child_body(right_finger_body, GELSIGHT_BODY_NAME_RIGHT)
_remove_named_child_body(left_finger_body, GELSIGHT_BODY_NAME_LEFT)

# Remove the stock finger collision pads AND visual meshes on BOTH fingers and replace that
# contact surface with a small, soft, high-friction gel pad (see _add_gelsight_sensor).
removed_right_contact_geoms, removed_right_visual_geoms = _strip_stock_finger_geoms(right_finger_body)
removed_left_contact_geoms, removed_left_visual_geoms = _strip_stock_finger_geoms(left_finger_body)

_add_finger_carrier_geom(right_finger_body, "right")
_add_finger_carrier_geom(left_finger_body, "left")

mount_body = ET.Element("body", name="agent_camera_mount", pos="0.08 0 -0.12")
ET.SubElement(
    mount_body,
    "geom",
    name="agent_camera_boom_geom",
    type="capsule",
    fromto="0 0 0 -0.08 0 0.12",
    size="0.008",
    rgba="0.02 0.02 0.02 1",
    contype="0",
    conaffinity="0",
    group="2",
)
ET.SubElement(
    mount_body,
    "geom",
    name="agent_camera_body_geom",
    type="box",
    size="0.020 0.016 0.012",
    rgba="0.02 0.02 0.02 1",
    contype="0",
    conaffinity="0",
    group="2",
)
ET.SubElement(
    mount_body,
    "camera",
    name="agent_wrist_camera",
    mode="fixed",
    fovy="82",
    xyaxes="-0.000007 -1.000000 0.000034 -0.976592 -0.000000 -0.215100",
)

right_sensor_names = _add_gelsight_sensor(right_finger_body, "right")
left_sensor_names = _add_gelsight_sensor(left_finger_body, "left")
hand_body.insert(0, mount_body)

ET.indent(panda_tree, space="  ")
panda_tree.write(PANDA_WITH_GRASP_XML, encoding="unicode")


# Deliberately STIFF/near-rigid contact material for the cube and its bump
# lattice, paired with the softened gel pad above -- a real probed object
# doesn't itself deform, only the sensor's gel does. This is the compiled
# -model source both the cube's own core geom and every bump sphere inherit
# via `core_properties` below, so the same rigid-object/soft-gel split
# applies uniformly across the whole cube.
CUBE_MATERIAL_SOLREF = "0.006 1"
CUBE_MATERIAL_SOLIMP = "0.97 0.99 0.0004"


def _cube_body_xml(bump_geoms_xml=""):
    pos_str = f"{_CELL_P_CUBE_BUILD_POS[0]:.17g} {_CELL_P_CUBE_BUILD_POS[1]:.17g} {_CELL_P_CUBE_BUILD_POS[2]:.17g}"
    size_str = f"{CUBE_HALF_SIZE_M:.17g} {CUBE_HALF_SIZE_M:.17g} {CUBE_HALF_SIZE_M:.17g}"
    return f"""    <body name="red_block" pos="{pos_str}">
      <freejoint name="red_block_freejoint"/>
      <geom name="red_block_geom" type="box" size="{size_str}" material="red_block_mat" mass="0.05" friction="1 {GEL_CUBE_TORSIONAL_FRICTION:.17g} 0.0001" solref="{CUBE_MATERIAL_SOLREF}" solimp="{CUBE_MATERIAL_SOLIMP}"/>
      <site name="{OBJECT_REF_SITE_NAME}" type="sphere" size="0.001" rgba="0 0 0 0"/>
{bump_geoms_xml}    </body>
"""


def _leafy_plant_xml(name, px, py, pot_r, pot_h, n_leaves, leaf_len, leaf_width, leaf_thick,
                      tilt_deg, seed, bias_angle, bias_spread_deg,
                      pot_material="occluder_pot_mat", leaf_material="occluder_leaf_mat"):
    """A small pot + a fan of flat leaf ellipsoids, biased toward `bias_angle` (radians)
    instead of spread evenly all the way around, the way a real plant's foliage tends to
    drape toward a nearby object. Verbatim port of Cell P-preview's leafy_plant_xml, only
    switched from inline rgba to this cell's existing occluder_pot_mat/occluder_leaf_mat
    materials -- geometry and RNG draw order are otherwise identical."""
    rng = np.random.default_rng(seed)
    pot_z = pot_h / 2
    base_z = pot_h * 0.85
    geoms = [f'<geom name="{name}_pot" type="cylinder" size="{pot_r:.4f} {pot_h/2:.4f}" '
             f'pos="0 0 {pot_z:.4f}" material="{pot_material}" contype="0" conaffinity="0"/>']
    spread = np.radians(bias_spread_deg)
    thetas = [bias_angle + rng.uniform(-spread / 2, spread / 2) for _ in range(n_leaves)]
    for i, theta in enumerate(thetas):
        tilt = np.radians(tilt_deg + rng.uniform(-8, 8))
        length = leaf_len * rng.uniform(0.85, 1.15)
        width = leaf_width * rng.uniform(0.85, 1.15)
        d = np.array([np.cos(theta) * np.cos(tilt), np.sin(theta) * np.cos(tilt), np.sin(tilt)])
        center = np.array([0, 0, base_z]) + d * (length / 2)
        y = np.array([-np.sin(theta), np.cos(theta), 0])
        geoms.append(
            f'<geom name="{name}_leaf{i}" type="ellipsoid" '
            f'size="{length/2:.4f} {width/2:.4f} {leaf_thick/2:.4f}" '
            f'pos="{center[0]:.4f} {center[1]:.4f} {center[2]:.4f}" '
            f'xyaxes="{d[0]:.4f} {d[1]:.4f} {d[2]:.4f}  {y[0]:.4f} {y[1]:.4f} {y[2]:.4f}" '
            f'material="{leaf_material}" contype="0" conaffinity="0"/>'
        )
    return f'<body name="{name}" pos="{px:.4f} {py:.4f} {TABLE_TOP_SURFACE_Z_M:.4f}">\n  ' + "\n  ".join(geoms) + "\n</body>\n"


def _occluder_plant_xml():
    if not OCCLUDER_PLANT_ENABLED:
        return ""
    k = OCCLUDER_PLANT_SCALE
    px, py = float(OCCLUDER_PLANT_POS_M[0]), float(OCCLUDER_PLANT_POS_M[1])
    nominal_cube_x = 0.5 * (CUBE_X_RANGE[0] + CUBE_X_RANGE[1])
    nominal_cube_y = 0.5 * (CUBE_Y_RANGE[0] + CUBE_Y_RANGE[1])
    bearing = float(np.arctan2(nominal_cube_y - py, nominal_cube_x - px))
    return _leafy_plant_xml(
        "occluder_plant", px, py,
        OCCLUDER_BASE_POT_RADIUS_M * k, OCCLUDER_BASE_POT_HEIGHT_M * k, OCCLUDER_LEAF_COUNT,
        OCCLUDER_BASE_LEAF_LENGTH_M * k, OCCLUDER_BASE_LEAF_WIDTH_M * k, OCCLUDER_BASE_LEAF_THICKNESS_M * k,
        OCCLUDER_LEAF_TILT_DEG, OCCLUDER_LEAF_SEED,
        bias_angle=bearing, bias_spread_deg=OCCLUDER_LEAF_BIAS_SPREAD_DEG,
    )


def _scene_xml_text(model_name, cube_body_xml):
    return f"""<mujoco model="{model_name}">
  <include file="{PANDA_WITH_GRASP_XML.name}"/>

  <statistic center="0.50 0 0.35" extent="1.1"/>

  <visual>
    <headlight diffuse="0.6 0.6 0.6" ambient="0.3 0.3 0.3" specular="0 0 0"/>
    <rgba haze="0.15 0.25 0.35 1"/>
    <global azimuth="120" elevation="-20" offwidth="1920" offheight="1080"/>
  </visual>

  <asset>
    <texture type="skybox" builtin="gradient" rgb1="0.3 0.5 0.7" rgb2="0 0 0" width="512" height="3072"/>
    <texture type="2d" name="groundplane" builtin="checker" mark="edge" rgb1="0.2 0.3 0.4" rgb2="0.1 0.2 0.3" markrgb="0.8 0.8 0.8" width="300" height="300"/>
    <material name="groundplane" texture="groundplane" texuniform="true" texrepeat="5 5" reflectance="0.2"/>
    <material name="table_mat" rgba="0.48 0.38 0.26 1"/>
    <material name="red_block_mat" rgba="1 0.03 0.02 1"/>
    <material name="occluder_pot_mat" rgba="0.62 0.42 0.27 1"/>
    <material name="occluder_leaf_mat" rgba="0.16 0.45 0.18 1"/>
  </asset>

  <worldbody>
    <light pos="0 0 1.5" dir="0 0 -1" directional="true"/>
    <geom name="floor" size="0 0 0.05" type="plane" material="groundplane"/>
    <body name="table" pos="0.55 0 0.025">
      <geom name="table_top" type="box" size="0.35 0.45 0.025" material="table_mat" friction="1 0.005 0.0001"/>
    </body>
{cube_body_xml}{_occluder_plant_xml()}  </worldbody>
</mujoco>
"""


# Pass 1: compile the scene with a bare cube (no bumps yet) purely to read
# back MuJoCo's compiled contact defaults (friction/solref/solimp/...) for
# red_block_geom, so every bump sphere can be given exactly those same
# contact properties instead of guessed values.
PRETEXTURE_SCENE_XML.write_text(
    _scene_xml_text("openvla36 cellP dual pretexture scene", _cube_body_xml()),
    encoding="utf-8",
)
pretexture_model = mujoco.MjModel.from_xml_path(str(PRETEXTURE_SCENE_XML))
_pretexture_cube_body_id = mujoco.mj_name2id(pretexture_model, mujoco.mjtObj.mjOBJ_BODY, "red_block")
_pretexture_core_geom_id = mujoco.mj_name2id(pretexture_model, mujoco.mjtObj.mjOBJ_GEOM, "red_block_geom")
assert _pretexture_cube_body_id >= 0 and _pretexture_core_geom_id >= 0
core_half_size = np.asarray(pretexture_model.geom_size[_pretexture_core_geom_id, :3], dtype=np.float64).copy()
assert CUBE_TEXTURE_GRID_HALF_SPAN_M + CUBE_TEXTURE_BUMP_RADIUS_M < float(np.min(core_half_size))
core_properties = {
    "friction": pretexture_model.geom_friction[_pretexture_core_geom_id].copy(),
    "solref": pretexture_model.geom_solref[_pretexture_core_geom_id].copy(),
    "solimp": pretexture_model.geom_solimp[_pretexture_core_geom_id].copy(),
    "solmix": float(pretexture_model.geom_solmix[_pretexture_core_geom_id]),
    "condim": int(pretexture_model.geom_condim[_pretexture_core_geom_id]),
    "contype": int(pretexture_model.geom_contype[_pretexture_core_geom_id]),
    "conaffinity": int(pretexture_model.geom_conaffinity[_pretexture_core_geom_id]),
    "priority": int(pretexture_model.geom_priority[_pretexture_core_geom_id]),
    "margin": float(pretexture_model.geom_margin[_pretexture_core_geom_id]),
    "gap": float(pretexture_model.geom_gap[_pretexture_core_geom_id]),
    "group": int(pretexture_model.geom_group[_pretexture_core_geom_id]),
    "matid": int(pretexture_model.geom_matid[_pretexture_core_geom_id]),
}
material_name = None
if core_properties["matid"] >= 0:
    material_name = mujoco.mj_id2name(pretexture_model, mujoco.mjtObj.mjOBJ_MATERIAL, core_properties["matid"])
    assert material_name is not None


def _texture_number_string(values):
    return " ".join(f"{float(value):.17g}" for value in np.asarray(values).reshape(-1))


def _texture_file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


common_bump_attributes = {
    "type": "sphere",
    "size": f"{CUBE_TEXTURE_BUMP_RADIUS_M:.17g}",
    "mass": "0",
    "friction": _texture_number_string(core_properties["friction"]),
    "solref": _texture_number_string(core_properties["solref"]),
    "solimp": _texture_number_string(core_properties["solimp"]),
    "solmix": f"{core_properties['solmix']:.17g}",
    "condim": str(core_properties["condim"]),
    "contype": str(core_properties["contype"]),
    "conaffinity": str(core_properties["conaffinity"]),
    "priority": str(core_properties["priority"]),
    "margin": f"{core_properties['margin']:.17g}",
    "gap": f"{core_properties['gap']:.17g}",
    "group": str(core_properties["group"]),
}
if material_name is not None:
    common_bump_attributes["material"] = material_name
common_bump_attributes["rgba"] = "0.02 0.02 0.02 1"

lattice = np.linspace(
    -CUBE_TEXTURE_GRID_HALF_SPAN_M, CUBE_TEXTURE_GRID_HALF_SPAN_M, CUBE_TEXTURE_GRID_SIZE, dtype=np.float64
)
bump_lines = []
bump_positions = {}
face_counts = {}
for face_name, normal_axis, sign, row_axis, column_axis in _CUBE_TEXTURE_FACE_SPECS:
    face_counts[face_name] = 0
    for row, row_value in enumerate(lattice):
        for column, column_value in enumerate(lattice):
            local_position = np.zeros(3, dtype=np.float64)
            local_position[normal_axis] = sign * core_half_size[normal_axis]
            local_position[row_axis] = row_value
            local_position[column_axis] = column_value
            geom_name = f"{CUBE_TEXTURE_GEOM_PREFIX}{face_name}_r{row:02d}_c{column:02d}"
            attrs = dict(common_bump_attributes)
            attrs.update(name=geom_name, pos=_texture_number_string(local_position))
            attr_str = " ".join(f'{key}="{value}"' for key, value in attrs.items())
            bump_lines.append(f"      <geom {attr_str}/>\n")
            bump_positions[geom_name] = local_position
            face_counts[face_name] += 1

assert len(bump_positions) == len(_CUBE_TEXTURE_FACE_SPECS) * CUBE_TEXTURE_GRID_SIZE ** 2
assert set(face_counts.values()) == {CUBE_TEXTURE_GRID_SIZE ** 2}

# Pass 2: the real scene, cube built with its bump lattice baked in from the
# start -- no live-model mutation, no reload-and-diff-validate dance.
PANDA_XML.write_text(
    _scene_xml_text("openvla36 cellP dual fots grasp scene", _cube_body_xml("".join(bump_lines))),
    encoding="utf-8",
)

model = mujoco.MjModel.from_xml_path(str(PANDA_XML))
data = mujoco.MjData(model)
mujoco.mj_forward(model, data)

# Dim every light source uniformly (scene light + the camera-attached headlight)
# so every camera -- policy, world, wrist, grasp diagnostics -- renders the same
# darker scene automatically, with no other rendering code changed.
model.light_diffuse[:] *= LIGHT_DIM_FACTOR
model.light_ambient[:] *= LIGHT_DIM_FACTOR
model.light_specular[:] *= LIGHT_DIM_FACTOR
model.vis.headlight.ambient[:] *= LIGHT_DIM_FACTOR
model.vis.headlight.diffuse[:] *= LIGHT_DIM_FACTOR
model.vis.headlight.specular[:] *= LIGHT_DIM_FACTOR

new_cube_body_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, "red_block")
new_core_geom_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_GEOM, "red_block_geom")
for geom_name, expected_position in bump_positions.items():
    geom_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_GEOM, geom_name)
    assert geom_id >= 0
    assert int(model.geom_type[geom_id]) == int(mujoco.mjtGeom.mjGEOM_SPHERE)
    assert int(model.geom_bodyid[geom_id]) == new_cube_body_id
    assert np.isclose(model.geom_size[geom_id, 0], CUBE_TEXTURE_BUMP_RADIUS_M, rtol=0.0, atol=1e-12)
    assert np.allclose(model.geom_pos[geom_id], expected_position, rtol=0.0, atol=1e-12)

# Mirroring sanity check: express both gel pads' world positions in the hand's own local frame
# and confirm they are actual mirror images of each other (equal x/z, opposite-signed y) --
# catches a mirroring mistake here instead of silently shipping a "sensor" that never touches
# anything.
_hand_body_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, "hand")
_right_gel_geom_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_GEOM, right_sensor_names["gel_pad_geom_name"])
_left_gel_geom_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_GEOM, left_sensor_names["gel_pad_geom_name"])
_hand_xmat = data.xmat[_hand_body_id].reshape(3, 3)
_right_gel_hand_local = _hand_xmat.T @ (data.geom_xpos[_right_gel_geom_id] - data.xpos[_hand_body_id])
_left_gel_hand_local = _hand_xmat.T @ (data.geom_xpos[_left_gel_geom_id] - data.xpos[_hand_body_id])
assert np.allclose(_right_gel_hand_local[[0, 2]], _left_gel_hand_local[[0, 2]], atol=1e-9), (
    f"Gel pads are not mirrored in x/z: right={_right_gel_hand_local} left={_left_gel_hand_local}"
)
assert _right_gel_hand_local[1] < 0.0 < _left_gel_hand_local[1], (
    f"Gel pads are not on opposite sides of the hand's y=0 plane: "
    f"right_y={_right_gel_hand_local[1]:.5f} left_y={_left_gel_hand_local[1]:.5f}"
)

CUBE_SURFACE_TEXTURE = {
    "geom_prefix": CUBE_TEXTURE_GEOM_PREFIX,
    "grid_size_per_face": int(CUBE_TEXTURE_GRID_SIZE),
    "grid_coordinates_m": lattice.tolist(),
    "face_names": [spec[0] for spec in _CUBE_TEXTURE_FACE_SPECS],
    "face_counts": face_counts,
    "bump_radius_m": float(CUBE_TEXTURE_BUMP_RADIUS_M),
    "hemisphere_protrusion_m": float(CUBE_TEXTURE_BUMP_RADIUS_M),
    "core_half_size_m": core_half_size.tolist(),
    "total_bump_geoms": int(len(bump_positions)),
    "added_inertial_mass_kg": 0.0,
    "cube_body_mass_kg_before": float(pretexture_model.body_mass[_pretexture_cube_body_id]),
    "cube_body_mass_kg_after": float(model.body_mass[new_cube_body_id]),
    "textured_xml": str(PANDA_XML),
    "textured_xml_sha256": _texture_file_sha256(PANDA_XML),
    "source_xml": str(PRETEXTURE_SCENE_XML),
    "source_xml_sha256": _texture_file_sha256(PRETEXTURE_SCENE_XML),
}

# OOD_CUBE_STARTS: XY corners outside the D3 training envelope (unchanged),
# z re-derived for the new cube half-size (table top + half-size), matching
# the same "core box center, not bump tip" convention randomize_start()
# below expects (it separately adds CUBE_TEXTURE_BUMP_RADIUS_M).
OOD_CUBE_STARTS = np.concatenate(
    [
        OOD_CUBE_STARTS_XY[:, :2],
        np.full((OOD_CUBE_STARTS_XY.shape[0], 1), TABLE_TOP_SURFACE_Z_M + CUBE_HALF_SIZE_M),
    ],
    axis=1,
)

IN_DISTRIBUTION_CUBE_STARTS = np.concatenate(
    [
        IN_DISTRIBUTION_CUBE_STARTS_XY,
        np.full((IN_DISTRIBUTION_CUBE_STARTS_XY.shape[0], 1), TABLE_TOP_SURFACE_Z_M + CUBE_HALF_SIZE_M),
    ],
    axis=1,
)


def _refresh_scene_bindings_after_texture():
    """(Re)bind every scene-ID/address global to the current `model`/`data`. Binds BOTH
    sensors' IDs (explicit _RIGHT/_LEFT globals) plus unsuffixed aliases pointing at the right
    sensor, the same names Cell A/B and the required_globals check use."""
    global body_ids, site_ids, camera_id
    global HAND_BODY_ID, LEFT_FINGER_BODY_ID, RIGHT_FINGER_BODY_ID
    global GELSIGHT_BODY_ID_RIGHT, GELSIGHT_SITE_ID_RIGHT, GELSIGHT_GEL_PAD_GEOM_ID_RIGHT, FOTS_TACTILE_CAM_ID_RIGHT
    global GELSIGHT_BODY_ID_LEFT, GELSIGHT_SITE_ID_LEFT, GELSIGHT_GEL_PAD_GEOM_ID_LEFT, FOTS_TACTILE_CAM_ID_LEFT
    global GELSIGHT_BODY_ID, GELSIGHT_SITE_ID, GELSIGHT_GEL_PAD_GEOM_ID, FOTS_TACTILE_CAM_ID
    global OBJECT_REF_SITE_ID
    global BLOCK_BODY_ID, BLOCK_JOINT_ID, BLOCK_QPOS_ADR
    global ARM_JOINT_IDS, ARM_QPOS_ADR, ARM_DOF_ADR
    global FINGER_JOINT_IDS, FINGER_QPOS_ADR

    body_ids = {
        "hand": require_mj_id(mujoco.mjtObj.mjOBJ_BODY, "hand"),
        "left_finger": require_mj_id(mujoco.mjtObj.mjOBJ_BODY, "left_finger"),
        "right_finger": require_mj_id(mujoco.mjtObj.mjOBJ_BODY, "right_finger"),
        "red_block": require_mj_id(mujoco.mjtObj.mjOBJ_BODY, "red_block"),
        "agent_camera_mount": require_mj_id(
            mujoco.mjtObj.mjOBJ_BODY, "agent_camera_mount"
        ),
        GELSIGHT_BODY_NAME_RIGHT: require_mj_id(mujoco.mjtObj.mjOBJ_BODY, GELSIGHT_BODY_NAME_RIGHT),
        GELSIGHT_BODY_NAME_LEFT: require_mj_id(mujoco.mjtObj.mjOBJ_BODY, GELSIGHT_BODY_NAME_LEFT),
    }
    site_ids = {
        GELSIGHT_SITE_NAME_RIGHT: require_mj_id(mujoco.mjtObj.mjOBJ_SITE, GELSIGHT_SITE_NAME_RIGHT),
        GELSIGHT_SITE_NAME_LEFT: require_mj_id(mujoco.mjtObj.mjOBJ_SITE, GELSIGHT_SITE_NAME_LEFT),
        OBJECT_REF_SITE_NAME: require_mj_id(mujoco.mjtObj.mjOBJ_SITE, OBJECT_REF_SITE_NAME),
    }
    camera_id = require_mj_id(mujoco.mjtObj.mjOBJ_CAMERA, AGENT_CAMERA_NAME)

    HAND_BODY_ID = body_ids["hand"]
    LEFT_FINGER_BODY_ID = body_ids["left_finger"]
    RIGHT_FINGER_BODY_ID = body_ids["right_finger"]

    GELSIGHT_BODY_ID_RIGHT = body_ids[GELSIGHT_BODY_NAME_RIGHT]
    GELSIGHT_SITE_ID_RIGHT = site_ids[GELSIGHT_SITE_NAME_RIGHT]
    GELSIGHT_GEL_PAD_GEOM_ID_RIGHT = require_mj_id(mujoco.mjtObj.mjOBJ_GEOM, GELSIGHT_GEL_PAD_GEOM_NAME_RIGHT)
    FOTS_TACTILE_CAM_ID_RIGHT = require_mj_id(mujoco.mjtObj.mjOBJ_CAMERA, GELSIGHT_TACTILE_CAMERA_NAME_RIGHT)

    GELSIGHT_BODY_ID_LEFT = body_ids[GELSIGHT_BODY_NAME_LEFT]
    GELSIGHT_SITE_ID_LEFT = site_ids[GELSIGHT_SITE_NAME_LEFT]
    GELSIGHT_GEL_PAD_GEOM_ID_LEFT = require_mj_id(mujoco.mjtObj.mjOBJ_GEOM, GELSIGHT_GEL_PAD_GEOM_NAME_LEFT)
    FOTS_TACTILE_CAM_ID_LEFT = require_mj_id(mujoco.mjtObj.mjOBJ_CAMERA, GELSIGHT_TACTILE_CAMERA_NAME_LEFT)

    # Unsuffixed aliases (right side).
    GELSIGHT_BODY_ID = GELSIGHT_BODY_ID_RIGHT
    GELSIGHT_SITE_ID = GELSIGHT_SITE_ID_RIGHT
    GELSIGHT_GEL_PAD_GEOM_ID = GELSIGHT_GEL_PAD_GEOM_ID_RIGHT
    FOTS_TACTILE_CAM_ID = FOTS_TACTILE_CAM_ID_RIGHT

    OBJECT_REF_SITE_ID = site_ids[OBJECT_REF_SITE_NAME]
    BLOCK_BODY_ID = body_ids["red_block"]
    BLOCK_JOINT_ID = require_mj_id(
        mujoco.mjtObj.mjOBJ_JOINT, "red_block_freejoint"
    )
    BLOCK_QPOS_ADR = int(model.jnt_qposadr[BLOCK_JOINT_ID])

    ARM_JOINT_IDS = [
        require_mj_id(mujoco.mjtObj.mjOBJ_JOINT, name)
        for name in ARM_JOINT_NAMES
    ]
    ARM_QPOS_ADR = [int(model.jnt_qposadr[joint_id]) for joint_id in ARM_JOINT_IDS]
    ARM_DOF_ADR = [int(model.jnt_dofadr[joint_id]) for joint_id in ARM_JOINT_IDS]
    FINGER_JOINT_IDS = [
        require_mj_id(mujoco.mjtObj.mjOBJ_JOINT, name)
        for name in FINGER_JOINT_NAMES
    ]
    FINGER_QPOS_ADR = [
        int(model.jnt_qposadr[joint_id]) for joint_id in FINGER_JOINT_IDS
    ]

    assert HAND_BODY_ID == require_mj_id(mujoco.mjtObj.mjOBJ_BODY, "hand")
    assert BLOCK_BODY_ID == require_mj_id(mujoco.mjtObj.mjOBJ_BODY, "red_block")
    assert GELSIGHT_BODY_ID_RIGHT == require_mj_id(mujoco.mjtObj.mjOBJ_BODY, GELSIGHT_BODY_NAME_RIGHT)
    assert GELSIGHT_BODY_ID_LEFT == require_mj_id(mujoco.mjtObj.mjOBJ_BODY, GELSIGHT_BODY_NAME_LEFT)
    assert GELSIGHT_BODY_ID_RIGHT != GELSIGHT_BODY_ID_LEFT
    assert camera_id == require_mj_id(
        mujoco.mjtObj.mjOBJ_CAMERA, AGENT_CAMERA_NAME
    )


_refresh_scene_bindings_after_texture()

print("Cell-P dual-GelSight scene rebuilt (self-contained, independent of Cell A):", PANDA_XML)
print("Right-finger stock contact geoms removed:", removed_right_contact_geoms)
print("Right-finger stock visual meshes removed:", removed_right_visual_geoms, "(replaced with primitive right_finger_carrier_geom)")
print("Left-finger stock contact geoms removed:", removed_left_contact_geoms)
print("Left-finger stock visual meshes removed:", removed_left_visual_geoms, "(replaced with primitive left_finger_carrier_geom)")
print(f"Cube half-size: {CUBE_HALF_SIZE_M * 1e3:.3f} mm (scale {CUBE_SIZE_SCALE:.4f} of the original {TACTILEMODEL03_CUBE_HALF_SIZE_M * 1e3:.1f} mm)")
print(f"Gel/cube torsional friction: {GEL_CUBE_TORSIONAL_FRICTION:.4g}")
print("Gel pad contact geoms:", GELSIGHT_GEL_PAD_GEOM_NAME_RIGHT, "/", GELSIGHT_GEL_PAD_GEOM_NAME_LEFT)
print(
    f"Cube bump texture: {CUBE_TEXTURE_GRID_SIZE}x{CUBE_TEXTURE_GRID_SIZE} hemispheres/face x 6 "
    f"= {CUBE_SURFACE_TEXTURE['total_bump_geoms']} collidable bumps "
    f"(radius {1e3 * CUBE_TEXTURE_BUMP_RADIUS_M:.2f} mm; mass/inertia unchanged)"
)
print("nq:", model.nq, "nv:", model.nv, "nu:", model.nu, "timestep:", model.opt.timestep)
print("OOD_CUBE_STARTS (core box center, z re-derived for the new cube):")
print(OOD_CUBE_STARTS)
print("IN_DISTRIBUTION_CUBE_STARTS (core box center, z re-derived for the new cube):")
print(IN_DISTRIBUTION_CUBE_STARTS)
print(
    "FOTS tactile cameras:", GELSIGHT_TACTILE_CAMERA_NAME_RIGHT, "(id", FOTS_TACTILE_CAM_ID_RIGHT, ") /",
    GELSIGHT_TACTILE_CAMERA_NAME_LEFT, "(id", FOTS_TACTILE_CAM_ID_LEFT, ")",
    " object reference site:", OBJECT_REF_SITE_NAME, "(id", OBJECT_REF_SITE_ID, ")",
)
print(
    f"Mirroring check OK: gel pads at hand-local xz={_right_gel_hand_local[[0, 2]]}, "
    f"y_right={_right_gel_hand_local[1]:.5f} m, y_left={_left_gel_hand_local[1]:.5f} m"
)

# ----- Base material values (captured once, from the freshly-compiled model, before any
# rollout mutates them) and the cube's own contact geom ids -- used by the per-rollout material
# randomization below. Friction/mass are edited live on `model` each rollout; capturing the TRUE
# compiled-in base values here means every rollout scales from the same reference, not from
# whatever the previous rollout happened to leave behind.
_cube_contact_geom_ids = [new_core_geom_id] + [
    require_mj_id(mujoco.mjtObj.mjOBJ_GEOM, name) for name in bump_positions
]
_base_cube_mass_kg = float(model.body_mass[BLOCK_BODY_ID])
_base_cube_inertia = model.body_inertia[BLOCK_BODY_ID].copy()
print(
    f"Slip-randomization base values: cube mass={_base_cube_mass_kg * 1e3:.1f} g, "
    f"{len(_cube_contact_geom_ids)} cube contact geoms tracked for live friction edits."
)


def maybe_apply_perturbation(perturbation_state):
    """Sets/clears data.xfrc_applied on the cube for a single, randomly-delayed external push per
    rollout -- independent of OpenVLA's own output entirely, so it gives a slip event with a KNOWN
    onset time without touching the policy's behavior at all. Called once per control step (every
    chunk_step, i.e. every 1/CONTROL_RATE_SCALE_FACTOR-scaled 100 ms), not every physics step --
    precise enough for a ~150 ms push, and avoids touching step_controller() itself."""
    if not perturbation_state["pending"]:
        return
    if perturbation_state["triggered"]:
        if data.time < perturbation_state["active_until_s"]:
            data.xfrc_applied[BLOCK_BODY_ID, :3] = perturbation_state["force_vec"]
        else:
            data.xfrc_applied[BLOCK_BODY_ID, :3] = 0.0
            perturbation_state["pending"] = False
        return
    if CONTROLLER_STATE["mode"] in SLIP_PERTURBATION_PHASES:
        if perturbation_state["phase_entry_time_s"] is None:
            perturbation_state["phase_entry_time_s"] = data.time
        elif data.time - perturbation_state["phase_entry_time_s"] >= perturbation_state["delay_s"]:
            perturbation_state["active_until_s"] = data.time + SLIP_PERTURBATION_DURATION_S
            perturbation_state["triggered"] = True


# -----------------------------------------------------------------------------
# FOTS tactile-sensor pipeline (Zhao et al., "FOTS: A Fast Optical Tactile
# Simulator...", IEEE RA-L 2024, https://arxiv.org/abs/2404.19217), ported
# verbatim from TactileModel05_Duo.ipynb's FOTS cell (itself unchanged from
# TactileModel04.ipynb's cell 4). Replaces this cell's earlier
# native-MuJoCo-tactile-sensor + Taxim pipeline:
#
#   MuJoCo depth (virtual camera behind the gel, built above)
#   -> diff against an analytical no-contact reference depth
#   -> contact mask + height map (FOTS's own pixel-height units)
#   -> a small Gaussian pre-blur (approximates real gel mechanical
#      diffusion; fixes a discrete-artifact shading bug found and fixed in
#      TactileModel04)
#   -> FOTS's pretrained MLP normal-to-RGB optical model + planar shadow
#   -> tactile RGB
#   -> FOTS's dilate/shear/twist marker-motion model, driven by the object's
#      pose relative to the sensor
#   -> marker flow image
#
# The optical/marker-motion equations, MLP architecture and weights, and
# GelSight calibration images are FOTS's own, vendored verbatim into
# external/fots/ (MIT-licensed; see external/fots/NOTICE.md).
# -----------------------------------------------------------------------------

from external.fots.mlp_model import MLP
from external.fots.fots_render import FOTSRender
from external.fots.marker_motion import MarkerMotion
from external.fots import params as fots_params

FOTS_DIR = PROJECT_DIR / "external" / "fots"

# Fixed by FOTS's pretrained MLP input normalization and its GelSight
# marker-grid calibration -- not a free resolution/grid choice.
FOTS_SENSOR_H = fots_params.sensor_h    # 320
FOTS_SENSOR_W = fots_params.sensor_w    # 240
FOTS_MM_TO_PIXEL = fots_params.mm_to_pixel  # 19.58 -- FOTS's own GelSight mm->pixel-height calibration
FOTS_MARKER_LAMB = [0.00125, 0.00021, 0.00038]  # FOTS's own default (dilate, shear, twist) kernel widths

# Tactile sampling rate: independent of the physics timestep (sampled by
# simulation time, not physics-step count).
FOTS_TACTILE_FPS = 30

# Contact threshold on depth_difference (meters). Temporary assumption (not
# a FOTS or GelSight-Mini published value): small enough to catch genuine
# sub-mm indentation, large enough to reject renderer/float noise.
FOTS_CONTACT_EPS_M = 2e-5

# Gaussian pre-blur (pixels) applied to height_map_px before FOTS's own
# rendering. See FOTSTactileSensor.compute_contact_geometry's docstring
# below for the full diagnosis (TactileModel04.ipynb / nb4_material_check/
# diagnose_shading.py): our height map comes from raw rigid-body
# interpenetration, sharper than a real gel's own mechanical diffusion,
# which produced discrete "fleck" shading artifacts against FOTS's
# pretrained MLP. sigma=4 px was the smallest value that reliably removed
# them while keeping the smallest/shallowest bump pattern still readable.
FOTS_HEIGHT_MAP_SMOOTH_SIGMA_PX = 4.0

DATASET_MIN_CONTACT_FRAMES = round(DATASET_MIN_CONTACT_SECONDS * FOTS_TACTILE_FPS)
print("FOTS sensor resolution (H,W):", FOTS_SENSOR_H, FOTS_SENSOR_W, "(fixed by pretrained MLP)")
print("FOTS mm_to_pixel:", FOTS_MM_TO_PIXEL, " tactile_fps:", FOTS_TACTILE_FPS, " contact_eps_m:", FOTS_CONTACT_EPS_M)
print("FOTS height-map pre-blur sigma (px):", FOTS_HEIGHT_MAP_SMOOTH_SIGMA_PX)

_fots_mlp_model = MLP().to(TORCH_DEVICE)
_fots_mlp_model.load_state_dict(
    torch.load(FOTS_DIR / "models" / "mlp_n2c_gelsight.pth", map_location=TORCH_DEVICE)
)
_fots_mlp_model.eval()
_fots_background_img = np.load(FOTS_DIR / "data" / "gelsight_bg.npy")
_fots_bg_render = np.load(FOTS_DIR / "data" / "ini_bg_fots_gelsight.npy")
print("Loaded FOTS GelSight calibration:", FOTS_DIR / "models" / "mlp_n2c_gelsight.pth", "on", TORCH_DEVICE)


def _build_fots_render():
    return FOTSRender(
        background_img=_fots_background_img,
        bg_render=_fots_bg_render.copy(),
        model=_fots_mlp_model,
    )


class FOTSTactileSensor:
    """FOTS-based optical tactile sensor bridged to a MuJoCo scene via a
    virtual camera mounted behind the gel pad."""

    def __init__(
        self,
        model,
        data,
        camera_id,
        gel_pad_geom_id,
        sensor_shell_geom_group,
        gel_pad_geom_group,
        gelsight_site_group,
        sensor_site_id,
        object_site_id,
        sensor_h=FOTS_SENSOR_H,
        sensor_w=FOTS_SENSOR_W,
        mm_to_pixel=FOTS_MM_TO_PIXEL,
        contact_eps_m=FOTS_CONTACT_EPS_M,
        marker_lamb=FOTS_MARKER_LAMB,
    ):
        self.model = model
        self.data = data
        self.camera_id = camera_id
        self.gel_pad_geom_id = gel_pad_geom_id
        self.sensor_site_id = sensor_site_id
        self.object_site_id = object_site_id
        self.sensor_h = sensor_h
        self.sensor_w = sensor_w
        self.mm_to_pixel = mm_to_pixel
        self.contact_eps_m = contact_eps_m
        self.marker_lamb = marker_lamb

        self._renderer = mujoco.Renderer(model, height=sensor_h, width=sensor_w)
        self._scene_option = mujoco.MjvOption()
        self._scene_option.geomgroup[sensor_shell_geom_group] = 0
        self._scene_option.geomgroup[gel_pad_geom_group] = 0
        self._scene_option.sitegroup[gelsight_site_group] = 0

        self.fots_render = _build_fots_render()
        self._traj = []  # relative-pose history (mm, mm, rad) while in contact; reset on contact loss

        cam_local_y = float(model.cam_pos[camera_id][1])
        gel_local_y = float(model.geom_pos[gel_pad_geom_id][1])
        gel_half_y = float(model.geom_size[gel_pad_geom_id][1])
        gel_front_local_y = gel_local_y - gel_half_y
        self.reference_depth_m = cam_local_y - gel_front_local_y

    def capture_depth(self):
        self._renderer.enable_depth_rendering()
        self._renderer.update_scene(self.data, camera=self.camera_id, scene_option=self._scene_option)
        depth_m = self._renderer.render().copy()
        self._renderer.disable_depth_rendering()
        depth_m = median_filter(depth_m, size=3)
        return depth_m

    def compute_contact_geometry(self, depth_m):
        """Depth difference, contact mask, and FOTS's own pixel-height
        units. `height_map_px` is masked to the contact region (a real
        GelSight/DIGIT gel is opaque, so nothing beyond it should be
        visible unless actually pressed against/touching it) and given a
        small Gaussian blur (FOTS_HEIGHT_MAP_SMOOTH_SIGMA_PX) approximating
        a real gel's mechanical diffusion, on top of (not instead of)
        FOTSRender.generate's own internal smooth_heightMap pyramid blur --
        see TactileModel04.ipynb's cell 4 for the full diagnosis of the
        shading-artifact bug this blur fixes. `depth_diff_m` itself is left
        unmasked/unsmoothed -- it's the raw diagnostic signal."""
        depth_diff_m = self.reference_depth_m - depth_m
        contact_mask = depth_diff_m > self.contact_eps_m
        masked_depth_diff_m = np.where(contact_mask, depth_diff_m, 0.0)
        height_map_px = (masked_depth_diff_m * 1000.0 * self.mm_to_pixel).astype(np.float32)
        height_map_px = gaussian_filter(height_map_px, sigma=FOTS_HEIGHT_MAP_SMOOTH_SIGMA_PX)
        return {
            "depth_diff_m": depth_diff_m,
            "contact_mask": contact_mask,
            "height_map_px": height_map_px,
        }

    def render_optical_image(self, height_map_px, contact_mask, shadow=True):
        return self.fots_render.generate(height_map_px.copy(), contact_mask, shadow=shadow)

    def _relative_pose(self):
        sensor_xpos = self.data.site_xpos[self.sensor_site_id].reshape(3, 1)
        sensor_xmat = self.data.site_xmat[self.sensor_site_id].reshape(3, 3)
        sensor_mat = np.vstack((np.hstack((sensor_xmat, sensor_xpos)), np.array([0, 0, 0, 1])))
        object_xpos = self.data.site_xpos[self.object_site_id].reshape(3, 1)
        object_xmat = self.data.site_xmat[self.object_site_id].reshape(3, 3)
        object_mat = np.vstack((np.hstack((object_xmat, object_xpos)), np.array([0, 0, 0, 1])))
        relative_mat = np.linalg.inv(sensor_mat) @ object_mat
        relative_xpos_m = relative_mat[:3, 3].copy()
        relative_xrot = cv2.Rodrigues(relative_mat[:3, :3])[0]
        return relative_xpos_m, relative_xrot

    def compute_marker_flow(self, tactile_rgb, depth_diff_m, contact_mask):
        relative_xpos_m, relative_xrot = self._relative_pose()
        if np.max(depth_diff_m) > self.contact_eps_m:
            self._traj.append([
                -float(relative_xpos_m[1]) * 1000.0,
                float(relative_xpos_m[0]) * 1000.0,
                -float(relative_xrot[2, 0]),
            ])
        else:
            self._traj = []

        marker = MarkerMotion(
            frame0_blur=tactile_rgb,
            depth=depth_diff_m * 1000.0,
            mask=contact_mask,
            traj=list(self._traj),
            lamb=self.marker_lamb,
            N=fots_params.N, M=fots_params.M, W=self.sensor_w, H=self.sensor_h,
        )
        marker_flow = marker._marker_motion()
        return marker_flow

    def get_observation(self):
        depth_m = self.capture_depth()
        geometry = self.compute_contact_geometry(depth_m)
        tactile_rgb = self.render_optical_image(geometry["height_map_px"], geometry["contact_mask"])
        marker_flow = self.compute_marker_flow(tactile_rgb, geometry["depth_diff_m"], geometry["contact_mask"])
        relative_xpos_m, relative_xrot = self._relative_pose()
        return {
            "sim_time_s": float(self.data.time),
            "tactile_rgb": tactile_rgb,
            "depth": depth_m,
            "depth_difference": geometry["depth_diff_m"],
            "contact_mask": geometry["contact_mask"],
            "height_map": geometry["height_map_px"],
            "marker_flow": marker_flow,
            "object_relative_pose": {
                "position_m": relative_xpos_m,
                "rotation_vector_rad": relative_xrot.reshape(-1),
            },
        }

    def close(self):
        self._renderer.close()


class FOTSRolloutRecorder:
    """Samples a FOTSTactileSensor at a fixed wall/sim-time rate (independent
    of the physics timestep) while the caller steps physics, and saves the
    real recorded tactile RGB + marker-flow videos plus per-frame
    quantitative arrays."""

    def __init__(self, rollout_dir, sensor, tactile_fps=FOTS_TACTILE_FPS, force_fn=None):
        self.rollout_dir = Path(rollout_dir)
        self.rollout_dir.mkdir(parents=True, exist_ok=True)
        self.sensor = sensor
        self.tactile_fps = tactile_fps
        self.sample_period_s = 1.0 / tactile_fps
        self.next_sample_time = float(sensor.data.time)

        self.tactile_rgb_path = self.rollout_dir / "fots_tactile_rgb.mp4"
        self.marker_flow_path = self.rollout_dir / "fots_marker_flow.mp4"
        writer_kwargs = dict(fps=tactile_fps, codec="libx264", quality=8, pixelformat="yuv420p", macro_block_size=None)
        self.rgb_writer = imageio.get_writer(str(self.tactile_rgb_path), **writer_kwargs)
        self.marker_writer = imageio.get_writer(str(self.marker_flow_path), **writer_kwargs)

        self.force_fn = force_fn  # zero-arg callable -> this finger's current normal contact force (N).
        self.sim_time_s = []
        self.depth_m_frames = []
        self.max_depth_diff_mm = []
        self.active_taxel_count = []
        self.relative_pos_mm = []
        self.relative_rot_rad = []
        self.normal_force_n = []
        self.frame_count = 0
        self.last_observation = None
        self.finalized_outputs = None

    def capture_initial(self):
        self._capture(record_time_gate=False)

    def after_physics_step(self):
        self._capture(record_time_gate=True)

    def _capture(self, record_time_gate):
        t = float(self.sensor.data.time)
        if record_time_gate and t < self.next_sample_time:
            return
        obs = self.sensor.get_observation()
        self.rgb_writer.append_data(obs["tactile_rgb"])
        self.marker_writer.append_data(obs["marker_flow"])
        self.sim_time_s.append(obs["sim_time_s"])
        if SAVE_RAW_DEPTH_STACK:
            self.depth_m_frames.append(obs["depth"].astype(np.float32))
        self.max_depth_diff_mm.append(float(obs["depth_difference"].max()) * 1000.0)
        self.active_taxel_count.append(int(obs["contact_mask"].sum()))
        self.relative_pos_mm.append(obs["object_relative_pose"]["position_m"] * 1000.0)
        self.relative_rot_rad.append(obs["object_relative_pose"]["rotation_vector_rad"])
        self.normal_force_n.append(float(self.force_fn()) if self.force_fn is not None else float("nan"))
        self.frame_count += 1
        self.next_sample_time = t + self.sample_period_s
        self.last_observation = obs

    def finalize(self):
        if self.finalized_outputs is not None:
            return self.finalized_outputs
        self.rgb_writer.close()
        self.marker_writer.close()

        npz_path = self.rollout_dir / "fots_quantitative_data.npz"
        _npz_fields = dict(
            sim_time_s=np.asarray(self.sim_time_s, dtype=np.float64),
            max_depth_diff_mm=np.asarray(self.max_depth_diff_mm, dtype=np.float64),
            active_taxel_count=np.asarray(self.active_taxel_count, dtype=np.int64),
            relative_pos_mm=np.asarray(self.relative_pos_mm, dtype=np.float64),
            relative_rot_rad=np.asarray(self.relative_rot_rad, dtype=np.float64),
            # Item 4 (contact force) and item 5 (contact state): both new in T3.
            normal_force_n=np.asarray(self.normal_force_n, dtype=np.float64),
            contact_state=np.asarray(self.active_taxel_count, dtype=np.int64) > 0,
            reference_depth_m=np.float64(self.sensor.reference_depth_m),
            contact_eps_m=np.float64(self.sensor.contact_eps_m),
            mm_to_pixel=np.float64(self.sensor.mm_to_pixel),
        )
        if SAVE_RAW_DEPTH_STACK:
            _npz_fields["depth_m"] = np.stack(self.depth_m_frames, axis=0)
        np.savez(npz_path, **_npz_fields)
        validation = {
            "status": "passed" if self.frame_count > 0 and max(self.max_depth_diff_mm, default=0.0) > 0.0 else "failed",
            "frame_count": self.frame_count,
            "tactile_fps": self.tactile_fps,
            "max_active_taxels": int(max(self.active_taxel_count, default=0)),
            "max_depth_diff_mm": float(max(self.max_depth_diff_mm, default=0.0)),
            "contact_held_at_end": bool(self.active_taxel_count[-1] > 0) if self.active_taxel_count else False,
        }
        validation_path = self.rollout_dir / "fots_validation.json"
        validation_path.write_text(json.dumps(validation, indent=2), encoding="utf-8")

        self.finalized_outputs = {
            "tactile_rgb_video": str(self.tactile_rgb_path),
            "marker_flow_video": str(self.marker_flow_path),
            "quantitative_npz": str(npz_path),
            "validation_json": str(validation_path),
            "validation": validation,
        }
        return self.finalized_outputs


FOTS_DEPTH_DIFF_DISPLAY_MAX_MM = 3.0
FOTS_DEPTH_DIFF_DISPLAY_GAMMA = 0.6


def depth_diff_to_display_rgb(depth_diff_m, max_mm=FOTS_DEPTH_DIFF_DISPLAY_MAX_MM, gamma=FOTS_DEPTH_DIFF_DISPLAY_GAMMA):
    """Fixed-scale visualization of depth_difference (mm), for photos/videos
    only -- display-only, no dependency on FOTS's own rendering."""
    value = np.clip(np.asarray(depth_diff_m, dtype=np.float32) * 1000.0 / max_mm, 0.0, 1.0)
    value = value ** gamma
    red = np.clip(2.0 * value - 0.5, 0.0, 1.0)
    green = np.clip(2.0 * value, 0.0, 1.0)
    blue = np.clip(4.0 * value, 0.0, 1.0) * (value < 0.5)
    rgb = np.stack([red, green, blue], axis=-1)
    return (rgb * 255.0).astype(np.uint8)


# --- One FOTSTactileSensor per finger (right + left), each reused across every rollout in
# this cell exactly like the single sensor was, each bound to its own tactile camera / gel pad /
# site, both sharing the cube's reference site. The class is parameterized purely by IDs and
# never hardcoded to "right", so it needed no change for the second sensor. Each rollout still
# gets its own fresh recorders (DualFOTSRecorder, below), matching the old one-recorder-per
# -rollout lifecycle. ---------------------------------------------------------------------
# Re-running this cell in the same kernel builds new renderers; free the previous run's GL
# contexts first so two sensors per run do not pile up.
for _old_sensor_name in ("FOTS_SENSOR_RIGHT", "FOTS_SENSOR_LEFT"):
    _old_sensor = globals().get(_old_sensor_name)
    if _old_sensor is not None:
        try:
            _old_sensor.close()
        except Exception:
            pass

FOTS_SENSOR_RIGHT = FOTSTactileSensor(
    model=model,
    data=data,
    camera_id=FOTS_TACTILE_CAM_ID_RIGHT,
    gel_pad_geom_id=GELSIGHT_GEL_PAD_GEOM_ID_RIGHT,
    sensor_shell_geom_group=SENSOR_SHELL_GEOM_GROUP,
    gel_pad_geom_group=GEL_PAD_GEOM_GROUP,
    gelsight_site_group=GELSIGHT_SITE_GROUP,
    sensor_site_id=GELSIGHT_SITE_ID_RIGHT,
    object_site_id=OBJECT_REF_SITE_ID,
)
FOTS_SENSOR_LEFT = FOTSTactileSensor(
    model=model,
    data=data,
    camera_id=FOTS_TACTILE_CAM_ID_LEFT,
    gel_pad_geom_id=GELSIGHT_GEL_PAD_GEOM_ID_LEFT,
    sensor_shell_geom_group=SENSOR_SHELL_GEOM_GROUP,
    gel_pad_geom_group=GEL_PAD_GEOM_GROUP,
    gelsight_site_group=GELSIGHT_SITE_GROUP,
    sensor_site_id=GELSIGHT_SITE_ID_LEFT,
    object_site_id=OBJECT_REF_SITE_ID,
)
FOTS_SENSOR = FOTS_SENSOR_RIGHT  # backward-compatible alias (right side)
print("FOTSTactileSensor built for both fingers. Analytical no-contact reference depth (m):")
print("  right:", FOTS_SENSOR_RIGHT.reference_depth_m, " left:", FOTS_SENSOR_LEFT.reference_depth_m)


class DualFOTSRecorder:
    """Records BOTH GelSight sensors in lock-step. step_controller() and settle_after_release()
    only know about a single `tactile_recorder` (they call after_physics_step() every physics
    step); this wrapper is that one object and drives one FOTSRolloutRecorder per finger, so
    neither function's signature changes. Right -> <rollout_dir>/right, left ->
    <rollout_dir>/left, so the two sides' video/npz/json files never collide."""

    def __init__(self, rollout_dir, sensor_right, sensor_left, tactile_fps=FOTS_TACTILE_FPS):
        self.rollout_dir = Path(rollout_dir)
        # The sensors are reused across rollouts; their marker-motion history (self._traj) must
        # not leak from the end of the previous rollout into the start of this one.
        sensor_right._traj = []
        sensor_left._traj = []
        self.recorders = {
            "right": FOTSRolloutRecorder(self.rollout_dir / "right", sensor_right, tactile_fps),
            "left": FOTSRolloutRecorder(self.rollout_dir / "left", sensor_left, tactile_fps),
        }

    def capture_initial(self):
        for recorder in self.recorders.values():
            recorder.capture_initial()

    def after_physics_step(self):
        for recorder in self.recorders.values():
            recorder.after_physics_step()

    @property
    def frame_count(self):
        counts = {side: recorder.frame_count for side, recorder in self.recorders.items()}
        assert len(set(counts.values())) == 1, f"Right/left tactile recorders fell out of step: {counts}"
        return counts["right"]

    def finalize(self):
        return {side: recorder.finalize() for side, recorder in self.recorders.items()}


def _pose_matrix(pos, quat_wxyz):
    """4x4 homogeneous transform from a (position, wxyz-quaternion) pose -- same convention
    FOTSTactileSensor._relative_pose already uses, just built from a body/site pose instead."""
    rot = np.empty(9, dtype=np.float64)
    mujoco.mju_quat2Mat(rot, np.asarray(quat_wxyz, dtype=np.float64))
    m = np.eye(4, dtype=np.float64)
    m[:3, :3] = rot.reshape(3, 3)
    m[:3, 3] = pos
    return m


def _finger_normal_force_n(gel_pad_geom_id):
    """Sum of |normal contact force| (N) over every contact currently active on this gel pad geom
    -- read directly from MuJoCo's own contact solver via mj_contactForce, not a fabricated or
    hacked-in "virtual loadcell": this is the exact force the physics engine already computed to
    resolve this step's contacts; the sensor only reads it out."""
    total = 0.0
    force_buf = np.zeros(6, dtype=np.float64)
    for i in range(data.ncon):
        con = data.contact[i]
        if con.geom1 == gel_pad_geom_id or con.geom2 == gel_pad_geom_id:
            mujoco.mj_contactForce(model, data, i, force_buf)
            total += abs(float(force_buf[0]))
    return total


class InstrumentedDualFOTSRecorder(DualFOTSRecorder):
    """DualFOTSRecorder plus the T3 slip ground truth (Plan B items 1-3): hand pose, cube pose, the
    object's pose relative to the HAND (not a single sensor -- symmetric, independent of finger
    placement), a rolling-window slip flag, the one-time grasp-establishment pose, and each
    finger's contact force -- all captured at the exact same instant as each tactile sample
    (piggybacked on the right recorder's own frame_count increment, so there is no separate timer
    that could drift out of sync with it)."""

    def __init__(self, rollout_dir, sensor_right, sensor_left, tactile_fps=FOTS_TACTILE_FPS):
        super().__init__(rollout_dir, sensor_right, sensor_left, tactile_fps)
        self.recorders["right"].force_fn = lambda: _finger_normal_force_n(GELSIGHT_GEL_PAD_GEOM_ID_RIGHT)
        self.recorders["left"].force_fn = lambda: _finger_normal_force_n(GELSIGHT_GEL_PAD_GEOM_ID_LEFT)

        self.sim_time_s = []
        self.controller_phase = []
        self.hand_pos_m, self.hand_quat_wxyz = [], []
        self.cube_pos_m, self.cube_quat_wxyz = [], []
        self.relative_pos_mm, self.relative_quat_wxyz = [], []
        self.grasp_pose = None  # set exactly once, at the instant the grip first locks (item 2).
        self.grasp_established_time_s = None
        self._grasp_pose_captured = False

    def _capture_ground_truth(self):
        hand = get_ee_pose()
        cube = get_cube_pose()
        hand_m = _pose_matrix(hand["position"], hand["quat_wxyz"])
        cube_m = _pose_matrix(cube["position"], cube["quat_wxyz"])
        relative_m = np.linalg.inv(hand_m) @ cube_m
        relative_quat = np.empty(4, dtype=np.float64)
        mujoco.mju_mat2Quat(relative_quat, relative_m[:3, :3].copy().reshape(-1))

        self.sim_time_s.append(float(data.time))
        self.controller_phase.append(CONTROLLER_STATE["mode"])
        self.hand_pos_m.append(hand["position"].copy())
        self.hand_quat_wxyz.append(hand["quat_wxyz"].copy())
        self.cube_pos_m.append(cube["position"].copy())
        self.cube_quat_wxyz.append(cube["quat_wxyz"].copy())
        self.relative_pos_mm.append(relative_m[:3, 3] * 1000.0)
        self.relative_quat_wxyz.append(relative_quat.copy())

        # Item 2 (grasp pose): a read-only observation of capped_grip_fraction()'s own existing
        # plateau-lock state (CONTROLLER_STATE["grip_locked_fraction"] going from None to a value)
        # -- no change to the controller itself, captured once, the first time it happens.
        if not self._grasp_pose_captured and CONTROLLER_STATE.get("grip_locked_fraction") is not None:
            self.grasp_pose = {
                "relative_pos_mm": (relative_m[:3, 3] * 1000.0).tolist(),
                "relative_quat_wxyz": relative_quat.tolist(),
            }
            self.grasp_established_time_s = float(data.time)
            self._grasp_pose_captured = True

    def capture_initial(self):
        super().capture_initial()
        self._capture_ground_truth()

    def after_physics_step(self):
        before = self.recorders["right"].frame_count
        super().after_physics_step()
        if self.recorders["right"].frame_count > before:
            self._capture_ground_truth()

    def finalize(self):
        outputs = super().finalize()
        n = len(self.sim_time_s)
        assert n == self.frame_count, (n, self.frame_count)

        sim_time_s = np.asarray(self.sim_time_s, dtype=np.float64)
        phases = np.asarray(self.controller_phase)
        relative_pos_mm = np.asarray(self.relative_pos_mm, dtype=np.float64)
        relative_quat_wxyz = np.asarray(self.relative_quat_wxyz, dtype=np.float64)
        holding = np.isin(phases, list(SLIP_HOLDING_PHASES))

        # Item 1 (slip): a ROLLING window (not cumulative-since-grasp drift -- see the knob's own
        # comment for why), only evaluated in phases where the object should be rigidly held.
        window_frames = max(1, round(SLIP_WINDOW_S * self.recorders["right"].tactile_fps))
        translation_delta_mm = np.zeros(n, dtype=np.float64)
        rotation_delta_deg = np.zeros(n, dtype=np.float64)
        for i in range(window_frames, n):
            translation_delta_mm[i] = float(np.linalg.norm(relative_pos_mm[i] - relative_pos_mm[i - window_frames]))
            error_rotvec = quat_error_world(relative_quat_wxyz[i], relative_quat_wxyz[i - window_frames])
            rotation_delta_deg[i] = float(np.degrees(np.linalg.norm(error_rotvec)))
        slip_binary = holding & (
            (translation_delta_mm > SLIP_TRANSLATION_THRESHOLD_MM) | (rotation_delta_deg > SLIP_ROTATION_THRESHOLD_DEG)
        )

        npz_path = self.rollout_dir / "slip_ground_truth.npz"
        np.savez(
            npz_path,
            sim_time_s=sim_time_s,
            controller_phase=phases,
            holding_phase=holding,
            hand_pos_m=np.asarray(self.hand_pos_m, dtype=np.float64),
            hand_quat_wxyz=np.asarray(self.hand_quat_wxyz, dtype=np.float64),
            cube_pos_m=np.asarray(self.cube_pos_m, dtype=np.float64),
            cube_quat_wxyz=np.asarray(self.cube_quat_wxyz, dtype=np.float64),
            relative_pos_mm=relative_pos_mm,          # item 3: in-hand object pose (position, mm)
            relative_quat_wxyz=relative_quat_wxyz,     # item 3: in-hand object pose (orientation)
            translation_delta_mm=translation_delta_mm,
            rotation_delta_deg=rotation_delta_deg,
            slip_binary=slip_binary,                   # item 1
            slip_window_s=np.float64(SLIP_WINDOW_S),
            slip_translation_threshold_mm=np.float64(SLIP_TRANSLATION_THRESHOLD_MM),
            slip_rotation_threshold_deg=np.float64(SLIP_ROTATION_THRESHOLD_DEG),
        )
        grasp_json_path = self.rollout_dir / "grasp_pose.json"
        grasp_json_path.write_text(json.dumps({
            "grasp_established": self.grasp_pose is not None,
            "grasp_established_time_s": self.grasp_established_time_s,
            **(self.grasp_pose or {"relative_pos_mm": None, "relative_quat_wxyz": None}),
        }, indent=2), encoding="utf-8")

        slip_frames = int(slip_binary.sum())
        holding_frames = int(holding.sum())
        outputs["ground_truth"] = {
            "quantitative_npz": str(npz_path),
            "grasp_pose_json": str(grasp_json_path),
            "grasp_established": self.grasp_pose is not None,
            "holding_frames": holding_frames,
            "slip_frames": slip_frames,
            "slip_fraction_of_holding": float(slip_frames / holding_frames) if holding_frames else float("nan"),
        }
        return outputs


# Sanity check for the exact bug this camera placement hit once already: if reference_depth_m is
# too close to the scene's near-clipping plane, any indentation deep enough to bring the object
# within znear of the camera gets silently clamped to a constant, physically-wrong depth instead
# of its true value. Fail loudly instead of producing quietly-wrong tactile frames. Checked for
# both sensors -- both share identical camera/gel-pad geometry.
_znear_abs_m = float(model.vis.map.znear) * float(model.stat.extent)
for _side_name, _sensor in (("right", FOTS_SENSOR_RIGHT), ("left", FOTS_SENSOR_LEFT)):
    _znear_margin_m = _sensor.reference_depth_m - _znear_abs_m
    print(f"Scene near-clip plane: {_znear_abs_m * 1e3:.2f} mm; {_side_name} margin to reference depth: {_znear_margin_m * 1e3:.2f} mm")
    assert _znear_margin_m > 0.010, (
        f"[{_side_name}] Tactile camera reference depth ({_sensor.reference_depth_m * 1e3:.2f} mm) is too close to the "
        f"scene's near-clipping plane ({_znear_abs_m * 1e3:.2f} mm) -- indentation would get clipped to a "
        f"constant depth instead of its true value. Increase FOTS_TACTILE_CAM_LOCAL_Y_M above."
    )

for _side_name, _sensor in (("right", FOTS_SENSOR_RIGHT), ("left", FOTS_SENSOR_LEFT)):
    _self_test_obs = _sensor.get_observation()
    print(
        f"[{_side_name}] Self-test observation (current, pre-grasp state):",
        "tactile_rgb", _self_test_obs["tactile_rgb"].shape, _self_test_obs["tactile_rgb"].dtype,
        "depth", _self_test_obs["depth"].shape,
        "contact px:", int(_self_test_obs["contact_mask"].sum()),
        "marker_flow", _self_test_obs["marker_flow"].shape,
    )
    _sensor._traj = []

worker_code = f"""
import json, sys, time
from pathlib import Path
from types import SimpleNamespace

sys.argv.append("panda_d3")
sys.path.insert(0, {json.dumps(str(OFT_REPO))})

import numpy as np
import torch
from PIL import Image
from experiments.robot.openvla_utils import (
    get_action_head,
    get_processor,
    get_proprio_projector,
    get_vla,
    get_vla_action,
)
from prismatic.vla.constants import ACTION_DIM, NUM_ACTIONS_CHUNK, PROPRIO_DIM

checkpoint_dir = Path({json.dumps(str(checkpoint_dir))})
dataset_name = {json.dumps(DATASET_NAME)}
assert (NUM_ACTIONS_CHUNK, ACTION_DIM, PROPRIO_DIM) == (8, 7, 8)
assert torch.cuda.is_available(), "CUDA is required for OpenVLA-OFT rollout inference."

cfg = SimpleNamespace(
    pretrained_checkpoint=str(checkpoint_dir),
    use_l1_regression=True,
    use_diffusion=False,
    use_film=False,
    num_images_in_input=1,
    use_proprio=True,
    load_in_8bit=False,
    load_in_4bit=False,
    center_crop=True,
    lora_rank=32,
    unnorm_key=dataset_name,
    num_diffusion_steps_train=50,
    num_diffusion_steps_inference=50,
)

print("OPENVLA_OFT_WORKER_LOADING", str(checkpoint_dir), flush=True)
vla = get_vla(cfg)
processor = get_processor(cfg)
action_head = get_action_head(cfg, vla.llm_dim)
proprio_projector = get_proprio_projector(cfg, vla.llm_dim, PROPRIO_DIM)
print("OPENVLA_OFT_WORKER_READY", flush=True)

for line in sys.stdin:
    try:
        req = json.loads(line)
        image = np.asarray(Image.open(req["image_path"]).convert("RGB"))           # model input
        proprio = np.asarray(req["proprio"], dtype=np.float32).reshape(-1)
        if proprio.shape != (PROPRIO_DIM,) or not np.all(np.isfinite(proprio)):
            raise ValueError(f"Invalid proprio: {{proprio}}")
        observation = {{"full_image": image, "state": proprio.copy()}}
        start = time.perf_counter()
        actions = np.asarray(
            get_vla_action(
                cfg,
                vla,
                processor,
                observation,
                req["task"],
                action_head,
                proprio_projector,
            ),
            dtype=np.float32,
        )
        if actions.shape != (NUM_ACTIONS_CHUNK, ACTION_DIM):
            raise ValueError(f"Expected 8x7 action chunk, got {{actions.shape}}")
        if not np.all(np.isfinite(actions)):
            raise ValueError("OFT action chunk contains NaN or infinity")
        print("OPENVLA_OFT_WORKER_RESULT " + json.dumps({{
            "id": req.get("id"),
            "actions": actions.tolist(),
            "inference_s": time.perf_counter() - start,
        }}), flush=True)
    except Exception as exc:
        print("OPENVLA_OFT_WORKER_ERROR " + json.dumps({{"error": repr(exc)}}), flush=True)
"""

def start_worker():
    env = oft_env({"TF_CPP_MIN_LOG_LEVEL": "3", "WANDB_MODE": "disabled"})
    proc = subprocess.Popen(
        [str(OFT_PYTHON), "-u", "-c", worker_code],
        cwd=str(OFT_REPO),
        env=env,
        stdin=subprocess.PIPE,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    tail = []
    start = time.time()
    while time.time() - start < 900:
        ready, _, _ = select.select([proc.stdout], [], [], 0.5)
        if ready:
            line = proc.stdout.readline()
            if line:
                tail.append(line.rstrip())
                print(line, end="")
                if line.startswith("OPENVLA_OFT_WORKER_READY"):
                    return proc, tail
        if proc.poll() is not None:
            raise RuntimeError("OpenVLA worker exited early:\n" + "\n".join(tail[-40:]))
    raise TimeoutError("Timed out while loading OpenVLA worker.")

def predict_action_chunk(proc, image_path, proprio, request_id):
    proprio = np.asarray(proprio, dtype=np.float32).reshape(-1)
    assert proprio.shape == (8,) and np.all(np.isfinite(proprio))
    proc.stdin.write(json.dumps({
        "id": int(request_id),
        "image_path": str(image_path),
        "task": TASK_PROMPT,
        "proprio": proprio.tolist(),
    }) + "\n")
    proc.stdin.flush()

    tail = []
    start = time.time()
    while time.time() - start < 600:
        ready, _, _ = select.select([proc.stdout], [], [], 0.5)
        if ready:
            line = proc.stdout.readline()
            if not line:
                continue
            tail.append(line.rstrip())
            if line.startswith("OPENVLA_OFT_WORKER_RESULT "):
                payload = json.loads(line.split(" ", 1)[1])
                actions = np.asarray(payload["actions"], dtype=np.float32)
                assert actions.shape == (8, 7), actions.shape
                assert np.all(np.isfinite(actions))
                return actions, payload
            if line.startswith("OPENVLA_OFT_WORKER_ERROR "):
                raise RuntimeError(line)
        if proc.poll() is not None:
            raise RuntimeError("OpenVLA-OFT worker exited during prediction:\n" + "\n".join(tail[-40:]))
    raise TimeoutError("Timed out waiting for OpenVLA-OFT action chunk.")

def render_rgb(renderer, camera):
    renderer.update_scene(data, camera=camera)
    return renderer.render()

def set_arm_qpos(qpos_dict):
    for name, value in qpos_dict.items():
        if name in ARM_JOINT_NAMES:
            data.qpos[ARM_QPOS_ADR[ARM_JOINT_NAMES.index(name)]] = float(value)
    set_arm_position_targets(qpos_dict)
    mujoco.mj_forward(model, data)

def randomize_start(rng, rollout_idx):
    if USE_IN_DISTRIBUTION_STARTS:
        cube_starts = IN_DISTRIBUTION_CUBE_STARTS
        nominal_core_cube_pos = cube_starts[rollout_idx % len(cube_starts)].copy()
        assert CUBE_X_RANGE[0] <= nominal_core_cube_pos[0] <= CUBE_X_RANGE[1]
        assert CUBE_Y_RANGE[0] <= nominal_core_cube_pos[1] <= CUBE_Y_RANGE[1]
    else:
        cube_starts = OOD_CUBE_STARTS
        nominal_core_cube_pos = cube_starts[rollout_idx % len(cube_starts)].copy()
        assert nominal_core_cube_pos[0] < 0.47 or nominal_core_cube_pos[0] > 0.53
        assert nominal_core_cube_pos[1] < -0.04 or nominal_core_cube_pos[1] > 0.04

    # Bottom-face hemisphere tips, rather than the hidden core box face, begin
    # exactly on the tabletop. This avoids an initialization overlap equal to
    # the bump radius.
    cube_pos = nominal_core_cube_pos.copy()
    cube_pos[2] += CUBE_TEXTURE_BUMP_RADIUS_M

    reset_task_state_randomized(cube_pos=cube_pos, robot_joint_noise=None, settle_steps=60)
    nominal_ee = get_ee_pose()["position"].copy()
    direction = rng.normal(size=3)
    direction /= max(np.linalg.norm(direction), 1e-12)
    ee_delta = direction * OOD_EE_JITTER_M
    ee_target = np.clip(nominal_ee + ee_delta, WORKSPACE_LOW, WORKSPACE_HIGH)

    qpos_target = solve_hand_position_ik(ee_target, seed_qpos=current_arm_qpos_dict())
    set_arm_qpos(qpos_target)
    set_gripper_opening(0.04)
    for _ in range(60):
        set_arm_position_targets(qpos_target)
        set_gripper_opening(0.04)
        mujoco.mj_step(model, data)
    mujoco.mj_forward(model, data)

    return {
        "cube_start": cube_pos.tolist(),
        "nominal_untextured_cube_start": nominal_core_cube_pos.tolist(),
        "texture_bottom_tip_initially_on_table": True,
        "robot_ee_start_delta_m": (get_ee_pose()["position"] - nominal_ee).tolist(),
        "outside_d3_cube_xy_envelope": not USE_IN_DISTRIBUTION_STARTS,
    }

def current_oft_proprio():
    joint_qpos = np.asarray(data.qpos[ARM_QPOS_ADR], dtype=np.float32)
    gripper_fraction = np.float32(np.clip(current_finger_opening_m() / 0.04, 0.0, 1.0))
    proprio = np.concatenate([joint_qpos, [gripper_fraction]]).astype(np.float32)
    assert proprio.shape == (8,) and np.all(np.isfinite(proprio))
    return proprio

CONTROLLER_STATE = {
    "mode": "open",
    "profile_step": 0,
    "applied_gripper_fraction": 1.0,
    "hold_position": None,
    "hold_quat": None,
    "reference_position": None,
    "reference_quat": None,
    "grip_opening_history": [],
    "grip_fraction_history": [],
    "grip_locked_fraction": None,
}


def reset_controller_state():
    initial_pose = get_ee_pose()
    CONTROLLER_STATE.update(
        mode="open",
        profile_step=0,
        applied_gripper_fraction=1.0,
        hold_position=None,
        hold_quat=None,
        reference_position=initial_pose["position"].copy(),
        reference_quat=initial_pose["quat_wxyz"].copy(),
        filtered_carry_translation=np.zeros(3, dtype=np.float64),
        filtered_carry_rotation=np.zeros(3, dtype=np.float64),
        last_wrist_alignment=None,
        grip_opening_history=[],
        grip_fraction_history=[],
        grip_locked_fraction=None,
    )


def smoothstep01(value):
    value = float(np.clip(value, 0.0, 1.0))
    return value * value * (3.0 - 2.0 * value)


def controller_gripper_fraction(raw_fraction):
    """Update grasp/release phase bookkeeping from the policy gripper output."""
    state = CONTROLLER_STATE
    raw_fraction = float(np.clip(raw_fraction, 0.0, 1.0))

    if state["mode"] == "open" and raw_fraction < GRIPPER_CLOSE_TRIGGER:
        pose = get_ee_pose()
        state.update(
            mode="closing",
            profile_step=0,
            hold_position=pose["position"].copy(),
            hold_quat=pose["quat_wxyz"].copy(),
        )
    elif (
        state["mode"] in {"closed_hold", "lift_bootstrap", "carrying"}
        and raw_fraction > GRIPPER_OPEN_TRIGGER
    ):
        pose = get_ee_pose()
        release_position = pose["position"].copy()
        release_position[2] = min(
            release_position[2] + RELEASE_CLEARANCE_M,
            WORKSPACE_HIGH[2],
        )
        state.update(
            mode="opening",
            profile_step=0,
            hold_position=release_position,
            hold_quat=pose["quat_wxyz"].copy(),
        )

    if state["mode"] == "closing":
        state["profile_step"] += 1
        alpha = smoothstep01(
            state["profile_step"] / GRIPPER_CLOSE_PROFILE_STEPS
        )
        state["applied_gripper_fraction"] = 1.0 - alpha
        if state["profile_step"] >= GRIPPER_CLOSE_PROFILE_STEPS:
            state["mode"] = "closed_hold"
    elif state["mode"] == "opening":
        state["profile_step"] += 1
        alpha = smoothstep01(
            state["profile_step"] / GRIPPER_OPEN_PROFILE_STEPS
        )
        state["applied_gripper_fraction"] = alpha
        if state["profile_step"] >= GRIPPER_OPEN_PROFILE_STEPS:
            state["mode"] = "released_hold"
    elif state["mode"] in {"closed_hold", "lift_bootstrap", "carrying"}:
        state["applied_gripper_fraction"] = 0.0
    else:
        state["applied_gripper_fraction"] = 1.0

    return float(state["applied_gripper_fraction"])


def select_controller_action(action_chunk):
    """Execute only horizon 0; later horizons are predictions, not commands."""
    action_chunk = np.asarray(action_chunk, dtype=np.float64)
    assert action_chunk.shape == (8, 7)
    state = CONTROLLER_STATE
    if (
        state["mode"] == "lift_bootstrap"
        and state["hold_position"] is not None
        and get_ee_pose()["position"][2]
        >= state["hold_position"][2] + LIFT_BOOTSTRAP_HEIGHT_M - 0.005
    ):
        state["mode"] = "carrying"
    action = action_chunk[0].copy()
    return action, 0, float(action[6])


def capped_grip_fraction(raw_fraction, mode):
    """Grip-plateau gripper-command hold -- see the constants' comment above
    GRIP_PLATEAU_STABLE_STEPS for the full diagnosis and reasoning.

    First version of this fix (re-derived a fresh "plateau minus a fixed
    overtravel margin" target the instant a plateau was detected) had a bug:
    the natural closing process had *already* built up a much larger implicit
    squeeze by that point (the raw policy fraction was already commanding a
    far tighter target than the object's measured stop, generating a large,
    already-established position error/clamping force) than the freshly
    -computed margin assumed. Flooring the command at that smaller
    freshly-computed value abruptly released most of the already-working
    grip in a single step -- confirmed empirically: the cube dropped to the
    table within one control step of the cap first engaging.

    Fixed design: don't recompute a target from geometry at all. Once the
    finger position has plateaued (stable within GRIP_PLATEAU_TOLERANCE_M for
    GRIP_PLATEAU_STABLE_STEPS consecutive steps), LOCK the tightest raw
    policy fraction already seen during that stability window as a floor for
    the rest of the grip -- i.e. freeze exactly the squeeze level the policy
    itself had already, naturally, established (which the friction-only
    sweep showed is generally enough to lift and carry), rather than
    substituting a different, independently-derived one. No abrupt jump in
    either direction. From then on the commanded fraction is never allowed
    to ratchet any tighter than that locked value -- removing the sustained
    -maximal-squeeze window that was creeping the compliant grip loose --
    but the policy can still freely command *more* opening at any time
    (release is never blocked)."""
    state = CONTROLLER_STATE
    if mode not in {"closed_hold", "lift_bootstrap", "carrying"}:
        state["grip_opening_history"] = []
        state["grip_fraction_history"] = []
        state["grip_locked_fraction"] = None
        return raw_fraction

    if state["grip_locked_fraction"] is not None:
        return max(raw_fraction, state["grip_locked_fraction"])

    measured_m = current_finger_opening_m()
    pos_history = state["grip_opening_history"]
    pos_history.append(measured_m)
    if len(pos_history) > GRIP_PLATEAU_STABLE_STEPS:
        del pos_history[0]

    frac_history = state.setdefault("grip_fraction_history", [])
    frac_history.append(raw_fraction)
    if len(frac_history) > GRIP_PLATEAU_STABLE_STEPS:
        del frac_history[0]

    if (
        len(pos_history) == GRIP_PLATEAU_STABLE_STEPS
        and (max(pos_history) - min(pos_history)) <= GRIP_PLATEAU_TOLERANCE_M
    ):
        # A small additional tightening margin *from* the already-locked
        # value (not a freshly-computed one -- see GRIP_LOCK_EXTRA_MARGIN's
        # own comment for why that distinction matters). Diagnosed from the
        # first 4-corner test: the plain lock (no extra margin) worked
        # perfectly for one corner (held 44+ steps through the full carry)
        # but was lost mid-lift on another corner despite locking cleanly at
        # a similar value (0.121) and never wavering -- a genuine physical
        # disturbance (most likely from the fast initial lift_bootstrap
        # motion) overcame a grip that just wasn't squeezed quite tightly
        # enough, not a creep/ratcheting failure this fix already handles.
        state["grip_locked_fraction"] = max(
            0.0, min(frac_history) - GRIP_LOCK_EXTRA_MARGIN
        )

    return raw_fraction


def wrist_visual_alignment_correction(current_pose):
    """Return an image-based XY correction without reading simulator object state."""
    inactive = {
        "active": False,
        "centroid_px": None,
        "error_px": None,
        "correction_xy_m": [0.0, 0.0],
    }
    if (
        not WRIST_VISUAL_SERVO_ENABLED
        or CONTROLLER_STATE["mode"] != "open"
        or current_pose["position"][2] > WRIST_VISUAL_SERVO_START_Z_M
        or current_pose["position"][2] < WRIST_VISUAL_SERVO_STOP_Z_M
    ):
        return np.zeros(2, dtype=np.float64), inactive

    rgb = render_rgb(wrist_alignment_renderer, AGENT_CAMERA_NAME)
    red = rgb[:, :, 0].astype(np.float64)
    green = rgb[:, :, 1].astype(np.float64)
    blue = rgb[:, :, 2].astype(np.float64)
    mask = (red > 140.0) & (red > 1.7 * green) & (red > 1.7 * blue)
    rows, cols = np.nonzero(mask)
    if len(cols) < 50:
        missing = dict(inactive)
        missing["reason"] = "red_target_not_found"
        return np.zeros(2, dtype=np.float64), missing

    centroid = np.array([cols.mean(), rows.mean()], dtype=np.float64)
    camera_rotation = data.cam_xmat[camera_id].reshape(3, 3).copy()
    camera_position = data.cam_xpos[camera_id].copy()
    image_height, image_width = rgb.shape[:2]
    focal_px = 0.5 * image_height / np.tan(
        np.deg2rad(model.cam_fovy[camera_id]) / 2.0
    )
    principal = np.array([0.5 * image_width, 0.5 * image_height])
    ray_camera = np.array([
        (centroid[0] - principal[0]) / focal_px,
        (principal[1] - centroid[1]) / focal_px,
        -1.0,
    ])
    ray_world = camera_rotation @ ray_camera
    if abs(ray_world[2]) < 1e-6:
        return np.zeros(2, dtype=np.float64), inactive
    distance = (WRIST_ALIGNMENT_PLANE_Z_M - camera_position[2]) / ray_world[2]
    estimated_point = camera_position + distance * ray_world

    desired_grasp_xy = estimated_point[:2] + WRIST_ESTIMATE_TO_GRASP_OFFSET_M
    world_error = desired_grasp_xy - current_pose["position"][:2]
    world_error_norm = float(np.linalg.norm(world_error))
    correction = WRIST_VISUAL_SERVO_GAIN * world_error
    if world_error_norm <= WRIST_ALIGNMENT_TOLERANCE_M:
        correction[:] = 0.0
    correction_norm = float(np.linalg.norm(correction))
    if correction_norm > WRIST_VISUAL_SERVO_MAX_STEP_M:
        correction *= WRIST_VISUAL_SERVO_MAX_STEP_M / correction_norm

    info = {
        "active": True,
        "centroid_px": centroid.tolist(),
        "correction_xy_m": correction.tolist(),
        "estimated_target_xy_m": estimated_point[:2].tolist(),
        "desired_grasp_xy_m": desired_grasp_xy.tolist(),
        "world_error_xy_m": world_error.tolist(),
        "world_error_norm_m": world_error_norm,
        "reason": (
            "within_tolerance"
            if world_error_norm <= WRIST_ALIGNMENT_TOLERANCE_M
            else "correcting"
        ),
    }
    return correction, info


def adapt_openvla_action(action):
    """Convert one policy delta into a smooth, contact-conscious 10 Hz pose target."""
    raw = np.asarray(action, dtype=np.float64).reshape(-1)
    assert raw.size == 7, f"Expected 7D OpenVLA action, got {raw}"

    clipped = raw.copy()
    clipped[:3] = np.clip(
        clipped[:3], -OPENVLA_MAX_TRANSLATION_M, OPENVLA_MAX_TRANSLATION_M
    )
    rotation_norm = float(np.linalg.norm(clipped[3:6]))
    if rotation_norm > OPENVLA_MAX_ROTATION_RAD:
        clipped[3:6] *= OPENVLA_MAX_ROTATION_RAD / rotation_norm
    clipped[6] = float(np.clip(clipped[6], 0.0, 1.0))

    state = CONTROLLER_STATE
    if state["mode"] == "closed_hold" and clipped[2] > LIFT_LOOKAHEAD_MIN_DZ_M:
        state["mode"] = "lift_bootstrap"

    current_pose = get_ee_pose()
    mode = state["mode"]
    translation_gain = (
        CARRY_TRANSLATION_GAIN if mode == "carrying" else APPROACH_TRANSLATION_GAIN
    )
    translation_delta = clipped[:3] * translation_gain
    rotation_delta = clipped[3:6] * ROTATION_GAIN

    visual_correction, visual_info = wrist_visual_alignment_correction(current_pose)
    state["last_wrist_alignment"] = visual_info
    if mode == "open" and visual_info["active"]:
        translation_delta[:2] += visual_correction
        if visual_info.get("world_error_norm_m", 0.0) > WRIST_ALIGNMENT_TOLERANCE_M:
            translation_delta[2] = max(
                translation_delta[2], -FINE_APPROACH_MAX_DESCENT_STEP_M
            )
    elif (
        WRIST_VISUAL_SERVO_ENABLED
        and mode == "open"
        and current_pose["position"][2] < WRIST_VISUAL_SERVO_STOP_Z_M
    ):
        # Keep the visually aligned XY target during the final straight descent.
        # The close-range mask is partially occluded by the fingers here.
        translation_delta[:2] = 0.0

    premature_close = (
        mode == "open"
        and clipped[6] < GRIPPER_CLOSE_TRIGGER
        and current_pose["position"][2] > MAX_VALID_GRASP_CLOSE_Z_M
    )
    if premature_close:
        clipped[6] = 1.0
        translation_delta[2] = min(
            translation_delta[2], -FINE_APPROACH_MAX_DESCENT_STEP_M
        )

    # Below the pregrasp clearance, retain the policy direction but limit lateral
    # sweeping and descent speed. This is a velocity limit, not a visual waypoint.
    fine_approach = mode == "open" and current_pose["position"][2] < FINE_APPROACH_START_Z_M
    if fine_approach:
        translation_delta[:2] = np.clip(
            translation_delta[:2],
            -FINE_APPROACH_MAX_XY_STEP_M,
            FINE_APPROACH_MAX_XY_STEP_M,
        )
        translation_delta[2] = max(
            translation_delta[2], -FINE_APPROACH_MAX_DESCENT_STEP_M
        )

    if mode == "carrying":
        previous_translation = state["filtered_carry_translation"]
        translation_delta = (
            CARRY_FILTER_ALPHA * translation_delta
            + (1.0 - CARRY_FILTER_ALPHA) * previous_translation
        )
        axis_limits = (
            PLACEMENT_AXIS_MAX_STEP_M
            if current_pose["position"][0] >= CARRY_DESCENT_START_X_M
            else CARRY_AXIS_MAX_STEP_M
        )
        translation_delta = np.clip(translation_delta, -axis_limits, axis_limits)
        state["filtered_carry_translation"] = translation_delta.copy()

        previous_rotation = state["filtered_carry_rotation"]
        rotation_delta = (
            CARRY_ROTATION_FILTER_ALPHA * rotation_delta
            + (1.0 - CARRY_ROTATION_FILTER_ALPHA) * previous_rotation
        )
        state["filtered_carry_rotation"] = rotation_delta.copy()
    else:
        state["filtered_carry_translation"][:] = 0.0
        state["filtered_carry_rotation"][:] = 0.0

    if mode == "closing" and state["hold_position"] is not None:
        # Let the physical fingers establish contact before lifting. The policy's
        # gripper output remains live, but pose deltas are deferred during closure.
        target_pos = state["hold_position"].copy()
        target_quat = state["hold_quat"].copy()
    else:
        target_pos = np.clip(
            state["reference_position"] + translation_delta,
            WORKSPACE_LOW,
            WORKSPACE_HIGH,
        )
        target_pos[2] = np.clip(
            current_pose["position"][2] + translation_delta[2],
            WORKSPACE_LOW[2],
            WORKSPACE_HIGH[2],
        )
        if mode == "carrying":
            target_pos[1] = np.clip(target_pos[1], *SAFE_CARRY_Y_BOUNDS_M)
        target_quat = quat_apply_world_delta(
            state["reference_quat"], rotation_delta
        )

    state["reference_position"] = target_pos.copy()
    state["reference_quat"] = target_quat.copy()
    qpos_target, ik_info = plan_pose_qpos(target_pos, target_quat)
    if not ik_info["converged"]:
        print("WARNING: 6D pose IK did not fully converge:", ik_info)

    mode_before_gripper_update = state["mode"]
    controller_gripper_fraction(clipped[6])
    if mode_before_gripper_update == "open" and state["mode"] == "closing":
        state["hold_position"] = target_pos.copy()
        state["hold_quat"] = target_quat.copy()

    applied_gripper_fraction = capped_grip_fraction(float(clipped[6]), state["mode"])
    state["applied_gripper_fraction"] = applied_gripper_fraction
    gripper_opening = (
        OPENVLA_GRIPPER_CLOSED_M
        + applied_gripper_fraction
        * (OPENVLA_GRIPPER_OPENING_M - OPENVLA_GRIPPER_CLOSED_M)
    )
    return (
        raw,
        clipped,
        qpos_target,
        target_pos,
        target_quat,
        gripper_opening,
        applied_gripper_fraction,
        ik_info,
    )


def current_gripper_command_m():
    tendon_actuator_ids = [
        actuator_id
        for actuator_id in range(model.nu)
        if model.actuator_trntype[actuator_id] == mujoco.mjtTrn.mjTRN_TENDON
    ]
    assert tendon_actuator_ids, "No Panda gripper tendon actuator found."
    actuator_id = tendon_actuator_ids[0]
    return float(np.clip(data.ctrl[actuator_id] / 255.0 * 0.04, 0.0, 0.04))


def step_controller(
    qpos_target, target_pos, target_quat, gripper_opening, qpos_frames, tactile_recorder=None
):
    """Execute one 10 Hz target through MuJoCo's physical position actuators."""
    target_pos = np.asarray(target_pos, dtype=np.float64)
    target_quat = quat_normalize(target_quat)
    desired_qpos = np.asarray(
        [qpos_target[name] for name in ARM_JOINT_NAMES], dtype=np.float64
    )
    current_qpos = data.qpos[ARM_QPOS_ADR].copy()
    start_arm_qpos = current_qpos.copy()
    target_values = current_qpos + ACTUATOR_LEAD * (desired_qpos - current_qpos)
    target_values = np.asarray([
        np.clip(value, *model.jnt_range[joint_id])
        for value, joint_id in zip(target_values, ARM_JOINT_IDS)
    ], dtype=np.float64)
    steps = max(1, int(round(CONTROL_HORIZON_S / model.opt.timestep)))
    frame_every = max(1, int(round((1.0 / VIDEO_FPS) / model.opt.timestep)))
    start_gripper_command = current_gripper_command_m()
    max_position_error_m = 0.0
    max_orientation_error_rad = 0.0

    for i in range(steps):
        alpha = smoothstep01((i + 1) / steps)
        gripper_command = (
            (1.0 - alpha) * start_gripper_command
            + alpha * gripper_opening
        )
        interpolated_arm_command = {
            name: float(
                (1.0 - alpha) * start_arm_qpos[index]
                + alpha * target_values[index]
            )
            for index, name in enumerate(ARM_JOINT_NAMES)
        }
        set_arm_position_targets(interpolated_arm_command)
        set_gripper_opening(gripper_command)
        mujoco.mj_step(model, data)
        if tactile_recorder is not None:
            tactile_recorder.after_physics_step()

        pose = get_ee_pose()
        max_position_error_m = max(
            max_position_error_m,
            float(np.linalg.norm(target_pos - pose["position"])),
        )
        max_orientation_error_rad = max(
            max_orientation_error_rad,
            float(np.linalg.norm(quat_error_world(
                target_quat, pose["quat_wxyz"]
            ))),
        )
        if (i + 1) % frame_every == 0 or i == steps - 1:
            qpos_frames.append(data.qpos.copy())

    final_pose = get_ee_pose()
    endpoint_position_error_m = float(np.linalg.norm(
        target_pos - final_pose["position"]
    ))
    endpoint_orientation_error_rad = float(np.linalg.norm(
        quat_error_world(target_quat, final_pose["quat_wxyz"])
    ))
    return {
        "mode": "mujoco_smoothed_position_actuators_no_state_projection",
        "physics_steps": int(steps),
        "feedback_projection_steps": 0,
        "reference_ik_failures": 0,
        "max_pre_correction_position_error_m": max_position_error_m,
        "max_pre_correction_orientation_error_rad": max_orientation_error_rad,
        "endpoint_position_error_m": endpoint_position_error_m,
        "endpoint_orientation_error_rad": endpoint_orientation_error_rad,
        "endpoint_within_tolerance": bool(
            endpoint_position_error_m < 0.003
            and endpoint_orientation_error_rad < np.deg2rad(1.0)
        ),
    }


def settle_after_release(qpos_target, qpos_frames, tactile_recorder=None):
    """Keep the released pose stable long enough for the cube to settle."""
    steps = max(1, int(round(RELEASE_SETTLE_S / model.opt.timestep)))
    frame_every = max(1, int(round((1.0 / VIDEO_FPS) / model.opt.timestep)))
    for i in range(steps):
        set_arm_position_targets(qpos_target)
        set_gripper_opening(OPENVLA_GRIPPER_OPENING_M)
        mujoco.mj_step(model, data)
        if tactile_recorder is not None:
            tactile_recorder.after_physics_step()
        if (i + 1) % frame_every == 0 or i == steps - 1:
            qpos_frames.append(data.qpos.copy())


def make_grasp_diagnostic_cameras(cube_start):
    cube_start = np.asarray(cube_start, dtype=np.float64)
    grasp_lookat = cube_start + np.array([0.0, 0.0, 0.055], dtype=np.float64)

    # Head-on view of the finger/cube contact at the initial grasp location.
    grasp_front_cam = mujoco.MjvCamera()
    grasp_front_cam.distance = 0.58
    grasp_front_cam.azimuth = 180
    grasp_front_cam.elevation = -12
    grasp_front_cam.lookat[:] = grasp_lookat

    # Lower view from the opposite table corner, also fixed on the initial cube.
    grasp_corner_cam = mujoco.MjvCamera()
    grasp_corner_cam.distance = 0.72
    grasp_corner_cam.azimuth = 315
    grasp_corner_cam.elevation = -16
    grasp_corner_cam.lookat[:] = grasp_lookat

    return grasp_front_cam, grasp_corner_cam


def render_rollout_videos(
    qpos_frames,
    cube_start,
    world_path,
    wrist_path,
    grasp_front_path,
    grasp_corner_path,
    final_frame_path=None,
):
    saved_qpos = data.qpos.copy()
    saved_qvel = data.qvel.copy()

    world_cam = make_third_person_camera() if "make_third_person_camera" in globals() else mujoco.MjvCamera()
    grasp_front_cam, grasp_corner_cam = make_grasp_diagnostic_cameras(cube_start)
    world_renderer = mujoco.Renderer(model, height=480, width=640)
    wrist_renderer = mujoco.Renderer(model, height=WRIST_IMAGE_SIZE, width=WRIST_IMAGE_SIZE)
    grasp_front_renderer = mujoco.Renderer(model, height=480, width=640)
    grasp_corner_renderer = mujoco.Renderer(model, height=480, width=640)

    world_frames, wrist_frames = [], []
    grasp_front_frames, grasp_corner_frames = [], []
    try:
        for qpos in qpos_frames:
            data.qpos[:] = qpos
            data.qvel[:] = 0.0
            mujoco.mj_forward(model, data)
            world_frames.append(render_rgb(world_renderer, world_cam))
            wrist_frames.append(render_rgb(wrist_renderer, AGENT_CAMERA_NAME))
            grasp_front_frames.append(render_rgb(grasp_front_renderer, grasp_front_cam))
            grasp_corner_frames.append(render_rgb(grasp_corner_renderer, grasp_corner_cam))
    finally:
        world_renderer.close()
        wrist_renderer.close()
        grasp_front_renderer.close()
        grasp_corner_renderer.close()
        data.qpos[:] = saved_qpos
        data.qvel[:] = saved_qvel
        mujoco.mj_forward(model, data)

    imageio.mimsave(world_path, world_frames, fps=VIDEO_FPS)
    imageio.mimsave(wrist_path, wrist_frames, fps=VIDEO_FPS)
    imageio.mimsave(grasp_front_path, grasp_front_frames, fps=VIDEO_FPS)
    imageio.mimsave(grasp_corner_path, grasp_corner_frames, fps=VIDEO_FPS)

    if final_frame_path is not None:
        Image.fromarray(world_frames[-1]).save(final_frame_path)

    return {
        "grasp_front": {
            "distance": float(grasp_front_cam.distance),
            "azimuth": float(grasp_front_cam.azimuth),
            "elevation": float(grasp_front_cam.elevation),
            "lookat": grasp_front_cam.lookat.astype(float).tolist(),
        },
        "grasp_corner_low": {
            "distance": float(grasp_corner_cam.distance),
            "azimuth": float(grasp_corner_cam.azimuth),
            "elevation": float(grasp_corner_cam.elevation),
            "lookat": grasp_corner_cam.lookat.astype(float).tolist(),
        },
    }

if RUN_CLOSED_LOOP_ROLLOUTS:
    worker = None
    rng = np.random.default_rng(ROLLOUT_SEED)
    all_logs = []
    dataset_kept_count = _resume_dataset_kept_count if "_resume_dataset_kept_count" in globals() else 0
    attempt_summary = []

    ## Start the model worker and each rollout
    try:
        worker, worker_tail = start_worker()

        rollout_corner_indices = (
            SELECTED_OOD_START_INDICES if SELECTED_OOD_START_INDICES is not None
            else list(range(N_ROLLOUTS))
        )
        for rollout_idx, corner_idx in enumerate(rollout_corner_indices):
            rollout_dir = run_dir / f"rollout_{rollout_idx:03d}"
            rollout_dir.mkdir(parents=True, exist_ok=True)

            start_info = randomize_start(rng, corner_idx)

            # --- Per-rollout randomization (Plan B): OpenVLA's own behavior is untouched by all of
            # this -- only how the physics/controller responds underneath it varies. -------------
            if SLIP_RANDOMIZE_MATERIALS:
                _torsional_friction = float(rng.uniform(*SLIP_TORSIONAL_FRICTION_RANGE))
                model.geom_friction[GELSIGHT_GEL_PAD_GEOM_ID_RIGHT, 1] = _torsional_friction
                model.geom_friction[GELSIGHT_GEL_PAD_GEOM_ID_LEFT, 1] = _torsional_friction
                model.geom_friction[_cube_contact_geom_ids, 1] = _torsional_friction
                _cube_mass_scale = float(rng.uniform(*SLIP_CUBE_MASS_SCALE_RANGE))
                model.body_mass[BLOCK_BODY_ID] = _base_cube_mass_kg * _cube_mass_scale
                model.body_inertia[BLOCK_BODY_ID] = _base_cube_inertia * _cube_mass_scale
            else:
                _torsional_friction, _cube_mass_scale = GEL_CUBE_TORSIONAL_FRICTION, 1.0
            if SLIP_RANDOMIZE_GRIP_MARGIN:
                GRIP_LOCK_EXTRA_MARGIN = float(rng.uniform(*SLIP_GRIP_LOCK_EXTRA_MARGIN_RANGE))
            _perturbation_direction = rng.normal(size=3)
            _perturbation_direction[2] *= 0.3  # mostly horizontal -- more like a real external bump.
            _perturbation_direction /= max(float(np.linalg.norm(_perturbation_direction)), 1e-9)
            perturbation_state = {
                "pending": bool(SLIP_PERTURBATION_ENABLED and rng.random() < SLIP_PERTURBATION_PROBABILITY),
                "triggered": False,
                "phase_entry_time_s": None,
                "active_until_s": None,
                "delay_s": float(rng.uniform(0.2, 1.5)),
                "force_vec": _perturbation_direction * float(rng.uniform(*SLIP_PERTURBATION_FORCE_RANGE_N)),
            }
            data.xfrc_applied[BLOCK_BODY_ID, :3] = 0.0  # clear any stale force from a previous rollout.

            reset_controller_state()
            qpos_frames = [data.qpos.copy()]
            tactile_recorder = InstrumentedDualFOTSRecorder(rollout_dir, FOTS_SENSOR_RIGHT, FOTS_SENSOR_LEFT)
            tactile_recorder.capture_initial()
            tactile_outputs = None
            records = []
            rollout_termination_reason = "maximum_action_budget"

            policy_camera = make_third_person_camera()
            policy_renderer = mujoco.Renderer(
                model,
                height=POLICY_IMAGE_HEIGHT,
                width=POLICY_IMAGE_WIDTH,
            )
            wrist_alignment_renderer = mujoco.Renderer(
                model, height=WRIST_IMAGE_SIZE, width=WRIST_IMAGE_SIZE
            )

            try:
                executed_action_count = 0
                query_idx = 0
                policy_queries = []
                while executed_action_count < MAX_EXECUTED_ACTIONS:
                    policy_rgb = render_rgb(policy_renderer, policy_camera)
                    image_path = rollout_dir / f"query_{query_idx:03d}_third_person.png"
                    Image.fromarray(policy_rgb).save(image_path)

                    query_proprio = current_oft_proprio()
                    action_chunk, pred_info = predict_action_chunk(
                        worker,
                        image_path,
                        query_proprio,
                        request_id=query_idx,
                    )
                    model_h0_action = action_chunk[0].copy()
                    (
                        controller_action,
                        selected_first_horizon,
                        gripper_lookahead_fraction,
                    ) = select_controller_action(action_chunk)
                    controller_phase_before = CONTROLLER_STATE["mode"]
                    steps_this_query = min(
                        ACTIONS_PER_CHUNK,
                        MAX_EXECUTED_ACTIONS - executed_action_count,
                    )
                    query_record = {
                        "query": int(query_idx),
                        "image": str(image_path),
                        "proprio_input": query_proprio.tolist(),
                        "predicted_action_chunk": action_chunk.tolist(),
                        "model_h0_action": model_h0_action.tolist(),
                        "controller_source_action": controller_action.tolist(),
                        "selected_first_horizon": int(selected_first_horizon),
                        "gripper_lookahead_fraction": float(gripper_lookahead_fraction),
                        "executed_chunk_actions": int(steps_this_query),
                        "inference_s": float(pred_info["inference_s"]),
                    }

                    destination_reached = False
                    for chunk_step in range(steps_this_query):
                        maybe_apply_perturbation(perturbation_state)
                        measured_proprio_before = current_oft_proprio()
                        (
                            controller_source_raw,
                            clipped,
                            qpos_target,
                            ee_target,
                            ee_target_quat,
                            gripper_opening,
                            applied_gripper_fraction,
                            ik_info,
                        ) = adapt_openvla_action(controller_action)
                        controller_phase_after = CONTROLLER_STATE["mode"]
                        gripper_before_m = current_finger_opening_m()
                        tactile_frame_start = tactile_recorder.frame_count
                        controller_info = step_controller(
                            qpos_target,
                            ee_target,
                            ee_target_quat,
                            gripper_opening,
                            qpos_frames,
                            tactile_recorder=tactile_recorder,
                        )
                        gripper_after_m = current_finger_opening_m()
                        finger_qpos_after_m = np.asarray(
                            data.qpos[FINGER_QPOS_ADR], dtype=np.float64
                        ).copy()
                        measured_proprio_after = current_oft_proprio()

                        ee_pose_after = get_ee_pose()
                        ee_pos = ee_pose_after["position"].copy()
                        orientation_error_deg = float(np.rad2deg(np.linalg.norm(
                            quat_error_world(ee_target_quat, ee_pose_after["quat_wxyz"])
                        )))
                        cube_pos = get_cube_pose()["position"].copy()
                        rec = {
                            "action_step": int(executed_action_count),
                            "query": int(query_idx),
                            "chunk_step": int(chunk_step),
                            "selected_chunk_horizon": int(selected_first_horizon),
                            "image": str(image_path),
                            "camera_view": POLICY_CAMERA_VIEW,
                            "source": "fine_tuned_openvla_oft_horizon_0_replanning",
                            "query_proprio_input": query_proprio.tolist(),
                            "measured_proprio_before_action": measured_proprio_before.tolist(),
                            "measured_proprio_after_action": measured_proprio_after.tolist(),
                            "raw_openvla_action": model_h0_action.tolist(),
                            "controller_source_action": controller_source_raw.tolist(),
                            "clipped_controller_action": clipped.tolist(),
                            "wrist_alignment": CONTROLLER_STATE.get("last_wrist_alignment"),
                            "controller_phase_before_action": controller_phase_before,
                            "controller_phase_after_planning": controller_phase_after,
                            "gripper_lookahead_fraction": float(gripper_lookahead_fraction),
                            "ee_target": ee_target.tolist(),
                            "ee_target_quat_wxyz": ee_target_quat.tolist(),
                            "ee_after": ee_pos.tolist(),
                            "ee_after_quat_wxyz": ee_pose_after["quat_wxyz"].tolist(),
                            "rotation_command_deg": np.rad2deg(clipped[3:6]).tolist(),
                            "orientation_tracking_error_deg": orientation_error_deg,
                            "pose_ik": ik_info,
                            "cartesian_controller": controller_info,
                            "cube_after_eval_only": cube_pos.tolist(),
                            "gripper_opening_m": float(gripper_opening),
                            "gripper_target_open_fraction": float(applied_gripper_fraction),
                            "gripper_controller_source_open_fraction": float(clipped[6]),
                            "gripper_target_per_finger_m": float(gripper_opening),
                            "gripper_before_per_finger_m": float(gripper_before_m),
                            "gripper_after_per_finger_m": float(gripper_after_m),
                            "gripper_after_total_width_m": float(2.0 * gripper_after_m),
                            "finger_qpos_after_m": finger_qpos_after_m.tolist(),
                            "tactile_frame_start": int(tactile_frame_start),
                            "tactile_frame_end_exclusive": int(tactile_recorder.frame_count),
                        }
                        records.append(rec)
                        print(
                            f"[rollout {rollout_idx} q{query_idx:02d} h0] "
                            f"step={executed_action_count:03d} "
                            f"phase={controller_phase_before}->{controller_phase_after} "
                            f"selected_h={selected_first_horizon} "
                            f"model_h0={np.round(model_h0_action, 4).tolist()} "
                            f"control={np.round(clipped, 4).tolist()} "
                            f"position_error_mm={controller_info['endpoint_position_error_m'] * 1000.0:.3f} "
                            f"ee={np.round(ee_pos, 3).tolist()} "
                            f"cube={np.round(cube_pos, 3).tolist()}"
                        )
                        executed_action_count += 1
                        if cube_at_destination(cube_pos):
                            destination_reached = True
                            break

                    policy_queries.append(query_record)
                    query_idx += 1
                    if destination_reached:
                        settle_after_release(
                            qpos_target, qpos_frames, tactile_recorder=tactile_recorder
                        )
                        rollout_termination_reason = "cube_reached_destination"
                        break
            finally:
                policy_renderer.close()
                wrist_alignment_renderer.close()
                tactile_outputs = tactile_recorder.finalize()

            final_cube = get_cube_pose()["position"].copy()
            success = cube_at_destination(final_cube)

            world_video = rollout_dir / "world.mp4"
            wrist_video = rollout_dir / "wrist.mp4"
            grasp_front_video = rollout_dir / "grasp_front.mp4"
            grasp_corner_video = rollout_dir / "grasp_corner_low.mp4"
            final_frame_photo = rollout_dir / "final_frame.png"
            diagnostic_camera_config = render_rollout_videos(
                qpos_frames,
                start_info["cube_start"],
                world_video,
                wrist_video,
                grasp_front_video,
                grasp_corner_video,
                final_frame_path=final_frame_photo,
            )

            payload = {
                "checkpoint": str(checkpoint_dir),
                "checkpoint_dataset": checkpoint_info["dataset"],
                "checkpoint_step": int(checkpoint_info["step"]),
                "tactile_sensors": ["right", "left"],
                "prompt": OPENVLA_ROLLOUT_PROMPT,
                "closed_loop_note": (
                    "No TFDS/expert actions are used. OpenVLA-OFT receives live image, "
                    "language, and 8D proprio, then replans after one horizon-0 action. "
                    "Horizons 1-7 are diagnostics only. Arm and gripper targets are "
                    "smoothly applied through MuJoCo actuators without state projection."
                ),
                "action_chunk_length": 8,
                "actions_executed_per_query": int(ACTIONS_PER_CHUNK),
                "max_executed_actions": int(MAX_EXECUTED_ACTIONS),
                "rollout_termination_reason": rollout_termination_reason,
                "proprio_convention": "[q1, q2, q3, q4, q5, q6, q7, measured_gripper_open_fraction]",
                "controller_mode": "hybrid_delta_reference_physical_actuators",
                "rotation_action_convention": "action[3:6] is a world-frame rotation vector in radians",
                "gripper_action_convention": (
                    "continuous policy open fraction applied directly and interpolated within "
                    "each 100 ms interval; 0=closed, 1=open; per-finger target is 0.00-0.04 m"
                ),
                "controller_parameters": {
                    "approach_translation_gain": APPROACH_TRANSLATION_GAIN.tolist(),
                    "carry_translation_gain": CARRY_TRANSLATION_GAIN.tolist(),
                    "carry_max_translation_m": float(CARRY_MAX_TRANSLATION_M),
                    "rotation_gain": float(ROTATION_GAIN),
                    "actuator_lead": float(ACTUATOR_LEAD),
                    "vertical_reference": "live measured end-effector z",
                    "horizontal_reference": "persistent commanded x/y",
                    "gripper_close_trigger": float(GRIPPER_CLOSE_TRIGGER),
                    "gripper_open_trigger": float(GRIPPER_OPEN_TRIGGER),
                    "lift_bootstrap_height_m": float(LIFT_BOOTSTRAP_HEIGHT_M),
                    "release_clearance_m": float(RELEASE_CLEARANCE_M),
                },
                "max_rotation_per_action_deg": float(np.rad2deg(OPENVLA_MAX_ROTATION_RAD)),
                "policy_camera": {
                    "view": POLICY_CAMERA_VIEW,
                    "width": int(POLICY_IMAGE_WIDTH),
                    "height": int(POLICY_IMAGE_HEIGHT),
                    "distance": float(policy_camera.distance),
                    "azimuth": float(policy_camera.azimuth),
                    "elevation": float(policy_camera.elevation),
                    "lookat": policy_camera.lookat.astype(float).tolist(),
                },
                "diagnostic_cameras": diagnostic_camera_config,
                "gelsight_tactile": tactile_outputs,
                "start_info": start_info,
                "experiment_dataset": EXPERIMENT_DATASET,
                "experiment_condition": EXPERIMENT_CONDITION,
                "cube_size_scale": float(CUBE_SIZE_SCALE),
                "light_dim_factor": float(LIGHT_DIM_FACTOR),
                "occluder_plant_enabled": bool(OCCLUDER_PLANT_ENABLED),
                "occluder_plant_position_m": [float(OCCLUDER_PLANT_POS_M[0]), float(OCCLUDER_PLANT_POS_M[1])],
                "occluder_plant_scale": float(OCCLUDER_PLANT_SCALE),
                "success_rough_eval_only": success,
                "outcome": "accepted" if success else "failed",
                "final_cube_pos": final_cube.tolist(),
                "policy_queries": policy_queries,
                "records": records,
                "videos": {
                    "world": str(world_video),
                    "wrist": str(wrist_video),
                    "grasp_front": str(grasp_front_video),
                    "grasp_corner_low": str(grasp_corner_video),
                    "final_frame": str(final_frame_photo),
                    "gelsight_right_rgb": tactile_outputs["right"]["tactile_rgb_video"],
                    "gelsight_right_marker_flow": tactile_outputs["right"]["marker_flow_video"],
                    "gelsight_left_rgb": tactile_outputs["left"]["tactile_rgb_video"],
                    "gelsight_left_marker_flow": tactile_outputs["left"]["marker_flow_video"],
                },
            }
            log_path = rollout_dir / "closed_loop_log.json"
            payload_text = json.dumps(payload, indent=2)
            log_path.write_text(payload_text, encoding="utf-8")
            all_logs.append(payload)

            print(f"\n===== {EXPERIMENT_DATASET}/{EXPERIMENT_CONDITION} -- rollout {rollout_idx:03d} of {len(rollout_corner_indices):03d} =====")
            print("Saved log:", log_path)
            print("World video:", world_video)
            print("Wrist video:", wrist_video)
            print("Grasp-front video:", grasp_front_video)
            print("Low corner video:", grasp_corner_video)
            print("Final frame photo:", final_frame_photo)
            for side in ("right", "left"):
                side_outputs = tactile_outputs[side]
                side_validation = side_outputs["validation"]
                print(f"[{side}] FOTS tactile RGB video:", side_outputs["tactile_rgb_video"])
                print(f"[{side}] FOTS marker flow video:", side_outputs["marker_flow_video"])
                print(f"[{side}] FOTS quantitative data:", side_outputs["quantitative_npz"])
                print(
                    f"[{side}] GelSight validation: {side_validation['status']} "
                    f"(frames={side_validation['frame_count']}, "
                    f"max active taxels={side_validation['max_active_taxels']}, "
                    f"max depth diff={side_validation['max_depth_diff_mm']:.3f} mm, "
                    f"contact held at end={side_validation['contact_held_at_end']})"
                )
            print("Termination:", rollout_termination_reason)
            print("Rough success:", success)
            print("Outcome:", "ACCEPTED (cube reached the far side)" if success else "FAILED")
            if DISPLAY_ROLLOUT_VIDEOS:
                display(Video(str(world_video), embed=False, width=640))
                display(Video(str(wrist_video), embed=False, width=320))
                display(Video(str(grasp_front_video), embed=False, width=480))
                display(Video(str(grasp_corner_video), embed=False, width=480))
            if DISPLAY_GELSIGHT_VIDEOS:
                for side in ("right", "left"):
                    print(f"{side.capitalize()} GelSight (tactile RGB / marker flow):")
                    display(Video(tactile_outputs[side]["tactile_rgb_video"], embed=False, width=320))
                    display(Video(tactile_outputs[side]["marker_flow_video"], embed=False, width=320))

            # --- T3 dataset curation: keep any demo with a REAL grasp attempt, success or not -----
            # Unlike T2, task success is NOT required -- a partial slip or an outright drop mid
            # -carry is exactly the positive-class data this dataset needs. Only total misses (the
            # gripper never meaningfully touched the cube on either side) are discarded.
            is_dataset_demo = False
            disqualified_reason = "no_real_grasp_attempt"
            _side_contact_stats = {}
            for side in ("right", "left"):
                _side_taxels = np.load(tactile_outputs[side]["quantitative_npz"])["active_taxel_count"]
                _contact_frames = int((_side_taxels > 0).sum())
                _peak_taxels = int(_side_taxels.max()) if _side_taxels.size else 0
                _side_contact_stats[side] = {"contact_frames": _contact_frames, "peak_taxels": _peak_taxels}
                if _contact_frames >= DATASET_MIN_CONTACT_FRAMES and _peak_taxels >= DATASET_MIN_PEAK_TAXELS:
                    is_dataset_demo = True
                    disqualified_reason = None
            if not is_dataset_demo:
                print(
                    f"  -> discarded, no real grasp attempt on either sensor "
                    f"(need >= {DATASET_MIN_CONTACT_FRAMES} contact frames [{DATASET_MIN_CONTACT_SECONDS:.1f}s] and >= {DATASET_MIN_PEAK_TAXELS} px "
                    f"on at least one side; right={_side_contact_stats['right']}, left={_side_contact_stats['left']})"
                )
            _ground_truth_summary = tactile_outputs.get("ground_truth", {})

            attempt_record = {
                "attempt": int(rollout_idx),
                "start_xy": [float(v) for v in start_info["nominal_untextured_cube_start"][:2]],
                "final_cube_pos": final_cube.tolist(),
                "outcome": payload["outcome"],
                "termination_reason": rollout_termination_reason,
                "kept_as_dataset_demo": bool(is_dataset_demo),
                "disqualified_reason": disqualified_reason,
                "torsional_friction": _torsional_friction,
                "cube_mass_scale": _cube_mass_scale,
                "grip_lock_extra_margin": float(GRIP_LOCK_EXTRA_MARGIN),
                "perturbation_applied": bool(perturbation_state["triggered"]),
                "grasp_established": bool(_ground_truth_summary.get("grasp_established", False)),
                "holding_frames": int(_ground_truth_summary.get("holding_frames", 0)),
                "slip_frames": int(_ground_truth_summary.get("slip_frames", 0)),
                "slip_fraction_of_holding": _ground_truth_summary.get("slip_fraction_of_holding"),
                "dataset_dir": None,
            }
            attempt_summary.append(attempt_record)

            if EXPERIMENT_DATASET and DATASET_KEEP_ONLY_QUALIFYING:
                if is_dataset_demo:
                    demo_dir = run_dir / f"demo_{dataset_kept_count:03d}"
                    old_prefix = str(rollout_dir)
                    rollout_dir.rename(demo_dir)
                    # Every path Cell P wrote into closed_loop_log.json (videos, tactile files) was
                    # built from the pre-rename rollout_dir; the rename physically moved those files,
                    # so patch the recorded strings to match with one substring replace (every path
                    # shares the exact same absolute prefix) instead of rebuilding the payload dict.
                    (demo_dir / "closed_loop_log.json").write_text(
                        payload_text.replace(old_prefix, str(demo_dir)), encoding="utf-8"
                    )
                    attempt_record["dataset_dir"] = str(demo_dir)
                    dataset_kept_count += 1
                    print(f"KEPT as {demo_dir.name}  (T3 dataset total so far: {dataset_kept_count}/{DATASET_TARGET_DEMOS})")
                else:
                    shutil.rmtree(rollout_dir)
                    print(f"DISCARDED rollout_{rollout_idx:03d} (not kept in the T3 dataset)")

                if dataset_kept_count >= DATASET_TARGET_DEMOS:
                    print(
                        f"\nReached the target of {DATASET_TARGET_DEMOS} demos after "
                        f"{rollout_idx + 1} attempt(s) -- stopping early (N_ROLLOUTS={N_ROLLOUTS} was only a ceiling)."
                    )
                    break

    finally:
        if worker is not None:
            try:
                worker.stdin.close()
            except Exception:
                pass
            worker.terminate()
            try:
                worker.wait(timeout=30)
            except subprocess.TimeoutExpired:
                worker.kill()
                worker.wait(timeout=30)

    print("Closed-loop output directory:", run_dir)

    if EXPERIMENT_DATASET:
        n_accepted = sum(1 for log in all_logs if log["success_rough_eval_only"])
        n_kept_this_run = sum(1 for a in attempt_summary if a["kept_as_dataset_demo"])
        n_kept_total = dataset_kept_count if "dataset_kept_count" in globals() else n_kept_this_run
        print(
            f"\n{EXPERIMENT_DATASET}/{EXPERIMENT_CONDITION}: {len(attempt_summary)} attempt(s) this run, "
            f"{n_accepted} reached the target region, {n_kept_this_run} kept as T3 dataset demos "
            f"(dataset total on disk: {n_kept_total}/{DATASET_TARGET_DEMOS} target)"
        )
        for a in attempt_summary:
            tag = "KEPT:" + Path(a["dataset_dir"]).name if a["dataset_dir"] else f"DISCARDED ({a['disqualified_reason']})"
            print(
                f"  attempt_{a['attempt']:03d}: start_xy={np.round(a['start_xy'], 4).tolist()} outcome={a['outcome']:8s} "
                f"friction={a['torsional_friction']:.3f} mass_x={a['cube_mass_scale']:.2f} "
                f"perturbed={a['perturbation_applied']} slip_frames={a['slip_frames']}/{a['holding_frames']} -> {tag}"
            )

        results_csv_path = run_dir / "results.csv"
        results_columns = [
            "attempt", "start_x", "start_y", "outcome",
            "final_cube_x", "final_cube_y", "final_cube_z",
            "termination_reason", "kept_as_dataset_demo", "disqualified_reason", "dataset_dir",
            "torsional_friction", "cube_mass_scale", "grip_lock_extra_margin", "perturbation_applied",
            "grasp_established", "holding_frames", "slip_frames", "slip_fraction_of_holding",
        ]
        # Merge with any earlier run's results.csv (e.g. an earlier session that produced
        # demo_000/demo_001 before this one resumed and added more) instead of overwriting it --
        # every demo_NNN folder on disk should stay indexed, however many sessions it took to
        # collect them. "attempt" numbers are per-run (reset each time this cell runs), so rows are
        # kept in file order (earlier runs first) rather than re-sorted by that column.
        results_rows = []
        if results_csv_path.is_file():
            with open(results_csv_path, newline="") as f:
                reader = csv.DictReader(f)
                if reader.fieldnames == results_columns:
                    results_rows = [[row[col] for col in results_columns] for row in reader]
                else:
                    print(f"WARNING: {results_csv_path} has an unexpected header; not merging its rows.")
        for a in attempt_summary:
            fc = a["final_cube_pos"]
            results_rows.append([
                a["attempt"], a["start_xy"][0], a["start_xy"][1], a["outcome"],
                fc[0], fc[1], fc[2], a["termination_reason"],
                a["kept_as_dataset_demo"], a["disqualified_reason"] or "", a["dataset_dir"] or "",
                a["torsional_friction"], a["cube_mass_scale"], a["grip_lock_extra_margin"], a["perturbation_applied"],
                a["grasp_established"], a["holding_frames"], a["slip_frames"], a["slip_fraction_of_holding"],
            ])
        with open(results_csv_path, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(results_columns)
            writer.writerows(results_rows)
        print(f"Per-attempt results table ({len(results_rows)} attempt(s) across every session so far): {results_csv_path}")
        if n_kept_total < DATASET_TARGET_DEMOS:
            print(
                f"\nWARNING: only {n_kept_total}/{DATASET_TARGET_DEMOS} demos on disk "
                f"after {len(attempt_summary)} attempt(s) (N_ROLLOUTS={N_ROLLOUTS} ceiling reached first). "
                "Re-run this cell -- it resumes from the existing demo_* folders (see the run_dir setup above) "
                "instead of starting over."
            )
else:
    print(
        "RUN_CLOSED_LOOP_ROLLOUTS=False: dual-GelSight scene and both FOTS sensors are built and "
        "self-tested; stopping before the OpenVLA worker starts. Run the two-sensor test cells at "
        "the end of the notebook next."
    )


In [ ]:
# Cell P2: OpenVLA-OFT chunk/controller/robot/proprio diagnostics.

print("Closed-loop output directory:", run_dir)

from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image as NotebookImage, display


DIAGNOSTIC_RUN_DIR = Path(run_dir)

SIGN_THRESHOLD_MM = 0.10
ROTATION_SIGN_THRESHOLD_DEG = 0.02

required_rotation_helpers = [
    "quat_apply_world_delta",
    "quat_error_world",
]
missing_helpers = [
    name for name in required_rotation_helpers if name not in globals()
]
assert not missing_helpers, (
    "Run Cell P0 first. Missing: " + ", ".join(missing_helpers)
)

axis_colors = {
    "X": "#376fd0",
    "Y": "#a0459b",
    "Z": "#e08b2c",
}
translation_names = ("X", "Y", "Z")
rotation_names = ("Roll", "Pitch", "Yaw")

# "rollout_*" (OOD-evaluation runs, e.g. T1/T1_duo) and "demo_*" (curated dataset runs, e.g. T2 --
# see Cell P's DATASET_KEEP_ONLY_SUCCESSFUL) are the two folder-naming schemes Cell P produces.
log_paths = sorted(DIAGNOSTIC_RUN_DIR.glob("rollout_*/closed_loop_log.json")) + sorted(
    DIAGNOSTIC_RUN_DIR.glob("demo_*/closed_loop_log.json")
)
assert log_paths, f"No rollout/demo logs found under {DIAGNOSTIC_RUN_DIR}"


for log_path in log_paths:
    payload = json.loads(log_path.read_text(encoding="utf-8"))
    records = payload["records"]
    assert records, f"No records in {log_path}"

    raw = np.asarray(
        [record["raw_openvla_action"] for record in records],
        dtype=np.float64,
    )
    clipped = np.asarray(
        [record["clipped_controller_action"] for record in records],
        dtype=np.float64,
    )

    # ------------------------------------------------------------------
    # Translation calculations
    # ------------------------------------------------------------------
    ee_target = np.asarray(
        [record["ee_target"] for record in records],
        dtype=np.float64,
    )
    ee_after = np.asarray(
        [record["ee_after"] for record in records],
        dtype=np.float64,
    )

    # Before action step s is the measured pose after action step s-1.
    ee_before = np.empty_like(ee_after)
    ee_before[1:] = ee_after[:-1]

    # Reconstruct the initial pose before the first command.
    ee_before[0] = ee_target[0] - clipped[0, :3]

    model_delta_mm = raw[:, :3] * 1000.0
    controller_delta_mm = (ee_target - ee_before) * 1000.0
    actual_delta_mm = (ee_after - ee_before) * 1000.0

    translation_mae = []
    translation_opposite_masks = []

    for axis_index, axis_name in enumerate(translation_names):
        model_values = model_delta_mm[:, axis_index]
        controller_values = controller_delta_mm[:, axis_index]
        actual_values = actual_delta_mm[:, axis_index]

        opposite_sign = (
            (np.abs(controller_values) >= SIGN_THRESHOLD_MM)
            & (np.abs(actual_values) >= SIGN_THRESHOLD_MM)
            & (np.sign(controller_values) != np.sign(actual_values))
        )
        mae_mm = float(
            np.mean(np.abs(controller_values - actual_values))
        )

        translation_mae.append(mae_mm)
        translation_opposite_masks.append(opposite_sign)

        print(
            f"{log_path.parent.name} {axis_name}: "
            f"OpenVLA-OFT requested total={model_values.sum():+.1f} mm | "
            f"controller target total={controller_values.sum():+.1f} mm | "
            f"robot moved total={actual_values.sum():+.1f} mm | "
            f"MAE={mae_mm:.3f} mm | "
            f"opposite-sign action steps={opposite_sign.sum()}"
        )

    # ------------------------------------------------------------------
    # Rotation calculations
    # ------------------------------------------------------------------
    target_quat = np.asarray(
        [record["ee_target_quat_wxyz"] for record in records],
        dtype=np.float64,
    )
    after_quat = np.asarray(
        [record["ee_after_quat_wxyz"] for record in records],
        dtype=np.float64,
    )

    # Before action step s is the measured orientation after action step s-1.
    before_quat = np.empty_like(after_quat)
    before_quat[1:] = after_quat[:-1]

    # Reconstruct the initial orientation before the first command.
    before_quat[0] = quat_apply_world_delta(
        target_quat[0],
        -clipped[0, 3:6],
    )

    model_rotation_deg = np.rad2deg(raw[:, 3:6])

    controller_rotation_deg = np.rad2deg(np.asarray([
        quat_error_world(target_quat[index], before_quat[index])
        for index in range(len(records))
    ]))

    actual_rotation_deg = np.rad2deg(np.asarray([
        quat_error_world(after_quat[index], before_quat[index])
        for index in range(len(records))
    ]))

    rotation_mae = []
    rotation_opposite_masks = []

    for axis_index, axis_name in enumerate(rotation_names):
        model_values = model_rotation_deg[:, axis_index]
        controller_values = controller_rotation_deg[:, axis_index]
        actual_values = actual_rotation_deg[:, axis_index]

        opposite_sign = (
            (np.abs(controller_values) >= ROTATION_SIGN_THRESHOLD_DEG)
            & (np.abs(actual_values) >= ROTATION_SIGN_THRESHOLD_DEG)
            & (np.sign(controller_values) != np.sign(actual_values))
        )
        mae_deg = float(
            np.mean(np.abs(controller_values - actual_values))
        )

        rotation_mae.append(mae_deg)
        rotation_opposite_masks.append(opposite_sign)

        print(
            f"{log_path.parent.name} {axis_name}: "
            f"model max={np.max(np.abs(model_values)):.3f} degrees | "
            f"controller max={np.max(np.abs(controller_values)):.3f} degrees | "
            f"robot max={np.max(np.abs(actual_values)):.3f} degrees | "
            f"MAE={mae_deg:.3f} degrees | "
            f"opposite-sign action steps={opposite_sign.sum()}"
        )

    # ------------------------------------------------------------------
    # Gripper calculations
    # ------------------------------------------------------------------
    missing_gripper_measurements = [
        record["query"]
        for record in records
        if "gripper_after_per_finger_m" not in record
    ]
    assert not missing_gripper_measurements, (
        "This rollout predates measured gripper logging. "
        "Run Cell P again, then run Cell P2 on its new run_dir."
    )

    gripper_open_m = float(
        globals().get("OPENVLA_GRIPPER_OPENING_M", 0.04)
    )
    gripper_closed_m = float(
        globals().get("OPENVLA_GRIPPER_CLOSED_M", 0.0)
    )
    opening_range = gripper_open_m - gripper_closed_m
    assert opening_range > 0.0, "Invalid gripper opening range."

    model_gripper = raw[:, 6]
    gripper_target_m = np.asarray(
        [record["gripper_target_per_finger_m"] for record in records],
        dtype=np.float64,
    )
    gripper_observed_m = np.asarray(
        [record["gripper_after_per_finger_m"] for record in records],
        dtype=np.float64,
    )
    controller_gripper = (
        gripper_target_m - gripper_closed_m
    ) / opening_range
    mujoco_gripper = (
        gripper_observed_m - gripper_closed_m
    ) / opening_range
    gripper_mae_fraction = float(
        np.mean(np.abs(controller_gripper - mujoco_gripper))
    )
    gripper_mae_per_finger_mm = (
        gripper_mae_fraction * opening_range * 1000.0
    )
    print(
        f"{log_path.parent.name} gripper: "
        f"controller-vs-MuJoCo MAE="
        f"{gripper_mae_per_finger_mm:.2f} mm per finger"
    )

    # ------------------------------------------------------------------
    # Combined figure: translation, rotation, and gripper
    # ------------------------------------------------------------------
    queries = np.arange(len(records))

    figure, axes = plt.subplots(
        3,
        1,
        figsize=(15, 13),
        sharex=True,
        constrained_layout=True,
    )

    figure.suptitle(
        f"{log_path.parent.name}: closed-loop command execution\n"
        "OpenVLA-OFT action vs controller target vs observed robot motion",
        fontsize=15,
    )

    # Translation panel
    axis = axes[0]

    translation_any_opposite = np.logical_or.reduce(
        translation_opposite_masks
    )
    axis.fill_between(
        queries,
        0,
        1,
        where=translation_any_opposite,
        transform=axis.get_xaxis_transform(),
        step="mid",
        color="#dc2626",
        alpha=0.10,
        label="Any controller/robot opposite sign",
    )

    for axis_index, axis_name in enumerate(translation_names):
        color = axis_colors[axis_name]

        axis.plot(
            queries,
            model_delta_mm[:, axis_index],
            color=color,
            linewidth=2.3,
            alpha=0.80,
            label=f"OpenVLA-OFT delta {axis_name}",
        )
        axis.plot(
            queries,
            controller_delta_mm[:, axis_index],
            color=color,
            linewidth=1.7,
            linestyle="--",
            label=f"Controller delta {axis_name}",
        )
        axis.plot(
            queries,
            actual_delta_mm[:, axis_index],
            color=color,
            linewidth=2.0,
            linestyle=":",
            label=f"Robot delta {axis_name}",
        )

    axis.axhline(0.0, color="#333333", linewidth=0.9)
    axis.set_title(
        "Translation commands | controller-vs-robot MAE: "
        f"X={translation_mae[0]:.3f}, "
        f"Y={translation_mae[1]:.3f}, "
        f"Z={translation_mae[2]:.3f} mm"
    )
    axis.set_ylabel("Millimetres / executed action")
    axis.grid(alpha=0.25)
    axis.legend(loc="upper right", fontsize=8, ncol=3)

    # Rotation panel
    axis = axes[1]

    rotation_any_opposite = np.logical_or.reduce(
        rotation_opposite_masks
    )
    axis.fill_between(
        queries,
        0,
        1,
        where=rotation_any_opposite,
        transform=axis.get_xaxis_transform(),
        step="mid",
        color="#dc2626",
        alpha=0.10,
        label="Any controller/robot opposite sign",
    )

    for axis_index, axis_name in enumerate(rotation_names):
        color = tuple(axis_colors.values())[axis_index]

        axis.plot(
            queries,
            model_rotation_deg[:, axis_index],
            color=color,
            linewidth=2.3,
            alpha=0.80,
            label=f"OpenVLA-OFT delta {axis_name}",
        )
        axis.plot(
            queries,
            controller_rotation_deg[:, axis_index],
            color=color,
            linewidth=1.7,
            linestyle="--",
            label=f"Controller delta {axis_name}",
        )
        axis.plot(
            queries,
            actual_rotation_deg[:, axis_index],
            color=color,
            linewidth=2.0,
            linestyle=":",
            label=f"Robot delta {axis_name}",
        )

    axis.axhline(0.0, color="#333333", linewidth=0.9)
    axis.set_title(
        "Rotational commands | controller-vs-robot MAE: "
        f"Roll={rotation_mae[0]:.3f}, "
        f"Pitch={rotation_mae[1]:.3f}, "
        f"Yaw={rotation_mae[2]:.3f} degrees"
    )
    axis.set_ylabel("Degrees / executed action")
    axis.grid(alpha=0.25)
    axis.legend(loc="upper right", fontsize=8, ncol=3)

    # Gripper panel
    axis = axes[2]

    axis.plot(
        queries,
        model_gripper,
        color="#2563eb",
        linewidth=2.3,
        label="OpenVLA-OFT gripper output",
    )
    axis.plot(
        queries,
        controller_gripper,
        color="#f59e0b",
        linewidth=1.9,
        linestyle="--",
        label="Controller target opening",
    )
    axis.plot(
        queries,
        mujoco_gripper,
        color="#16a34a",
        linewidth=2.1,
        label="MuJoCo observed opening",
    )
    axis.axhline(
        0.5,
        color="#20242a",
        linewidth=1.0,
        linestyle=":",
        label="Half-open reference",
    )

    axis.set_title(
        "Continuous gripper behavior | "
        f"controller-vs-MuJoCo MAE = "
        f"{gripper_mae_per_finger_mm:.2f} mm per finger"
    )
    axis.set_xlabel(
        f"Executed action step ({CONTROL_HORIZON_S:.3f} s control horizon)"
    )
    axis.set_ylabel("Open fraction")
    axis.set_ylim(-0.05, 1.05)
    axis.grid(alpha=0.25)
    axis.legend(loc="upper right", fontsize=8)

    output_path = (
        log_path.parent / "closed_loop_command_comparison.png"
    )
    figure.savefig(output_path, dpi=150)
    plt.close(figure)

    print("Saved:", output_path)
    display(NotebookImage(filename=str(output_path)))

    # ------------------------------------------------------------------
    # Measured proprio and the state supplied at each chunk boundary
    # ------------------------------------------------------------------
    measured_proprio = np.asarray(
        [record["measured_proprio_before_action"] for record in records],
        dtype=np.float64,
    )
    query_proprio = np.asarray(
        [record["query_proprio_input"] for record in records],
        dtype=np.float64,
    )
    chunk_steps = np.asarray(
        [record["chunk_step"] for record in records], dtype=np.int64
    )
    chunk_start_steps = np.flatnonzero(chunk_steps == 0)
    assert measured_proprio.shape == query_proprio.shape == (len(records), 8)

    proprio_figure, proprio_axes = plt.subplots(
        2, 1, figsize=(15, 9), sharex=True, constrained_layout=True
    )
    proprio_figure.suptitle(
        f"{log_path.parent.name}: live OFT proprioception",
        fontsize=15,
    )
    for joint_index in range(7):
        line = proprio_axes[0].plot(
            queries,
            measured_proprio[:, joint_index],
            linewidth=1.7,
            label=f"measured q{joint_index + 1}",
        )[0]
        proprio_axes[0].scatter(
            chunk_start_steps,
            query_proprio[chunk_start_steps, joint_index],
            color=line.get_color(),
            marker="o",
            s=22,
        )
    proprio_axes[0].set(
        title="Joint positions; circles mark the state supplied at each policy query",
        ylabel="Radians",
    )
    proprio_axes[0].grid(alpha=0.25)
    proprio_axes[0].legend(ncol=4, fontsize=8)

    proprio_axes[1].plot(
        queries,
        measured_proprio[:, 7],
        color="#18a058",
        linewidth=2.2,
        label="Measured before each action",
    )
    proprio_axes[1].scatter(
        chunk_start_steps,
        query_proprio[chunk_start_steps, 7],
        color="#dc3a3e",
        s=30,
        label="OFT query input",
    )
    proprio_axes[1].set(
        title="Measured gripper proprioception",
        xlabel=f"Executed action step ({CONTROL_HORIZON_S:.3f} s each)",
        ylabel="Open fraction",
        ylim=(-0.05, 1.05),
    )
    proprio_axes[1].grid(alpha=0.25)
    proprio_axes[1].legend()

    proprio_output_path = log_path.parent / "closed_loop_proprio.png"
    proprio_figure.savefig(proprio_output_path, dpi=150)
    plt.close(proprio_figure)
    print("Saved:", proprio_output_path)
    display(NotebookImage(filename=str(proprio_output_path)))


In [ ]:
# Cell P3: World video with live OFT translation, rotation, and gripper diagrams.
# Run Cell P2 immediately before this cell.
from pathlib import Path
import imageio.v2 as imageio
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from IPython.display import Video, display

required_m2_values = [
    "log_path", "payload", "records", "queries", "axis_colors",
    "translation_names", "rotation_names", "model_delta_mm",
    "controller_delta_mm", "actual_delta_mm", "model_rotation_deg",
    "controller_rotation_deg", "actual_rotation_deg", "model_gripper",
    "controller_gripper", "mujoco_gripper",
]
missing = [name for name in required_m2_values if name not in globals()]
assert not missing, "Run Cell P2 first. Missing: " + ", ".join(missing)

source_video = Path(payload["videos"]["world"])
if not source_video.is_absolute():
    source_video = log_path.parent / source_video
assert source_video.exists(), f"Missing world video: {source_video}"

motion_panels = [
    (
        "Translation commands",
        "mm / executed action",
        translation_names,
        (model_delta_mm, controller_delta_mm, actual_delta_mm),
        1.0,
    ),
    (
        "Rotational commands",
        "degrees / executed action",
        rotation_names,
        (
            model_rotation_deg,
            controller_rotation_deg,
            actual_rotation_deg,
        ),
        0.1,
    ),
]

source_labels = ("OpenVLA-OFT", "Controller", "MuJoCo")
source_styles = (("-", 1.8), ("--", 1.5), (":", 1.7))

output_path = log_path.parent / "world_with_live_diagnostics.mp4"
reader = imageio.get_reader(str(source_video))
output_fps = float(
    reader.get_meta_data().get("fps", globals().get("VIDEO_FPS", 10))
)
writer = imageio.get_writer(
    str(output_path),
    fps=output_fps,
    codec="libx264",
    pixelformat="yuv420p",
    macro_block_size=None,
)

figure, axes = plt.subplots(
    3,
    1,
    figsize=(9.6, 7.2),
    dpi=100,
    sharex=True,
    constrained_layout=True,
)

# Translation and rotation panels.
motion_lines = []
for panel_index, (title, ylabel, names, arrays, minimum) in enumerate(
    motion_panels
):
    panel_lines = []

    for dimension, name in enumerate(names):
        color = tuple(axis_colors.values())[dimension]
        dimension_lines = []

        for source_index, source_label in enumerate(source_labels):
            style, width = source_styles[source_index]
            line, = axes[panel_index].plot(
                [],
                [],
                color=color,
                linestyle=style,
                linewidth=width,
                alpha=0.85,
                label=f"{source_label} {name}",
            )
            dimension_lines.append(line)

        panel_lines.append(dimension_lines)

    all_values = np.concatenate([
        np.asarray(values).reshape(-1) for values in arrays
    ])
    limit = max(
        minimum,
        1.08 * float(np.nanmax(np.abs(all_values))),
    )

    axes[panel_index].set(
        title=title,
        ylabel=ylabel,
        ylim=(-limit, limit),
    )
    axes[panel_index].legend(
        loc="upper right",
        fontsize=6.5,
        ncol=3,
    )
    motion_lines.append(panel_lines)

# Gripper panel.
gripper_lines = [
    axes[2].plot(
        [],
        [],
        color=color,
        linestyle=style,
        linewidth=width,
        label=label,
    )[0]
    for color, style, width, label in (
        ("#2563eb", "-", 2.0, "OpenVLA-OFT gripper output"),
        ("#f59e0b", "--", 1.7, "Controller target opening"),
        ("#16a34a", "-", 1.9, "MuJoCo observed opening"),
    )
]

axes[2].axhline(
    0.5,
    color="#20242a",
    linewidth=1.0,
    linestyle=":",
    label="Half-open reference",
)
axes[2].set(
    title="Continuous gripper behavior",
    ylabel="Open fraction",
    ylim=(-0.05, 1.05),
)
axes[2].set_xlabel(
    f"Executed action step ({float(CONTROL_HORIZON_S):.3f} s control horizon)"
)
axes[2].legend(loc="upper right", fontsize=7, ncol=2)

# Fixed axes and current-action cursors.
query_cursors = []
for axis in axes:
    axis.axhline(
        0.0,
        color="#333333",
        linewidth=0.7,
        alpha=0.55,
    )
    axis.set_xlim(0, max(1, len(records) - 1))
    axis.grid(alpha=0.25)

    cursor = axis.axvline(
        0,
        color="#dc2626",
        linewidth=1.1,
        alpha=0.75,
    )
    cursor.set_visible(False)
    query_cursors.append(cursor)

gripper_arrays = (
    model_gripper,
    controller_gripper,
    mujoco_gripper,
)

frame_count = 0

try:
    for frame_index, source_frame in enumerate(reader):
        # Frame 0 is before action step 0.
        # Frame s+1 is the simulated state after executed action step s.
        completed = min(frame_index, len(records))
        shown_queries = queries[:completed]

        # Update translation and rotation lines.
        for panel_index, (_, _, names, arrays, _) in enumerate(
            motion_panels
        ):
            for dimension in range(len(names)):
                for source_index in range(3):
                    motion_lines[
                        panel_index
                    ][dimension][source_index].set_data(
                        shown_queries,
                        arrays[source_index][:completed, dimension],
                    )

        # Update gripper lines.
        for line, values in zip(gripper_lines, gripper_arrays):
            line.set_data(shown_queries, values[:completed])

        if completed:
            current_step = completed - 1

            for cursor in query_cursors:
                cursor.set_xdata([current_step, current_step])
                cursor.set_visible(True)

            current_record = records[current_step]
            status = (
                f"action {current_step:03d} | "
                f"query {current_record['query']:03d} "
                f"h{current_record['chunk_step']}"
            )
        else:
            status = "before action 000"

        figure.suptitle(
            f"{log_path.parent.name}: live command execution | {status}",
            fontsize=13,
        )

        figure.canvas.draw()
        plot_frame = np.asarray(
            figure.canvas.buffer_rgba(),
            dtype=np.uint8,
        )[:, :, :3].copy()

        # Resize the world video without changing its aspect ratio.
        source_frame = np.asarray(
            source_frame,
            dtype=np.uint8,
        )[:, :, :3]

        panel_height = plot_frame.shape[0]
        source_width = int(round(
            panel_height
            * source_frame.shape[1]
            / source_frame.shape[0]
        ))
        source_width += source_width % 2

        resampling = getattr(Image, "Resampling", Image).LANCZOS
        source_frame = np.asarray(
            Image.fromarray(source_frame).resize(
                (source_width, panel_height),
                resampling,
            )
        )

        combined_frame = np.concatenate(
            [source_frame, plot_frame],
            axis=1,
        )
        writer.append_data(combined_frame)
        frame_count += 1

finally:
    reader.close()
    writer.close()
    plt.close(figure)

assert frame_count > 0, f"No frames read from {source_video}"

action_aligned_frames = len(records) + 1
assert frame_count >= action_aligned_frames, (
    f"Video ended at {frame_count} frames before all "
    f"{action_aligned_frames} action-aligned frames were rendered."
)
settle_frames = frame_count - action_aligned_frames
if settle_frames:
    print(
        f"Included {settle_frames} post-release settle frames; "
        "diagnostic traces hold their final action value."
    )

print("Saved synchronized live-diagnostics video:", output_path)
display(Video(str(output_path), embed=False, width=1200))

In [ ]:
# Cell P4: Judgment videos -- zoomed grasp view and FOTS tactile RGB / depth_difference side by
# side, time-synchronized, for visually judging whether each tactile sensor tracks real contact
# geometry (e.g. the cube edge, the bump texture). Built once per sensor (right, then left)
# against the same grasp view, since both sensors watch the same physical grasp from their own
# finger. Reads only files already written by Cell P for a given rollout (rollout_NNN/right and
# rollout_NNN/left); it does not re-run the policy or touch the live rollout model. Ported from
# TactileModel05_Duo.ipynb's judgment video: each panel keeps its own native aspect ratio (grasp
# view is 640x480 landscape, FOTS's own canvas is 240x320 portrait) instead of being
# force-resized into one shared box, which visibly stretched the grasp-view panel in an earlier
# version.

# Which rollouts to build. None (default) processes every rollout_* folder
# found under run_dir. To look at specific ones instead, set this to a list
# of rollout indices, e.g. JUDGMENT_ROLLOUT_INDICES = [1, 3] for only
# rollout_001 and rollout_003.
JUDGMENT_ROLLOUT_INDICES = [1]


def _resize_keep_aspect(img, target_h):
    w, h = img.size
    target_w = max(1, round(w * target_h / h))
    return img.resize((target_w, target_h), Image.Resampling.LANCZOS)


def render_tactile_judgment_video(rollout_dir, side):
    import imageio.v3 as imageio_v3

    rollout_dir = Path(rollout_dir)
    out_name = f"tactile_judgment_{side}.mp4"
    grasp_front_path = rollout_dir / "grasp_front.mp4"
    tactile_rgb_path = rollout_dir / side / "fots_tactile_rgb.mp4"
    quant_path = rollout_dir / side / "fots_quantitative_data.npz"
    for required in (grasp_front_path, tactile_rgb_path, quant_path):
        assert required.is_file(), f"Missing {required}; run Cell P for this rollout first."

    quant = np.load(quant_path)
    depth_m = quant["depth_m"]
    reference_depth_m = float(quant["reference_depth_m"])
    sim_time_s = quant["sim_time_s"]
    n_frames = depth_m.shape[0]

    # imageio.v2's ffmpeg reader can mis-report frame count as inf and over-read
    # on repeated iteration; v3's imiter reads exactly the encoded frame stream.
    grasp_front_frames = list(imageio_v3.imiter(str(grasp_front_path)))
    tactile_rgb_frames = list(imageio_v3.imiter(str(tactile_rgb_path)))
    assert len(tactile_rgb_frames) == n_frames, (len(tactile_rgb_frames), n_frames)

    panel_h = 480
    label_h = 30
    foot_h = 22

    writer = imageio.get_writer(
        str(rollout_dir / out_name),
        fps=FOTS_TACTILE_FPS,
        codec="libx264",
        quality=8,
        pixelformat="yuv420p",
        macro_block_size=None,
    )
    try:
        for i in range(n_frames):
            grasp_idx = int(np.clip(round(sim_time_s[i] * VIDEO_FPS), 0, len(grasp_front_frames) - 1))
            front = _resize_keep_aspect(Image.fromarray(grasp_front_frames[grasp_idx]), panel_h)
            tactile = _resize_keep_aspect(Image.fromarray(tactile_rgb_frames[i]), panel_h)
            depth_diff_m = reference_depth_m - depth_m[i]
            depth = _resize_keep_aspect(
                Image.fromarray(depth_diff_to_display_rgb(depth_diff_m)), panel_h
            )

            total_w = front.width + tactile.width + depth.width
            panel = Image.new("RGB", (total_w, panel_h + label_h + foot_h), (18, 18, 22))
            draw = ImageDraw.Draw(panel)
            x = 0
            for img, label in (
                (front, "Zoomed grasp view (front camera)"),
                (tactile, f"FOTS tactile RGB ({side})"),
                (depth, f"depth_difference (fixed 0-{FOTS_DEPTH_DIFF_DISPLAY_MAX_MM:.1f} mm)"),
            ):
                panel.paste(img, (x, label_h))
                draw.text((x + 6, 7), label, fill=(240, 240, 240))
                x += img.width
            draw.text(
                (6, panel.height - foot_h + 4),
                f"t={sim_time_s[i]:.3f}s   grasp-view frame {grasp_idx} (10 Hz)   tactile frame {i} (30 Hz)",
                fill=(200, 200, 200),
            )
            writer.append_data(np.asarray(panel))
    finally:
        writer.close()

    return rollout_dir / out_name


if "run_dir" in globals():
    _judgment_video_paths = []
    if JUDGMENT_ROLLOUT_INDICES is None:
        # "rollout_*" (OOD-evaluation runs) and "demo_*" (curated dataset runs, e.g. T2) --
        # see Cell P's DATASET_KEEP_ONLY_SUCCESSFUL.
        _rollout_dirs = sorted(run_dir.glob("rollout_*")) + sorted(run_dir.glob("demo_*"))
    else:
        # An int is a rollout_NNN index; a str (e.g. "demo_003") is used as the folder name directly.
        _rollout_dirs = [
            run_dir / (f"rollout_{i:03d}" if isinstance(i, int) else i) for i in JUDGMENT_ROLLOUT_INDICES
        ]
        _missing = [d for d in _rollout_dirs if not d.is_dir()]
        assert not _missing, (
            f"No such rollout folder(s): {_missing}; available: {[d.name for d in sorted(run_dir.glob('rollout_*'))]}. "
            "Set JUDGMENT_ROLLOUT_INDICES to rollouts that exist."
        )
    for _rollout_dir in _rollout_dirs:
        for _side in ("right", "left"):
            _judgment_video_paths.append(render_tactile_judgment_video(_rollout_dir, _side))
    print(f"Wrote {len(_judgment_video_paths)} tactile judgment video(s):")
    for _p in _judgment_video_paths:
        print(" ", _p)
    if DISPLAY_GELSIGHT_VIDEOS and _judgment_video_paths:
        for _p in _judgment_video_paths[-2:]:   # right + left of the last rollout
            display(Video(str(_p), embed=False, width=3 * 480))
else:
    print("run_dir not in globals(); call render_tactile_judgment_video(<rollout_dir>, 'right' or 'left') manually.")


In [ ]:
# Cell P5: DROPO-style replay augmentation for the T3 dataset.
#
# Generates MORE T3 demos far faster than Cell P, without ever calling OpenVLA-OFT again. For
# each new attempt, it picks an EXISTING real demo (one Cell P already collected with the live
# D3 policy driving it), resets the sim to that demo's exact recorded starting state, and
# replays -- open-loop -- the exact sequence of `controller_source_action` values OpenVLA-OFT
# chose during that original rollout (saved per-step in its closed_loop_log.json["records"]).
# The only thing that's new is the physics underneath: fresh, independently-randomized
# friction/cube-mass/grip-margin/perturbation draws (same distributions Cell P itself uses),
# so the replayed action sequence produces a genuinely different physical outcome -- a different
# slip event, a different grasp stability, sometimes a drop that didn't happen the first time.
#
# What makes this fast: OpenVLA-OFT inference (~250 forward passes/attempt, the dominant real
# cost measured in the production run) is skipped entirely -- there is no worker subprocess in
# this cell. What is NOT skipped, and never synthesized: every physics step (`mujoco.mj_step`)
# is real, and every tactile frame is a REAL fresh render -- `adapt_openvla_action` and
# `step_controller` are the exact same functions Cell P itself calls, and
# `InstrumentedDualFOTSRecorder` (also unmodified) calls the real FOTS sensors and MuJoCo's own
# `mj_contactForce` every sampled step, exactly like a live rollout. Nothing here copies or
# interpolates a previous demo's tactile pixels or label values.
#
# The one honest caveat (see the "DROPO-style replay" discussion this cell was built from): a
# replayed action was chosen by OpenVLA in response to the ORIGINAL dynamics, not these new
# ones. Early in an attempt, before the new randomization has caused the physical state to
# diverge much from the seed, this is very close to authentic closed-loop behavior. The more the
# state diverges, the more this becomes "what a frozen action sequence does under new physics"
# rather than "what OpenVLA-OFT would actually have done" -- a real, live policy would have
# reacted to the divergence; this replay can't. Use these demos as a genuine physics/tactile
# augmentation, not as a claim that OpenVLA-OFT itself produced this exact behavior.
#
# Folder naming is deliberately NOT "demo_*": Cell P's own resume/target-count logic globs
# run_dir/"demo_*" to decide how many real demos already exist and when to stop the live
# collection. A name that matched would silently inflate that count and could cut the real
# production run short. DROPO_FOLDER_PREFIX keeps these fully separate on disk, fully visible
# here, and fully identical in file *schema* to a real demo_NNN folder (same closed_loop_log.json
# keys, same slip_ground_truth.npz / grasp_pose.json / per-sensor npz+mp4 layout) so a dataset
# loader for the NN only needs to glob both prefixes -- no special-casing of file contents.
#
# Requires Cell P to have already run at least once in this kernel (needs the live MuJoCo model
# /data, the two FOTS sensors, and Cell M's checkpoint metadata) -- exactly like Cell D1 needs
# Cell P's saved files, this cell needs Cell P's live objects. It does NOT need
# RUN_CLOSED_LOOP_ROLLOUTS=True and does not start (or need) the OpenVLA-OFT worker subprocess.

import time

import imageio.v3 as imageio_v3
import matplotlib.pyplot as plt
from IPython.display import Video, display
from PIL import Image

DROPO_ENABLED = True

DROPO_TIME_BUDGET_S = 3600.0   # stopping mechanism 1: hard wall-clock ceiling for this whole cell.
DROPO_SAFETY_MARGIN_S = 90.0   # stop STARTING a new replay once budget - elapsed < this much is left.
DROPO_TARGET_DEMOS = 700       # stopping mechanism 2: stop once this many DROPO demos are KEPT on disk
                                # in total (counts dropo_demo_* folders already there too, exactly like
                                # Cell P's own DATASET_TARGET_DEMOS -- a later resumed session keeps
                                # counting toward the same target instead of restarting it).
DROPO_SHOWCASE_COUNT = 10      # how many of the (now on-disk) dropo_demo_* folders to showcase at the end.

DROPO_SEED_ORDER_SEED = 1234       # deterministic shuffle of which existing real demo to replay next.
DROPO_RANDOMIZATION_SEED = 4242    # independent RNG stream from Cell P's own ROLLOUT_SEED, so the new
                                    # friction/mass/grip-margin/perturbation draws here are not correlated
                                    # with (or a subset of) what Cell P already drew for its own rollouts.

DROPO_FOLDER_PREFIX = "dropo_demo_"          # kept, qualifying replays.
DROPO_TEMP_PREFIX = "dropo_rollout_"         # in-progress / to-be-discarded, mirrors Cell P's rollout_NNN.

if DROPO_ENABLED:
    assert "model" in globals() and "data" in globals(), (
        "Run Cell P first -- this cell reuses its live MuJoCo model/data, it does not rebuild the scene."
    )
    assert "run_dir" in globals(), "Run Cell P first (needs run_dir, Cell P's T3/baseline output folder)."
    assert "FOTS_SENSOR_RIGHT" in globals() and "FOTS_SENSOR_LEFT" in globals(), (
        "Run Cell P first (needs both live FOTSTactileSensor objects)."
    )
    assert "checkpoint_dir" in globals() and "checkpoint_info" in globals(), "Run Cell M first."

    _seed_demo_dirs = sorted(
        p for p in run_dir.glob("demo_*")
        if p.is_dir() and p.name.split("_")[1].isdigit() and (p / "closed_loop_log.json").is_file()
    )
    assert _seed_demo_dirs, f"No demo_* folders with a closed_loop_log.json under {run_dir} to replay from -- run Cell P first."

    _dropo_existing = sorted(run_dir.glob(f"{DROPO_FOLDER_PREFIX}*"))
    dropo_kept_count = len(_dropo_existing)
    if _dropo_existing:
        print(
            f"Resuming: {dropo_kept_count} {DROPO_FOLDER_PREFIX}* folder(s) already saved under {run_dir}; "
            f"new ones continue at {DROPO_FOLDER_PREFIX}{dropo_kept_count:03d}."
        )

    _seed_order_rng = np.random.default_rng(DROPO_SEED_ORDER_SEED)
    _seed_cycle = list(_seed_demo_dirs)
    _seed_order_rng.shuffle(_seed_cycle)

    dropo_rng = np.random.default_rng(DROPO_RANDOMIZATION_SEED)
    _observed_step_seconds = []          # refined from real replays as they complete.
    _default_step_seconds_guess = 0.12   # conservative prior for the very first attempt only.

    dropo_attempt_summary = []
    _wall_start = time.monotonic()
    _attempt_idx = 0
    _seed_pos = 0

    print(
        f"DROPO replay augmentation: {len(_seed_demo_dirs)} real demo(s) available as seeds. "
        f"Two stopping conditions: (1) time budget={DROPO_TIME_BUDGET_S / 60.0:.0f} min "
        f"(safety margin={DROPO_SAFETY_MARGIN_S:.0f}s), (2) {DROPO_TARGET_DEMOS} kept DROPO demos "
        f"(currently {dropo_kept_count}/{DROPO_TARGET_DEMOS})."
    )

    if dropo_kept_count >= DROPO_TARGET_DEMOS:
        print(f"Already at {dropo_kept_count}/{DROPO_TARGET_DEMOS} DROPO demos -- nothing to do.")

    while dropo_kept_count < DROPO_TARGET_DEMOS:
        _elapsed_s = time.monotonic() - _wall_start
        _per_step_estimate_s = (
            sum(_observed_step_seconds) / len(_observed_step_seconds)
            if _observed_step_seconds else _default_step_seconds_guess
        )
        # Worst case for the NEXT attempt: a full MAX_EXECUTED_ACTIONS-length replay (Cell P's own
        # ceiling), so one unlucky long seed can never blow past the budget.
        _worst_case_next_s = _per_step_estimate_s * MAX_EXECUTED_ACTIONS + 8.0  # +8s video/IO margin.
        if _elapsed_s + _worst_case_next_s > DROPO_TIME_BUDGET_S - DROPO_SAFETY_MARGIN_S:
            print(
                f"\nStopping: {_elapsed_s:.0f}s elapsed, next replay could take up to ~{_worst_case_next_s:.0f}s "
                f"-- starting it would risk exceeding the {DROPO_TIME_BUDGET_S / 60.0:.0f} min budget."
            )
            break

        seed_dir = _seed_cycle[_seed_pos % len(_seed_cycle)]
        _seed_epoch = _seed_pos // len(_seed_cycle)   # >0 once a seed is reused -- still an independent replay.
        _seed_pos += 1

        seed_log = json.loads((seed_dir / "closed_loop_log.json").read_text())
        seed_records = seed_log.get("records") or []
        seed_start_info = seed_log["start_info"]
        if not seed_records:
            print(f"Skipping seed {seed_dir.name}: no recorded actions to replay.")
            continue

        attempt_dir = run_dir / f"{DROPO_TEMP_PREFIX}{_attempt_idx:04d}"
        if attempt_dir.exists():
            shutil.rmtree(attempt_dir)
        attempt_dir.mkdir(parents=True, exist_ok=True)
        _attempt_wall_start = time.monotonic()

        # --- Reconstruct the EXACT starting state the seed demo began from. DROPO-style replay
        # resets to a real visited state and lets everything diverge from THERE, purely through the
        # new randomized dynamics below -- not through an incidentally different starting pose. ----
        _seed_cube_start = np.array(seed_start_info["cube_start"], dtype=np.float64)
        reset_task_state_randomized(cube_pos=_seed_cube_start, robot_joint_noise=None, settle_steps=60)
        _nominal_ee = get_ee_pose()["position"].copy()
        _ee_target = np.clip(
            _nominal_ee + np.array(seed_start_info["robot_ee_start_delta_m"], dtype=np.float64),
            WORKSPACE_LOW, WORKSPACE_HIGH,
        )
        _qpos_target0 = solve_hand_position_ik(_ee_target, seed_qpos=current_arm_qpos_dict())
        set_arm_qpos(_qpos_target0)
        set_gripper_opening(0.04)
        for _ in range(60):
            set_arm_position_targets(_qpos_target0)
            set_gripper_opening(0.04)
            mujoco.mj_step(model, data)
        mujoco.mj_forward(model, data)
        start_info = {
            "cube_start": _seed_cube_start.tolist(),
            "nominal_untextured_cube_start": seed_start_info["nominal_untextured_cube_start"],
            "texture_bottom_tip_initially_on_table": True,
            "robot_ee_start_delta_m": (get_ee_pose()["position"] - _nominal_ee).tolist(),
            "outside_d3_cube_xy_envelope": seed_start_info.get("outside_d3_cube_xy_envelope"),
        }

        # --- New, independently-randomized dynamics for THIS replay -- same distributions Cell P
        # itself draws from; this, not the frozen action sequence, is what makes each replay a
        # genuinely new physical outcome. --------------------------------------------------------
        if SLIP_RANDOMIZE_MATERIALS:
            _torsional_friction = float(dropo_rng.uniform(*SLIP_TORSIONAL_FRICTION_RANGE))
            model.geom_friction[GELSIGHT_GEL_PAD_GEOM_ID_RIGHT, 1] = _torsional_friction
            model.geom_friction[GELSIGHT_GEL_PAD_GEOM_ID_LEFT, 1] = _torsional_friction
            model.geom_friction[_cube_contact_geom_ids, 1] = _torsional_friction
            _cube_mass_scale = float(dropo_rng.uniform(*SLIP_CUBE_MASS_SCALE_RANGE))
            model.body_mass[BLOCK_BODY_ID] = _base_cube_mass_kg * _cube_mass_scale
            model.body_inertia[BLOCK_BODY_ID] = _base_cube_inertia * _cube_mass_scale
        else:
            _torsional_friction, _cube_mass_scale = GEL_CUBE_TORSIONAL_FRICTION, 1.0
        if SLIP_RANDOMIZE_GRIP_MARGIN:
            GRIP_LOCK_EXTRA_MARGIN = float(dropo_rng.uniform(*SLIP_GRIP_LOCK_EXTRA_MARGIN_RANGE))
        _perturbation_direction = dropo_rng.normal(size=3)
        _perturbation_direction[2] *= 0.3
        _perturbation_direction /= max(float(np.linalg.norm(_perturbation_direction)), 1e-9)
        perturbation_state = {
            "pending": bool(SLIP_PERTURBATION_ENABLED and dropo_rng.random() < SLIP_PERTURBATION_PROBABILITY),
            "triggered": False,
            "phase_entry_time_s": None,
            "active_until_s": None,
            "delay_s": float(dropo_rng.uniform(0.2, 1.5)),
            "force_vec": _perturbation_direction * float(dropo_rng.uniform(*SLIP_PERTURBATION_FORCE_RANGE_N)),
        }
        data.xfrc_applied[BLOCK_BODY_ID, :3] = 0.0

        reset_controller_state()
        qpos_frames = [data.qpos.copy()]
        tactile_recorder = InstrumentedDualFOTSRecorder(attempt_dir, FOTS_SENSOR_RIGHT, FOTS_SENSOR_LEFT)
        tactile_recorder.capture_initial()
        wrist_alignment_renderer = mujoco.Renderer(model, height=WRIST_IMAGE_SIZE, width=WRIST_IMAGE_SIZE)

        records_out = []
        rollout_termination_reason = "seed_replay_exhausted"  # neither budget nor destination -- ran out of script.
        qpos_target = _qpos_target0
        try:
            destination_reached = False
            for step_idx, seed_rec in enumerate(seed_records):
                maybe_apply_perturbation(perturbation_state)
                controller_action = np.asarray(seed_rec["controller_source_action"], dtype=np.float64)
                controller_phase_before = CONTROLLER_STATE["mode"]
                gripper_before_m = current_finger_opening_m()
                tactile_frame_start = tactile_recorder.frame_count
                (
                    controller_source_raw, clipped, qpos_target, ee_target, ee_target_quat,
                    gripper_opening, applied_gripper_fraction, ik_info,
                ) = adapt_openvla_action(controller_action)
                controller_phase_after = CONTROLLER_STATE["mode"]
                controller_info = step_controller(
                    qpos_target, ee_target, ee_target_quat, gripper_opening, qpos_frames,
                    tactile_recorder=tactile_recorder,
                )
                gripper_after_m = current_finger_opening_m()
                ee_pose_after = get_ee_pose()
                cube_pos = get_cube_pose()["position"].copy()
                records_out.append({
                    "action_step": int(step_idx),
                    "replayed_from": seed_dir.name,
                    "replayed_step_index": int(step_idx),
                    "controller_source_action": controller_source_raw.tolist(),
                    "clipped_controller_action": clipped.tolist(),
                    "controller_phase_before_action": controller_phase_before,
                    "controller_phase_after_planning": controller_phase_after,
                    "ee_target": ee_target.tolist(),
                    "ee_target_quat_wxyz": ee_target_quat.tolist(),
                    "ee_after": ee_pose_after["position"].tolist(),
                    "ee_after_quat_wxyz": ee_pose_after["quat_wxyz"].tolist(),
                    "cartesian_controller": controller_info,
                    "cube_after_eval_only": cube_pos.tolist(),
                    "gripper_opening_m": float(gripper_opening),
                    "gripper_target_open_fraction": float(applied_gripper_fraction),
                    "gripper_before_per_finger_m": float(gripper_before_m),
                    "gripper_after_per_finger_m": float(gripper_after_m),
                    "tactile_frame_start": int(tactile_frame_start),
                    "tactile_frame_end_exclusive": int(tactile_recorder.frame_count),
                })
                if cube_at_destination(cube_pos):
                    destination_reached = True
                    break
            if destination_reached:
                settle_after_release(qpos_target, qpos_frames, tactile_recorder=tactile_recorder)
                rollout_termination_reason = "cube_reached_destination"
        finally:
            wrist_alignment_renderer.close()
            tactile_outputs = tactile_recorder.finalize()

        final_cube = get_cube_pose()["position"].copy()
        success = cube_at_destination(final_cube)

        world_video = attempt_dir / "world.mp4"
        wrist_video = attempt_dir / "wrist.mp4"
        grasp_front_video = attempt_dir / "grasp_front.mp4"
        grasp_corner_video = attempt_dir / "grasp_corner_low.mp4"
        final_frame_photo = attempt_dir / "final_frame.png"
        diagnostic_camera_config = render_rollout_videos(
            qpos_frames, start_info["cube_start"], world_video, wrist_video,
            grasp_front_video, grasp_corner_video, final_frame_path=final_frame_photo,
        )

        payload = {
            "checkpoint": str(checkpoint_dir),
            "checkpoint_dataset": checkpoint_info["dataset"],
            "checkpoint_step": int(checkpoint_info["step"]),
            "tactile_sensors": ["right", "left"],
            "prompt": OPENVLA_ROLLOUT_PROMPT,
            "closed_loop_note": (
                "DROPO-style replay: no OpenVLA-OFT inference happened in this attempt. The exact "
                f"controller_source_action sequence OpenVLA-OFT chose in {seed_dir.name} (same pinned "
                "D3 checkpoint) is replayed open-loop through the real controller and physics under "
                "newly randomized friction/mass/grip-margin/perturbation. Every tactile frame and every "
                "ground-truth field below was captured live from this real re-simulation -- nothing here "
                "is copied or synthesized from the seed demo."
            ),
            "data_source": "dropo_replay",
            "dropo_seed_demo": seed_dir.name,
            "dropo_seed_action_count": len(seed_records),
            "dropo_seed_epoch": int(_seed_epoch),
            "dropo_randomization": {
                "torsional_friction": _torsional_friction,
                "cube_mass_scale": _cube_mass_scale,
                "grip_lock_extra_margin": float(GRIP_LOCK_EXTRA_MARGIN),
                "perturbation_pending_or_triggered": bool(perturbation_state["pending"] or perturbation_state["triggered"]),
                "perturbation_triggered": bool(perturbation_state["triggered"]),
            },
            "action_chunk_length": 8,
            "actions_executed_per_query": int(ACTIONS_PER_CHUNK),
            "max_executed_actions": int(MAX_EXECUTED_ACTIONS),
            "rollout_termination_reason": rollout_termination_reason,
            "proprio_convention": "[q1, q2, q3, q4, q5, q6, q7, measured_gripper_open_fraction]",
            "controller_mode": "hybrid_delta_reference_physical_actuators",
            "rotation_action_convention": "action[3:6] is a world-frame rotation vector in radians",
            "gripper_action_convention": (
                "continuous policy open fraction applied directly and interpolated within "
                "each 100 ms interval; 0=closed, 1=open; per-finger target is 0.00-0.04 m"
            ),
            "max_rotation_per_action_deg": float(np.rad2deg(OPENVLA_MAX_ROTATION_RAD)),
            "diagnostic_cameras": diagnostic_camera_config,
            "gelsight_tactile": tactile_outputs,
            "start_info": start_info,
            "experiment_dataset": EXPERIMENT_DATASET,
            "experiment_condition": EXPERIMENT_CONDITION,
            "cube_size_scale": float(CUBE_SIZE_SCALE),
            "light_dim_factor": float(LIGHT_DIM_FACTOR),
            "occluder_plant_enabled": bool(OCCLUDER_PLANT_ENABLED),
            "occluder_plant_position_m": [float(OCCLUDER_PLANT_POS_M[0]), float(OCCLUDER_PLANT_POS_M[1])],
            "occluder_plant_scale": float(OCCLUDER_PLANT_SCALE),
            "success_rough_eval_only": success,
            "outcome": "accepted" if success else "failed",
            "final_cube_pos": final_cube.tolist(),
            "policy_queries": [],
            "records": records_out,
            "videos": {
                "world": str(world_video),
                "wrist": str(wrist_video),
                "grasp_front": str(grasp_front_video),
                "grasp_corner_low": str(grasp_corner_video),
                "final_frame": str(final_frame_photo),
                "gelsight_right_rgb": tactile_outputs["right"]["tactile_rgb_video"],
                "gelsight_right_marker_flow": tactile_outputs["right"]["marker_flow_video"],
                "gelsight_left_rgb": tactile_outputs["left"]["tactile_rgb_video"],
                "gelsight_left_marker_flow": tactile_outputs["left"]["marker_flow_video"],
            },
        }

        _step_seconds_this_attempt = (time.monotonic() - _attempt_wall_start) / max(1, len(records_out))
        _observed_step_seconds.append(_step_seconds_this_attempt)

        # Keep criterion: more than DATASET_MIN_CONTACT_SECONDS (2.0s) of real cube<->GelSight
        # contact on at least one side -- just contact duration, no separate peak-area gate (unlike
        # Cell P's own DATASET_MIN_PEAK_TAXELS floor).
        is_dataset_demo = False
        disqualified_reason = "insufficient_contact_duration"
        _side_contact_stats = {}
        for side in ("right", "left"):
            _side_taxels = np.load(tactile_outputs[side]["quantitative_npz"])["active_taxel_count"]
            _contact_frames = int((_side_taxels > 0).sum())
            _peak_taxels = int(_side_taxels.max()) if _side_taxels.size else 0
            _side_contact_stats[side] = {"contact_frames": _contact_frames, "peak_taxels": _peak_taxels}
            if _contact_frames >= DATASET_MIN_CONTACT_FRAMES:
                is_dataset_demo = True
                disqualified_reason = None

        _ground_truth_summary = tactile_outputs.get("ground_truth", {})
        attempt_record = {
            "attempt": _attempt_idx,
            "seed_demo": seed_dir.name,
            "seed_epoch": int(_seed_epoch),
            "outcome": payload["outcome"],
            "final_cube_pos": final_cube.tolist(),
            "termination_reason": rollout_termination_reason,
            "kept_as_dataset_demo": bool(is_dataset_demo),
            "disqualified_reason": disqualified_reason,
            "torsional_friction": _torsional_friction,
            "cube_mass_scale": _cube_mass_scale,
            "grip_lock_extra_margin": float(GRIP_LOCK_EXTRA_MARGIN),
            "perturbation_applied": bool(perturbation_state["triggered"]),
            "grasp_established": bool(_ground_truth_summary.get("grasp_established", False)),
            "holding_frames": int(_ground_truth_summary.get("holding_frames", 0)),
            "slip_frames": int(_ground_truth_summary.get("slip_frames", 0)),
            "slip_fraction_of_holding": _ground_truth_summary.get("slip_fraction_of_holding"),
            "step_seconds": _step_seconds_this_attempt,
            "dataset_dir": None,
        }

        if is_dataset_demo:
            demo_dir = run_dir / f"{DROPO_FOLDER_PREFIX}{dropo_kept_count:03d}"
            old_prefix = str(attempt_dir)
            attempt_dir.rename(demo_dir)
            payload_text = json.dumps(payload, indent=2).replace(old_prefix, str(demo_dir))
            (demo_dir / "closed_loop_log.json").write_text(payload_text, encoding="utf-8")
            attempt_record["dataset_dir"] = str(demo_dir)
            dropo_kept_count += 1
            print(
                f"KEPT as {demo_dir.name}  (dropo total so far: {dropo_kept_count})  "
                f"seed={seed_dir.name} friction={_torsional_friction:.3f} mass_x={_cube_mass_scale:.2f} "
                f"perturbed={attempt_record['perturbation_applied']} "
                f"slip={attempt_record['slip_frames']}/{attempt_record['holding_frames']} "
                f"outcome={attempt_record['outcome']}"
            )
        else:
            shutil.rmtree(attempt_dir)
            print(
                f"DISCARDED {attempt_dir.name} (seed={seed_dir.name}): {disqualified_reason} "
                f"(need >= {DATASET_MIN_CONTACT_FRAMES} contact frames [{DATASET_MIN_CONTACT_SECONDS:.1f}s] on at "
                f"least one side; right={_side_contact_stats['right']}, left={_side_contact_stats['left']})"
            )

        dropo_attempt_summary.append(attempt_record)
        _attempt_idx += 1

        if dropo_kept_count >= DROPO_TARGET_DEMOS:
            print(f"\nReached the target of {DROPO_TARGET_DEMOS} DROPO demos after {_attempt_idx} attempt(s) -- stopping early.")
            break

    _elapsed_total_s = time.monotonic() - _wall_start
    n_kept_this_run = sum(1 for a in dropo_attempt_summary if a["kept_as_dataset_demo"])
    print(
        f"\nDROPO replay augmentation finished: {len(dropo_attempt_summary)} attempt(s) in "
        f"{_elapsed_total_s / 60.0:.1f} min, {n_kept_this_run} kept this run "
        f"(dataset total: {dropo_kept_count} {DROPO_FOLDER_PREFIX}* demo(s) under {run_dir}, "
        f"plus {len(_seed_demo_dirs)} real demo_* demo(s))."
    )

    results_dropo_path = run_dir / "results_dropo.csv"
    if dropo_attempt_summary:
        results_dropo_columns = [
            "attempt", "seed_demo", "seed_epoch", "outcome",
            "final_cube_x", "final_cube_y", "final_cube_z",
            "termination_reason", "kept_as_dataset_demo", "disqualified_reason", "dataset_dir",
            "torsional_friction", "cube_mass_scale", "grip_lock_extra_margin", "perturbation_applied",
            "grasp_established", "holding_frames", "slip_frames", "slip_fraction_of_holding", "step_seconds",
        ]
        # Kept deliberately separate from Cell P's own results.csv (different schema, different
        # attempt numbering) so re-running Cell P can never see or be confused by these rows --
        # only run_dir/"demo_*" and run_dir/"results.csv" matter to Cell P's own logic.
        results_dropo_rows = []
        if results_dropo_path.is_file():
            with open(results_dropo_path, newline="") as f:
                reader = csv.DictReader(f)
                if reader.fieldnames == results_dropo_columns:
                    results_dropo_rows = [[row[col] for col in results_dropo_columns] for row in reader]
                else:
                    print(f"WARNING: {results_dropo_path} has an unexpected header; not merging its rows.")
        for a in dropo_attempt_summary:
            fc = a["final_cube_pos"]
            results_dropo_rows.append([
                a["attempt"], a["seed_demo"], a["seed_epoch"], a["outcome"],
                fc[0], fc[1], fc[2], a["termination_reason"],
                a["kept_as_dataset_demo"], a["disqualified_reason"] or "", a["dataset_dir"] or "",
                a["torsional_friction"], a["cube_mass_scale"], a["grip_lock_extra_margin"], a["perturbation_applied"],
                a["grasp_established"], a["holding_frames"], a["slip_frames"], a["slip_fraction_of_holding"],
                a["step_seconds"],
            ])
        with open(results_dropo_path, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(results_dropo_columns)
            writer.writerows(results_dropo_rows)
        print(f"Per-attempt DROPO results table ({len(results_dropo_rows)} attempt(s) across every session so far): {results_dropo_path}")

    print(
        "\nDataset layout for training: enumerate BOTH "
        f"sorted(run_dir.glob('demo_*')) and sorted(run_dir.glob('{DROPO_FOLDER_PREFIX}*')) -- every "
        "folder from either glob has an identical closed_loop_log.json / slip_ground_truth.npz / "
        "grasp_pose.json / right/left fots_*.{npz,mp4} layout. Only closed_loop_log.json['data_source'] "
        "differs ('dropo_replay' here vs absent/live on a real Cell P demo), for provenance only -- no "
        "training code needs to branch on it unless you explicitly want to weight or filter the two "
        "populations differently."
    )

    # ===== Showcase: DROPO_SHOWCASE_COUNT of the dropo_demo_* demos now on disk, same format as
    # Cell D1 (still-frame summary, per-demo slip/force/contact time series, stats table, videos). =
    _dropo_all_dirs = sorted(run_dir.glob(f"{DROPO_FOLDER_PREFIX}*"))
    if not _dropo_all_dirs:
        print("\nNo dropo_demo_* folders on disk yet -- nothing to showcase.")
    else:
        _dropo_showcase_dirs = sorted(
            {_dropo_all_dirs[i] for i in np.linspace(
                0, len(_dropo_all_dirs) - 1, min(DROPO_SHOWCASE_COUNT, len(_dropo_all_dirs))
            ).astype(int)},
            key=lambda p: p.name,
        )
        print(
            f"\nShowcasing {len(_dropo_showcase_dirs)} of {len(_dropo_all_dirs)} {DROPO_FOLDER_PREFIX}* demo(s): "
            f"{[p.name for p in _dropo_showcase_dirs]}"
        )

        def _dropo_peak_contact_frame(demo_dir, side):
            side_dir = demo_dir / side
            quant = np.load(side_dir / "fots_quantitative_data.npz")
            taxels = quant["active_taxel_count"]
            peak_idx = int(np.argmax(taxels))
            frame = imageio_v3.imread(side_dir / "fots_tactile_rgb.mp4", index=peak_idx)
            return {
                "frame": frame, "peak_idx": peak_idx, "peak_taxels": int(taxels[peak_idx]),
                "contact_frames": int((taxels > 0).sum()), "total_frames": int(len(taxels)),
                "peak_force_n": float(np.nanmax(quant["normal_force_n"])) if "normal_force_n" in quant.files else float("nan"),
            }

        dropo_showcase_rows = []
        for demo_dir in _dropo_showcase_dirs:
            log = json.loads((demo_dir / "closed_loop_log.json").read_text())
            grasp = json.loads((demo_dir / "grasp_pose.json").read_text()) if (demo_dir / "grasp_pose.json").is_file() else {}
            gt = np.load(demo_dir / "slip_ground_truth.npz", allow_pickle=True) if (demo_dir / "slip_ground_truth.npz").is_file() else None
            row = {
                "demo": demo_dir.name, "dir": demo_dir,
                "final_frame": np.asarray(Image.open(demo_dir / "final_frame.png").convert("RGB")),
                "outcome": log["outcome"],
                "seed_demo": log.get("dropo_seed_demo", "n/a"),
                "start_xy": log["start_info"]["nominal_untextured_cube_start"][:2],
                "final_xyz": log["final_cube_pos"],
                "grasp_pose": grasp,
                "ground_truth": gt,
            }
            for side in ("right", "left"):
                row[side] = _dropo_peak_contact_frame(demo_dir, side)
            dropo_showcase_rows.append(row)

        # Figure 1: still-frame summary (identical layout to Cell D1).
        n_rows = len(dropo_showcase_rows)
        fig, axes = plt.subplots(n_rows, 3, figsize=(9, 3.15 * n_rows), squeeze=False)
        col_titles = ("Third-person, episode end", "RIGHT sensor @ peak contact", "LEFT sensor @ peak contact")
        for row_idx, row in enumerate(dropo_showcase_rows):
            for col_idx, (image, title) in enumerate((
                (row["final_frame"], col_titles[0]), (row["right"]["frame"], col_titles[1]), (row["left"]["frame"], col_titles[2]),
            )):
                ax = axes[row_idx, col_idx]
                ax.imshow(image)
                ax.set_xticks([]); ax.set_yticks([])
                if row_idx == 0:
                    ax.set_title(title, fontsize=9)
            axes[row_idx, 0].set_ylabel(
                f"{row['demo']}\n{row['outcome']}\nfrom {row['seed_demo']}",
                fontsize=8, rotation=0, ha="right", va="center", labelpad=42,
            )
            for col, side in ((1, "right"), (2, "left")):
                s = row[side]
                axes[row_idx, col].set_xlabel(
                    f"peak {s['peak_taxels']} px, {s['contact_frames']}/{s['total_frames']} contact, {s['peak_force_n']:.1f} N peak",
                    fontsize=7,
                )
        fig.suptitle(f"DROPO replay showcase -- {n_rows} of {len(_dropo_all_dirs)} demos ({run_dir})", fontsize=11)
        fig.tight_layout()
        dropo_showcase_path = run_dir / f"dropo_showcase_{'_'.join(p.name.split('_')[-1] for p in _dropo_showcase_dirs)}.png"
        fig.savefig(dropo_showcase_path, dpi=110)
        plt.show()

        # Figure 2: per-demo ground-truth time series (identical panel layout to Cell D1).
        _PHASE_COLORS = {
            "open": "0.92", "closing": "#ffe4b3", "closed_hold": "#ffd699",
            "lift_bootstrap": "#ffb3b3", "carrying": "#b3d9ff", "opening": "#c9f7c9", "released_hold": "0.92",
        }
        for row in dropo_showcase_rows:
            gt = row["ground_truth"]
            if gt is None:
                print(f"{row['demo']}: no slip_ground_truth.npz -- skipping time-series plot.")
                continue
            t = gt["sim_time_s"]
            phase = gt["controller_phase"]
            td, rd = gt["translation_delta_mm"], gt["rotation_delta_deg"]
            slip, holding = gt["slip_binary"], gt["holding_phase"]
            trans_thr = float(gt["slip_translation_threshold_mm"])
            rot_thr = float(gt["slip_rotation_threshold_deg"])
            right_q = np.load(row["dir"] / "right" / "fots_quantitative_data.npz")
            left_q = np.load(row["dir"] / "left" / "fots_quantitative_data.npz")

            fig, axes = plt.subplots(4, 1, figsize=(11, 8.5), sharex=True)
            phase_bounds = np.flatnonzero(phase[1:] != phase[:-1]) + 1
            starts = np.concatenate(([0], phase_bounds))
            ends = np.concatenate((phase_bounds, [len(phase)]))
            for ax in axes:
                for s, e in zip(starts, ends):
                    ax.axvspan(t[s], t[min(e, len(t) - 1)], color=_PHASE_COLORS.get(str(phase[s]), "0.95"), alpha=0.5, lw=0)

            axes[0].plot(t, td, color="tab:purple", lw=1.3, label="translation delta (mm)")
            axes[0].axhline(trans_thr, color="k", ls="--", lw=1, label=f"threshold ({trans_thr:.1f} mm)")
            axes[0].fill_between(t, 0, float(np.nanmax(td)) * 1.05 if np.isfinite(td).any() else 1.0, where=slip, color="red", alpha=0.15, step="mid", label="slip=1")
            axes[0].set_ylabel("translation\ndelta (mm)")
            axes[0].legend(loc="upper right", fontsize=7)

            axes[1].plot(t, rd, color="tab:orange", lw=1.3, label="rotation delta (deg)")
            axes[1].axhline(rot_thr, color="k", ls="--", lw=1, label=f"threshold ({rot_thr:.1f} deg)")
            axes[1].set_ylabel("rotation\ndelta (deg)")
            axes[1].legend(loc="upper right", fontsize=7)

            axes[2].plot(t, right_q["normal_force_n"], color="tab:red", lw=1.3, label="right finger")
            axes[2].plot(t, left_q["normal_force_n"], color="tab:blue", lw=1.3, label="left finger")
            axes[2].set_ylabel("normal force (N)")
            axes[2].legend(loc="upper right", fontsize=7)

            axes[3].step(t, right_q["contact_state"].astype(int) + 0.02, color="tab:red", lw=1.3, where="post", label="right contact")
            axes[3].step(t, left_q["contact_state"].astype(int) - 0.02, color="tab:blue", lw=1.3, where="post", label="left contact")
            axes[3].step(t, holding.astype(int) * 0.5 + 0.25, color="k", lw=1.0, where="post", label="holding phase (x0.5)")
            axes[3].set_ylim(-0.15, 1.15)
            axes[3].set_ylabel("contact state")
            axes[3].set_xlabel("simulation time (s)")
            axes[3].legend(loc="upper right", fontsize=7)

            grasp = row["grasp_pose"]
            grasp_txt = (
                f"grasp pose (mm, hand frame): {np.round(grasp['relative_pos_mm'], 1).tolist()} @ t={grasp['grasp_established_time_s']:.2f}s"
                if grasp.get("grasp_established") else "grasp never locked this episode"
            )
            holding_frames_n = int(holding.sum())
            slip_frames_n = int(slip.sum())
            fig.suptitle(
                f"{row['demo']} (seed={row['seed_demo']})  outcome={row['outcome']}  |  {grasp_txt}  |  "
                f"slip {slip_frames_n}/{holding_frames_n} holding frames ({100 * slip_frames_n / max(holding_frames_n, 1):.0f}%)",
                fontsize=10,
            )
            fig.tight_layout()
            dropo_timeseries_path = row["dir"] / "slip_timeseries.png"
            fig.savefig(dropo_timeseries_path, dpi=110)
            plt.show()

        # Stats table, sourced from results_dropo.csv (friction/mass/perturbation per attempt --
        # same idea as Cell D1 sourcing these from Cell P's own results.csv).
        _dropo_results_by_dir = {}
        if results_dropo_path.is_file():
            with open(results_dropo_path, newline="") as f:
                for r in csv.DictReader(f):
                    if r.get("dataset_dir"):
                        _dropo_results_by_dir[str(Path(r["dataset_dir"]).resolve())] = r

        print(f"\n{'demo':16s}{'seed':13s}{'outcome':10s}{'friction':>9s}{'mass_x':>7s}{'perturbed':>10s}{'holding':>9s}{'slip':>6s}{'slip%':>7s}")
        for row in dropo_showcase_rows:
            gt = row["ground_truth"]
            holding_frames_n = int(gt["holding_phase"].sum()) if gt is not None else 0
            slip_frames_n = int(gt["slip_binary"].sum()) if gt is not None else 0
            r = _dropo_results_by_dir.get(str(row["dir"].resolve()), {})
            friction_s = f"{float(r['torsional_friction']):.3f}" if r.get("torsional_friction") else "n/a"
            mass_s = f"{float(r['cube_mass_scale']):.2f}" if r.get("cube_mass_scale") else "n/a"
            print(
                f"{row['demo']:16s}{row['seed_demo']:13s}{row['outcome']:10s}"
                f"{friction_s:>9s}{mass_s:>7s}{r.get('perturbation_applied', 'n/a'):>10s}"
                f"{holding_frames_n:9d}{slip_frames_n:6d}{100 * slip_frames_n / max(holding_frames_n, 1):6.0f}%"
            )
        print(f"\nSaved DROPO still-frame summary: {dropo_showcase_path}")
        print("Saved per-demo time-series plots inside each demo's own folder (slip_timeseries.png).")

        # Full videos: third-person + both sensors' tactile RGB (Cell D1's own default groups).
        _dropo_video_group_paths = {
            "world": lambda d: [("world", d / "world.mp4")],
            "gelsight_rgb": lambda d: [(f"{side} tactile RGB", d / side / "fots_tactile_rgb.mp4") for side in ("right", "left")],
        }
        for row in dropo_showcase_rows:
            print(f"\n===== {row['demo']} (seed={row['seed_demo']}, {row['outcome']}) =====")
            for group, path_fn in _dropo_video_group_paths.items():
                for label, video_path in path_fn(row["dir"]):
                    if not video_path.is_file():
                        print(f"  (missing {video_path})")
                        continue
                    print(label)
                    display(Video(str(video_path), embed=False, width=640 if group == "world" else 320))
else:
    print("DROPO_ENABLED=False: this cell does nothing.")


# Testing both tactile sensors

Cells T1-T3 check that the two GelSight/FOTS sensors of this notebook's robot setup produce good, high-quality output. They use the scene and sensors that Cell P builds, so run Cell P first (with `RUN_CLOSED_LOOP_ROLLOUTS = False` if you only want to test the sensors and not run OpenVLA).

- **Cell T1** runs a scripted grasp, lift, carry, place and release (no policy involved) while both sensors record, so any difference between them comes from the sensors and gripper, not from policy behaviour.
- **Cell T2** judges both sensors. Per sensor: clean no-contact baseline, contact on grasp, physically plausible indentation, stable contact under load, the bump lattice resolved with the right pitch and patch size, tactile RGB and marker-flow response, recovery after release, saved-video fidelity, and the cube's measured distance. Across sensors: aligned timestamps, balanced squeeze, same contact window, mirrored object pose. It writes a PASS/FAIL table, two figures, a side-by-side video and a JSON report to `openvla36_dual_sensor_test/<timestamp>/`, and raises an error if any check fails.
- **Cell T3** audits the tactile data saved by the closed-loop policy rollouts (both sensors) and flags dead, one-sided or impact-damaged recordings.


In [ ]:
# Cell T1: Two-sensor test episode -- scripted grasp, lift, carry, place and release with BOTH
# GelSight sensors recording. No OpenVLA and no learned policy: a deterministic waypoint script
# (the same recipe as TactileModel05_Duo.ipynb's scripted pick-and-place, with an extra
# hold-closed phase) drives the arm, so any difference between the two sensors' outputs comes
# from the sensors/gripper and not from policy behaviour. It runs on the scene and the two FOTS
# sensors that Cell P built -- run Cell P first (RUN_CLOSED_LOOP_ROLLOUTS=False builds them in
# seconds without loading OpenVLA) -- and needs Cell P0's 6D IK helpers.
#
# Phases (each is physically simulated and recorded by both sensors at FOTS_TACTILE_FPS):
#   hold_open   fingers fully open at the grasp pose: the sensors must read "no contact"
#   close       gripper closes on the cube: contact must appear
#   hold_closed squeeze held still: the steady-state reading the quality checks are based on
#   lift / transfer / place   the grip carries the cube against gravity to the far side
#   release / retreat   gripper opens and moves away: contact must disappear again
# Cell T2 then judges both sensors' outputs; Cell T3 audits saved policy rollouts.

required_globals = [
    "model", "data", "FOTS_SENSOR_RIGHT", "FOTS_SENSOR_LEFT", "DualFOTSRecorder",
    "GELSIGHT_GEL_PAD_GEOM_ID_RIGHT", "GELSIGHT_GEL_PAD_GEOM_ID_LEFT",
    "solve_hand_pose_ik", "reset_task_state_randomized", "get_ee_pose", "current_arm_qpos_dict",
    "set_arm_position_targets", "set_gripper_opening", "current_finger_opening_m",
    "ARM_JOINT_NAMES", "ARM_QPOS_ADR", "FINGER_QPOS_ADR", "BLOCK_QPOS_ADR", "BLOCK_BODY_ID", "HAND_BODY_ID",
    "TABLE_TOP_SURFACE_Z_M", "CUBE_HALF_SIZE_M", "CUBE_TEXTURE_BUMP_RADIUS_M", "FOTS_TACTILE_FPS", "PROJECT_DIR",
]
missing = [name for name in required_globals if name not in globals()]
assert not missing, (
    "Run Cell P first (set RUN_CLOSED_LOOP_ROLLOUTS=False to build only the scene and the two FOTS "
    "sensors, without OpenVLA), after Cells 2, A, B, C, F, M and P0. Missing: " + ", ".join(missing)
)

SENSOR_TEST_CUBE_XY = (0.50, 0.00)   # the D3 nominal cube start
SENSOR_TEST_PLACE_X_M = 0.72         # far-side x used by TactileModel05_Duo's pick-and-place
SENSOR_TEST_GRIPPER_OPEN_M = 0.04
SENSOR_TEST_GRIPPER_CLOSED_M = 0.00
SENSOR_TEST_HOME_XYZ = np.array([0.307, 0.00, 0.590])
SENSOR_TEST_PREGRASP_Z_M = 0.280
SENSOR_TEST_GRASP_SEED_Z_M = 0.180
SENSOR_TEST_LIFT_Z_M = 0.360
SENSOR_TEST_RETREAT_Z_M = 0.320
SENSOR_TEST_SERIES_DT_S = 0.01       # cube/gripper state sampling period for the time series

sensor_test_dir = Path(PROJECT_DIR) / "openvla36_dual_sensor_test" / datetime.now().strftime("%Y%m%d_%H%M%S")
sensor_test_dir.mkdir(parents=True, exist_ok=True)

# --- Plan every waypoint's arm pose first (pure IK, nothing physically stepped yet). -----------
_cube_pos = np.array(
    [*SENSOR_TEST_CUBE_XY, TABLE_TOP_SURFACE_Z_M + CUBE_HALF_SIZE_M + CUBE_TEXTURE_BUMP_RADIUS_M],
    dtype=np.float64,
)
reset_task_state_randomized(cube_pos=_cube_pos, robot_joint_noise=None, settle_steps=60)
_cube_center = data.xpos[BLOCK_BODY_ID].copy()            # settled cube center (the grasp target)
_home_quat = get_ee_pose()["quat_wxyz"].copy()            # exactly vertical, by construction of HOME_QPOS
_seed = current_arm_qpos_dict()

_ik_reports = {}
for _name, _target in (
    ("home", SENSOR_TEST_HOME_XYZ),
    ("pregrasp", np.array([_cube_center[0], _cube_center[1], SENSOR_TEST_PREGRASP_Z_M])),
    ("grasp_seed", np.array([_cube_center[0], _cube_center[1], SENSOR_TEST_GRASP_SEED_Z_M])),
):
    _seed, _ik_reports[_name] = solve_hand_pose_ik(_target, _home_quat, seed_qpos=_seed)

# Center BOTH gel pads (not just the hand origin) on the cube's face-center height: with the wrist
# locked vertical the hand-to-pad offset is a fixed rigid transform, so one Z-only correction is enough.
_gel_mid_z = 0.5 * float(data.geom_xpos[GELSIGHT_GEL_PAD_GEOM_ID_RIGHT, 2] + data.geom_xpos[GELSIGHT_GEL_PAD_GEOM_ID_LEFT, 2])
_grasp_hand_z = float(data.xpos[HAND_BODY_ID][2]) + (float(_cube_center[2]) - _gel_mid_z)
grasp_qpos, _ik_reports["grasp"] = solve_hand_pose_ik(
    np.array([_cube_center[0], _cube_center[1], _grasp_hand_z]), _home_quat, seed_qpos=_seed
)
_gel_right_xz_offset_mm = 1e3 * float(np.linalg.norm(data.geom_xpos[GELSIGHT_GEL_PAD_GEOM_ID_RIGHT, [0, 2]] - _cube_center[[0, 2]]))
_gel_left_xz_offset_mm = 1e3 * float(np.linalg.norm(data.geom_xpos[GELSIGHT_GEL_PAD_GEOM_ID_LEFT, [0, 2]] - _cube_center[[0, 2]]))
_hand_tilt_deg = float(np.degrees(np.arccos(np.clip(-data.xmat[HAND_BODY_ID].reshape(3, 3)[2, 2], -1.0, 1.0))))

_place_xy = np.array([SENSOR_TEST_PLACE_X_M, _cube_center[1]])
lift_qpos, _ik_reports["lift"] = solve_hand_pose_ik(
    np.array([_cube_center[0], _cube_center[1], SENSOR_TEST_LIFT_Z_M]), _home_quat, seed_qpos=grasp_qpos)
transfer_qpos, _ik_reports["transfer"] = solve_hand_pose_ik(
    np.array([_place_xy[0], _place_xy[1], SENSOR_TEST_LIFT_Z_M]), _home_quat, seed_qpos=lift_qpos)
place_qpos, _ik_reports["place"] = solve_hand_pose_ik(
    np.array([_place_xy[0], _place_xy[1], _grasp_hand_z]), _home_quat, seed_qpos=transfer_qpos)
retreat_qpos, _ik_reports["retreat"] = solve_hand_pose_ik(
    np.array([_place_xy[0], _place_xy[1], SENSOR_TEST_RETREAT_Z_M]), _home_quat, seed_qpos=place_qpos)

_unconverged = {name: report for name, report in _ik_reports.items() if not report["converged"]}
print(f"Grasp pose: hand z={_grasp_hand_z:.4f} m, tilt from vertical {_hand_tilt_deg:.3f} deg; "
      f"gel pad centers vs cube face center (x,z): right {_gel_right_xz_offset_mm:.2f} mm, left {_gel_left_xz_offset_mm:.2f} mm")
if _unconverged:
    print("WARNING: IK did not fully converge for:", {k: round(v["position_error_m"] * 1e3, 3) for k, v in _unconverged.items()}, "(position error, mm)")

# (name, arm_qpos_dict, gripper_opening_m, duration_s)
SENSOR_TEST_PHASES = [
    ("hold_open", grasp_qpos, SENSOR_TEST_GRIPPER_OPEN_M, 1.0),
    ("close", grasp_qpos, SENSOR_TEST_GRIPPER_CLOSED_M, 1.0),
    ("hold_closed", grasp_qpos, SENSOR_TEST_GRIPPER_CLOSED_M, 1.0),
    ("lift", lift_qpos, SENSOR_TEST_GRIPPER_CLOSED_M, 1.0),
    ("transfer", transfer_qpos, SENSOR_TEST_GRIPPER_CLOSED_M, 1.5),
    ("place", place_qpos, SENSOR_TEST_GRIPPER_CLOSED_M, 1.0),
    ("release", place_qpos, SENSOR_TEST_GRIPPER_OPEN_M, 0.7),
    ("retreat", retreat_qpos, SENSOR_TEST_GRIPPER_OPEN_M, 1.0),
]

# --- Start already grasp-ready: teleport into the grasp pose with the fingers fully open. -------
mujoco.mj_resetData(model, data)
data.qvel[:] = 0.0
data.qpos[BLOCK_QPOS_ADR:BLOCK_QPOS_ADR + 7] = np.array([*_cube_pos, 1.0, 0.0, 0.0, 0.0])
for _joint_name, _qpos_adr in zip(ARM_JOINT_NAMES, ARM_QPOS_ADR):
    data.qpos[_qpos_adr] = grasp_qpos[_joint_name]
for _qpos_adr in FINGER_QPOS_ADR:
    data.qpos[_qpos_adr] = SENSOR_TEST_GRIPPER_OPEN_M
set_arm_position_targets(grasp_qpos)
set_gripper_opening(SENSOR_TEST_GRIPPER_OPEN_M)
mujoco.mj_forward(model, data)
for _ in range(40):
    set_arm_position_targets(grasp_qpos)
    set_gripper_opening(SENSOR_TEST_GRIPPER_OPEN_M)
    mujoco.mj_step(model, data)
print(f"Start: hand={np.round(data.xpos[HAND_BODY_ID], 4)} cube={np.round(data.xpos[BLOCK_BODY_ID], 4)} "
      f"finger opening={current_finger_opening_m():.4f} m")

# --- Record BOTH sensors through the whole episode. -----------------------------------------------
sensor_test_recorder = DualFOTSRecorder(sensor_test_dir / "scripted", FOTS_SENSOR_RIGHT, FOTS_SENSOR_LEFT)
sensor_test_recorder.capture_initial()

_snapshot_keys = ("tactile_rgb", "marker_flow", "depth_difference", "contact_mask")
sensor_test_snapshots = {}      # phase name -> {side: raw observation arrays at the END of the phase}
sensor_test_phase_windows = {}  # phase name -> (start_time_s, end_time_s) on MuJoCo's own sim clock
sensor_test_series = {"t": [], "cube_xyz": [], "finger_opening_m": []}

_series_every = max(1, int(round(SENSOR_TEST_SERIES_DT_S / model.opt.timestep)))
_previous_qpos = grasp_qpos
_previous_gripper = SENSOR_TEST_GRIPPER_OPEN_M
_step_count = 0
try:
    for _phase_name, _qpos_target, _gripper_target_m, _duration_s in SENSOR_TEST_PHASES:
        _phase_start = float(data.time)
        _steps = max(1, int(round(_duration_s / model.opt.timestep)))
        for _local in range(_steps):
            _alpha = (_local + 1) / _steps
            set_arm_position_targets({
                joint: (1.0 - _alpha) * _previous_qpos[joint] + _alpha * _qpos_target[joint] for joint in ARM_JOINT_NAMES
            })
            set_gripper_opening((1.0 - _alpha) * _previous_gripper + _alpha * _gripper_target_m)
            mujoco.mj_step(model, data)
            sensor_test_recorder.after_physics_step()
            _step_count += 1
            if _step_count % _series_every == 0:
                sensor_test_series["t"].append(float(data.time))
                sensor_test_series["cube_xyz"].append(data.xpos[BLOCK_BODY_ID].copy())
                sensor_test_series["finger_opening_m"].append(current_finger_opening_m())
        _previous_qpos, _previous_gripper = _qpos_target, _gripper_target_m
        sensor_test_phase_windows[_phase_name] = (_phase_start, float(data.time))
        sensor_test_snapshots[_phase_name] = {
            side: {key: np.array(recorder.last_observation[key]) for key in _snapshot_keys}
            for side, recorder in sensor_test_recorder.recorders.items()
        }
        print(f"after {_phase_name:12s} t={data.time:5.2f}s hand={np.round(data.xpos[HAND_BODY_ID], 3)} "
              f"cube={np.round(data.xpos[BLOCK_BODY_ID], 3)} finger opening={current_finger_opening_m():.4f} m")
finally:
    sensor_test_outputs = sensor_test_recorder.finalize()
    for _sensor in (FOTS_SENSOR_RIGHT, FOTS_SENSOR_LEFT):
        _sensor._traj = []   # leave the shared sensors with a clean marker-motion history

for _key in sensor_test_series:
    sensor_test_series[_key] = np.asarray(sensor_test_series[_key])
sensor_test_meta = {
    "cube_start_xyz": _cube_center.tolist(),
    "cube_final_xyz": data.xpos[BLOCK_BODY_ID].tolist(),
    "grasp_hand_z_m": _grasp_hand_z,
    "gel_pad_xz_offset_mm": {"right": _gel_right_xz_offset_mm, "left": _gel_left_xz_offset_mm},
    "hand_tilt_from_vertical_deg": _hand_tilt_deg,
    "place_x_m": SENSOR_TEST_PLACE_X_M,
    "phase_windows_s": {name: list(window) for name, window in sensor_test_phase_windows.items()},
    "ik_unconverged": {name: report["position_error_m"] for name, report in _unconverged.items()},
}
(sensor_test_dir / "scripted" / "episode_meta.json").write_text(json.dumps(sensor_test_meta, indent=2), encoding="utf-8")
np.savez_compressed(
    sensor_test_dir / "scripted" / "phase_snapshots.npz",
    **{f"{phase}__{side}__{key}": value for phase, per_side in sensor_test_snapshots.items()
       for side, arrays in per_side.items() for key, value in arrays.items()},
)

for _side, _outputs in sensor_test_outputs.items():
    print(f"[{_side}] recorder validation: {_outputs['validation']}")
print("Cube start:", np.round(_cube_center, 4), " final:", np.round(data.xpos[BLOCK_BODY_ID], 4))
print("Saved test episode to:", sensor_test_dir / "scripted")


In [ ]:
# Cell T2: Judge BOTH sensors' outputs from Cell T1's scripted episode.
#   * per-sensor quality checks (recording integrity, clean no-contact baseline, contact on grasp,
#     physically plausible indentation, stable/persistent contact under load, the bump lattice
#     showing up with the right geometry, tactile RGB + marker-flow response, recovery after
#     release, saved-video fidelity, object pose read-out);
#   * cross-sensor checks (time alignment, balanced squeeze, same contact window, mirrored
#     object pose, same lattice pitch);
#   * one gripper check (the two gel pads actually lifted and carried the cube -- the sensor
#     readings under load only mean something if the grasp held);
# then a PASS/FAIL table, figures, a side-by-side video and a JSON report. The cell raises
# AssertionError if any check fails, after all artifacts have been written.
#
# Thresholds are set from the measured behaviour of this scene (TactileModel05_Duo's scripted
# runs and this notebook's own T1 run: both sensors ~4,700 taxels, ~0.77 mm peak indentation, a
# regular 3x3 bump lattice) with wide margins, and where possible against physical ground truth
# (lattice pitch, contact-patch area, cube distance) instead of a hand-picked number.
import imageio.v3 as imageio_v3
import matplotlib.pyplot as plt
from scipy import ndimage as ndi

assert "sensor_test_outputs" in globals(), "Run Cell T1 first."

SIDES = ("right", "left")
_gel_geom_id = {"right": GELSIGHT_GEL_PAD_GEOM_ID_RIGHT, "left": GELSIGHT_GEL_PAD_GEOM_ID_LEFT}
_site_id = {"right": GELSIGHT_SITE_ID_RIGHT, "left": GELSIGHT_SITE_ID_LEFT}
_camera_id = {"right": FOTS_TACTILE_CAM_ID_RIGHT, "left": FOTS_TACTILE_CAM_ID_LEFT}
_sensor = {"right": FOTS_SENSOR_RIGHT, "left": FOTS_SENSOR_LEFT}

# --- Thresholds -------------------------------------------------------------------------------------
GEL_THICKNESS_MM = 2e3 * float(model.geom_size[_gel_geom_id["right"], 1])   # deeper than this is not gel deformation
MIN_STEADY_TAXELS = 1500          # measured ~4,700 per sensor while squeezing the dotted cube
MIN_PEAK_DEPTH_MM = 0.1           # measured 0.29-0.78 mm (smaller cube = shallower squeeze); the contact threshold itself is 0.02 mm
MAX_STEADY_TAXEL_CV = 0.25        # measured ~0.01 (std/mean of the contact area while the squeeze is held)
MIN_LOAD_CONTACT_FRACTION = 0.98  # over hold_closed + lift + transfer; measured 1.00
MIN_LATTICE_BLOBS = 4             # gel window covers >= 2 x 3 bumps of the lattice; measured 9 (full-size cube), 16 (2/3-scale cube)
MAX_TINY_BLOBS = 1                # contact fragments < 30 px away from the image border (speckle/fleck artifacts); measured 0
LATTICE_PITCH_TOL = 0.03          # measured pitch vs bump_pitch * focal_px / reference_depth; measured < 0.1 %
PATCH_AREA_TOL = 0.25             # measured vs pi*(2*R*d - d^2) for the bump sphere; measured ~ 2 %
OBJECT_DISTANCE_TOL_MM = 1.5      # cube-center distance vs half-size + bump - indentation - site offset; measured < 0.1 mm
MIN_RGB_RESPONSE_ALL = 4.0        # mean |RGB - baseline| over the whole image (0-255); measured ~14
MIN_RGB_RESPONSE_IN_CONTACT = 20.0  # same, over contact pixels only; measured ~59
MIN_MARKER_RESPONSE = 1.0         # mean |marker_flow - baseline| (0-255); measured ~4.7
BASELINE_MAX_ABS_DIFF = 1         # no-contact RGB must equal FOTS's own zero-height render (measured 0)
VIDEO_MAX_MEAN_ABS_DIFF = 6.0     # saved H.264 tactile-RGB frame vs raw sensor frame (0-255)
MIN_BALANCE_RATIO = 0.8           # min/max of right vs left steady taxels and steady indentation; measured ~0.96-0.99
MAX_ONSET_DIFF_S = 0.10
MIN_TAXEL_CORRELATION = 0.9       # Pearson r of the two contact-area time series; measured 0.98
MIRROR_TOL_MM = 1.5
MIN_CUBE_LIFT_M = 0.10            # cube center must rise at least this much during "lift" (measured ~0.14)
PLACE_TOL_M = 0.05

SENSOR_TEST_THRESHOLD_NAMES = (
    "GEL_THICKNESS_MM", "MIN_STEADY_TAXELS", "MIN_PEAK_DEPTH_MM", "MAX_STEADY_TAXEL_CV", "MIN_LOAD_CONTACT_FRACTION",
    "MIN_LATTICE_BLOBS", "MAX_TINY_BLOBS", "LATTICE_PITCH_TOL", "PATCH_AREA_TOL", "OBJECT_DISTANCE_TOL_MM",
    "MIN_RGB_RESPONSE_ALL", "MIN_RGB_RESPONSE_IN_CONTACT", "MIN_MARKER_RESPONSE", "BASELINE_MAX_ABS_DIFF",
    "VIDEO_MAX_MEAN_ABS_DIFF", "MIN_BALANCE_RATIO", "MAX_ONSET_DIFF_S", "MIN_TAXEL_CORRELATION", "MIRROR_TOL_MM",
    "MIN_CUBE_LIFT_M", "PLACE_TOL_M",
)
_windows = sensor_test_phase_windows
_load_phases = ("hold_closed", "lift", "transfer")   # sustained grip under load ("place" is the cube re-seating on the table)


def _frames_in(t, phase_names):
    mask = np.zeros(len(t), dtype=bool)
    for name in phase_names:
        start, end = _windows[name]
        mask |= (t > start) & (t <= end)
    return np.flatnonzero(mask)


def _snapshot_index(t, phase_name):
    """Index of the last recorded tactile frame at or before the END of `phase_name` -- the frame
    Cell T1 kept the raw arrays of."""
    return int(np.searchsorted(t, _windows[phase_name][1], side="right") - 1)


def _blob_stats(mask):
    """Connected contact blobs. `whole` marks blobs that do not touch the image border and are at
    least 60 % of the median interior blob area: their centroid and area are unbiased, so the
    lattice pitch and patch size are measured on them only."""
    labels, count = ndi.label(mask)
    if count == 0:
        return {"count": 0, "areas": np.zeros(0), "centroids": np.zeros((0, 2)), "on_border": np.zeros(0, dtype=bool), "whole": np.zeros(0, dtype=bool)}
    index = range(1, count + 1)
    height, width = mask.shape
    on_border = np.array([
        rows.start == 0 or cols.start == 0 or rows.stop == height or cols.stop == width
        for rows, cols in ndi.find_objects(labels)
    ])
    areas = np.asarray(ndi.sum(mask, labels, index), dtype=np.float64)
    reference_area = np.median(areas[~on_border]) if (~on_border).any() else np.median(areas)
    return {
        "count": int(count),
        "areas": areas,
        "centroids": np.asarray(ndi.center_of_mass(mask, labels, index), dtype=np.float64),
        "on_border": on_border,
        "whole": (~on_border) & (areas >= 0.6 * reference_area),
    }


def _nearest_neighbour_pitch_px(blobs):
    """Median nearest-neighbour centroid spacing of the whole (unclipped) blobs."""
    if blobs["count"] < 2:
        return float("nan")
    centroids = blobs["centroids"][blobs["whole"]]
    if len(centroids) < 2:
        return float("nan")
    distances = np.linalg.norm(centroids[:, None, :] - centroids[None, :, :], axis=-1)
    np.fill_diagonal(distances, np.inf)
    return float(np.median(distances.min(axis=1)))


def _analyze_side(side):
    outputs = sensor_test_outputs[side]
    quant = np.load(outputs["quantitative_npz"])
    t = quant["sim_time_s"]
    taxels = quant["active_taxel_count"].astype(np.float64)
    max_depth_mm = quant["max_depth_diff_mm"]
    rel_pos_mm = quant["relative_pos_mm"]
    reference_depth_m = float(quant["reference_depth_m"])
    contact_eps_m = float(quant["contact_eps_m"])
    finite_depth = bool(np.isfinite(quant["depth_m"]).all())

    rgb_video = list(imageio_v3.imiter(outputs["tactile_rgb_video"]))
    marker_video = list(imageio_v3.imiter(outputs["marker_flow_video"]))
    snapshots = {phase: sensor_test_snapshots[phase][side] for phase in sensor_test_snapshots}

    hold_open = _frames_in(t, ("hold_open",))
    hold_closed = _frames_in(t, ("hold_closed",))
    load = _frames_in(t, _load_phases)
    steady_depth_mm = float(np.median(max_depth_mm[hold_closed]))
    steady_taxels = float(np.median(taxels[hold_closed]))

    steady_mask = snapshots["hold_closed"]["contact_mask"]
    blobs = _blob_stats(steady_mask)
    baseline_rgb = snapshots["hold_open"]["tactile_rgb"].astype(np.float64)
    steady_rgb = snapshots["hold_closed"]["tactile_rgb"].astype(np.float64)
    reference_rgb = _sensor[side].render_optical_image(
        np.zeros((FOTS_SENSOR_H, FOTS_SENSOR_W), dtype=np.float32), np.zeros((FOTS_SENSOR_H, FOTS_SENSOR_W), dtype=bool)
    ).astype(np.float64)
    baseline_marker = snapshots["hold_open"]["marker_flow"].astype(np.float64)
    steady_marker = snapshots["hold_closed"]["marker_flow"].astype(np.float64)
    released_rgb = snapshots["retreat"]["tactile_rgb"].astype(np.float64)
    released_marker = snapshots["retreat"]["marker_flow"].astype(np.float64)

    steady_idx = _snapshot_index(t, "hold_closed")
    focal_px = 0.5 * FOTS_SENSOR_H / np.tan(np.deg2rad(float(model.cam_fovy[_camera_id[side]])) / 2.0)
    px_per_mm = focal_px / (reference_depth_m * 1e3)
    bump_pitch_m = 2.0 * CUBE_TEXTURE_GRID_HALF_SPAN_M / (CUBE_TEXTURE_GRID_SIZE - 1)
    depth_eff_mm = max(steady_depth_mm - contact_eps_m * 1e3, 0.0)
    radius_mm = CUBE_TEXTURE_BUMP_RADIUS_M * 1e3
    patch_area_mm2 = np.pi * max(2.0 * radius_mm * depth_eff_mm - depth_eff_mm ** 2, 0.0)
    site_ahead_of_gel_mm = 1e3 * (
        (float(model.geom_pos[_gel_geom_id[side]][1]) - float(model.geom_size[_gel_geom_id[side]][1]))
        - float(model.site_pos[_site_id[side]][1])
    )

    onset = np.flatnonzero(taxels > 0)
    return {
        "t": t, "taxels": taxels, "max_depth_mm": max_depth_mm, "rel_pos_mm": rel_pos_mm,
        "n_frames": len(t), "n_rgb_video_frames": len(rgb_video), "n_marker_video_frames": len(marker_video),
        "finite_depth": finite_depth, "finite_pose": bool(np.isfinite(rel_pos_mm).all()),
        "hold_open": hold_open, "hold_closed": hold_closed, "load": load,
        "steady_taxels": steady_taxels, "steady_depth_mm": steady_depth_mm,
        "steady_taxel_cv": float(np.std(taxels[hold_closed]) / max(np.mean(taxels[hold_closed]), 1e-9)),
        "blobs": blobs, "n_tiny_blobs": int(((blobs["areas"] < 30) & ~blobs["on_border"]).sum()),
        "pitch_px": _nearest_neighbour_pitch_px(blobs),
        "expected_pitch_px": bump_pitch_m * focal_px / reference_depth_m,
        "patch_area_px": float(np.median(blobs["areas"][blobs["whole"]])) if blobs["whole"].any() else float("nan"),
        "expected_patch_area_px": patch_area_mm2 * px_per_mm ** 2,
        "rgb_shape_ok": all(
            snapshots[p][k].shape == (FOTS_SENSOR_H, FOTS_SENSOR_W, 3) and snapshots[p][k].dtype == np.uint8
            for p in snapshots for k in ("tactile_rgb", "marker_flow")
        ),
        "baseline_vs_reference": float(np.abs(baseline_rgb - reference_rgb).max()),
        "rgb_response_all": float(np.abs(steady_rgb - baseline_rgb).mean()),
        "rgb_response_in_contact": float(np.abs(steady_rgb - baseline_rgb)[steady_mask].mean()) if steady_mask.any() else 0.0,
        "marker_response": float(np.abs(steady_marker - baseline_marker).mean()),
        "released_rgb_diff": float(np.abs(released_rgb - baseline_rgb).max()),
        "released_marker_diff": float(np.abs(released_marker - baseline_marker).max()),
        "released_taxels_last10": float(taxels[-10:].max()),
        "video_vs_raw": float(np.abs(rgb_video[steady_idx].astype(np.float64) - steady_rgb).mean()),
        "steady_rel_pos_mm": np.median(rel_pos_mm[hold_closed], axis=0),
        "expected_distance_mm": CUBE_HALF_SIZE_M * 1e3 + radius_mm - steady_depth_mm - site_ahead_of_gel_mm,
        "onset_t": float(t[onset[0]]) if len(onset) else float("nan"),
        "last_contact_t": float(t[onset[-1]]) if len(onset) else float("nan"),
        "snapshots": snapshots, "reference_depth_m": reference_depth_m, "contact_eps_m": contact_eps_m,
        "expected_frames": (max(w[1] for w in _windows.values()) - min(w[0] for w in _windows.values())) * FOTS_TACTILE_FPS,
        "recorder_frames": sensor_test_recorder.recorders[side].frame_count,
    }


analysis = {side: _analyze_side(side) for side in SIDES}
R, L = analysis["right"], analysis["left"]

# --- Checks ----------------------------------------------------------------------------------------
checks = []


def _add(group, name, side, value, criterion, passed):
    checks.append({"group": group, "name": name, "side": side, "value": value, "criterion": criterion, "passed": bool(passed)})


for side in SIDES:
    a = analysis[side]
    _add("sensor", "recording complete", side,
         f"npz={a['n_frames']} rgb={a['n_rgb_video_frames']} marker={a['n_marker_video_frames']} recorder={a['recorder_frames']}",
         "all equal and >= 95% of the expected frame count",
         a["n_frames"] == a["n_rgb_video_frames"] == a["n_marker_video_frames"] == a["recorder_frames"]
         and a["n_frames"] >= 0.95 * a["expected_frames"])
    _add("sensor", "outputs finite, correct shape", side,
         f"depth finite={a['finite_depth']} pose finite={a['finite_pose']} rgb/marker (320,240,3) uint8={a['rgb_shape_ok']}",
         "all True", a["finite_depth"] and a["finite_pose"] and a["rgb_shape_ok"])
    _add("sensor", "clean no-contact baseline", side,
         f"max taxels while open={a['taxels'][a['hold_open']].max():.0f}, nearest object {-a['max_depth_mm'][a['hold_open']].max():.2f} mm beyond the gel",
         "0 taxels and nothing at/inside the gel plane",
         a["taxels"][a["hold_open"]].max() == 0 and a["max_depth_mm"][a["hold_open"]].max() < 0.0)
    _add("sensor", "baseline RGB = zero-height render", side,
         f"max |RGB - reference| = {a['baseline_vs_reference']:.1f}", f"<= {BASELINE_MAX_ABS_DIFF}",
         a["baseline_vs_reference"] <= BASELINE_MAX_ABS_DIFF)
    _add("sensor", "contact on grasp", side,
         f"steady taxels min={a['taxels'][a['hold_closed']].min():.0f} median={a['steady_taxels']:.0f}",
         f"min >= {MIN_STEADY_TAXELS}", a["taxels"][a["hold_closed"]].min() >= MIN_STEADY_TAXELS)
    _add("sensor", "physical indentation", side,
         f"steady {a['steady_depth_mm']:.3f} mm, episode max {a['max_depth_mm'].max():.3f} mm",
         f"steady >= {MIN_PEAK_DEPTH_MM} mm and episode max <= gel thickness {GEL_THICKNESS_MM:.1f} mm",
         MIN_PEAK_DEPTH_MM <= a["steady_depth_mm"] and a["max_depth_mm"].max() <= GEL_THICKNESS_MM)
    _add("sensor", "stable steady contact", side, f"cv of contact area while held = {a['steady_taxel_cv']:.3f}",
         f"<= {MAX_STEADY_TAXEL_CV}", a["steady_taxel_cv"] <= MAX_STEADY_TAXEL_CV)
    load_fraction = float((a["taxels"][a["load"]] > 0).mean())
    _add("sensor", "contact persists under load", side, f"{100 * load_fraction:.1f} % of hold/lift/transfer frames",
         f">= {100 * MIN_LOAD_CONTACT_FRACTION:.0f} %", load_fraction >= MIN_LOAD_CONTACT_FRACTION)
    _add("sensor", "bump lattice resolved", side,
         f"{a['blobs']['count']} contact blobs ({int(a['blobs']['whole'].sum())} whole), {a['n_tiny_blobs']} tiny (<30 px) away from the border",
         f">= {MIN_LATTICE_BLOBS} blobs and <= {MAX_TINY_BLOBS} tiny",
         a["blobs"]["count"] >= MIN_LATTICE_BLOBS and a["n_tiny_blobs"] <= MAX_TINY_BLOBS)
    pitch_error = abs(a["pitch_px"] - a["expected_pitch_px"]) / a["expected_pitch_px"] if np.isfinite(a["pitch_px"]) else float("inf")
    _add("sensor", "lattice pitch = physical pitch", side,
         f"{a['pitch_px']:.1f} px vs {a['expected_pitch_px']:.1f} px expected ({100 * pitch_error:.2f} % off)",
         f"within {100 * LATTICE_PITCH_TOL:.0f} %", pitch_error <= LATTICE_PITCH_TOL)
    area_error = abs(a["patch_area_px"] - a["expected_patch_area_px"]) / a["expected_patch_area_px"] if np.isfinite(a["patch_area_px"]) else float("inf")
    _add("sensor", "contact patch = bump geometry", side,
         f"{a['patch_area_px']:.0f} px vs {a['expected_patch_area_px']:.0f} px expected ({100 * area_error:.1f} % off)",
         f"within {100 * PATCH_AREA_TOL:.0f} %", area_error <= PATCH_AREA_TOL)
    _add("sensor", "tactile RGB responds", side,
         f"mean |RGB - baseline|: image {a['rgb_response_all']:.1f}, contact pixels {a['rgb_response_in_contact']:.1f}",
         f"image >= {MIN_RGB_RESPONSE_ALL} and contact >= {MIN_RGB_RESPONSE_IN_CONTACT}",
         a["rgb_response_all"] >= MIN_RGB_RESPONSE_ALL and a["rgb_response_in_contact"] >= MIN_RGB_RESPONSE_IN_CONTACT)
    _add("sensor", "marker flow responds", side, f"mean |flow - baseline| = {a['marker_response']:.2f}",
         f">= {MIN_MARKER_RESPONSE}", a["marker_response"] >= MIN_MARKER_RESPONSE)
    _add("sensor", "recovers after release", side,
         f"last-10-frame taxels max={a['released_taxels_last10']:.0f}, RGB diff={a['released_rgb_diff']:.0f}, marker diff={a['released_marker_diff']:.0f}",
         f"0 taxels, RGB and marker flow back to baseline (<= {BASELINE_MAX_ABS_DIFF})",
         a["released_taxels_last10"] == 0 and a["released_rgb_diff"] <= BASELINE_MAX_ABS_DIFF
         and a["released_marker_diff"] <= BASELINE_MAX_ABS_DIFF)
    _add("sensor", "saved video = raw output", side, f"mean |video frame - raw frame| = {a['video_vs_raw']:.2f}",
         f"<= {VIDEO_MAX_MEAN_ABS_DIFF}", a["video_vs_raw"] <= VIDEO_MAX_MEAN_ABS_DIFF)
    measured_distance_mm = abs(float(a["steady_rel_pos_mm"][1]))
    _add("sensor", "cube distance = geometry", side,
         f"{measured_distance_mm:.2f} mm vs {a['expected_distance_mm']:.2f} mm expected",
         f"within {OBJECT_DISTANCE_TOL_MM} mm", abs(measured_distance_mm - a["expected_distance_mm"]) <= OBJECT_DISTANCE_TOL_MM)

# Cross-sensor consistency.
_ratio = lambda x, y: float(min(x, y) / max(x, y)) if max(x, y) > 0 else 0.0
_common = min(R["n_frames"], L["n_frames"])   # unequal counts are reported by "recording complete", not a crash here
both_in_contact = (R["taxels"][:_common] > 0) | (L["taxels"][:_common] > 0)
taxel_correlation = (
    float(np.corrcoef(R["taxels"][:_common][both_in_contact], L["taxels"][:_common][both_in_contact])[0, 1])
    if both_in_contact.sum() > 2 else float("nan")
)
rel_r, rel_l = R["steady_rel_pos_mm"], L["steady_rel_pos_mm"]
_add("cross", "time-aligned frames", "R vs L",
     f"identical timestamps={R['n_frames'] == L['n_frames'] and np.array_equal(R['t'], L['t'])} ({R['n_frames']} vs {L['n_frames']} frames)",
     "identical", R["n_frames"] == L["n_frames"] and np.array_equal(R["t"], L["t"]))
_add("cross", "balanced squeeze (contact area)", "R vs L",
     f"{R['steady_taxels']:.0f} vs {L['steady_taxels']:.0f} taxels (min/max {_ratio(R['steady_taxels'], L['steady_taxels']):.2f})",
     f"min/max >= {MIN_BALANCE_RATIO}", _ratio(R["steady_taxels"], L["steady_taxels"]) >= MIN_BALANCE_RATIO)
_add("cross", "balanced squeeze (indentation)", "R vs L",
     f"{R['steady_depth_mm']:.3f} vs {L['steady_depth_mm']:.3f} mm (min/max {_ratio(R['steady_depth_mm'], L['steady_depth_mm']):.2f})",
     f"min/max >= {MIN_BALANCE_RATIO}", _ratio(R["steady_depth_mm"], L["steady_depth_mm"]) >= MIN_BALANCE_RATIO)
_add("cross", "same contact window", "R vs L",
     f"onset {R['onset_t']:.3f}/{L['onset_t']:.3f} s, last contact {R['last_contact_t']:.3f}/{L['last_contact_t']:.3f} s",
     f"onset and release within {MAX_ONSET_DIFF_S} s",
     abs(R["onset_t"] - L["onset_t"]) <= MAX_ONSET_DIFF_S and abs(R["last_contact_t"] - L["last_contact_t"]) <= MAX_ONSET_DIFF_S)
_add("cross", "contact-area series agree", "R vs L", f"Pearson r = {taxel_correlation:.3f}", f">= {MIN_TAXEL_CORRELATION}",
     np.isfinite(taxel_correlation) and taxel_correlation >= MIN_TAXEL_CORRELATION)
_add("cross", "mirrored object pose", "R vs L",
     f"cube in sensor frame: right {np.round(rel_r, 2).tolist()} mm, left {np.round(rel_l, 2).tolist()} mm",
     f"x mirrored (sum), y and z equal, all within {MIRROR_TOL_MM} mm",
     abs(rel_r[0] + rel_l[0]) <= MIRROR_TOL_MM and abs(rel_r[1] - rel_l[1]) <= MIRROR_TOL_MM and abs(rel_r[2] - rel_l[2]) <= MIRROR_TOL_MM)
pitch_ratio = _ratio(R["pitch_px"], L["pitch_px"]) if np.isfinite(R["pitch_px"]) and np.isfinite(L["pitch_px"]) else 0.0
_add("cross", "same lattice pitch", "R vs L", f"{R['pitch_px']:.1f} vs {L['pitch_px']:.1f} px", "within 2 %", pitch_ratio >= 0.98)

# Gripper context: the readings under load only mean something if the grasp really held the cube.
cube_series = sensor_test_series["cube_xyz"]
lift_end_z = float(cube_series[np.searchsorted(sensor_test_series["t"], _windows["lift"][1]) - 1][2])
cube_start_xyz = np.asarray(sensor_test_meta["cube_start_xyz"])
cube_final_xyz = np.asarray(sensor_test_meta["cube_final_xyz"])
_add("gripper", "cube lifted by the two gel pads", "both",
     f"cube z rose {1e3 * (lift_end_z - cube_start_xyz[2]):.0f} mm during lift", f">= {1e3 * MIN_CUBE_LIFT_M:.0f} mm",
     lift_end_z - cube_start_xyz[2] >= MIN_CUBE_LIFT_M)
_add("gripper", "cube carried and placed", "both",
     f"final x={cube_final_xyz[0]:.3f} m (target {sensor_test_meta['place_x_m']:.2f}), z={cube_final_xyz[2]:.3f} m (start {cube_start_xyz[2]:.3f})",
     f"x within {PLACE_TOL_M} m of target, resting back on the table",
     abs(cube_final_xyz[0] - sensor_test_meta["place_x_m"]) <= PLACE_TOL_M and abs(cube_final_xyz[2] - cube_start_xyz[2]) <= 0.005)

SENSOR_TEST_PASSED = all(c["passed"] for c in checks)

# --- Report table ----------------------------------------------------------------------------------
print(f"{'':9s}{'check':36s}{'sensor':8s}{'result':7s}measured  |  criterion")
for c in checks:
    print(f"{c['group']:9s}{c['name']:36s}{c['side']:8s}{'PASS' if c['passed'] else 'FAIL':7s}{c['value']}  |  {c['criterion']}")
n_failed = sum(not c["passed"] for c in checks)
print(f"\n{len(checks) - n_failed}/{len(checks)} checks passed.")

report_path = sensor_test_dir / "sensor_test_report.json"
report_path.write_text(json.dumps({
    "passed": bool(SENSOR_TEST_PASSED),
    "checks": checks,
    "thresholds": {name: globals()[name] for name in SENSOR_TEST_THRESHOLD_NAMES},
    "episode": sensor_test_meta,
}, indent=2, default=lambda o: o.tolist() if hasattr(o, "tolist") else str(o)), encoding="utf-8")

# --- Figure 1: what each sensor saw at the steady grip and while carrying ---------------------------
_fig_phases = (("hold_closed", "steady grip"), ("transfer", "carrying the cube"))
_columns = (("tactile_rgb", "FOTS tactile RGB"), ("depth_difference", f"depth_difference (0-{FOTS_DEPTH_DIFF_DISPLAY_MAX_MM:.0f} mm)"),
            ("contact_mask", "contact mask"), ("marker_flow", "marker flow"))
fig, axes = plt.subplots(2 * len(_fig_phases), len(_columns), figsize=(2.6 * len(_columns), 3.55 * 2 * len(_fig_phases)))
for row, (phase, phase_label) in enumerate([(p, l) for p, l in _fig_phases for _ in SIDES]):
    side = SIDES[row % 2]
    snap = sensor_test_snapshots[phase][side]
    for col, (key, col_label) in enumerate(_columns):
        image = snap[key]
        if key == "depth_difference":
            image = depth_diff_to_display_rgb(image)
        elif key == "contact_mask":
            image = (image.astype(np.uint8) * 255)
        axes[row, col].imshow(image, cmap="gray" if image.ndim == 2 else None)
        axes[row, col].set_xticks([])
        axes[row, col].set_yticks([])
        if row == 0:
            axes[row, col].set_title(col_label, fontsize=9)
    axes[row, 0].set_ylabel(f"{side.upper()}\n{phase_label}", fontsize=9)
fig.suptitle(f"Both GelSight sensors -- {'PASS' if SENSOR_TEST_PASSED else 'FAIL'} ({len(checks) - n_failed}/{len(checks)} checks)", fontsize=11)
fig.tight_layout()
figure_photo_path = sensor_test_dir / "sensor_test_photo.png"
fig.savefig(figure_photo_path, dpi=110)
plt.show()
plt.close(fig)

# --- Figure 2: both sensors over the whole episode -----------------------------------------------------
fig, axes = plt.subplots(4, 1, figsize=(11, 10), sharex=True)
_series_t = sensor_test_series["t"]
_colors = {"right": "tab:red", "left": "tab:blue"}
for side in SIDES:
    a = analysis[side]
    axes[0].plot(a["t"], a["taxels"], color=_colors[side], label=f"{side} sensor", lw=1.6, alpha=0.85)
    axes[1].plot(a["t"], np.clip(a["max_depth_mm"], -1.0, None), color=_colors[side], label=f"{side} sensor", lw=1.6, alpha=0.85)
    axes[2].plot(a["t"], np.abs(a["rel_pos_mm"][:, 1]), color=_colors[side], label=f"{side} sensor", lw=1.6, alpha=0.85)
axes[0].set_ylabel("active taxels (px)")
axes[1].axhline(GEL_THICKNESS_MM, color="k", ls="--", lw=1)
axes[1].text(0.01, GEL_THICKNESS_MM, " gel thickness (max physical indentation)", va="bottom", fontsize=8, transform=axes[1].get_yaxis_transform())
axes[1].set_ylabel("max depth diff (mm)\n(clipped at -1)")
axes[1].set_ylim(-1.05, GEL_THICKNESS_MM * 1.15)
axes[2].set_ylabel("cube-center distance from\nsensor site (mm)")
axes[3].plot(_series_t, 1e3 * sensor_test_series["cube_xyz"][:, 2], color="tab:green", label="cube z (mm)")
axes[3].plot(_series_t, 1e3 * sensor_test_series["finger_opening_m"], color="tab:orange", label="finger opening (mm)")
axes[3].set_ylabel("cube z / finger opening (mm)")
axes[3].set_xlabel("simulation time (s)")
for ax in axes:
    for phase, (start, end) in _windows.items():
        ax.axvspan(start, end, color="0.5", alpha=0.07 if list(_windows).index(phase) % 2 else 0.14, lw=0)
    ax.grid(alpha=0.25)
    ax.legend(loc="upper right", fontsize=8)
for phase, (start, end) in _windows.items():
    axes[0].text(0.5 * (start + end), 1.0, phase, transform=axes[0].get_xaxis_transform(), ha="center", va="bottom", fontsize=8)
fig.suptitle("Right vs left sensor over the scripted grasp / lift / carry / place / release episode", fontsize=11, y=0.995)
fig.tight_layout()
figure_series_path = sensor_test_dir / "sensor_test_timeseries.png"
fig.savefig(figure_series_path, dpi=110)
plt.show()
plt.close(fig)

# --- Side-by-side video of both sensors ------------------------------------------------------------------
def _phase_at(time_s):
    for name, (start, end) in _windows.items():
        if start < time_s <= end:
            return name
    return "start"


both_video_path = sensor_test_dir / "both_sensors_tactile.mp4"
_rgb_streams = {side: list(imageio_v3.imiter(sensor_test_outputs[side]["tactile_rgb_video"])) for side in SIDES}
_writer = imageio.get_writer(str(both_video_path), fps=FOTS_TACTILE_FPS, codec="libx264", quality=8, pixelformat="yuv420p", macro_block_size=None)
try:
    for i in range(analysis["right"]["n_frames"]):
        canvas = Image.new("RGB", (2 * FOTS_SENSOR_W, FOTS_SENSOR_H + 34), (18, 18, 22))
        draw = ImageDraw.Draw(canvas)
        for k, side in enumerate(SIDES):
            canvas.paste(Image.fromarray(_rgb_streams[side][i]), (k * FOTS_SENSOR_W, 34))
            draw.text((k * FOTS_SENSOR_W + 6, 4), f"{side.upper()}  taxels={int(analysis[side]['taxels'][i])}", fill=(240, 240, 240))
        draw.text((6, 19), f"t={analysis['right']['t'][i]:.2f}s  phase: {_phase_at(analysis['right']['t'][i])}", fill=(200, 200, 200))
        _writer.append_data(np.asarray(canvas))
finally:
    _writer.close()

print("Saved:", report_path, figure_photo_path, figure_series_path, both_video_path, sep="\n  ")
if DISPLAY_GELSIGHT_VIDEOS:
    display(Video(str(both_video_path), embed=False, width=2 * FOTS_SENSOR_W))

failed = [f"{c['name']} [{c['side']}]: {c['value']}  (need {c['criterion']})" for c in checks if not c["passed"]]
print("\nTWO-SENSOR TEST:", "PASS -- both GelSight sensors produce clean, symmetric, physically consistent output." if SENSOR_TEST_PASSED else "FAIL")
assert SENSOR_TEST_PASSED, "Two-sensor quality checks failed:\n  " + "\n  ".join(failed)


In [ ]:
# Cell T3: Audit the tactile data that the closed-loop OpenVLA rollouts saved -- BOTH sensors.
# Reads only files Cell P already wrote (rollout_NNN/right|left/fots_validation.json plus the small
# arrays of fots_quantitative_data.npz and closed_loop_log.json's outcome), so it never re-runs the
# policy or the simulator and works after a kernel restart. Cell T1/T2 is the controlled test of the
# sensors; this cell answers a different question: did both sensors also behave sensibly during the
# policy's own, messier rollouts?
#
# It is a diagnostic table, not a gate, because a policy rollout can legitimately fail to grasp
# (there is then no contact to judge). One physical invariant IS enforced: a rollout the policy
# completed successfully carried the cube with both gel pads, so both sensors must have registered
# a sustained grip in it -- a dead or one-sided sensor there is a sensor problem, not a policy one.
# None -> the run_dir of the last Cell P run in this kernel, else PROJECT_DIR/EXPERIMENT_DATASET/EXPERIMENT_CONDITION.
AUDIT_RUN_DIR = None
AUDIT_MIN_SUSTAINED_CONTACT_FRAMES = 30   # about 1 s at 30 Hz: what "sustained grip" means for a successful rollout
AUDIT_MIN_PEAK_TAXELS = 1000

if AUDIT_RUN_DIR is not None:
    _audit_dir = Path(AUDIT_RUN_DIR)
elif "run_dir" in globals() and Path(run_dir).is_dir():
    _audit_dir = Path(run_dir)
else:
    _audit_dir = Path(PROJECT_DIR) / EXPERIMENT_DATASET / EXPERIMENT_CONDITION

_gel_thickness_mm = 2e3 * float(model.geom_size[GELSIGHT_GEL_PAD_GEOM_ID_RIGHT, 1]) if "model" in globals() else 2.4
_rollout_dirs = sorted(_audit_dir.glob("rollout_*")) if _audit_dir.is_dir() else []
print("Auditing:", _audit_dir, f"({len(_rollout_dirs)} rollout folders)")


def _load_side(rollout_dir, side):
    side_dir = rollout_dir / side
    validation_path, npz_path = side_dir / "fots_validation.json", side_dir / "fots_quantitative_data.npz"
    if not (validation_path.is_file() and npz_path.is_file()):
        return None
    quant = np.load(npz_path)   # lazily reads only the small arrays below, never the depth stack
    t, taxels, depth_mm = quant["sim_time_s"], quant["active_taxel_count"], quant["max_depth_diff_mm"]
    contact = taxels > 0
    return {
        "status": json.loads(validation_path.read_text())["status"],
        "t": t, "taxels": taxels.astype(np.float64), "frames": int(len(t)),
        "contact_frames": int(contact.sum()), "peak_taxels": int(taxels.max()) if len(t) else 0,
        "peak_depth_mm": float(depth_mm.max()) if len(t) else float("nan"),
        "impact_frames": int((depth_mm > _gel_thickness_mm).sum()),
        "held_at_end": bool(contact[-1]) if len(t) else False,
    }


audit_rows = []
for _rollout_dir in _rollout_dirs:
    _row = {"rollout": _rollout_dir.name, "outcome": "?", "flags": []}
    _log_path = _rollout_dir / "closed_loop_log.json"
    if _log_path.is_file():
        _row["outcome"] = json.loads(_log_path.read_text()).get("outcome", "?")
    _sides = {side: _load_side(_rollout_dir, side) for side in ("right", "left")}
    if (_rollout_dir / "fots_quantitative_data.npz").is_file() and not any(_sides.values()):
        _row["flags"].append("single-sensor format (no right/ left/ folders)")
    for _side, _data in _sides.items():
        if _data is None and not any("single-sensor" in f for f in _row["flags"]):
            _row["flags"].append(f"missing {_side}/ output")
    _row["sides"] = _sides
    if all(_sides.values()):
        _r, _l = _sides["right"], _sides["left"]
        if _r["frames"] != _l["frames"] or not np.array_equal(_r["t"], _l["t"]):
            _row["flags"].append("frames not aligned")
        else:
            _either = (_r["taxels"] > 0) | (_l["taxels"] > 0)
            _both = (_r["taxels"] > 0) & (_l["taxels"] > 0)
            _row["contact_iou"] = float(_both.sum() / _either.sum()) if _either.any() else float("nan")
            _row["taxel_r"] = float(np.corrcoef(_r["taxels"][_either], _l["taxels"][_either])[0, 1]) if _either.sum() > 2 else float("nan")
            if _either.any() and _row["contact_iou"] < 0.5:
                _row["flags"].append("asymmetric contact (one pad touched the cube much more than the other)")
        for _side, _data in _sides.items():
            if _data["status"] != "passed":
                _row["flags"].append(f"{_side} validation {_data['status']}")
            if _data["impact_frames"]:
                _row["flags"].append(f"{_side}: {_data['impact_frames']} impact frame(s) deeper than the {_gel_thickness_mm:.1f} mm gel (peak {_data['peak_depth_mm']:.1f} mm)")
    audit_rows.append(_row)

_dual_rows = [row for row in audit_rows if all(row["sides"].values())]
if not audit_rows:
    print("No rollout folders yet. Run Cell P with RUN_CLOSED_LOOP_ROLLOUTS=True first (or set AUDIT_RUN_DIR).")
elif not _dual_rows:
    print("Rollouts found, but none has the dual-sensor layout (rollout_NNN/right + left). Flags:")
    for row in audit_rows[:5]:
        print(" ", row["rollout"], row["flags"])
else:
    print(f"{'rollout':13s}{'outcome':10s}{'frames':>7s} | {'R contact/peak taxels/peak mm':>31s} | {'L contact/peak taxels/peak mm':>31s} | {'IoU':>5s} {'r':>6s}  flags")
    for row in _dual_rows:
        r, l = row["sides"]["right"], row["sides"]["left"]
        print(
            f"{row['rollout']:13s}{row['outcome']:10s}{r['frames']:7d} | "
            f"{r['contact_frames']:9d} / {r['peak_taxels']:6d} / {r['peak_depth_mm']:7.2f} | "
            f"{l['contact_frames']:9d} / {l['peak_taxels']:6d} / {l['peak_depth_mm']:7.2f} | "
            f"{row.get('contact_iou', float('nan')):5.2f} {row.get('taxel_r', float('nan')):6.2f}  {'; '.join(row['flags'])}"
        )

    def _grip_registered(side_data):
        return side_data["contact_frames"] >= AUDIT_MIN_SUSTAINED_CONTACT_FRAMES and side_data["peak_taxels"] >= AUDIT_MIN_PEAK_TAXELS

    _accepted = [row for row in _dual_rows if row["outcome"] == "accepted"]
    _both_registered = [row for row in _dual_rows if _grip_registered(row["sides"]["right"]) and _grip_registered(row["sides"]["left"])]
    _one_sided = [row for row in _dual_rows if _grip_registered(row["sides"]["right"]) != _grip_registered(row["sides"]["left"])]
    print(f"\n{len(_dual_rows)} dual-sensor rollouts: {len(_accepted)} accepted (cube reached the far side), {len(_dual_rows) - len(_accepted)} failed.")
    print(f"Both sensors registered a sustained grip in {len(_both_registered)}/{len(_dual_rows)} rollouts; exactly one sensor did in {len(_one_sided)}.")
    print(f"Rollouts with impact frames deeper than the gel: {sum(any('impact' in f for f in row['flags']) for row in _dual_rows)}.")
    _bad = [row["rollout"] for row in _accepted if not (_grip_registered(row["sides"]["right"]) and _grip_registered(row["sides"]["left"]))]
    if _accepted:
        print(f"Successful rollouts where both sensors saw the grip: {len(_accepted) - len(_bad)}/{len(_accepted)}.")
    assert not _bad, (
        "A rollout that carried the cube to the far side has a sensor with no sustained contact "
        f"(>= {AUDIT_MIN_SUSTAINED_CONTACT_FRAMES} frames, >= {AUDIT_MIN_PEAK_TAXELS} px): {_bad}"
    )


# Reviewing the T3 dataset (10-demo smoke test)

This smoke test ran Cell P with `N_ROLLOUTS=20` (ceiling), `DATASET_TARGET_DEMOS=10`: **10/10 attempts kept** (7 succeeded, 3 failed -- both outcomes kept, per T3's inverted keep criterion; all 10 cleared the `DATASET_MIN_CONTACT_SECONDS=2.0` bar), friction/mass/perturbation all varied across attempts. Cell D1 below shows the recalibrated slip signal (12 mm/10 deg): successful demos read 3-9% slip, failed demos read 43-56%, and one failed demo's plot shows a textbook, tightly-isolated drop event.

In [ ]:
# Cell D1: Showcase a hand-picked subset of the T3 dataset -- slip ground truth included.
# Self-contained: reads only files Cell P already saved (closed_loop_log.json, grasp_pose.json,
# final_frame.png, world/wrist/grasp videos, both sensors' fots_quantitative_data.npz/tactile_rgb/
# marker_flow, and slip_ground_truth.npz) -- no MuJoCo/FOTS objects needed, so this works after a
# kernel restart, pointed at any completed or in-progress T3 run.
#
# The main knob is SHOWCASE_DEMO_SELECTION: leave it as None to auto-pick SHOWCASE_COUNT demos
# evenly spaced across the whole dataset (default), or set it to an explicit list to choose exactly
# which ones -- entries can be an int (demo index, e.g. 37 -> "demo_037") or a str (an exact folder
# name, e.g. "demo_037"), mixed freely, in any order, any length.
#
# For each selected demo this shows, in order: (1) a still-frame summary (third-person + each
# sensor's peak-contact tactile frame, like T2's Cell D1), (2) a NEW time-series panel -- slip
# binary, translation/rotation delta vs the SLIP_TRANSLATION_THRESHOLD_MM/SLIP_ROTATION_THRESHOLD_DEG
# thresholds, per-finger normal force, per-finger contact state, all phase-shaded -- so you can see
# directly which controller phase (open/closing/closed_hold/lift_bootstrap/carrying) is driving any
# slip flag, and (3) optionally the full videos (same as before).
import json
from pathlib import Path

import imageio.v3 as imageio_v3
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Video, display
from PIL import Image

# ----- The knobs -----
SHOWCASE_DEMO_SELECTION = None  # e.g. [0, 37, "demo_112", 199] to pick specific demos instead.
SHOWCASE_COUNT = 10             # how many when SHOWCASE_DEMO_SELECTION is None.
SHOWCASE_RANDOM_SEED = None     # set an int (with SHOWCASE_DEMO_SELECTION=None) to sample randomly
                                 # instead of evenly spaced across the dataset.

SHOWCASE_SHOW_VIDEOS = True  # play the full videos below the still-frame summary, per selected demo.
SHOWCASE_VIDEO_GROUPS = ("world", "gelsight_rgb")  # any of: "world", "wrist", "grasp_front",
                                 # "grasp_corner_low", "gelsight_rgb", "gelsight_marker_flow".

# Where the dataset lives: this notebook's own T3/baseline dataset by default. Deliberately NOT
# inferred from live EXPERIMENT_DATASET/EXPERIMENT_CONDITION kernel globals -- a stale value left
# over from an earlier cell run in the same kernel session would silently point here at the wrong
# dataset instead of failing loudly (this bit T2's Cell D1 once already). Set SHOWCASE_DATASET_DIR
# explicitly to look at a different dataset/run.
SHOWCASE_DATASET_DIR = None  # e.g. Path("/some/other/T3/baseline") to point at a different run.

_default_project_dir = Path(globals().get("PROJECT_DIR", "/home/fariborz/projects/force-vla-colab"))
_dataset_dir = Path(SHOWCASE_DATASET_DIR) if SHOWCASE_DATASET_DIR is not None else _default_project_dir / "T3" / "baseline"
assert _dataset_dir.is_dir(), f"No T3 dataset found at {_dataset_dir}. Run Cell P first, or set SHOWCASE_DATASET_DIR."

_all_demo_dirs = {int(p.name.split("_")[1]): p for p in sorted(_dataset_dir.glob("demo_*")) if p.name.split("_")[1].isdigit()}
assert _all_demo_dirs, f"No demo_* folders under {_dataset_dir} yet."
print(f"T3 dataset: {_dataset_dir}  ({len(_all_demo_dirs)} demos on disk)")

if SHOWCASE_DEMO_SELECTION is not None:
    _picked_indices = [entry if isinstance(entry, int) else int(str(entry).split("_")[1]) for entry in SHOWCASE_DEMO_SELECTION]
    _missing = [i for i in _picked_indices if i not in _all_demo_dirs]
    assert not _missing, f"No such demo index/folder: {_missing}. Available indices: {sorted(_all_demo_dirs)}"
elif SHOWCASE_RANDOM_SEED is not None:
    _picked_indices = sorted(np.random.default_rng(SHOWCASE_RANDOM_SEED).choice(
        sorted(_all_demo_dirs), size=min(SHOWCASE_COUNT, len(_all_demo_dirs)), replace=False
    ).tolist())
else:
    _available = sorted(_all_demo_dirs)
    _picked_indices = sorted({_available[i] for i in np.linspace(0, len(_available) - 1, min(SHOWCASE_COUNT, len(_available))).astype(int)})
print(f"Showcasing {len(_picked_indices)} demo(s):", [f"demo_{i:03d}" for i in _picked_indices])


def _peak_contact_frame(demo_dir, side):
    side_dir = demo_dir / side
    quant = np.load(side_dir / "fots_quantitative_data.npz")
    taxels = quant["active_taxel_count"]
    peak_idx = int(np.argmax(taxels))
    frame = imageio_v3.imread(side_dir / "fots_tactile_rgb.mp4", index=peak_idx)
    return {
        "frame": frame, "peak_idx": peak_idx, "peak_taxels": int(taxels[peak_idx]),
        "contact_frames": int((taxels > 0).sum()), "total_frames": int(len(taxels)),
        "peak_force_n": float(np.nanmax(quant["normal_force_n"])) if "normal_force_n" in quant.files else float("nan"),
    }


showcase_rows = []
for demo_idx in _picked_indices:
    demo_dir = _all_demo_dirs[demo_idx]
    log = json.loads((demo_dir / "closed_loop_log.json").read_text())
    grasp = json.loads((demo_dir / "grasp_pose.json").read_text()) if (demo_dir / "grasp_pose.json").is_file() else {}
    gt = np.load(demo_dir / "slip_ground_truth.npz", allow_pickle=True) if (demo_dir / "slip_ground_truth.npz").is_file() else None
    row = {
        "demo": demo_dir.name, "dir": demo_dir,
        "final_frame": np.asarray(Image.open(demo_dir / "final_frame.png").convert("RGB")),
        "outcome": log["outcome"],
        "termination_reason": log["rollout_termination_reason"],
        "start_xy": log["start_info"]["nominal_untextured_cube_start"][:2],
        "final_xyz": log["final_cube_pos"],
        "grasp_pose": grasp,
        "ground_truth": gt,
    }
    for side in ("right", "left"):
        row[side] = _peak_contact_frame(demo_dir, side)
    showcase_rows.append(row)

# ----- Figure 1: still-frame summary (unchanged style from T2's Cell D1) --------------------------
n_rows = len(showcase_rows)
fig, axes = plt.subplots(n_rows, 3, figsize=(9, 3.15 * n_rows), squeeze=False)
col_titles = ("Third-person, episode end", "RIGHT sensor @ peak contact", "LEFT sensor @ peak contact")
for row_idx, row in enumerate(showcase_rows):
    for col_idx, (image, title) in enumerate((
        (row["final_frame"], col_titles[0]), (row["right"]["frame"], col_titles[1]), (row["left"]["frame"], col_titles[2]),
    )):
        ax = axes[row_idx, col_idx]
        ax.imshow(image)
        ax.set_xticks([]); ax.set_yticks([])
        if row_idx == 0:
            ax.set_title(title, fontsize=9)
    axes[row_idx, 0].set_ylabel(f"{row['demo']}\n{row['outcome']}", fontsize=9, rotation=0, ha="right", va="center", labelpad=30)
    for col, side in ((1, "right"), (2, "left")):
        s = row[side]
        axes[row_idx, col].set_xlabel(
            f"peak {s['peak_taxels']} px, {s['contact_frames']}/{s['total_frames']} contact, "
            f"{s['peak_force_n']:.1f} N peak", fontsize=7,
        )
fig.suptitle(f"T3 dataset showcase -- {n_rows} of {len(_all_demo_dirs)} demos ({_dataset_dir})", fontsize=11)
fig.tight_layout()
showcase_path = _dataset_dir / f"showcase_{'_'.join(f'{i:03d}' for i in _picked_indices)}.png"
fig.savefig(showcase_path, dpi=110)
plt.show()

# ----- Figure 2: per-demo ground-truth time series (slip, force, contact) -------------------------
_PHASE_COLORS = {
    "open": "0.92", "closing": "#ffe4b3", "closed_hold": "#ffd699",
    "lift_bootstrap": "#ffb3b3", "carrying": "#b3d9ff", "opening": "#c9f7c9", "released_hold": "0.92",
}
for row in showcase_rows:
    gt = row["ground_truth"]
    if gt is None:
        print(f"{row['demo']}: no slip_ground_truth.npz (older/incompatible demo) -- skipping time-series plot.")
        continue
    t = gt["sim_time_s"]
    phase = gt["controller_phase"]
    td, rd = gt["translation_delta_mm"], gt["rotation_delta_deg"]
    slip, holding = gt["slip_binary"], gt["holding_phase"]
    trans_thr = float(gt["slip_translation_threshold_mm"])
    rot_thr = float(gt["slip_rotation_threshold_deg"])
    right_q = np.load(row["dir"] / "right" / "fots_quantitative_data.npz")
    left_q = np.load(row["dir"] / "left" / "fots_quantitative_data.npz")

    fig, axes = plt.subplots(4, 1, figsize=(11, 8.5), sharex=True)
    # Phase-shaded background on every panel.
    phase_bounds = np.flatnonzero(phase[1:] != phase[:-1]) + 1
    starts = np.concatenate(([0], phase_bounds))
    ends = np.concatenate((phase_bounds, [len(phase)]))
    for ax in axes:
        for s, e in zip(starts, ends):
            ax.axvspan(t[s], t[min(e, len(t) - 1)], color=_PHASE_COLORS.get(str(phase[s]), "0.95"), alpha=0.5, lw=0)

    axes[0].plot(t, td, color="tab:purple", lw=1.3, label="translation delta (mm)")
    axes[0].axhline(trans_thr, color="k", ls="--", lw=1, label=f"threshold ({trans_thr:.1f} mm)")
    axes[0].fill_between(t, 0, float(np.nanmax(td)) * 1.05 if np.isfinite(td).any() else 1.0, where=slip, color="red", alpha=0.15, step="mid", label="slip=1")
    axes[0].set_ylabel("translation\ndelta (mm)")
    axes[0].legend(loc="upper right", fontsize=7)

    axes[1].plot(t, rd, color="tab:orange", lw=1.3, label="rotation delta (deg)")
    axes[1].axhline(rot_thr, color="k", ls="--", lw=1, label=f"threshold ({rot_thr:.1f} deg)")
    axes[1].set_ylabel("rotation\ndelta (deg)")
    axes[1].legend(loc="upper right", fontsize=7)

    axes[2].plot(t, right_q["normal_force_n"], color="tab:red", lw=1.3, label="right finger")
    axes[2].plot(t, left_q["normal_force_n"], color="tab:blue", lw=1.3, label="left finger")
    axes[2].set_ylabel("normal force (N)")
    axes[2].legend(loc="upper right", fontsize=7)

    axes[3].step(t, right_q["contact_state"].astype(int) + 0.02, color="tab:red", lw=1.3, where="post", label="right contact")
    axes[3].step(t, left_q["contact_state"].astype(int) - 0.02, color="tab:blue", lw=1.3, where="post", label="left contact")
    axes[3].step(t, holding.astype(int) * 0.5 + 0.25, color="k", lw=1.0, where="post", label="holding phase (x0.5)")
    axes[3].set_ylim(-0.15, 1.15)
    axes[3].set_ylabel("contact state")
    axes[3].set_xlabel("simulation time (s)")
    axes[3].legend(loc="upper right", fontsize=7)

    grasp = row["grasp_pose"]
    grasp_txt = (
        f"grasp pose (mm, hand frame): {np.round(grasp['relative_pos_mm'], 1).tolist()} @ t={grasp['grasp_established_time_s']:.2f}s"
        if grasp.get("grasp_established") else "grasp never locked this episode"
    )
    holding_frames = int(holding.sum())
    slip_frames = int(slip.sum())
    fig.suptitle(
        f"{row['demo']}  outcome={row['outcome']}  |  {grasp_txt}  |  "
        f"slip {slip_frames}/{holding_frames} holding frames ({100 * slip_frames / max(holding_frames, 1):.0f}%)",
        fontsize=10,
    )
    fig.tight_layout()
    timeseries_path = row["dir"] / "slip_timeseries.png"
    fig.savefig(timeseries_path, dpi=110)
    plt.show()

# ----- Compact stats table (friction/mass/perturbation come from results.csv -- Cell P records
# them per attempt there, not inside each demo's own closed_loop_log.json). ------------------------
_results_by_dir = {}
_results_csv_path = _dataset_dir / "results.csv"
if _results_csv_path.is_file():
    import csv as _csv
    with open(_results_csv_path, newline="") as _f:
        for _r in _csv.DictReader(_f):
            if _r.get("dataset_dir"):
                _results_by_dir[str(Path(_r["dataset_dir"]).resolve())] = _r

print(f"\n{'demo':10s}{'outcome':10s}{'friction':>9s}{'mass_x':>7s}{'perturbed':>10s}{'holding':>9s}{'slip':>6s}{'slip%':>7s}")
for row in showcase_rows:
    gt = row["ground_truth"]
    holding_frames = int(gt["holding_phase"].sum()) if gt is not None else 0
    slip_frames = int(gt["slip_binary"].sum()) if gt is not None else 0
    _r = _results_by_dir.get(str(row["dir"].resolve()), {})
    _friction_s = f"{float(_r['torsional_friction']):.3f}" if _r.get("torsional_friction") else "n/a"
    _mass_s = f"{float(_r['cube_mass_scale']):.2f}" if _r.get("cube_mass_scale") else "n/a"
    print(
        f"{row['demo']:10s}{row['outcome']:10s}"
        f"{_friction_s:>9s}{_mass_s:>7s}{_r.get('perturbation_applied', 'n/a'):>10s}"
        f"{holding_frames:9d}{slip_frames:6d}{100 * slip_frames / max(holding_frames, 1):6.0f}%"
    )
print(f"\nSaved still-frame summary: {showcase_path}")
print("Saved per-demo time-series plots inside each demo's own folder (slip_timeseries.png).")

# ----- Full videos, per selected demo (unchanged from T2's Cell D1) -------------------------------
_VIDEO_GROUP_PATHS = {
    "world": lambda d: [("world", d / "world.mp4")],
    "wrist": lambda d: [("wrist", d / "wrist.mp4")],
    "grasp_front": lambda d: [("grasp_front", d / "grasp_front.mp4")],
    "grasp_corner_low": lambda d: [("grasp_corner_low", d / "grasp_corner_low.mp4")],
    "gelsight_rgb": lambda d: [(f"{side} tactile RGB", d / side / "fots_tactile_rgb.mp4") for side in ("right", "left")],
    "gelsight_marker_flow": lambda d: [(f"{side} marker flow", d / side / "fots_marker_flow.mp4") for side in ("right", "left")],
}
_VIDEO_WIDTHS = {"world": 640, "wrist": 320, "grasp_front": 480, "grasp_corner_low": 480}

if SHOWCASE_SHOW_VIDEOS:
    _unknown_groups = [g for g in SHOWCASE_VIDEO_GROUPS if g not in _VIDEO_GROUP_PATHS]
    assert not _unknown_groups, f"Unknown SHOWCASE_VIDEO_GROUPS entries: {_unknown_groups}. Valid: {list(_VIDEO_GROUP_PATHS)}"
    for row in showcase_rows:
        print(f"\n===== {row['demo']} ({row['outcome']}) =====")
        for group in SHOWCASE_VIDEO_GROUPS:
            for label, video_path in _VIDEO_GROUP_PATHS[group](row["dir"]):
                assert video_path.is_file(), f"Missing {video_path}"
                print(label)
                display(Video(str(video_path), embed=False, width=_VIDEO_WIDTHS.get(group, 320)))
else:
    print("\nSHOWCASE_SHOW_VIDEOS=False: full videos not displayed (summaries above only).")
